# Project 3 — notebook gộp: `du-lieu-do-thi` + pipeline ảnh/video

Notebook này gộp hai pipeline của Project 3 (Truy vấn đa phương thức có giải thích bằng đồ thị) để chạy một lần trên Kaggle.

| Phần | Nguồn | Nội dung | Thư mục kết quả |
|---|---|---|---|
| **A** | `hieutrinh123/du-lieu-do-thi` (version 9/11) | Flickr8k (text → ảnh) và Action Genome (text → video): CLIP → đồ thị cảnh (node đối tượng, quan hệ, hành động/bối cảnh Charades) → GNN + đường đi + nhánh khớp truy vấn↔node → xếp hạng lại; giải thích, đối chứng, kiểm định cặp. Phần 4 (MSVD → MSR-VTT) cũ vẫn tắt (`%%script true`). | `/kaggle/working/results/`, `/kaggle/working/output/action_genome/results/` |
| **C** | `lamvo30ai/project03-level4b-v2-graphfix` (version 1), mục 4u trong phần B | Text → video MSVD + MSR-VTT: đồ thị v3 có vị từ, TypedSAGE / QC-RGAT, tagger học, ConceptNet, xAI; tạo cache đặc trưng frame cho mục 4v. Chạy trong tiến trình riêng (papermill). | `/kaggle/working/project03_level4b_v2/`, `/kaggle/working/p3_out/video_v2/` |
| **B** | `lamvo30ai/project3-full-pipeline` (version 8) | Visual Genome + COCO (text → ảnh, scene graph GT; GraphSAGE; SCENIR), ARO + SugarCrepe (ảnh → caption, RelTR), COCO-spatial (cạnh hình học), XAI, video MSVD (Level4B v5). | `/kaggle/working/p3_out/` |

**Kaggle:** GPU T4, Internet On. Datasets: `hieutrinh123/project3-merged-assets` (payload nhúng của phần B và notebook phần C, tách ra vì Kaggle giới hạn mã nguồn 1 MB), `hieutrinh123/action-genome`, `dannywu375/visualgenome`, `mathurinache/visual-genome`, `clkmuhammed/microsoft-coco-2017-common-objects-in-context`. Notebook nguồn: `hieutrinh123/du-lieu-do-thi` (cache CLIP), `lamvo30ai/project03-level4b-v2-graphfix` (cache video của phần B — nếu không có quyền truy cập, nhánh video của phần B ghi lỗi và phần còn lại vẫn chạy).

**Thời gian ước tính:** phần A khoảng 4–5 giờ (ViT-L/14), phần B khoảng 3 giờ ⇒ 7–8 giờ. Phần C: gần như 0 nếu output `lamvo30ai/project03-level4b-v2-graphfix` đã được Add Input (chế độ `auto` chỉ chép kết quả), ngược lại +2,4 giờ (tổng ~10 giờ, sát giới hạn 12 giờ của Kaggle).

**Thay đổi khi gộp:** ngoài việc nhúng phần C (mục 4u), chỉ hai chỗ — (1) phần A đặt backbone ViT-L/14 như mô hình cuối (version 9); (2) phần B đổi tên hàm `log` thành `p3_log` để không đè hàm `log` của phần A. Ngoài ra thêm một cell giải phóng bộ nhớ GPU giữa hai phần. Không đổi logic, tham số hay dữ liệu của phần nào.

**Bản Kaggle:** đã bỏ các cell bị tắt sẵn (`%%script true`, gồm Phần 4 MSVD → MSR-VTT cũ) và chuyển hai payload lớn sang dataset để mã nguồn dưới 1 MB; logic chạy không đổi.


---
# PHẦN A — `du-lieu-do-thi`: Flickr8k + Action Genome


# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị

**Pipeline:** `truy vấn → CLIP → top-K ứng viên → điểm đồ thị (GNN + đường đi) → xếp hạng lại → giải thích → fidelity`

Điểm số cuối của một ứng viên:

$$\text{score} = \alpha \cdot \cos_{\text{CLIP}} + (1-\alpha)\big[\lambda \cdot s_{\text{GNN}} + (1-\lambda)\cdot s_{\text{path}}\big]$$

* `s_GNN` — GraphSAGE/GAT tự cài trên đồ thị dị thể (ảnh / lớp đối tượng / khái niệm).
  Mục 1.7.2 thay nhánh này bằng **SCENIR** (ICML 2025): Graph Autoencoder tự giám sát trên scene graph
  của từng ảnh — phương pháp mới cho cấp độ 2 (`CFG.gnn_type = "scenir"`).
* `s_path` — tổng **cộng tính** theo từng đường đi nối khái niệm truy vấn tới node ảnh.
  Vì cộng tính nên bỏ một đường đi làm điểm giảm đúng bằng phần đóng góp của nó ⇒ giải thích
  đến thẳng từ cơ chế xếp hạng, không phải hợp lý hoá sau khi có kết quả.
* `α`, `λ` chọn trên **validation**, không bao giờ trên test.

**Chạy trên Kaggle:** Settings → Accelerator → GPU P100 (hoặc T4 x2), Internet → On.
Notebook tự dò dữ liệu trong `/kaggle/input`; không thấy thì chạy chế độ `SYNTHETIC` để smoke-test toàn bộ mã.

## Cấu trúc notebook

| Phần | Nội dung | Vị trí |
|---|---|---|
| **1** | Nền tảng + Flickr8k: cấu hình, đồ thị, GNN cổ điển, huấn luyện, giải thích, fidelity, ablation | mục 1.1 – 1.17 |
| **2** | GNN của SCENIR thay GNN cổ điển (Flickr8k) | Phần 2 |
| **3** | Action Genome (dataset thứ hai): liên kết khái niệm, ablation, SCENIR | mục 3.1 – 3.5 |
| **4** | Cấp độ 4(b): truy vấn văn bản → video (MSVD + MSR-VTT) | mục 4.1 – 4.16 |


> **Bản v5 (chạy lại Phần 1–3).** Thay đổi so với bản trước:
> 1. Node **bộ ba cấp ảnh** + mẫu đường đi `P6` (mục 1.2b); nhãn đối tượng Flickr8k bằng linear probe ngoài-fold.
> 2. Đối chứng `no_triples`, `shuffled_triples`; **bộ đánh giá quan hệ khó** (cùng đối tượng, khác quan hệ) cho Flickr8k và AG.
> 3. Sửa đối chứng fidelity: giải thích luôn nhỏ hơn toàn bộ đường đi (bản trước in `sparsity 0.00` cho các ca ≤ 5 đường đi).
> 4. Dùng dataset Kaggle thay Drive; cache CLIP của lần chạy trước được dùng lại nếu output cũ được Add Input.
> 5. **Phần 4 (MSVD → MSR-VTT) không chạy lại** (các cell có `%%script true`); kết quả của nó nằm ở version trước.
>
> **Bản v8 (version 6).** (a) AG: đồ thị không–thời gian — thêm node hành động, bối cảnh và bộ ba thời gian từ nhãn Charades (cell v8 đầu Phần 3), đối chứng `no_actions` / `shuffled_actions`, cấu hình `no_class_edges`. (b) Flickr8k: chế độ `oracle_grams` — gallery ĐÃ ĐƯỢC CHÚ THÍCH (caption 1–4 của ảnh làm nguồn node, cùng điều kiện với annotation của AG), node = từ + cụm 2 từ + bộ ba. Kết quả đồ thị BLIP ở version 5.
>
> **Bản v7 (version 5).** Flickr8k: đồ thị cảnh dựng từ mô hình mô tả ảnh ĐỘC LẬP với CLIP (BLIP), xem mục 1.2c; thêm kiểm định cặp CLIP+Graph vs CLIP, đối chứng `shuffled_vlm`, và backbone ViT-B/32 trên cùng đồ thị. Phần 2–4 không chạy lại (AG: version 4).
>
> **Bản v6 (version 3).** (a) Flickr8k chạy với backbone YẾU hơn — CLIP **ViT-B/32** — để đo hiệu quả của đồ thị theo độ mạnh của backbone (kết quả ViT-L/14 nằm ở version 2). (b) AG thêm 2 đối chứng: `composite_swap` (gán nhầm quan hệ, giữ nguyên vật) và **truy vấn phản thực tế** (đổi vị từ trong truy vấn). (c) Không chạy lại oracle, SCENIR, liên kết ngữ nghĩa AG (đã có ở version 2) để tiết kiệm GPU.


---
# PHẦN 1 — Nền tảng và Flickr8k

_Mục 1.1 – 1.17: toàn bộ mã dùng chung (đồ thị, mô hình, truy vấn, đánh giá, giải thích) rồi chạy trên Flickr8k._


In [ ]:
import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

RUN_ON_KAGGLE = Path("/kaggle/input").exists()
print("Kaggle environment:", RUN_ON_KAGGLE)

In [ ]:
# Cài đặt phụ thuộc. Trên Kaggle torch/numpy đã có sẵn; chỉ cần open_clip + nltk.
def pip_install(pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

if RUN_ON_KAGGLE:
    pip_install(["open_clip_torch", "nltk"])   # không ghim phiên bản: Kaggle đổi torch thường xuyên

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 1.1 Cấu hình

Mọi siêu tham số nằm ở một chỗ. `CFG.n_images` là núm vặn chính cho thời gian chạy:
4000 ảnh chạy khoảng 12-15 phút trên P100 (gồm cả trích đặc trưng CLIP).

In [ ]:
class CFG:
    # --- dữ liệu ---
    dataset      = "flickr8k"
    n_images     = 8000
    min_obj_freq = 5           # lớp đối tượng hiếm hơn ngưỡng này bị loại — ĐẾM TRÊN TRAIN
    max_concepts = 4000

    # --- nhãn ảnh Flickr8k ---
    # "tagger": mọi ảnh gắn nhãn bằng CLIP zero-shot trên từ vựng rút từ caption TRAIN
    #           (không ảnh val/test nào được dùng caption của chính nó).
    # "oracle": nhãn rút từ caption 1..4 của chính ảnh (cách cũ) — chỉ để so sánh, KHÔNG báo cáo là kết quả chính.
    flickr_annot = "oracle_grams"   # v8: gallery đã chú thích (caption 1–4); "vlm" = v7 (BLIP), "probe" = v5/v6
    _flickr_annot_v7 = "vlm"       # v7: node từ mô hình mô tả ảnh độc lập với CLIP (1.2c); "probe" = v5/v6
    _flickr_annot_v5 = "probe"     # v5: linear probe khởi tạo từ văn bản CLIP, ngoài-fold (1.2b); "tagger" = zero-shot cũ
    tag_k_grid   = [3, 5, 8, 10, 12]          # số nhãn/ảnh — chọn trên VAL theo F1
    tag_prompts  = ["a photo of a {}.", "a photo of {}.", "a photo containing a {}."]

    # --- chia dữ liệu (chia theo ẢNH, không theo caption) ---
    split_seed   = 20240101
    val_ratio    = 0.15
    test_ratio   = 0.20

    # --- CLIP ---
    clip_model   = "ViT-L-14"                 # trước: ViT-B-32. Tagger + baseline mạnh hơn rõ rệt
    clip_weights = "laion2b_s32b_b82k"
    clip_bs      = 128                        # ViT-L tốn bộ nhớ hơn; L4 24GB chạy 128 ổn

    # --- đồ thị ---
    max_path_len      = 3
    scene_min_count   = 2      # cạnh scene phải xuất hiện ≥ ngưỡng này trong caption TRAIN (trước đây = 1)
    scene_top_k       = 10     # mỗi lớp đối tượng giữ tối đa k cạnh scene mạnh nhất
    cooc_min_count    = 10
    cooc_top_k        = 10
    use_wordnet       = True
    isa_min_depth     = 4      # bỏ khái niệm quá chung (entity, object, whole, ...)
    use_idf           = True   # trọng số đường đi = IDF của node đích (tính trên TRAIN)
    drop_redundant_indirect = True   # bỏ đường đi gián tiếp tới khái niệm ảnh đã chứa trực tiếp

    # --- v5: quan hệ CẤP ẢNH (node bộ ba, mẫu P6) — xem mục 1.2b ---
    use_triples      = True
    triple_min_count = 5                      # bộ ba phải xuất hiện ≥ ngưỡng này trong caption TRAIN
    triple_k_grid    = [1, 2, 3, 5, 8]        # số bộ ba/ảnh — chọn trên VAL theo F1
    triple_tau_grid  = [0.0, 0.05, 0.1, 0.2, 0.3]
    probe_folds      = 5                      # dự đoán ngoài-fold cho ảnh train
    probe_steps      = 300
    probe_lr         = 1e-3
    probe_wd_init    = 1e-3                   # phạt ‖W − text‖²
    rel_neg_min      = 3                      # bộ quan hệ khó: tối thiểu / tối đa số âm mỗi truy vấn
    rel_neg_max      = 30

    # --- sửa ablation âm ---
    # (1) IDF: "taxonomy" = đối tượng đếm df qua P0, khái niệm qua P2 (isa thật, trước khi xáo).
    #     "all" = cách cũ: đếm qua MỌI mẫu ⇒ thêm quan hệ làm df phình, IDF của khớp trực tiếp
    #     tụt về ≈0 — đây là lý do chính khiến 'none' thua 'no_relations'.
    idf_from        = "taxonomy"
    # (2) Cạnh related: xếp hạng theo NPMI thay vì đếm thô (đếm thô ⇒ mọi lớp nối vào các hub
    #     phổ biến như person/man/floor). Trọng số cạnh = P(đích | nguồn) trên TRAIN.
    cooc_score      = "npmi"         # "npmi" | "count" (cũ)
    cooc_min_npmi   = 0.10
    edge_weighting  = True           # trọng số đường đi = idf(c) × Π P(đích|nguồn); False = đếm (cũ)
    # (3) GNN tách theo loại cạnh + cổng học được, khởi tạo gần như đóng ⇒ mô hình bắt đầu
    #     từ 'no_relations' và chỉ mở quan hệ khi nó thực sự giảm loss.
    gnn_typed       = True
    gate_init       = -1.5           # sigmoid(-1.5) ≈ 0.18 cho cạnh scene/related/isa; contains mở sẵn
    path_init_indirect = -1.0        # path_w khởi tạo cho P1..P5 (softplus(-1) ≈ 0.31; P0 = 0.69)
    scalar_lr_mult  = 10.0           # lr của cổng + path_w = lr × hệ số này
    # (4) Huấn luyện giống lúc suy luận: mẫu âm khó lấy từ CLIP top-K trên gallery TRAIN,
    #     và có z(s_clip) trong logit ⇒ đồ thị học phần CLIP còn thiếu, không học lại CLIP.
    hard_neg_k      = 31             # 0 = tắt (chỉ dùng âm trong batch như cũ)
    train_with_clip = True
    # Hệ số của z(s_clip) trong logit HUẤN LUYỆN có học không. AG (CLIP yếu, MRR 0.12): học được
    # thì tốt hơn (clip_w → 0.07). Flickr (CLIP mạnh): học làm clip_w → 0.4 và MRR giảm.
    # CHỌN THEO VAL MRR cho từng bộ dữ liệu, không theo test.
    learn_clip_w    = True

    # --- GNN ---
    gnn_type     = "sage"
    gnn_dim      = 256
    gnn_layers   = 2
    gnn_dropout  = 0.1
    lr           = 1e-3
    weight_decay = 1e-4
    epochs       = 20
    train_bs     = 128
    temperature  = 0.07

    # --- truy vấn / xếp hạng ---
    rerank_k     = 100

    # --- thí nghiệm ---
    seeds        = [0, 1, 2]
    alpha_grid   = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lambda_grid  = [0.0, 0.25, 0.5, 0.75]     # bỏ 1.0: λ=1 ⇒ s_path có trọng số 0 ⇒ giải thích không trung thực với điểm cuối
    ablations    = ("none", "random_edges", "no_relations", "no_scene", "no_related", "no_concept")

    adaptive_fusion = True
    fusion_steps    = 300
    fusion_tau      = 0.2
    fusion_wd       = 1e-2

    # --- xAI ---
    n_explain_queries = 20
    top_m_paths       = 5

    # --- đường dẫn ---
    out_dir = Path("/kaggle/working") if RUN_ON_KAGGLE else Path("./work")

# Flickr8k chạy thêm 2 đối chứng của node bộ ba; AG không có bộ ba dự đoán nên giữ CFG.ablations.
FLICKR_ABLATIONS = ("none", "no_class_edges", "no_relations")   # v6: rút gọn

CFG.out_dir.mkdir(parents=True, exist_ok=True)
(CFG.out_dir / "cache").mkdir(exist_ok=True)
(CFG.out_dir / "results").mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 1.2 Nạp dữ liệu

Notebook dò tự động các tệp Visual Genome trong `/kaggle/input` theo tên
(`objects.json`, `relationships.json`, `image_data.json`, `region_descriptions.json`).
Không tìm thấy ⇒ tự sinh dữ liệu tổng hợp để toàn bộ mã vẫn chạy hết (smoke-test).

**Cấu trúc chuẩn hoá** dùng chung cho mọi nguồn:
```python
images = [{"image_id": int, "caption": str, "objects": [str], "relations": [(subj, pred, obj)]}]
```

In [ ]:
# Lớp khái niệm dự phòng: dùng khi WordNet/nltk không tải được (Kaggle tắt Internet).
# Đây là một MMKG tí hon viết tay, đủ để tạo đường đi gián tiếp cho smoke-test và
# vẫn hoạt động trên Visual Genome vì các lớp đối tượng phổ biến đều có mặt.
FALLBACK_ISA = {
    "ball": ["sports equipment", "toy"], "field": ["open area", "sports venue"],
    "grass": ["vegetation", "ground"], "goal": ["sports equipment"],
    "shoe": ["clothing", "footwear"], "child": ["person"], "boy": ["person", "child"],
    "girl": ["person", "child"], "man": ["person"], "woman": ["person"],
    "person": ["human"], "keeper": ["person"],
    "pot": ["cookware", "container"], "stove": ["appliance", "cooking device"],
    "plate": ["tableware", "container"], "spoon": ["tableware", "utensil"],
    "table": ["furniture"], "bed": ["furniture"], "lamp": ["furniture", "light source"],
    "pillow": ["bedding"], "curtain": ["window covering"], "window": ["building part"],
    "book": ["reading material"], "sand": ["ground"], "umbrella": ["shade device"],
    "wave": ["water"], "towel": ["cloth"], "boat": ["vehicle", "watercraft"],
    "car": ["vehicle"], "truck": ["vehicle"], "bus": ["vehicle"], "bike": ["vehicle"],
    "sign": ["signage"], "road": ["open area", "transport way"],
    "building": ["structure"], "light": ["light source"], "elephant": ["animal"],
    "dog": ["animal"], "cat": ["animal"], "horse": ["animal"], "bird": ["animal"],
    "tree": ["vegetation", "plant"], "fence": ["barrier", "structure"],
    "water": ["liquid"], "rock": ["natural object"], "shirt": ["clothing"],
    "hat": ["clothing"], "chair": ["furniture"], "food": ["edible"],
    "pizza": ["food", "edible"], "plane": ["vehicle"], "train": ["vehicle"],
}


def find_vg_files():
    """Dò các tệp Visual Genome trong /kaggle/input theo tên tệp."""
    wanted = ["objects.json", "relationships.json", "image_data.json",
              "region_descriptions.json", "attributes.json"]
    found = {}
    if not Path("/kaggle/input").exists():
        return found
    for name in wanted:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            found[name.replace(".json", "")] = sorted(hits, key=lambda p: -os.path.getsize(p))[0]
    return found


def _norm(s):
    """Chuẩn hoá chuỗi nhãn: chữ thường, bỏ ký tự lạ, gộp khoảng trắng."""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_visual_genome(paths, n_images):
    """Nạp VG. Trả về danh sách dict đã chuẩn hoá."""
    print("Nạp Visual Genome từ:", {k: v for k, v in paths.items()})
    with open(paths["objects"]) as f:
        objects = json.load(f)
    obj_by_img, objid2name = {}, {}
    for rec in objects:
        iid = rec.get("image_id") or rec.get("id")
        names = []
        for o in rec.get("objects", []):
            nm = _norm((o.get("names") or [o.get("name", "")])[0])
            if nm:
                names.append(nm)
                objid2name[o.get("object_id")] = nm
        if names:
            obj_by_img[iid] = names
    del objects

    rel_by_img = defaultdict(list)
    if "relationships" in paths:
        with open(paths["relationships"]) as f:
            rels = json.load(f)
        for rec in rels:
            iid = rec.get("image_id") or rec.get("id")
            for r in rec.get("relationships", []):
                pred = _norm(r.get("predicate", ""))
                s = r.get("subject", {}); o = r.get("object", {})
                sn = _norm((s.get("names") or [s.get("name", "")])[0]) or objid2name.get(s.get("object_id"), "")
                on = _norm((o.get("names") or [o.get("name", "")])[0]) or objid2name.get(o.get("object_id"), "")
                if pred and sn and on:
                    rel_by_img[iid].append((sn, pred, on))
        del rels

    cap_by_img = {}
    if "region_descriptions" in paths:
        with open(paths["region_descriptions"]) as f:
            regions = json.load(f)
        for rec in regions:
            iid = rec.get("id") or rec.get("image_id")
            phrases = [r.get("phrase", "").strip() for r in rec.get("regions", [])]
            phrases = [p for p in phrases if 15 <= len(p) <= 160]
            if phrases:
                # ghép 3 vùng dài nhất thành một caption cấp ảnh
                phrases = sorted(phrases, key=len, reverse=True)[:3]
                cap_by_img[iid] = ". ".join(phrases)
        del regions

    url_by_img = {}
    if "image_data" in paths:
        with open(paths["image_data"]) as f:
            meta = json.load(f)
        for m in meta:
            url_by_img[m.get("image_id") or m.get("id")] = m.get("url", "")
        del meta

    ids = [i for i in obj_by_img if i in cap_by_img] or list(obj_by_img)
    ids = sorted(ids)[:n_images]
    out = []
    for iid in ids:
        out.append(dict(image_id=int(iid),
                        caption=cap_by_img.get(iid, " ".join(obj_by_img[iid][:8])),
                        objects=sorted(set(obj_by_img[iid])),
                        relations=list({tuple(r) for r in rel_by_img.get(iid, [])}),
                        url=url_by_img.get(iid, "")))
    print(f"  → {len(out)} ảnh, "
          f"{np.mean([len(r['objects']) for r in out]):.1f} đối tượng/ảnh, "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/ảnh")
    return out


def make_synthetic(n_images, seed=0):
    """Dữ liệu tổng hợp có cấu trúc quan hệ thật, dùng để smoke-test toàn bộ pipeline.

    Thiết kế có chủ ý: mỗi ảnh thuộc một 'chủ đề', chủ đề quyết định đối tượng và
    quan hệ, nên đồ thị MANG thông tin — nếu pipeline đúng thì điểm đồ thị phải
    cải thiện được kết quả so với đặc trưng nhiễu.
    """
    rng = random.Random(seed)
    themes = {
        "football":  (["ball", "field", "child", "goal", "grass", "shoe"],
                      [("child", "plays", "ball"), ("ball", "on", "grass"), ("child", "wears", "shoe")]),
        "kitchen":   (["pot", "stove", "woman", "plate", "spoon", "table"],
                      [("woman", "cooks with", "pot"), ("pot", "on", "stove"), ("plate", "on", "table")]),
        "beach":     (["sand", "umbrella", "wave", "man", "towel", "boat"],
                      [("man", "sits on", "towel"), ("umbrella", "on", "sand"), ("boat", "in", "wave")]),
        "street":    (["car", "sign", "road", "building", "light", "person"],
                      [("car", "on", "road"), ("sign", "near", "light"), ("person", "crosses", "road")]),
        "zoo":       (["elephant", "fence", "tree", "keeper", "water", "rock"],
                      [("elephant", "near", "fence"), ("tree", "behind", "fence"), ("keeper", "feeds", "elephant")]),
        "bedroom":   (["bed", "lamp", "pillow", "window", "curtain", "book"],
                      [("lamp", "next to", "bed"), ("pillow", "on", "bed"), ("curtain", "on", "window")]),
    }
    names = list(themes)
    out = []
    for i in range(n_images):
        th = names[i % len(names)]
        objs, rels = themes[th]
        k = rng.randint(3, len(objs))
        chosen = rng.sample(objs, k)
        r = [t for t in rels if t[0] in chosen and t[2] in chosen]
        # nhiễu: thỉnh thoảng mượn một đối tượng của chủ đề khác
        if rng.random() < 0.3:
            other = themes[rng.choice(names)][0]
            chosen.append(rng.choice(other))
        # 40% truy vấn nhắc tới KHÁI NIỆM (không có trong danh sách đối tượng) thay vì
        # gọi thẳng tên đối tượng ⇒ tạo ra nhóm truy vấn gián tiếp cần đường đi P2/P4/P5.
        if rng.random() < 0.4:
            concepts = sorted({c for o in chosen for c in FALLBACK_ISA.get(o, [])}
                              - set(chosen))
            if concepts:
                cap = (f"a photo showing {' and '.join(rng.sample(concepts, min(2, len(concepts))))}"
                       f" in a {th} scene")
            else:
                cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        else:
            cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        out.append(dict(image_id=i, caption=cap, objects=sorted(set(chosen)),
                        relations=r, url="", theme=th))
    return out


def load_dataset():
    mode = CFG.dataset
    if mode in ("auto", "vg"):
        paths = find_vg_files()
        if "objects" in paths:
            return load_visual_genome(paths, CFG.n_images), "visual_genome"
        if mode == "vg":
            raise FileNotFoundError("Không tìm thấy objects.json của Visual Genome trong /kaggle/input")
    print("⚠ Không tìm thấy dữ liệu công khai → chạy chế độ SYNTHETIC (smoke-test).")
    return make_synthetic(CFG.n_images), "synthetic"

## 1.3 Chia dữ liệu và phòng rò rỉ

Yêu cầu mục 6 của đề, cài đặt cụ thể:

| Quy định | Cách thực hiện trong mã |
|---|---|
| Test không dùng để huấn luyện / chọn siêu tham số | Chia **theo ảnh**; caption của test không vào tập huấn luyện |
| Cạnh suy ra từ thống kê phải xây từ train | `build_graph(..., stat_image_ids=train_ids)`; cạnh đồng xuất hiện chỉ đếm trên train |
| α và siêu tham số chọn trên validation | `tune_on_val()` quét lưới trên val, test chỉ chạy một lần |
| Nộp split cố định hoặc seed + mã tạo split | `split.json` được ghi ra kèm seed và hash |

Node ảnh của test **vẫn nằm trong đồ thị** (chúng là ứng viên cần xếp hạng), nhưng
cạnh của chúng chỉ đến từ annotation của chính bức ảnh, không từ thống kê toàn cục,
và không có gradient nào chảy qua chúng khi huấn luyện.

In [ ]:
def make_split(records, seed=CFG.split_seed):
    ids = sorted(r["image_id"] for r in records)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_test = int(n * CFG.test_ratio)
    n_val  = int(n * CFG.val_ratio)
    split = dict(test=sorted(ids[:n_test]),
                 val=sorted(ids[n_test:n_test + n_val]),
                 train=sorted(ids[n_test + n_val:]),
                 seed=seed)
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split (seed={seed}, sha1={split['sha1']}): "
          f"train={len(split['train'])} val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])), "train/test chồng lấn!"
    assert not (set(split["val"]) & set(split["test"])), "val/test chồng lấn!"
    return split

## 1.4 Đặc trưng CLIP (đóng băng)

CLIP được **đóng băng hoàn toàn** và chỉ chạy một lần: đặc trưng lưu ra `.npy` rồi tái sử dụng
cho mọi thí nghiệm. Đây là baseline của Nhiệm vụ 1.

Thứ tự ưu tiên tìm ảnh: tệp ảnh trong `/kaggle/input` → tải theo URL → đặc trưng tổng hợp.

In [ ]:
def index_local_images():
    """Lập chỉ mục mọi tệp ảnh trong /kaggle/input theo tên tệp (không đuôi)."""
    idx = {}
    roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    for root in roots:
        for p in root.rglob("*"):
            if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                idx.setdefault(p.stem, str(p))
    return idx


def encode_with_clip(records, tag):
    """Trả về (img_feat, txt_feat) đã chuẩn hoá L2, shape [N, D]."""
    cache = CFG.out_dir / "cache" / f"clip_{tag}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache)
        print(f"Dùng lại đặc trưng CLIP từ cache: {cache.name}")
        return z["img"], z["txt"]

    try:
        import open_clip
        from PIL import Image
    except ImportError:
        open_clip = None

    img_index = index_local_images()
    have_images = sum(1 for r in records if str(r["image_id"]) in img_index)
    print(f"Ảnh tìm thấy cục bộ: {have_images}/{len(records)}")

    if open_clip is None or have_images < 0.5 * len(records):
        print("⚠ Không đủ ảnh thật hoặc thiếu open_clip → sinh đặc trưng tổng hợp "
              "(vẫn chạy hết pipeline, nhưng con số không dùng để báo cáo).")
        return synthetic_features(records)

    model, _, preprocess = open_clip.create_model_and_transforms(
        CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters():          # đóng băng tuyệt đối
        p.requires_grad_(False)

    # --- ảnh ---
    feats = []
    from torch.utils.data import Dataset, DataLoader

    class ImgDS(Dataset):
        def __len__(self): return len(records)
        def __getitem__(self, i):
            p = img_index.get(str(records[i]["image_id"]))
            try:
                im = Image.open(p).convert("RGB")
            except Exception:
                im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    dl = DataLoader(ImgDS(), batch_size=CFG.clip_bs, num_workers=2)
    with torch.no_grad():
        for batch in dl:
            f = model.encode_image(batch.to(DEVICE))
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    img_feat = np.concatenate(feats)

    # --- văn bản ---
    feats = []
    caps = [r["caption"] for r in records]
    with torch.no_grad():
        for i in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[i:i + CFG.clip_bs]).to(DEVICE)
            f = model.encode_text(tok)
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    txt_feat = np.concatenate(feats)

    del model
    torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img_feat, txt=txt_feat)
    return img_feat, txt_feat


def synthetic_features(records, dim=512, seed=0):
    """Đặc trưng giả có tín hiệu thật: ảnh cùng chủ đề/đối tượng thì gần nhau."""
    rng = np.random.RandomState(seed)
    vocab = sorted({o for r in records for o in r["objects"]})
    basis = rng.randn(len(vocab), dim).astype(np.float32)
    v2i = {v: i for i, v in enumerate(vocab)}
    img, txt = [], []
    for r in records:
        idx = [v2i[o] for o in r["objects"]]
        core = basis[idx].mean(0) if idx else rng.randn(dim).astype(np.float32)
        img.append(core + 0.9 * rng.randn(dim).astype(np.float32))
        txt.append(core + 0.9 * rng.randn(dim).astype(np.float32))
    f = lambda x: (lambda a: a / np.linalg.norm(a, axis=1, keepdims=True))(np.stack(x))
    return f(img), f(txt)

## 1.5 Xây đồ thị

**Đỉnh:** ảnh · lớp đối tượng · khái niệm.
Đối tượng là **lớp** (một node `dog` dùng chung mọi ảnh) chứ không phải từng thể hiện —
nhờ vậy đồ thị nối được các ảnh với nhau và message passing mới có tác dụng.

**Cạnh:**

| Loại | Nguồn | Có phải thống kê? |
|---|---|---|
| `contains` ảnh → lớp đối tượng | annotation của chính ảnh đó | không → an toàn cho test |
| `scene` đối tượng → đối tượng | scene graph, gộp trên **train** | có → chỉ train |
| `related` đối tượng → đối tượng | đồng xuất hiện trên **train** | có → chỉ train |
| `isa` đối tượng → khái niệm | WordNet hypernym | không (tri thức ngoài) |

**Mẫu đường đi** dùng cho `s_path` (từ node ảnh tới node khái niệm của truy vấn):

| Mẫu | Đường đi | Dài | Loại |
|---|---|---|---|
| P0 | `img –contains→ c` | 1 | trực tiếp |
| P1 | `img –contains→ o –scene→ c` | 2 | gián tiếp |
| P2 | `img –contains→ o –isa→ c` | 2 | gián tiếp |
| P3 | `img –contains→ o –related→ c` | 2 | gián tiếp |
| P4 | `img –contains→ o –scene→ o₂ –isa→ c` | 3 | gián tiếp |
| P5 | `img –contains→ o –related→ o₂ –isa→ c` | 3 | gián tiếp |

`child plays football` với ảnh chỉ nhận diện được `ball`, `field` được bắt qua P1/P2/P4.

In [ ]:
import scipy.sparse as sp

PATTERNS = ["P0_contains", "P1_scene", "P2_isa", "P3_related", "P4_scene_isa", "P5_related_isa",
            "P6_triple"]                      # v5: ảnh –thể_hiện→ ⟨s p o⟩ (quan hệ CẤP ẢNH)
PATTERN_LEN = [1, 2, 2, 2, 3, 3, 1]
PIDX = {p: i for i, p in enumerate(PATTERNS)}
PATTERN_DESC = {
    "P0_contains":  "ảnh –chứa→ {c}",
    "P1_scene":     "ảnh –chứa→ {o} –{rel}→ {c}",
    "P2_isa":       "ảnh –chứa→ {o} –là_một→ {c}",
    "P3_related":   "ảnh –chứa→ {o} –liên_quan→ {c}",
    "P4_scene_isa": "ảnh –chứa→ {o} –{rel}→ {o2} –là_một→ {c}",
    "P5_related_isa": "ảnh –chứa→ {o} –liên_quan→ {o2} –là_một→ {c}",
    "P6_triple":    "ảnh –thể_hiện→ ⟨{c}⟩",
}

# ---------------------------------------------------------------------------------
# Lọc WordNet: chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC.
#   - nghĩa danh từ phổ biến nhất thuộc VISUAL_LEX (hoặc là môn thể thao/trò chơi),
#   - và từ đó được dùng như danh từ ít nhất ngang như động từ (loại stand, shake, jump, walk...).
# Loại được các node rác ở bản trước: pink, blue, beige, snowy, walking, shake, stand, bit...
# ---------------------------------------------------------------------------------
VISUAL_LEX = {"noun.person", "noun.animal", "noun.artifact", "noun.object", "noun.food",
              "noun.plant", "noun.substance", "noun.body", "noun.location", "noun.group",
              "noun.phenomenon"}
SPORT_ROOTS = {"sport.n.01", "game.n.01"}
GENERIC_CONCEPTS = {"entity", "physical entity", "abstraction", "object", "whole", "thing",
                    "matter", "artifact", "instrumentality", "organism", "living thing",
                    "causal agent", "location", "region", "group", "act", "event",
                    "unit", "part", "piece", "material", "substance", "device", "commodity",
                    "consumer goods", "covering", "creation", "structure", "area"}
# khái niệm "cấp cơ bản" — luôn thêm nếu nằm trên đường hypernym (dog→animal, car→vehicle...)
BASIC_CATEGORIES = {"person", "animal", "dog", "bird", "vehicle", "boat", "aircraft",
                    "clothing", "footwear", "headdress", "food", "plant", "tree", "flower",
                    "furniture", "building", "toy", "sport", "ball", "musical instrument",
                    "body of water", "geological formation", "precipitation", "child",
                    "game equipment", "sports equipment"}

_WN = None


def _wordnet():
    """Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA)."""
    global _WN
    if _WN is None:
        try:
            from nltk.corpus import wordnet as wn
            wn.synsets("dog"); _WN = wn
        except Exception:
            try:
                import nltk
                nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
                from nltk.corpus import wordnet as wn
                wn.synsets("dog"); _WN = wn
            except Exception:
                _WN = False
    return _WN or None


def _main_noun_synset(w):
    wn = _wordnet()
    if wn is None:
        return None
    syns = wn.synsets(w.replace(" ", "_"), pos="n")
    return syns[0] if syns else None


_NOUN_CACHE = {}


def is_visual_noun(w):
    if w in _NOUN_CACHE:
        return _NOUN_CACHE[w]
    wn = _wordnet()
    if wn is None:
        _NOUN_CACHE[w] = True          # không có WordNet → không lọc được
        return True
    s, ok = _main_noun_synset(w), False
    if s is not None:
        lex = s.lexname()
        if lex in VISUAL_LEX:
            ok = True
        elif lex == "noun.act":       # football, soccer, tennis... vẫn giữ
            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}
            ok = bool(anc & SPORT_ROOTS)
        if ok:
            key = w.replace(" ", "_")
            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)
                                  for l in x.lemmas() if l.name().lower() == key)
            ok = cnt("n") >= cnt("v")
    _NOUN_CACHE[w] = ok
    return ok


def get_wordnet_hypernyms(words):
    """IsA: cha + ông (đủ sâu, không quá chung) + mọi tổ tiên thuộc BASIC_CATEGORIES.
    Chỉ dùng nghĩa phổ biến nhất ⇒ không còn 'bicycle là_một ride', 'beige là_một brown'."""
    if not CFG.use_wordnet:
        return {}
    wn = _wordnet()
    if wn is None:
        fb = {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}
        print(f"⚠ WordNet không dùng được → FALLBACK_ISA ({len(fb)}/{len(words)} lớp có khái niệm cha).")
        return fb
    out = {}
    for w in words:
        s = _main_noun_synset(w)
        if s is None:
            continue
        hy = []
        for path in s.hypernym_paths()[:2]:
            for depth in range(len(path) - 2, -1, -1):        # path[-1] chính là w
                h = path[depth]
                name = h.lemmas()[0].name().replace("_", " ").lower()
                near = depth >= len(path) - 3                   # cha hoặc ông
                if name == w or name in GENERIC_CONCEPTS or name in hy:
                    continue
                if (near and depth >= CFG.isa_min_depth and
                        (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) \
                        or name in BASIC_CATEGORIES:
                    hy.append(name)
        if hy:
            out[w] = hy[:5]
    for w in words:
        if w not in out and w in FALLBACK_ISA:
            out[w] = FALLBACK_ISA[w]
    print(f"WordNet(+fallback): {len(out)}/{len(words)} lớp đối tượng có khái niệm cha")
    return out


class MultimodalGraph:
    """Đồ thị dị thể ảnh–đối tượng–khái niệm, kèm ma trận đường đi (đã nhân IDF) theo mẫu."""

    def __init__(self, records, train_ids, ablation="none", seed=0):
        """ablation: none | no_relations | random_edges | no_scene | no_related | no_concept"""
        self.records = records
        self.ablation = ablation
        self.n_img = len(records)
        self.img_pos = {r["image_id"]: i for i, r in enumerate(records)}
        train_set = set(train_ids)
        rng = np.random.RandomState(seed)

        # ---------- từ vựng lớp đối tượng: tần suất CHỈ trên TRAIN ----------
        freq = Counter(o for r in records if r["image_id"] in train_set for o in r["objects"])
        objs = sorted([o for o, c in freq.items() if c >= CFG.min_obj_freq])
        self.obj2id = {o: self.n_img + i for i, o in enumerate(objs)}
        self.n_obj = len(objs)

        # ---------- khái niệm (WordNet) ----------
        hyper = {} if ablation in ("no_relations", "no_concept") else get_wordnet_hypernyms(objs)
        # Khái niệm trùng tên một lớp đối tượng (vd 'dog' là cha của 'terrier') được nối
        # THẲNG vào node đối tượng đó — bản trước tạo 2 node 'dog' và truy vấn 'dog' chỉ
        # khớp node khái niệm, làm mất đường đi P0.
        concepts = sorted({c for v in hyper.values() for c in v
                           if c not in self.obj2id})[:CFG.max_concepts]
        base = self.n_img + self.n_obj
        self.con2id = {c: base + i for i, c in enumerate(concepts)}
        self.n_con = len(concepts)
        # ---------- v5: node BỘ BA cấp ảnh (s, p, o) — bỏ khi no_relations / no_triples ----------
        self.tri_base = self.n_img + self.n_obj + self.n_con
        use_tri = (getattr(CFG, "use_triples", False) and ablation not in ("no_relations", "no_triples")
                   and any(r.get("img_triples") for r in records))
        tri_keys = sorted({k for r in records for k, _ in r.get("img_triples", [])
                           if k[0] in self.obj2id and k[2] in self.obj2id}) if use_tri else []
        self.tri2id = {k: self.tri_base + i for i, k in enumerate(tri_keys)}
        self.id2key = {v: k for k, v in self.tri2id.items()}
        self.n_tri = len(tri_keys)
        self.n_nodes = self.tri_base + self.n_tri

        self.id2name = {}
        for o, i in self.obj2id.items(): self.id2name[i] = o
        for c, i in self.con2id.items(): self.id2name[i] = c
        for k, i in self.tri2id.items(): self.id2name[i] = " ".join(k)

        # ---------- cạnh contains (nhãn riêng của mỗi ảnh) ----------
        rows, cols = [], []
        self.img_objects = []
        for i, r in enumerate(records):
            ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
            self.img_objects.append(ids)
            rows += [i] * len(ids); cols += ids
        A_con = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                              shape=(self.n_img, self.n_nodes))

        # ---------- cạnh scene: CHỈ train, có ngưỡng + top-k ----------
        scene = Counter(); pred_cnt = defaultdict(Counter)
        if ablation not in ("no_relations", "no_scene"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue                      # ← chốt chặn rò rỉ
                for s, p, o in r["relations"]:
                    if s in self.obj2id and o in self.obj2id and s != o:
                        key = (self.obj2id[s], self.obj2id[o])
                        scene[key] += 1; pred_cnt[key][p] += 1
            by_src = defaultdict(list)
            for (a, b), v in scene.items():
                if v >= CFG.scene_min_count:
                    by_src[a].append((v, b))
            scene = {(a, b): v for a, lst in by_src.items()
                     for v, b in sorted(lst, reverse=True)[:CFG.scene_top_k]}
        # vị từ hiển thị = vị từ PHỔ BIẾN NHẤT của cặp (bản trước lấy vị từ gặp đầu tiên)
        self.scene_pred = {k: pred_cnt[k].most_common(1)[0][0] for k in scene}
        # số ảnh TRAIN chứa mỗi lớp — mẫu số cho P(đích | nguồn)
        n_tr = max(sum(1 for r in records if r["image_id"] in train_set), 1)
        obj_df = Counter(self.obj2id[o] for r in records if r["image_id"] in train_set
                         for o in set(r["objects"]) if o in self.obj2id)
        scene_w = {(a, b): min(1.0, v / max(obj_df[a], 1)) for (a, b), v in scene.items()}
        A_scene = self._mat(scene_w)

        # ---------- cạnh related: đồng xuất hiện, CHỈ train ----------
        cooc = Counter()
        if ablation not in ("no_relations", "no_related"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue
                ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
                for a, b in itertools.combinations(ids, 2):
                    cooc[(a, b)] += 1; cooc[(b, a)] += 1
            cooc = {k: v for k, v in cooc.items() if v >= CFG.cooc_min_count}
            by_src = defaultdict(list)
            for (a, b), v in cooc.items():
                if CFG.cooc_score == "npmi":
                    # NPMI ∈ [-1, 1]: 0 = độc lập. Đếm thô ưu ái lớp phổ biến (hub) ⇒ mọi lớp
                    # đều nối tới person/man/floor và P3/P5 thành nhiễu phủ khắp gallery.
                    p_ab = v / n_tr
                    pmi = math.log(p_ab / ((obj_df[a] / n_tr) * (obj_df[b] / n_tr)))
                    s = pmi / max(-math.log(p_ab), 1e-6)
                    if s < CFG.cooc_min_npmi:
                        continue
                else:
                    s = v
                by_src[a].append((s, b))
            cooc = {(a, b): cooc[(a, b)] for a, lst in by_src.items()
                    for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]}
        cooc_w = {(a, b): v / max(obj_df[a], 1) for (a, b), v in cooc.items()}   # P(b | a)
        A_rel = self._mat(cooc_w)

        # ---------- cạnh isa (đích có thể là node khái niệm HOẶC node đối tượng) ----------
        isa = {}
        for o, cs in hyper.items():
            if o not in self.obj2id:
                continue
            for c in cs:
                t = self.obj2id.get(c, self.con2id.get(c))
                if t is not None and t != self.obj2id[o]:
                    isa[(self.obj2id[o], t)] = 1.0
        A_isa = self._mat(isa)
        A_isa_true = A_isa.copy()                 # cho IDF: không phụ thuộc ablation random_edges

        # ---------- ablation: cạnh ngẫu nhiên, GIỮ NGUYÊN số cạnh ----------
        if ablation == "random_edges":
            A_scene = self._randomize(A_scene, rng)
            A_rel   = self._randomize(A_rel, rng)
            A_isa   = self._randomize(A_isa, rng)
        if ablation in ("no_relations", "no_class_edges"):     # no_class_edges (v8): bỏ cạnh CẤP LỚP, GIỮ bộ ba cấp ảnh/video
            A_scene = A_rel = A_isa = self._mat({})

        # ---------- v5: cạnh ảnh –thể_hiện→ bộ ba, trọng số = xác suất probe của CHÍNH ảnh ----------
        # shuffled_triples / random_edges: ảnh i nhận bộ ba của ảnh π(i) ⇒ đối chứng gắn-với-ảnh.
        trows, tcols, tvals = [], [], []
        if self.n_tri:
            src_img = (rng.permutation(self.n_img) if ablation in ("shuffled_triples", "random_edges")
                       else np.arange(self.n_img))
            for i in range(self.n_img):
                for k, pr in records[int(src_img[i])].get("img_triples", []):
                    t = self.tri2id.get(k)
                    if t is not None:
                        trows.append(i); tcols.append(t); tvals.append(float(pr))
        A_tri = sp.csr_matrix((np.array(tvals, np.float32), (np.array(trows, int), np.array(tcols, int))),
                              shape=(self.n_img, self.n_nodes))
        A_tri.sum_duplicates()
        self.A_triple = A_tri

        self.scene_count = dict(scene)
        self.cooc_count = dict(cooc)
        for M in (A_scene, A_rel, A_isa):
            if CFG.edge_weighting:
                M.data[:] = np.clip(M.data, 0.0, 1.0)   # P(đích|nguồn); xáo cạnh có thể cộng dồn >1
            else:
                M.data[:] = 1.0                   # R_p[i,c] = SỐ đường đi thật sự (cách cũ)
        self.A_contains, self.A_scene, self.A_related, self.A_isa = A_con, A_scene, A_rel, A_isa

        # ---------- ma trận đếm đường đi theo mẫu ----------
        R = {
            "P0_contains":    A_con.tocsr(),
            "P1_scene":       (A_con @ A_scene).tocsr(),
            "P2_isa":         (A_con @ A_isa).tocsr(),
            "P3_related":     (A_con @ A_rel).tocsr(),
            "P4_scene_isa":   (A_con @ A_scene @ A_isa).tocsr(),
            "P5_related_isa": (A_con @ A_rel @ A_isa).tocsr(),
            "P6_triple":      A_tri.tocsr(),
        }
        # Bỏ đường đi gián tiếp tới khái niệm mà ảnh ĐÃ chứa trực tiếp: chúng chỉ đếm lại
        # tín hiệu P0 (vd 'dog –pushing→ terrier –là_một→ dog' ở bản trước).
        if CFG.drop_redundant_indirect:
            direct = (A_con > 0).astype(np.float32)
            for p in PATTERNS[1:]:
                R[p] = (R[p] - R[p].multiply(direct)).tocsr()
                R[p].eliminate_zeros()

        # IDF của node đích, tính CHỈ trên ảnh TRAIN: khái niệm mà hầu như ảnh nào cũng
        # với tới (person, man...) không phân biệt được ứng viên ⇒ trọng số ≈ 0.
        tr_rows = np.array(sorted(self.img_pos[i] for i in train_set if i in self.img_pos))
        if CFG.idf_from == "taxonomy":
            # df(đối tượng) = số ảnh train CHỨA nó; df(khái niệm) = số ảnh train chứa một hạ vị
            # của nó (isa thật). Không đếm scene/related ⇒ IDF của khớp trực tiếp giống hệt
            # nhau ở mọi ablation, và quan hệ không còn "pha loãng" P0.
            df_con = np.asarray((A_con[tr_rows] > 0).sum(0)).ravel()
            df_isa = np.asarray(((A_con @ A_isa_true)[tr_rows] > 0).sum(0)).ravel()
            df_tri = np.asarray((A_tri[tr_rows] > 0).sum(0)).ravel()      # v5: df của bộ ba = số ảnh train mang nó
            ar = np.arange(self.n_nodes)
            df = np.where(ar < self.n_img + self.n_obj, df_con, np.where(ar < self.tri_base, df_isa, df_tri))
        else:
            reach = None
            for p in PATTERNS:
                m = (R[p][tr_rows] > 0).astype(np.float32)
                reach = m if reach is None else reach + m
            df = np.asarray((reach > 0).sum(0)).ravel()
        idf = np.log((len(tr_rows) + 1.0) / (df + 1.0)).astype(np.float32)
        if not CFG.use_idf:
            idf = np.ones_like(idf)
        idf[:self.n_img] = 0.0
        self.idf = idf
        D = sp.diags(idf)
        # R[p][i, c] = Σ_{đường đi i→c theo mẫu p} idf(c). Liệt kê đường đi dùng ĐÚNG
        # trọng số này nên giải thích vẫn khớp điểm (self-test kiểm tra).
        self.R = {p: (R[p] @ D).tocsr() for p in PATTERNS}
        for p in PATTERNS:
            self.R[p].eliminate_zeros()

        self.name2id = dict(self.obj2id); self.name2id.update(self.con2id)
        self.edge_index, self.edge_type = self._build_edge_index()

    def _mat(self, d):
        if not d:
            return sp.csr_matrix((self.n_nodes, self.n_nodes), dtype=np.float32)
        keys = list(d.keys())
        r = np.fromiter((k[0] for k in keys), int, len(keys))
        c = np.fromiter((k[1] for k in keys), int, len(keys))
        v = np.fromiter(d.values(), np.float32, len(keys))
        return sp.csr_matrix((v, (r, c)), shape=(self.n_nodes, self.n_nodes))

    def _randomize(self, A, rng):
        """Giữ nguyên SỐ LƯỢNG cạnh, xáo trộn đích đến — đối chứng cho ablation."""
        A = A.tocoo()
        if A.nnz == 0:
            return A.tocsr()
        new_col = rng.randint(self.n_img, getattr(self, "tri_base", self.n_nodes), size=A.nnz)
        M = sp.csr_matrix((A.data, (A.row, new_col)), shape=A.shape)
        M.sum_duplicates()
        return M

    def _build_edge_index(self):
        parts, types = [], []
        for t, A in enumerate([self.A_contains.tocoo(), self.A_scene.tocoo(),
                               self.A_related.tocoo(), self.A_isa.tocoo()]):
            if A.nnz == 0:
                continue
            parts.append(np.stack([A.row, A.col])); types.append(np.full(A.nnz, t))
            parts.append(np.stack([A.col, A.row])); types.append(np.full(A.nnz, t + 4))
        if not parts:
            return (torch.zeros(2, 0, dtype=torch.long), torch.zeros(0, dtype=torch.long))
        ei = torch.tensor(np.concatenate(parts, 1), dtype=torch.long)
        et = torch.tensor(np.concatenate(types), dtype=torch.long)
        return ei, et

    def summary(self):
        return dict(nodes=self.n_nodes, images=self.n_img, object_classes=self.n_obj,
                    concepts=self.n_con, edges=int(self.edge_index.shape[1] // 2),
                    contains=int(self.A_contains.nnz), scene=int(self.A_scene.nnz),
                    related=int(self.A_related.nnz), isa=int(self.A_isa.nnz),
                    triple_nodes=int(getattr(self, "n_tri", 0)),
                    triple_edges=int(self.A_triple.nnz) if hasattr(self, "A_triple") else 0,
                    paths={p: int(self.R[p].nnz) for p in PATTERNS},
                    ablation=self.ablation)

## 1.6 Phân tích truy vấn → khái niệm trên đồ thị

Một truy vấn được quy về tập node (lớp đối tượng hoặc khái niệm) mà nó nhắc tới.
Đây là điểm neo để `s_path` tìm đường đi.

In [ ]:
STOP = set("a an the of on in at to with and or is are was were this that these those "
           "there here it its his her their some any for from by as be being been very "
           "photo image picture showing shows scene view near next".split())


def parse_query(text, graph):
    """Trả về [(node_id, tên, n-gram khớp)] cho các khái niệm xuất hiện trong truy vấn."""
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):                                    # ưu tiên cụm 2 từ
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits


def path_score_matrix(graph, concept_ids, cand_idx):
    """[len(cand), n_patterns] — số đường đi (đã log1p) theo từng mẫu, cho mỗi ứng viên."""
    out = np.zeros((len(cand_idx), len(PATTERNS)), np.float32)
    if not concept_ids:
        return out
    cols = np.array(sorted(set(concept_ids)))
    for p, name in enumerate(PATTERNS):
        sub = graph.R[name][cand_idx][:, cols]
        out[:, p] = np.asarray(sub.sum(1)).ravel()
    return np.log1p(out)

## 1.7 Mô hình

* **1.7.1 GNN cổ điển (cấp độ 1)**: GraphSAGE/GAT + PathScorer.
* **1.7.2 GNN của SCENIR (nhiệm vụ 2)**: thay nhánh `s_GNN` bằng encoder của SCENIR, bật bằng `CFG.gnn_type = "scenir"`.

### 1.7.1 GNN cổ điển (cấp độ 1)

GNN **tự cài bằng PyTorch thuần** (không cần PyTorch Geometric — tránh hẳn rủi ro cài đặt
trên Kaggle, và là thứ nhóm phải tự giải thích được khi vấn đáp).

* **GraphSAGE** — `h' = W₁h + W₂·mean(h_neighbours)`. Inductive: node ảnh test chưa từng
  tham gia huấn luyện vẫn có biểu diễn hợp lệ.
* **GAT** — hệ số chú ý từng cạnh, softmax theo node đích (dùng `index_reduce_` để ổn định số học).
* **PathScorer** — một trọng số học được cho mỗi mẫu đường đi, nhân hệ số suy giảm theo độ dài.
  Vì điểm là **tổng cộng tính** nên đóng góp của từng đường đi tách bạch được ⇒ fidelity đo được.

In [ ]:
def scatter_mean(src, index, n):
    out = torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    cnt = torch.zeros(n, 1, device=src.device, dtype=src.dtype)
    cnt.index_add_(0, index, torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype))
    return out / cnt.clamp(min=1)


def scatter_softmax(logits, index, n):
    m = torch.full((n,), -1e30, device=logits.device, dtype=logits.dtype)
    m = m.index_reduce_(0, index, logits, "amax", include_self=True)
    e = (logits - m[index]).exp()
    s = torch.zeros(n, device=logits.device, dtype=logits.dtype).index_add_(0, index, e)
    return e / s[index].clamp(min=1e-16)


class SAGELayer(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.Linear(din, dout)

    def forward(self, x, edge_index, edge_weight=None):
        src, dst = edge_index
        msg = x[src] if edge_weight is None else x[src] * edge_weight.unsqueeze(-1)
        agg = scatter_mean(msg, dst, x.size(0))
        return self.self_w(x) + self.neigh_w(agg)


class TypedSAGELayer(nn.Module):
    """h' = W₀h + Σ_t σ(g_t)·W_t·mean_{u∈N_t(v)} h_u — mỗi loại cạnh một ma trận và một cổng.
    SAGELayer cũ gộp contains/scene/related/isa vào CHUNG một trung bình ⇒ cạnh nhiễu pha
    loãng tín hiệu ảnh–đối tượng, và mô hình không có cách nào tắt riêng một loại cạnh.
    Cổng các loại quan hệ khởi tạo gần đóng ⇒ điểm xuất phát ≈ no_relations."""

    N_TYPES = 8                                   # 4 loại × 2 chiều (xem _build_edge_index)

    def __init__(self, din, dout, gate_init=-1.5):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(self.N_TYPES)])
        g = torch.full((self.N_TYPES,), float(gate_init))
        g[0] = g[4] = 3.0                         # contains (hai chiều) mở sẵn
        self.gate = nn.Parameter(g)

    def forward(self, x, edge_index, edge_type):
        src, dst = edge_index
        out = self.self_w(x)
        for t in range(self.N_TYPES):
            m = edge_type == t
            if not bool(m.any()):
                continue
            agg = scatter_mean(x[src[m]], dst[m], x.size(0))
            out = out + torch.sigmoid(self.gate[t]) * self.neigh_w[t](agg)
        return out


class GATLayer(nn.Module):
    def __init__(self, din, dout, heads=4):
        super().__init__()
        assert dout % heads == 0
        self.h, self.dk = heads, dout // heads
        self.lin = nn.Linear(din, dout)
        self.att_src = nn.Parameter(torch.empty(heads, self.dk))
        self.att_dst = nn.Parameter(torch.empty(heads, self.dk))
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, edge_index, edge_weight=None):
        N = x.size(0)
        src, dst = edge_index
        z = self.lin(x).view(N, self.h, self.dk)
        a = (z * self.att_src).sum(-1)[src] + (z * self.att_dst).sum(-1)[dst]   # [E,H]
        a = F.leaky_relu(a, 0.2)
        if edge_weight is not None:
            a = a + edge_weight.log().clamp(min=-5).unsqueeze(-1)
        out = torch.zeros(N, self.h, self.dk, device=x.device, dtype=z.dtype)
        for hh in range(self.h):                      # softmax riêng từng head
            al = scatter_softmax(a[:, hh], dst, N)
            out[:, hh].index_add_(0, dst, z[src, hh] * al.unsqueeze(-1))
        return out.reshape(N, -1)


class GraphScorer(nn.Module):
    """Sinh biểu diễn node bằng GNN, chiếu CLIP-text vào cùng không gian, cộng điểm đường đi."""

    def __init__(self, n_nodes, clip_dim, img_feat, cfg=CFG, edge_type=None):
        super().__init__()
        self.n_img = img_feat.shape[0]
        self.typed = cfg.gnn_typed and cfg.gnn_type == "sage"
        # loại cạnh của đồ thị ĐẦY ĐỦ — dùng khi encode_graph được gọi không kèm edge_type
        self.register_buffer("full_et", (edge_type if edge_type is not None
                                         else torch.zeros(0, dtype=torch.long)).clone())
        # Node ẢNH: chiếu từ đặc trưng CLIP đóng băng ⇒ inductive, ảnh test chưa từng
        # tham gia huấn luyện vẫn có biểu diễn hợp lệ (không phải vector ngẫu nhiên).
        self.register_buffer("img_feat", img_feat.clone())
        self.img_proj = nn.Linear(clip_dim, cfg.gnn_dim)
        # Node ĐỐI TƯỢNG / KHÁI NIỆM: embedding học được (chúng lặp lại giữa các ảnh).
        self.sym_emb = nn.Embedding(n_nodes - self.n_img, cfg.gnn_dim)
        nn.init.normal_(self.sym_emb.weight, std=0.02)
        if self.typed:
            self.layers = nn.ModuleList([TypedSAGELayer(cfg.gnn_dim, cfg.gnn_dim, cfg.gate_init)
                                         for _ in range(cfg.gnn_layers)])
        else:
            Layer = SAGELayer if cfg.gnn_type == "sage" else GATLayer
            self.layers = nn.ModuleList(
                [Layer(cfg.gnn_dim, cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.norms = nn.ModuleList(
            [nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.drop = nn.Dropout(cfg.gnn_dropout)
        self.q_proj = nn.Sequential(                     # CLIP-text → không gian đồ thị
            nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(),
            nn.Linear(cfg.gnn_dim, cfg.gnn_dim))
        # PathScorer: 1 trọng số / mẫu, softplus để luôn ≥0 (giải thích mới diễn giải được)
        # P0 khởi tạo 0 (softplus = 0.69), P1..P5 khởi tạo thấp: đường đi gián tiếp phải
        # "tự chứng minh" trên mẫu âm khó thì trọng số mới tăng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        # path_gain CỐ ĐỊNH (buffer, không học): sau znorm_per_row điểm bất biến theo tỉ lệ nên
        # gain không có tín hiệu thật — gradient chỉ đi qua eps=1e-6 của z-norm, rất nhỏ nhưng
        # cùng dấu, và Adam vẫn bước đủ lr ⇒ gain trôi về ~1e-7, nhánh path bị TẮT NGẦM và mọi
        # "đóng góp" trong giải thích hiện +0.0000 (đã gặp trên AG). Tắt nhánh path là việc của λ.
        self.register_buffer("path_gain", torch.tensor(1.0))
        # trọng số của z(s_clip) trong logit lúc HUẤN LUYỆN, học được (softplus, khởi tạo = 1):
        # trên AG CLIP rất yếu (MRR 0.12) nên mô hình cần hạ được trọng số này.
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        # Hệ số của z(s_path) trong logit HUẤN LUYỆN (softplus, khởi tạo = 1). Thiếu nó, nhánh path
        # có hệ số cứng = 1, nên khi nó gây hại mô hình tự tắt bằng đường thoái hoá: đợt 2 kéo
        # path_gain → 1e-7, đợt 3 dồn trọng số chuẩn hoá sang mẫu không có dữ liệu (P0 → 0.000).
        # Lúc suy luận vẫn trộn bằng λ chọn trên val; hệ số này chỉ dùng khi huấn luyện.
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def node_features(self):
        return torch.cat([self.img_proj(self.img_feat), self.sym_emb.weight], 0)

    def edge_gates(self):
        """σ(g_t) của từng lớp — in ra để biết mô hình có thực sự dùng quan hệ hay không."""
        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers] if self.typed else []

    def encode_graph(self, edge_index, edge_type=None):
        h = self.node_features()
        if self.typed and edge_type is None:
            assert self.full_et.numel() == edge_index.size(1), \
                "encode_graph trên đồ thị con cần truyền edge_type tương ứng"
            edge_type = self.full_et
        for i, (l, n) in enumerate(zip(self.layers, self.norms)):
            h = l(h, edge_index, edge_type) if self.typed else l(h, edge_index)
            h = n(h)
            if i < len(self.layers) - 1:
                h = self.drop(F.gelu(h))
        return F.normalize(h, dim=-1)

    def pattern_weights(self):
        """Trọng số hiệu dụng của mỗi mẫu: P0 NEO = 1, P_k = softplus(w_k) · decay^(len_k − 1).
        Neo P0 thay cho chuẩn hoá tổng = 1: tỉ lệ vẫn định danh được, nhưng trọng số không thể
        trôi sang mẫu vắng dữ liệu (no_relations seed 2 đợt 3: P0 = 0.000, nhánh path chết)."""
        d = torch.sigmoid(self.decay)
        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)
        w = F.softplus(self.path_w) * d.pow(lens - 1)
        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])

    def gnn_score(self, h, q_clip, cand_idx):
        q = F.normalize(self.q_proj(q_clip), dim=-1)          # [B, d]
        return torch.einsum("bd,bkd->bk", q, h[cand_idx])      # [B, K]

    def path_score(self, path_mat):
        """path_mat: [B, K, n_patterns] → [B, K]. Cộng tính theo mẫu ⇒ tách được đóng góp."""
        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)

### 1.7.2 GNN của SCENIR thay cho GNN cổ điển

SCENIR (ICML 2025, [arXiv 2505.15867](https://arxiv.org/html/2505.15867v1)). Chỉ thay nhánh `s_GNN`; split, ứng viên CLIP,
`s_path`, chọn α/λ trên val, ablation và fidelity giữ nguyên. Bật bằng `CFG.gnn_type = "scenir"`.

* **Đầu vào:** mỗi ảnh một scene graph. Đỉnh = lớp đối tượng của ảnh + khái niệm IsA của chúng; cạnh = cạnh
  `scene`/`related` (xây chỉ từ train) và `isa` của `MultimodalGraph` nằm trong tập đỉnh đó. Đặc trưng đỉnh =
  CLIP-text của tên đỉnh.
* **Encoder:** SCENIR GAE — 3 lớp GIN với hai nhánh `mu`/`logstd` tách rời, feature decoder và edge decoder MLP,
  discriminator đối kháng. Huấn luyện tự giám sát trên scene graph của ảnh **train** rồi đóng băng.
* **Điểm:** `z_I = L2norm(Σ_v mu_v)`, `s_GNN = cos(q_proj(CLIP_text(q)), z_I)`. `q_proj` và trọng số đường đi học
  bằng đúng vòng InfoNCE của GNN cổ điển trong `train_graph_scorer`.

In [ ]:
from contextlib import contextmanager

# Siêu tham số SCENIR = config.json của repo gốc; chiều đặc trưng đỉnh lấy theo CLIP-text (512 thay vì 768).
for _k, _v in dict(scenir_latent=1000, scenir_layers=3, scenir_disc_hidden=32,
                   scenir_edge_hidden=256, scenir_edge_out=32, scenir_lr=1e-3,
                   scenir_lr_gamma=0.95, scenir_bs=64, scenir_disc_steps=5,
                   scenir_epochs=30, scenir_with_concepts=True,
                   scenir_proj_hidden=1024).items():
    setattr(CFG, _k, _v)


@contextmanager
def use_gnn(kind):
    """Tạm đổi CFG.gnn_type ('sage' | 'gat' | 'scenir'); khôi phục khi ra khỏi khối."""
    old = CFG.gnn_type
    CFG.gnn_type = kind
    try:
        yield
    finally:
        CFG.gnn_type = old


def image_scene_graphs(graph):
    """Hợp rời (disjoint union) scene graph của mọi ảnh, cache trên chính đồ thị.
    node_gid: id node toàn cục của từng đỉnh cục bộ; node_ptr/edge_ptr: đỉnh/cạnh của ảnh i nằm liền
    nhau trong [ptr[i], ptr[i+1]); edge_index vô hướng (mỗi cạnh lưu hai chiều)."""
    if hasattr(graph, "_scenir_sg"):
        return graph._scenir_sg
    A_obj = ((graph.A_scene + graph.A_related) > 0).tocsr()
    A_isa = graph.A_isa.tocsr()
    nbrs = lambda M, n: M.indices[M.indptr[n]:M.indptr[n + 1]]
    node_gid, node_ptr, edge_ptr, src, dst = [], [0], [0], [], []
    for i in range(graph.n_img):
        objs = [int(o) for o in graph.img_objects[i]]
        local = {o: k for k, o in enumerate(objs)}
        pairs = set()
        for o in objs:
            for t in nbrs(A_obj, o):
                t = int(t)
                if t in local and t != o:
                    pairs.add((min(local[o], local[t]), max(local[o], local[t])))
        if CFG.scenir_with_concepts:
            for o in objs:
                for c in nbrs(A_isa, o):
                    c = int(c)
                    if c == o:
                        continue
                    if c not in local:
                        local[c] = len(local)
                    pairs.add((min(local[o], local[c]), max(local[o], local[c])))
        base = len(node_gid)
        node_gid.extend(local)                 # dict giữ thứ tự chèn = thứ tự id cục bộ
        node_ptr.append(len(node_gid))
        for a, b in sorted(pairs):
            src += [base + a, base + b]; dst += [base + b, base + a]
        edge_ptr.append(len(src))
    sg = dict(node_gid=np.array(node_gid, np.int64), node_ptr=np.array(node_ptr, np.int64),
              edge_ptr=np.array(edge_ptr, np.int64),
              edge_index=np.array([src, dst], np.int64).reshape(2, -1))
    graph._scenir_sg = sg
    return sg


_SCENIR_NAME_EMB = {}


def _encode_node_names(names):
    """CLIP-text đóng băng (cùng encoder với truy vấn). Không có CLIP (synthetic/offline) thì dùng
    vector ngẫu nhiên cố định theo tên — chỉ để smoke-test."""
    try:
        return clip_text_features(names)
    except Exception as e:
        print(f"⚠ Không mã hoá được tên node bằng CLIP ({type(e).__name__}) → vector ngẫu nhiên theo tên.")
        out = []
        for n in names:
            v = np.random.RandomState(int(hashlib.md5(n.encode()).hexdigest()[:8], 16)).randn(512)
            out.append((v / np.linalg.norm(v)).astype(np.float32))
        return np.stack(out)


def node_name_features(graph):
    """[n_nodes, D] đặc trưng đỉnh; dòng của node ảnh để 0 (scene graph không chứa node ảnh)."""
    names = [graph.id2name[i] for i in range(graph.n_img, graph.n_nodes)]
    new = [n for n in dict.fromkeys(names) if n not in _SCENIR_NAME_EMB]
    if new:
        _SCENIR_NAME_EMB.update(zip(new, _encode_node_names(new)))
    dim = len(next(iter(_SCENIR_NAME_EMB.values()))) if _SCENIR_NAME_EMB else 512
    X = np.zeros((graph.n_nodes, dim), np.float32)
    if names:
        X[graph.n_img:] = np.stack([_SCENIR_NAME_EMB[n] for n in names])
    return torch.tensor(X)

In [ ]:
def scatter_sum(src, index, n):
    return torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype).index_add_(0, index, src)


class GINLayer(nn.Module):
    """GINConv(nn.Linear) của PyG với ε = 0: h'_v = W·(h_v + Σ_{u∈N(v)} h_u)."""

    def __init__(self, din, dout):
        super().__init__()
        self.lin = nn.Linear(din, dout)

    def forward(self, x, edge_index):
        src, dst = edge_index
        return self.lin(x + scatter_sum(x[src], dst, x.size(0)))


class ScenirEncoder(nn.Module):
    """MultiGINEncoder của SCENIR: hai nhánh GIN tách rời cho mu và logstd, không ReLU ở lớp cuối."""

    def __init__(self, din, dhid, dout, layers=3):
        super().__init__()
        dims = [din] + [dhid] * (layers - 1) + [dout]
        self.gnn_mu = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.gnn_logstd = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])

    @staticmethod
    def _run(layers, x, edge_index):
        for i, layer in enumerate(layers):
            x = layer(x, edge_index)
            if i < len(layers) - 1:
                x = F.relu(x)
        return x

    def forward(self, x, edge_index, mu_only=False):
        mu = self._run(self.gnn_mu, x, edge_index)
        return mu if mu_only else (mu, self._run(self.gnn_logstd, x, edge_index))


class ScenirMLP(nn.Module):
    """MLP 2 lớp — feature decoder và edge decoder của SCENIR."""

    def __init__(self, din, dhid, dout):
        super().__init__()
        self.layer_1, self.layer_2 = nn.Linear(din, dhid), nn.Linear(dhid, dout)

    def forward(self, x):
        return self.layer_2(F.relu(self.layer_1(x)))


class ScenirDiscriminator(nn.Module):
    def __init__(self, din, dhid):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(din, dhid), nn.ReLU(),
                                 nn.Linear(dhid, dhid), nn.ReLU(), nn.Linear(dhid, 1))

    def forward(self, z):
        return self.net(z)


SCENIR_EPS, SCENIR_MAX_LOGSTD = 1e-15, 10


def scenir_negative_edges(edge_index, batch, n_samples):
    """Cặp (u, v) cùng một đồ thị trong batch, không tự nối, không trùng cạnh thật."""
    N = batch.numel()
    cnt = torch.bincount(batch)
    ptr = torch.cat([cnt.new_zeros(1), cnt.cumsum(0)[:-1]])
    u = torch.randint(0, N, (2 * n_samples,), device=batch.device)
    b = batch[u]
    off = torch.minimum((torch.rand(u.numel(), device=batch.device) * cnt[b]).long(), cnt[b] - 1)
    v = ptr[b] + off
    keep = (u != v) & ~torch.isin(u * N + v, edge_index[0] * N + edge_index[1])
    return torch.stack([u[keep], v[keep]])[:, :n_samples]


class ScenirGAE(nn.Module):
    """SCENIR = ARGVA (VGAE + điều chuẩn đối kháng) + feature decoder MLP + edge decoder MLP."""

    def __init__(self, din, cfg=CFG):
        super().__init__()
        L = cfg.scenir_latent
        self.encoder = ScenirEncoder(din, L, L, cfg.scenir_layers)
        self.discriminator = ScenirDiscriminator(L, cfg.scenir_disc_hidden)
        self.feature_decoder = ScenirMLP(L, din, din)
        self.edge_decoder = ScenirMLP(L, cfg.scenir_edge_hidden, cfg.scenir_edge_out)

    def encode(self, x, edge_index):
        mu, logstd = self.encoder(x, edge_index)
        logstd = logstd.clamp(max=SCENIR_MAX_LOGSTD)
        z = mu + torch.randn_like(logstd) * logstd.exp() if self.training else mu
        return z, mu, logstd

    @staticmethod
    def _inner(z, edge_index):
        return torch.sigmoid((z[edge_index[0]] * z[edge_index[1]]).sum(-1))

    @staticmethod
    def kl_loss(mu, logstd):
        return -0.5 * torch.mean(torch.sum(1 + 2 * logstd - mu ** 2 - logstd.exp() ** 2, dim=1))

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.discriminator(z)) + SCENIR_EPS).mean()

    def discriminator_loss(self, z):
        real = torch.sigmoid(self.discriminator(torch.randn_like(z)))
        fake = torch.sigmoid(self.discriminator(z.detach()))
        return -torch.log(real + SCENIR_EPS).mean() - torch.log(1 - fake + SCENIR_EPS).mean()

    def recon_loss(self, x, edge_index, batch):
        """recon_loss_batched của repo: MSE tái tạo đặc trưng + BCE cạnh dương/âm trên đầu ra edge decoder.
        Encode lại như bản gốc; trả về (loss, mu, logstd) để kl_loss dùng đúng lần encode này."""
        z, mu, logstd = self.encode(x, edge_index)
        loss = F.mse_loss(self.feature_decoder(z), x)
        if edge_index.size(1) > 0:
            z_e = self.edge_decoder(z)
            loss = loss - torch.log(self._inner(z_e, edge_index) + SCENIR_EPS).mean()
            neg = scenir_negative_edges(edge_index, batch, edge_index.size(1))
            if neg.size(1) > 0:
                loss = loss - torch.log(1 - self._inner(z_e, neg) + SCENIR_EPS).mean()
        return loss, mu, logstd

In [ ]:
def _sg_batch(sg, X, img_idx):
    """Gom scene graph của các ảnh img_idx thành một batch (x, edge_index, batch) trên DEVICE.
    X phải nằm sẵn trên DEVICE."""
    img_idx = np.asarray(img_idx)
    ranges = lambda p: np.concatenate([np.arange(p[i], p[i + 1]) for i in img_idx])
    nodes, edges = ranges(sg["node_ptr"]), ranges(sg["edge_ptr"])
    counts = sg["node_ptr"][img_idx + 1] - sg["node_ptr"][img_idx]
    remap = np.full(len(sg["node_gid"]), -1, np.int64)
    remap[nodes] = np.arange(len(nodes))
    ei = remap[sg["edge_index"][:, edges]]
    x = X[torch.from_numpy(sg["node_gid"][nodes]).to(X.device)]
    return (x, torch.from_numpy(ei).to(X.device),
            torch.from_numpy(np.repeat(np.arange(len(img_idx)), counts)).to(X.device))


def train_scenir_gae(sg, X, img_idx, seed=0, verbose=True):
    """Huấn luyện SCENIR GAE tự giám sát CHỈ trên scene graph của ảnh train (không truy vấn, không nhãn).
    loss = (1/6)·reg + 3·recon + (1/3)·(1/num_nodes)·kl; discriminator cập nhật disc_steps bước/lần."""
    set_seed(seed)
    # Giữ cả đồ thị không cạnh (recon_loss chịu được), nếu không ablation no_relations không còn gì để huấn luyện.
    n_nodes = np.diff(sg["node_ptr"])
    idx = np.array([i for i in img_idx if n_nodes[i] > 0], np.int64)
    assert len(idx) > 0, "Không ảnh train nào có scene graph — kiểm tra nhãn đối tượng."
    gae = ScenirGAE(X.shape[1]).to(DEVICE)
    disc_opt = torch.optim.AdamW(gae.discriminator.parameters(), lr=CFG.scenir_lr)
    enc_opt = torch.optim.AdamW(list(gae.encoder.parameters()) + list(gae.edge_decoder.parameters())
                                + list(gae.feature_decoder.parameters()), lr=CFG.scenir_lr)
    sched = torch.optim.lr_scheduler.ExponentialLR(enc_opt, gamma=CFG.scenir_lr_gamma)
    hist, t0 = [], time.time()
    for ep in range(CFG.scenir_epochs):
        gae.train()
        perm = np.random.permutation(idx)
        tot_e = tot_d = 0.0
        nb = 0
        for s in range(0, len(perm), CFG.scenir_bs):
            x, ei, b = _sg_batch(sg, X, perm[s:s + CFG.scenir_bs])
            enc_opt.zero_grad()
            z, _, _ = gae.encode(x, ei)
            d_acc = 0.0
            for _ in range(CFG.scenir_disc_steps):
                disc_opt.zero_grad()
                d_loss = gae.discriminator_loss(z)
                d_loss.backward()
                disc_opt.step()
                d_acc += d_loss.item()
            rec, mu, logstd = gae.recon_loss(x, ei, b)
            loss = ((1 / 6) * gae.reg_loss(z) + 3 * rec
                    + (1 / 3) * (1 / x.size(0)) * gae.kl_loss(mu, logstd))
            loss.backward()
            enc_opt.step()
            tot_e += loss.item(); tot_d += d_acc / CFG.scenir_disc_steps; nb += 1
        sched.step()
        hist.append((tot_e / max(nb, 1), tot_d / max(nb, 1)))
        if verbose and (ep % 5 == 0 or ep == CFG.scenir_epochs - 1):
            print(f"  [SCENIR GAE] epoch {ep:2d} | enc {hist[-1][0]:.4f} | disc {hist[-1][1]:.4f} "
                  f"| {len(idx)} đồ thị train | {time.time() - t0:.0f}s")
    gae.eval()
    for p in gae.parameters():
        p.requires_grad_(False)
    return gae, hist


@torch.no_grad()
def scenir_embed_images(gae, sg, X, chunk=1024):
    """z_I = Σ_v mu_v (eval ⇒ không lấy mẫu) cho MỌI ảnh; ảnh không có đỉnh nào → vector 0."""
    gae.eval()
    n_img = len(sg["node_ptr"]) - 1
    out = torch.zeros(n_img, CFG.scenir_latent)
    for s in range(0, n_img, chunk):
        idx = np.arange(s, min(s + chunk, n_img))
        x, ei, b = _sg_batch(sg, X, idx)
        if x.size(0) == 0:
            continue
        mu = gae.encoder(x, ei, mu_only=True)
        out[torch.from_numpy(idx)] = scatter_sum(mu, b, len(idx)).cpu()
    return out

In [ ]:
class ScenirGraphScorer(nn.Module):
    """Cùng giao diện với GraphScorer (encode_graph / gnn_score / path_score / pattern_weights), nên
    score_queries, tune_on_val, fidelity và demo phía sau dùng lại nguyên vẹn."""

    def __init__(self, gae, z_img, clip_dim, cfg=CFG):
        super().__init__()
        self.gae = gae
        self.register_buffer("z_img", F.normalize(z_img, dim=-1))      # sum-pool ⇒ bắt buộc L2
        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.scenir_proj_hidden), nn.ReLU(),
                                    nn.Linear(cfg.scenir_proj_hidden, z_img.shape[1]))
        # Phần chấm đường đi khởi tạo giống GraphScorer (mục 7.1) để hai nhánh GNN so sánh công bằng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        self.register_buffer("path_gain", torch.tensor(1.0))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def encode_graph(self, edge_index=None, edge_type=None):
        """GAE đã đóng băng ⇒ nhúng ảnh tính một lần; edge_index/edge_type chỉ để khớp giao diện GraphScorer."""
        return self.z_img

    def edge_gates(self):
        return []                                # SCENIR không có cổng theo loại cạnh

    pattern_weights = GraphScorer.pattern_weights
    gnn_score = GraphScorer.gnn_score
    path_score = GraphScorer.path_score


def build_scenir_scorer(engine, train_img_idx, seed=0, verbose=True):
    """Dựng scene graph từng ảnh, huấn luyện GAE trên ảnh train, nhúng mọi ảnh."""
    g = engine.graph
    sg = image_scene_graphs(g)
    X = node_name_features(g).to(DEVICE)
    if verbose:
        nv, ne = np.diff(sg["node_ptr"]), np.diff(sg["edge_ptr"]) // 2
        print(f"  SCENIR scene graph/ảnh: {nv.mean():.1f} đỉnh, {ne.mean():.1f} cạnh | "
              f"ảnh không có cạnh: {(ne == 0).mean():.1%}")
    gae, _ = train_scenir_gae(sg, X, train_img_idx, seed=seed, verbose=verbose)
    return ScenirGraphScorer(gae, scenir_embed_images(gae, sg, X), engine.txt_feat.shape[1])

## 1.8 Bộ máy truy vấn: ứng viên, huấn luyện, xếp hạng lại

Để tính `s_path` cho mọi cặp (truy vấn, ảnh) mà không phải lặp Python, ta dựng
ma trận chỉ báo khái niệm `Q [n_query × n_node]` rồi nhân thưa một lần cho mỗi mẫu:
`S_p = R_p · Qᵀ` → `[n_img × n_query]`. Sau đó chỉ việc tra.

In [ ]:
def znorm_per_row(x, eps=1e-6):
    """Chuẩn hoá z theo từng truy vấn. Dùng CHUNG cho train và eval để α, λ chọn
    trên val vẫn đúng khi áp lên test."""
    if isinstance(x, torch.Tensor):
        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)


minmax_per_row = znorm_per_row      # giữ tên cũ cho tương thích


class RetrievalEngine:
    def __init__(self, records, graph, img_feat, txt_feat, queries):
        """queries: list[dict(text, gold_image_id, group)] — group ∈ {direct, indirect}"""
        self.records, self.graph = records, graph
        self.img_feat = torch.tensor(img_feat)
        self.txt_feat = torch.tensor(txt_feat)
        self.queries = queries
        self.q_concepts = [parse_query(q["text"], graph) for q in queries]
        self._build_S()

    def _build_S(self):
        g = self.graph
        rows, cols = [], []
        for j, hits in enumerate(self.q_concepts):
            for nid, _, _ in hits:
                rows.append(nid); cols.append(j)
        Q = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                          shape=(g.n_nodes, len(self.queries)))
        self.S = {p: (g.R[p] @ Q).tocsr() for p in PATTERNS}   # [n_img × n_query]

    def path_feats(self, img_idx, q_idx):
        """img_idx [B,K] hoặc [B]; q_idx [B] → [.., n_patterns]"""
        img_idx = np.asarray(img_idx); q_idx = np.asarray(q_idx)
        if img_idx.ndim == 1:
            img_idx = img_idx[:, None]
        B, K = img_idx.shape
        rr = img_idx.ravel(); cc = np.repeat(q_idx, K)
        out = np.zeros((B * K, len(PATTERNS)), np.float32)
        for p, name in enumerate(PATTERNS):
            out[:, p] = np.asarray(self.S[name][rr, cc]).ravel()
        return np.log1p(out).reshape(B, K, len(PATTERNS))   # log1p SAU khi cộng

    def clip_candidates(self, q_idx, gallery_idx, k):
        qf = self.txt_feat[q_idx]                       # [B, D]
        gf = self.img_feat[gallery_idx]                 # [G, D]
        sim = qf @ gf.T
        k = min(k, len(gallery_idx))
        top = sim.topk(k, dim=1)
        return gallery_idx[top.indices.numpy()], top.values.numpy()


def train_graph_scorer(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    """InfoNCE với mẫu âm trong batch. CLIP đóng băng; chỉ học GNN + trọng số đường đi."""
    set_seed(seed)
    g = engine.graph
    if CFG.gnn_type == "scenir":                 # mục 7.2: GAE tự giám sát trên ảnh train, rồi đóng băng
        model = build_scenir_scorer(engine, train_img_idx, seed=seed, verbose=verbose).to(DEVICE)
    else:
        model = GraphScorer(g.n_nodes, engine.txt_feat.shape[1],
                            engine.img_feat, edge_type=g.edge_type).to(DEVICE)
    # CFG.learn_clip_w = False ⇒ cố định hệ số z(s_clip) ở softplus(0.5413) = 1 (không vào optimizer).
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    # Cổng cạnh + trọng số mẫu đường đi chỉ là vài vô hướng: cho lr riêng (×CFG.scalar_lr_mult),
    # không weight decay — nếu không chúng gần như đứng yên ở giá trị khởi tạo.
    # Tham số đã đóng băng (GAE của SCENIR) không đưa vào optimizer.
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    scalar = [p for n, p in params
              if n.endswith("gate") or n in ("path_w", "decay", "clip_logit")]
    sid = {id(p) for p in scalar}
    opt = torch.optim.AdamW(
        [dict(params=[p for _, p in params if id(p) not in sid]),
         dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],
        lr=CFG.lr, weight_decay=CFG.weight_decay)
    # INDUCTIVE: khi huấn luyện bỏ mọi cạnh chạm node ảnh KHÔNG thuộc train (val/test).
    # Node ảnh val/test thành node cô lập ⇒ không gửi thông điệp, không nhận gradient,
    # không ảnh hưởng tới biểu diễn đối tượng/khái niệm học được. Lúc đánh giá
    # (score_queries) mới dùng đồ thị đầy đủ — GraphSAGE inductive nên hợp lệ.
    keep_img = np.zeros(g.n_img, bool)
    keep_img[np.asarray(train_img_idx)] = True
    src, dst = g.edge_index.numpy()
    is_bad = lambda v: (v < g.n_img) & ~keep_img[np.minimum(v, g.n_img - 1)]
    mask = ~(is_bad(src) | is_bad(dst))
    tmask = torch.from_numpy(mask)
    ei = g.edge_index[:, tmask].to(DEVICE)
    et = g.edge_type[tmask].to(DEVICE)
    if verbose:
        print(f"  cạnh dùng khi huấn luyện: {int(mask.sum())}/{len(mask)} "
              f"(đã bỏ {int((~mask).sum())} cạnh chạm ảnh val/test)")
    hist = []
    tr_q, tr_img = np.asarray(train_q_idx), np.asarray(train_img_idx)
    img_dev = engine.img_feat.to(DEVICE)

    # Mẫu âm KHÓ: CLIP top-K trong gallery TRAIN (bỏ ảnh đúng). Lúc test đồ thị chỉ xếp lại
    # top-100 của CLIP, nên phải học phân biệt ĐÚNG những ứng viên đó — với âm ngẫu nhiên
    # trong batch, đường đi gián tiếp "dễ thắng" và học trọng số không dùng được ở test.
    hard = None
    if CFG.hard_neg_k > 0:
        cand_hn, _ = engine.clip_candidates(tr_q, tr_img, CFG.hard_neg_k + 1)
        hard = np.stack([np.concatenate([row[row != gold], row[:1]])[:CFG.hard_neg_k]
                         for row, gold in zip(cand_hn, tr_img)])

    def logits_for(cand_np, qi, h, qf):
        cand = torch.tensor(cand_np, device=DEVICE)
        pm = torch.tensor(engine.path_feats(cand_np, qi)).to(DEVICE)
        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))
        if CFG.train_with_clip:      # CLIP không có gradient; chỉ trọng số clip_w học được
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", qf, img_dev[cand]))
        return z / CFG.temperature

    for ep in range(CFG.epochs):
        model.train()
        order = np.random.permutation(len(train_q_idx))
        tot, nb = 0.0, 0
        for s in range(0, len(order), CFG.train_bs):
            sel = order[s:s + CFG.train_bs]
            if len(sel) < 8:
                continue
            qi = tr_q[sel]
            gi = tr_img[sel]                              # ảnh đúng của từng truy vấn
            qf = engine.txt_feat[qi].to(DEVICE)
            h = model.encode_graph(ei, et)

            # (a) âm trong batch như cũ: [B, B], đáp án trên đường chéo
            loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf),
                                   torch.arange(len(sel), device=DEVICE))
            # (b) âm khó từ CLIP: [B, 1+K], đáp án ở cột 0
            if hard is not None:
                cand_h = np.concatenate([gi[:, None], hard[sel]], 1)
                loss = loss + F.cross_entropy(logits_for(cand_h, qi, h, qf),
                                              torch.zeros(len(sel), dtype=torch.long, device=DEVICE))

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 3 == 0 or ep == CFG.epochs - 1):
            w = model.pattern_weights().detach().cpu().numpy()
            gates = model.edge_gates()
            # Cổng của LỚP 0: ở lớp cuối chỉ node ảnh được dùng, mà ảnh chỉ nhận cạnh contains
            # ⇒ cổng scene/related/isa lớp cuối không có gradient (luôn đứng ở giá trị khởi tạo).
            gs = (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4])) if gates else ""
            print(f"  epoch {ep:2d} | loss {hist[-1]:.4f} | w_path " +
                  " ".join(f"{x:.3f}" for x in w) + gs +
                  f" | clip_w {F.softplus(model.clip_logit).item():.2f}")
    return model, hist


@torch.no_grad()
def score_queries(model, engine, q_idx, gallery_idx, k=None):
    """Trả về (cand_idx, s_clip, s_gnn, s_path) — mọi điểm đã chuẩn hoá z theo hàng. Dùng đồ thị ĐẦY ĐỦ."""
    k = k or CFG.rerank_k
    model.eval()
    ei = engine.graph.edge_index.to(DEVICE)
    h = model.encode_graph(ei)
    cand, s_clip = engine.clip_candidates(q_idx, gallery_idx, k)
    pm = torch.tensor(engine.path_feats(cand, q_idx)).to(DEVICE)
    qf = engine.txt_feat[q_idx].to(DEVICE)
    s_gnn = model.gnn_score(h, qf, torch.tensor(cand, device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    return cand, znorm_per_row(s_clip), znorm_per_row(s_gnn), znorm_per_row(s_path)


def combine(s_clip, s_gnn, s_path, alpha, lam):
    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)

## 1.9 Chỉ số đánh giá

In [ ]:
def retrieval_metrics(cand, scores, gold_pos, ks=(1, 5, 10)):
    """gold_pos: chỉ số ảnh đúng (trong không gian toàn cục) cho từng truy vấn."""
    order = np.argsort(-scores, axis=1)
    ranked = np.take_along_axis(cand, order, axis=1)
    hit = ranked == np.asarray(gold_pos)[:, None]
    has = hit.any(1)
    rank = np.where(has, hit.argmax(1) + 1, 10**9)
    out = {f"R@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR"] = float(np.where(has, 1.0 / rank, 0.0).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    return out


def fmt_metrics(runs):
    """runs: list[dict] từ nhiều seed → chuỗi 'mean ± std'."""
    keys = runs[0].keys()
    return {k: f"{np.mean([r[k] for r in runs]):.4f} ± {np.std([r[k] for r in runs]):.4f}"
            for k in keys}

## 1.10 Bộ truy vấn đánh giá: trực tiếp vs gián tiếp

Đây là bằng chứng chính cho câu hỏi "tri thức quan hệ giúp ở đâu".

* **Trực tiếp** — truy vấn nhắc tới lớp đối tượng có sẵn trong annotation của ảnh đúng
  (tồn tại đường đi P0). CLIP thường đã làm tốt nhóm này.
* **Gián tiếp** — truy vấn nhắc tới khái niệm **không** nằm trong danh sách đối tượng của ảnh,
  nhưng nối được qua quan hệ (P1–P5). Đây là nhóm đồ thị phải chứng minh giá trị.

Cách phân nhóm này tự động và kiểm tra được, tránh việc gán nhãn cảm tính.

In [ ]:
def build_query_set(records, graph, img_ids, seed=0, verbose=True):
    pos = graph.img_pos
    queries = []
    for iid in img_ids:
        i = pos[iid]
        r = records[i]
        hits = parse_query(r["caption"], graph)
        # nhóm direct/indirect tính trên đối tượng/khái niệm như bản trước (không tính node bộ ba)
        hits = [h for h in hits if h[0] < getattr(graph, "tri_base", graph.n_nodes)]
        own = set(graph.img_objects[i])
        hit_ids = {n for n, _, _ in hits}
        direct = bool(hit_ids & own)
        indirect_ids = hit_ids - own
        reach = 0
        if indirect_ids:
            cols = np.array(sorted(indirect_ids))
            for p in PATTERNS[1:]:
                reach += int(graph.R[p][i][:, cols].nnz)
        group = "indirect" if (indirect_ids and reach > 0 and not direct) else \
                ("direct" if direct else "other")
        queries.append(dict(text=r["caption"], gold=iid, group=group,
                            n_concepts=len(hits), n_indirect=len(indirect_ids)))
    if verbose:
        print(f"Nhóm truy vấn: {dict(Counter(q['group'] for q in queries))}")
    return queries


def export_eval_queries(queries, graph, path, n=20, allowed_ids=None):
    """Xuất bộ ≥20 truy vấn đánh giá của cấp độ 2. allowed_ids = id ảnh TEST (bắt buộc)."""
    assert allowed_ids is not None, "Truyền allowed_ids=set(split['test']) — chỉ lấy truy vấn của tập test"
    allowed_ids = set(allowed_ids)
    pool = [q for q in queries if q["gold"] in allowed_ids]
    ind = [q for q in pool if q["group"] == "indirect"]
    dir_ = [q for q in pool if q["group"] == "direct"]
    rng = random.Random(0)
    sel = rng.sample(ind, min(n // 2, len(ind))) + rng.sample(dir_, min(n // 2, len(dir_)))
    payload = dict(
        n=len(sel),
        construction=("Truy vấn lấy từ caption của ảnh trong tập test. Nhóm 'indirect' là "
                      "truy vấn có khái niệm KHÔNG nằm trong annotation đối tượng của ảnh "
                      "đúng nhưng nối được qua đường đi P1-P5; nhóm 'direct' có ít nhất một "
                      "khái niệm trùng annotation (đường đi P0). Phân nhóm tự động, kiểm tra "
                      "lại được bằng hàm build_query_set."),
        usage="CHỈ dùng để đánh giá. Không tinh chỉnh mô hình trên bộ này.",
        queries=[{k: q[k] for k in ("text", "gold", "group", "n_concepts", "n_indirect")}
                 for q in sel])
    with open(path, "w") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Đã ghi {len(sel)} truy vấn đánh giá → {path}")
    return sel

## 1.11 Chọn siêu tham số trên VALIDATION

Quét lưới `(α, λ)` trên tập val. Test chưa hề được chạm vào ở bước này.

In [ ]:
def tune_on_val(model, engine, val_q, val_gallery, verbose=True):
    cand, sc, sg, sp_ = score_queries(model, engine, val_q["q_idx"], val_gallery)
    best = (-1, None)
    grid = []
    for a in CFG.alpha_grid:
        for l in CFG.lambda_grid:
            m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), val_q["gold_pos"])
            grid.append(dict(alpha=a, lam=l, **m))
            if m["MRR"] > best[0]:
                best = (m["MRR"], (a, l))
    if verbose:
        print(f"  Tốt nhất trên val: α={best[1][0]}, λ={best[1][1]} (MRR={best[0]:.4f})")
    return best[1], grid

## 1.12 Giải thích và đo fidelity (Nhiệm vụ 3)

Giải thích = danh sách **đường đi cụ thể** đóng góp nhiều nhất vào `s_path` của ứng viên đó.
Phần đóng góp không phải suy đoán: nó được đo bằng cách **bỏ đúng đường đi đó rồi tính lại điểm**.

| Chỉ số | Định nghĩa | Kỳ vọng |
|---|---|---|
| fidelity− (cần thiết) | bỏ đường đi giải thích → Δ điểm, Δ thứ hạng | điểm giảm rõ, hạng tụt |
| fidelity+ (đủ) | **chỉ giữ** đường đi giải thích → Δ điểm | điểm gần như không đổi |
| sparsity | 1 − |đường đi giải thích| / |toàn bộ đường đi| | càng cao càng gọn |
| validity | đường đi có thật và nối đúng khái niệm truy vấn ↔ ảnh | 100% |
| **đối chứng** | bỏ **ngẫu nhiên** cùng số đường đi | phải ÍT ảnh hưởng hơn |

Dòng đối chứng là dòng quan trọng nhất: nếu bỏ đường đi giải thích không tệ hơn bỏ ngẫu nhiên
thì giải thích vô nghĩa. Đây gần như chắc chắn là câu hỏi vấn đáp.

In [ ]:
def enumerate_paths(graph, img_pos_idx, concept_ids, cap=100000):
    """Liệt kê đường đi CỤ THỂ từ node ảnh tới các khái niệm truy vấn.
    Mỗi đường đi mang trọng số w = idf(node đích) × Π trọng số cạnh (P(đích|nguồn) nếu
    CFG.edge_weighting) — đúng trọng số trong graph.R, nên tổng theo mẫu khớp tuyệt đối
    với điểm tính bằng ma trận."""
    target = set(concept_ids)
    objs = graph.img_objects[img_pos_idx]
    own = set(objs)
    A_s, A_r, A_i = graph.A_scene, graph.A_related, graph.A_isa
    idf = graph.idf
    out = []

    def nbrs(M, n):
        sl = slice(M.indptr[n], M.indptr[n + 1])
        return zip(M.indices[sl], M.data[sl])

    def add(pattern, nodes, rel=None, ew=1.0):
        c = nodes[-1]
        if pattern != "P0_contains" and CFG.drop_redundant_indirect and c in own:
            return                                  # cùng quy tắc với ma trận R
        w = float(idf[c]) * float(ew)
        if w > 0:
            out.append(dict(pattern=pattern, nodes=nodes, rel=rel, w=w))

    for o in objs:
        if o in target:
            add("P0_contains", [o])
        for o2, w1 in nbrs(A_s, o):
            rel = graph.scene_pred.get((o, o2), "quan hệ")
            if o2 in target:
                add("P1_scene", [o, o2], rel, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P4_scene_isa", [o, o2, c], rel, w1 * w2)
        for c, w1 in nbrs(A_i, o):
            if c in target:
                add("P2_isa", [o, c], None, w1)
        for o2, w1 in nbrs(A_r, o):
            if o2 in target:
                add("P3_related", [o, o2], None, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P5_related_isa", [o, o2, c], None, w1 * w2)
        if len(out) > cap:
            break
    # v5: ảnh –thể_hiện→ bộ ba (trọng số = xác suất probe × idf, đúng như graph.R['P6_triple'])
    A_t = getattr(graph, "A_triple", None)
    if A_t is not None and A_t.nnz:
        sl = slice(A_t.indptr[img_pos_idx], A_t.indptr[img_pos_idx + 1])
        for t, v in zip(A_t.indices[sl], A_t.data[sl]):
            if int(t) in target:
                add("P6_triple", [int(t)], None, v)
    return out[:cap]


def paths_to_counts(paths):
    """Đặc trưng mẫu = log1p(Σ trọng số đường đi của mẫu) — giống hệt RetrievalEngine.path_feats."""
    v = np.zeros(len(PATTERNS), np.float64)
    for p in paths:
        v[PIDX[p["pattern"]]] += p["w"]
    return np.log1p(v).astype(np.float32)


def path_text(graph, p):
    n = [graph.id2name.get(x, f"#{x}") for x in p["nodes"]]
    t = PATTERN_DESC[p["pattern"]]
    d = dict(c=n[-1], o=n[0], rel=p.get("rel") or "quan_hệ")
    if len(n) >= 3:
        d["o2"] = n[1]
    return t.format(**d)


@torch.no_grad()
def explain_and_score(model, graph, img_idx, concept_ids, top_m=CFG.top_m_paths):
    """Trả về (giải thích, điểm đầy đủ, chi tiết fidelity) cho MỘT cặp (truy vấn, ảnh)."""
    paths = enumerate_paths(graph, img_idx, concept_ids)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    score = lambda ps: float(gain * (paths_to_counts(ps) * w).sum())

    full = score(paths)
    # đóng góp biên của từng đường đi: bỏ đúng nó ra rồi tính lại
    contrib = []
    for i, p in enumerate(paths):
        rest = paths[:i] + paths[i + 1:]
        contrib.append(full - score(rest))
    order = np.argsort(-np.array(contrib)) if contrib else np.array([], int)
    # v5: giải thích phải NHỎ HƠN toàn bộ đường đi, nếu không đối chứng ngẫu nhiên trùng hệt giải thích
    # (bản trước in 'fidelity− = đối chứng, sparsity 0.00' cho mọi ca có ≤ top_m đường đi).
    m = min(top_m, len(paths) - 1) if len(paths) >= 2 else len(paths)
    expl_idx = list(order[:m])
    expl = [dict(text=path_text(graph, paths[i]), pattern=paths[i]["pattern"],
                 contribution=float(contrib[i])) for i in expl_idx]

    keep = [paths[i] for i in expl_idx]
    drop = [p for i, p in enumerate(paths) if i not in set(expl_idx)]
    # Đối chứng: bỏ NGẪU NHIÊN cùng số đường đi, lấy trung bình 10 lần bốc.
    # Bốc một lần thôi thì với truy vấn ít đường đi nó hay trùng đúng tập giải thích
    # và đối chứng mất ý nghĩa.
    rng = np.random.RandomState(0)
    rnd_scores = []
    for _ in range(10):
        ridx = set(rng.choice(len(paths), size=min(len(expl_idx), len(paths)),
                              replace=False).tolist()) if paths else set()
        rnd_scores.append(score([p for i, p in enumerate(paths) if i not in ridx]))
    rnd_score = float(np.mean(rnd_scores)) if (rnd_scores and len(paths) >= 2) else float("nan")

    detail = dict(
        n_paths=len(paths),
        score_full=full,
        score_without_explanation=score(drop),      # fidelity−
        score_only_explanation=score(keep),         # fidelity+
        score_without_random=rnd_score,             # ĐỐI CHỨNG (trung bình 10 lần)
        sparsity=1 - len(expl_idx) / max(len(paths), 1),
        validity=float(all(p["nodes"][-1] in set(concept_ids) for p in keep)) if keep else 0.0,
    )
    return expl, full, detail

## 1.13 Chạy thí nghiệm chính

Một hàm `run_experiment` cho cả 3 cấu hình ablation, mỗi cấu hình 3 seed.

In [ ]:
_GROUP_CACHE = {}
PER_QUERY = {}          # (ablation, seed) → MRR từng truy vấn test; khoá "<ab>+adaptive" cho bản thích nghi
ART_BY_ABL = {}         # ablation → mô hình + đồ thị của seed đầu
FUSION_LOG = []         # nhật ký kết hợp thích nghi theo (ablation, seed)


def reciprocal_ranks(cand, scores, gold_pos):
    order = np.argsort(-scores, axis=1)
    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold_pos)[:, None]
    return np.where(hit.any(1), 1.0 / (hit.argmax(1) + 1), 0.0)


def prepare(records, split, img_feat, txt_feat, ablation="none", seed=0):
    graph = MultimodalGraph(records, split["train"], ablation=ablation, seed=seed)
    # Nhóm trực tiếp/gián tiếp chốt MỘT LẦN trên đồ thị đầy đủ rồi dùng lại cho mọi ablation.
    if "groups" not in _GROUP_CACHE:
        base = graph if ablation == "none" else MultimodalGraph(records, split["train"])
        _GROUP_CACHE["groups"] = {q["gold"]: q["group"] for q in
                                  build_query_set(records, base, [r["image_id"] for r in records])}
    queries = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    for q in queries:
        q["group"] = _GROUP_CACHE["groups"].get(q["gold"], q["group"])
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, queries)
    pos = graph.img_pos
    idx = lambda ids: np.array([pos[i] for i in ids])
    parts = {}
    for name in ("train", "val", "test"):
        ids = split[name]
        parts[name] = dict(q_idx=idx(ids), gold_pos=idx(ids), img_idx=idx(ids),
                           groups=[queries[pos[i]]["group"] for i in ids])
    return graph, eng, parts


def evaluate_split(model, engine, part, alpha, lam, by_group=True):
    gallery = part["img_idx"]
    cand, sc, sg, sp_ = score_queries(model, engine, part["q_idx"], gallery)
    res = {}
    full = combine(sc, sg, sp_, alpha, lam)
    res["CLIP"] = retrieval_metrics(cand, sc, part["gold_pos"])
    res["CLIP+Graph"] = retrieval_metrics(cand, full, part["gold_pos"])
    if by_group:
        g = np.array(part["groups"])
        for grp in ("direct", "indirect"):
            m = g == grp
            if m.sum() >= 5:
                res[f"CLIP [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], sc[m], np.array(part["gold_pos"])[m])
                res[f"CLIP+Graph [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], full[m], np.array(part["gold_pos"])[m])
    return res, (cand, sc, sg, sp_)


# ============================ KẾT HỢP ĐIỂM THÍCH NGHI ============================
FUSION_FEATS = ["has_concept", "log_n_concepts", "path_coverage", "clip_margin"]


def fusion_features(engine, q_idx, cand, sc):
    """Đặc trưng của TRUY VẤN, không dùng nhãn đúng: có khái niệm không, bao nhiêu khái niệm,
    tỉ lệ ứng viên top-K có ≥1 đường đi, độ chênh top1–top2 của CLIP (CLIP tự tin thì ít cần đồ thị)."""
    ncon = np.array([len(engine.q_concepts[q]) for q in q_idx], np.float32)
    cov = (engine.path_feats(cand, q_idx).sum(-1) > 0).mean(1)
    srt = -np.sort(-sc, axis=1)
    return np.stack([(ncon > 0).astype(np.float32), np.log1p(ncon), cov,
                     srt[:, 0] - srt[:, 1]], 1).astype(np.float32)


def _fuse(a, l, sc, sg, sp):
    return a[:, None] * sc + (1 - a[:, None]) * (l[:, None] * sg + (1 - l[:, None]) * sp)


class AdaptiveFusion(nn.Module):
    def __init__(self, d, a0, l0, mu, sd):
        super().__init__()
        self.wa, self.wl = nn.Linear(d, 1), nn.Linear(d, 1)
        for lin, v in ((self.wa, a0), (self.wl, l0)):
            v = min(max(float(v), 0.02), 0.98)
            nn.init.zeros_(lin.weight); nn.init.constant_(lin.bias, math.log(v / (1 - v)))
        self.register_buffer("mu", torch.tensor(mu, dtype=torch.float32))
        self.register_buffer("sd", torch.tensor(sd, dtype=torch.float32))

    def gates(self, Xn):
        return torch.sigmoid(self.wa(Xn)).squeeze(-1), torch.sigmoid(self.wl(Xn)).squeeze(-1)

    @torch.no_grad()
    def score(self, X, sc, sg, sp):
        a, l = self.gates((torch.tensor(X) - self.mu) / self.sd)
        a, l = a.numpy(), l.numpy()
        return _fuse(a, l, sc, sg, sp), a, l


def fit_fusion(X, sc, sg, sp, cand, gold, a0, l0):
    """Listwise CE trên top-K ứng viên của VAL. Khởi tạo w=0 ⇒ xuất phát đúng bằng (α, λ) toàn cục."""
    gold = np.asarray(gold); hit = cand == gold[:, None]; keep = hit.any(1)
    mu, sd = X.mean(0), X.std(0) + 1e-6
    m = AdaptiveFusion(X.shape[1], a0, l0, mu, sd)
    m.wa.bias.requires_grad_(False); m.wl.bias.requires_grad_(False)   # α(q) chỉ lệch khỏi α toàn cục theo đặc trưng
    if keep.sum() < 10:
        return m
    t = lambda a: torch.tensor(a[keep], dtype=torch.float32)
    Xn, SC, SG, SP = t((X - mu) / sd), t(sc), t(sg), t(sp)
    y = torch.tensor(hit[keep].argmax(1))
    opt = torch.optim.Adam([m.wa.weight, m.wl.weight], lr=0.05)
    for _ in range(CFG.fusion_steps):
        a, l = m.gates(Xn)
        loss = F.cross_entropy(_fuse(a, l, SC, SG, SP) / CFG.fusion_tau, y) \
            + CFG.fusion_wd * (m.wa.weight.pow(2).sum() + m.wl.weight.pow(2).sum())
        opt.zero_grad(); loss.backward(); opt.step()
    return m


def cv_fusion(X, sc, sg, sp, cand, gold, a0, l0, k=5, seed=0):
    """MRR ngoài-fold trên VAL: thích nghi vs (α, λ) toàn cục. Chống overfit khi chỉ có ~100–1000 truy vấn val."""
    gold = np.asarray(gold); n = len(X)
    idx = np.random.RandomState(seed).permutation(n)
    rr_a, rr_g = np.zeros(n), np.zeros(n)
    for f in np.array_split(idx, k):
        tr = np.setdiff1d(idx, f)
        m = fit_fusion(X[tr], sc[tr], sg[tr], sp[tr], cand[tr], gold[tr], a0, l0)
        rr_a[f] = reciprocal_ranks(cand[f], m.score(X[f], sc[f], sg[f], sp[f])[0], gold[f])
        rr_g[f] = reciprocal_ranks(cand[f], combine(sc[f], sg[f], sp[f], a0, l0), gold[f])
    return float(rr_a.mean()), float(rr_g.mean())


def adaptive_rows(fus, engine, part, raw):
    cand, sc, sg, sp_ = raw
    X = fusion_features(engine, part["q_idx"], cand, sc)
    s, a, l = fus.score(X, sc, sg, sp_)
    gold = np.asarray(part["gold_pos"])
    res = {"CLIP+Graph (adaptive)": retrieval_metrics(cand, s, gold)}
    g = np.array(part["groups"])
    for grp in ("direct", "indirect"):
        m = g == grp
        if m.sum() >= 5:
            res[f"CLIP+Graph (adaptive) [{grp}] (n={int(m.sum())})"] = retrieval_metrics(cand[m], s[m], gold[m])
    return res, s, a, l, X


def _adaptive_step(model, eng, parts, alpha, lam, test_raw):
    pv = parts["val"]
    vc, vsc, vsg, vsp = score_queries(model, eng, pv["q_idx"], pv["img_idx"])
    Xv = fusion_features(eng, pv["q_idx"], vc, vsc)
    cv_a, cv_g = cv_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    fus = fit_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    ad_res, s_ad, a_q, l_q, Xt = adaptive_rows(fus, eng, parts["test"], test_raw)
    has = Xt[:, 0] > 0
    flog = dict(alpha=alpha, lam=lam, cv_global=cv_g, cv_adaptive=cv_a,
                alpha_q_with_concept=float(a_q[has].mean()) if has.any() else float("nan"),
                alpha_q_no_concept=float(a_q[~has].mean()) if (~has).any() else float("nan"),
                w_alpha=dict(zip(FUSION_FEATS, fus.wa.weight.detach().numpy().ravel().round(3).tolist())))
    return fus, flog, ad_res, s_ad


def run_experiment(records, split, img_feat, txt_feat, ablations=("none",), seeds=None):
    seeds = seeds or CFG.seeds
    _GROUP_CACHE.clear(); PER_QUERY.clear(); ART_BY_ABL.clear(); FUSION_LOG.clear()
    all_res = defaultdict(list)
    artefacts = {}
    for ab in ablations:
        print(f"\n{'='*66}\nABLATION = {ab}\n{'='*66}")
        for si, seed in enumerate(seeds):
            graph, eng, parts = prepare(records, split, img_feat, txt_feat, ab, seed)
            if si == 0:
                print("Đồ thị:", graph.summary())
            model, hist = train_graph_scorer(eng, parts["train"]["q_idx"],
                                             parts["train"]["img_idx"], seed=seed,
                                             verbose=(si == 0))
            (alpha, lam), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"],
                                             verbose=(si == 0))
            res, raw = evaluate_split(model, eng, parts["test"], alpha, lam)
            cand, sc, sg, sp_ = raw
            gold_t, groups_t = parts["test"]["gold_pos"], np.array(parts["test"]["groups"])
            rr_clip = reciprocal_ranks(cand, sc, gold_t)
            PER_QUERY[(ab, seed)] = dict(rr=reciprocal_ranks(cand, combine(sc, sg, sp_, alpha, lam), gold_t),
                                         rr_clip=rr_clip, groups=groups_t, lam=lam)
            if "relprobe_hook" in globals():          # v5: bộ quan hệ khó (α, λ giữ nguyên từ VAL)
                relprobe_hook(ab, seed, model, eng, parts, alpha, lam)
            if "match_hook" in globals():             # v9: nhánh khớp truy vấn ↔ node (α, λ giữ nguyên từ VAL)
                match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res)
            fus, flog = None, None
            if CFG.adaptive_fusion:
                fus, flog, ad_res, s_ad = _adaptive_step(model, eng, parts, alpha, lam, raw)
                res.update(ad_res)
                FUSION_LOG.append(dict(ablation=ab, seed=seed, **flog))
                PER_QUERY[(f"{ab}+adaptive", seed)] = dict(rr=reciprocal_ranks(cand, s_ad, gold_t),
                                                           rr_clip=rr_clip, groups=groups_t, lam=lam)
            for k, v in res.items():
                all_res[f"{ab} | {k}"].append(v)
            if si == 0:
                ART_BY_ABL[ab] = dict(graph=graph, engine=eng, model=model, parts=parts, alpha=alpha,
                                      lam=lam, grid=grid, raw=raw, hist=hist, fusion=fus)
            if ab == "none" and si == 0:
                artefacts = ART_BY_ABL[ab]
            msg = (f"  seed {seed}: R@1={res['CLIP+Graph']['R@1']:.4f} "
                   f"MRR={res['CLIP+Graph']['MRR']:.4f} (α={alpha}, λ={lam})")
            if flog:
                msg += (f" | adaptive MRR={res['CLIP+Graph (adaptive)']['MRR']:.4f} "
                        f"[CV val {flog['cv_adaptive']:.4f} vs {flog['cv_global']:.4f}; "
                        f"α(q) có/không khái niệm {flog['alpha_q_with_concept']:.2f}/{flog['alpha_q_no_concept']:.2f}]")
            print(msg)
    return all_res, artefacts


def results_table(all_res):
    import pandas as pd
    rows = []
    for k, runs in all_res.items():
        ab, name = k.split(" | ", 1)
        rows.append(dict(ablation=ab, model=name, **fmt_metrics(runs)))
    return pd.DataFrame(rows)


def final_model_table(all_res, fusion_log):
    """MÔ HÌNH CUỐI = (cấu trúc đồ thị, global|adaptive) có MRR VAL cao nhất (adaptive tính ngoài-fold).
    Test chỉ được ĐỌC cho cấu hình đã chọn. Đây là con số chính đưa vào báo cáo."""
    import pandas as pd
    fl = pd.DataFrame(fusion_log)
    fl = fl[fl.ablation == "none"]          # ablation là phân tích, không phải ứng viên mô hình cuối
    sel = pd.concat([fl.groupby("ablation").cv_global.mean().rename("val_MRR").reset_index().assign(mode="global"),
                     fl.groupby("ablation").cv_adaptive.mean().rename("val_MRR").reset_index().assign(mode="adaptive")]
                    ).sort_values("val_MRR", ascending=False).reset_index(drop=True)
    best = sel.iloc[0]
    tab = results_table(all_res)
    pat = r"^CLIP\+Graph( \[|$)" if best["mode"] == "global" else r"^CLIP\+Graph \(adaptive\)"
    test = tab[(tab.ablation == best.ablation) & (tab.model.str.match(pat) | tab.model.str.match(r"^CLIP( \[|$)"))]
    print(f"→ Mô hình cuối (chọn trên VAL): đồ thị='{best.ablation}', kết hợp={best['mode']} "
          f"(VAL MRR {best.val_MRR:.4f})")
    return sel, test

In [ ]:
# =====================================================================================
# v9/v10 — NHÁNH KHỚP TRỰC TIẾP TRUY VẤN ↔ NODE ĐỒ THỊ (learned query → scene-graph matcher)
#   Vì sao: ở v8, một ánh xạ TUYẾN TÍNH truy vấn → vector node của video (trên CPU) đã cho MRR
#   cao hơn cả hệ GNN ⇒ GNN chưa tận dụng hết thông tin trong node. Nhánh này học (chỉ trên TRAIN)
#   một ridge (dạng đối ngẫu) từ đặc trưng truy vấn [TF-IDF 1–2 gram ⊕ CLIP-text] sang vector node
#   (đối tượng / hành động / bối cảnh / bộ ba, trọng số IDF) của ảnh–video đúng. Điểm khớp
#   s_match(q, v) = ŷ_q · y_v. Dùng s_match để (1) bổ sung ứng viên vào top-K (γ) và
#   (2) cộng vào điểm cuối (β). λ_ridge, γ, β chọn trên VAL; α, λ của CLIP+Graph giữ nguyên.
#   Ghi thêm 2 mô hình:  CLIP+Graph+Match  (đầy đủ)  và  CLIP+Match  (không GNN/path — đối chứng
#   để biết GNN còn đóng góp gì khi đã có nhánh khớp). Mọi ablation/đối chứng đồ thị (xáo node,
#   bỏ hành động, bỏ quan hệ…) tự áp vào nhánh khớp vì vector node lấy từ CHÍNH đồ thị đó.
# =====================================================================================
import time, traceback
import pandas as pd
import scipy.sparse as _sps
from sklearn.feature_extraction.text import TfidfVectorizer

MATCH_ON = True
MATCH_RIDGE = (0.3, 1.0, 3.0)
MATCH_GAMMA = (0.0, 0.25, 0.5, 1.0, 2.0)
MATCH_BETA = tuple(float(b) for b in np.round(np.arange(0, 1.01, 0.1), 2))
_MATCH_CORE, MATCH_LOG = {}, []


def _l2n(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)


def _match_core(eng, train_pos):
    tf = eng.txt_feat.cpu().numpy() if isinstance(eng.txt_feat, torch.Tensor) else np.asarray(eng.txt_feat)
    texts = [q["text"] for q in eng.queries]
    tr = np.asarray(train_pos, np.int64)
    key = (len(texts), len(tr), tf.shape[1], round(float(tf[0, 0]), 6), texts[0][:80], int(tr[:50].sum()))
    if key in _MATCH_CORE:
        return _MATCH_CORE[key]
    t0 = time.time()
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    vec.fit([texts[i] for i in tr])
    Xt = vec.transform(texts).tocsr()
    Xc = _l2n(tf.astype(np.float32))
    K_all = (Xt @ Xt[tr].T).toarray().astype(np.float32) + Xc @ Xc[tr].T          # [n_img × n_tr]
    Ktr = torch.tensor(K_all[tr], dtype=torch.float64, device=DEVICE)
    ev, U = torch.linalg.eigh((Ktr + Ktr.T) / 2)
    M = {lr: ((U * (1.0 / (ev.clamp_min(0) + lr))) @ U.T).float() for lr in MATCH_RIDGE}
    del Ktr, U
    core = dict(vec=vec, tr=tr, Xt_tr=Xt[tr], Xc_tr=Xc[tr], M=M, K_all=torch.tensor(K_all, device=DEVICE))
    _MATCH_CORE.clear(); _MATCH_CORE[key] = core
    print(f"    [match] lõi ridge: {len(tr)} truy vấn train, TF-IDF {Xt.shape[1]} chiều, {time.time() - t0:.0f}s")
    return core


MATCH_TRI_W = (0.0, 0.25, 0.5, 1.0)        # v10: trọng số khối node BỘ BA trong vector node (chọn trên VAL)


def _idf_l2(Y, tr):
    df_ = np.asarray((Y[tr] > 0).sum(0)).ravel()
    Y = Y.multiply((np.log((1 + len(tr)) / (1 + df_)) + 1)[None, :]).tocsr()
    nrm = np.sqrt(np.asarray(Y.multiply(Y).sum(1)).ravel())
    return (_sps.diags(1.0 / (nrm + 1e-8)) @ Y).tocsr(), (nrm > 0).astype(np.float32)


def _match_G(graph, tr):
    """Trả về (G_obj, G_tri, có_bộ_ba[train], có_bộ_ba[tất cả]); y = [a, w·b] / ‖·‖ với a = node đối tượng/
    hành động/bối cảnh, b = node bộ ba (mỗi khối IDF + L2 riêng). v9 gộp chung ⇒ hàng nghìn node bộ ba
    thưa lấn át node đối tượng (match-only VAL trên AG: 0.32 có bộ ba vs 0.67 không bộ ba)."""
    n_img, ro, co, rt, ct = graph.n_img, [], [], [], []
    tri = getattr(graph, "tri2id", None) or {}
    n_obj_cols = max(1, graph.n_nodes - n_img)
    for i in range(n_img):
        ids = sorted(set(graph.img_objects[i])); ro += [i] * len(ids); co += [j - n_img for j in ids]
        if tri:
            t = sorted({tri[k] for k, _ in graph.records[i].get("img_triples", []) if k in tri})
            rt += [i] * len(t); ct += [j - n_img for j in t]
    A, _ = _idf_l2(_sps.csr_matrix((np.ones(len(ro), np.float32), (ro, co)), shape=(n_img, n_obj_cols)), tr)
    B, hb = _idf_l2(_sps.csr_matrix((np.ones(len(rt), np.float32), (rt, ct)), shape=(n_img, n_obj_cols)), tr)
    T = lambda X: torch.tensor((X[tr] @ X.T).toarray().astype(np.float32), device=DEVICE)
    return (T(A), T(B), torch.tensor(hb[tr], device=DEVICE), torch.tensor(hb, device=DEVICE))


def _G_w(Gs, w):
    Ga, Gb, ht, hv = Gs
    if w == 0:
        return Ga
    nt, nv = torch.sqrt(1 + w * w * ht), torch.sqrt(1 + w * w * hv)
    return (Ga + w * w * Gb) / (nt[:, None] * nv[None, :])


def _match_scores(core, G, lr, q_idx):
    qi = torch.as_tensor(np.asarray(q_idx, np.int64), device=DEVICE)
    return (core["K_all"][qi] @ core["M"][lr]) @ G                                     # [B × n_img]


def _match_text_scores(core, G, lr, text, clipfeat):
    k = (core["vec"].transform([text]) @ core["Xt_tr"].T).toarray().astype(np.float32)[0] + \
        _l2n(np.asarray(clipfeat, np.float32).reshape(1, -1))[0] @ core["Xc_tr"].T
    return ((torch.tensor(k, device=DEVICE)[None] @ core["M"][lr]) @ G)[0].cpu().numpy()


def _pick(S, cand):
    c = torch.as_tensor(np.asarray(cand, np.int64), device=DEVICE)
    return torch.gather(S, 1, c).cpu().numpy()


def _match_cands(eng, q_idx, gallery, S, gamma, k):
    gallery = np.asarray(gallery)
    sim = (eng.txt_feat[np.asarray(q_idx)] @ eng.img_feat[gallery].T).numpy()
    if gamma > 0:
        sim = znorm_per_row(sim) + gamma * znorm_per_row(S[:, torch.as_tensor(gallery, device=DEVICE)].cpu().numpy())
    k = min(k, len(gallery))
    return gallery[np.argpartition(-sim, k - 1, axis=1)[:, :k]]


def _graph_scores(model, eng, h, q_idx, cand):
    q_idx, cand = np.asarray(q_idx), np.asarray(cand)
    qf = eng.txt_feat[q_idx]
    s_clip = torch.einsum("bd,bkd->bk", qf, eng.img_feat[torch.from_numpy(cand)]).numpy()
    pm = torch.tensor(eng.path_feats(cand, q_idx)).to(DEVICE)
    s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand, device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    z = znorm_per_row
    return z(s_clip), z(s_gnn), z(s_path)


def _final(tag, b, sc, sg, spp, sm, alpha, lam):
    return (1 - b) * combine(sc, sg, spp, alpha, lam) + b * sm if tag == "gm" else (1 - b) * sc + b * sm


@torch.no_grad()
def _match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res):
    t0 = time.time()
    tr = parts["train"]["q_idx"]
    core = _match_core(eng, tr)
    Gs = _match_G(eng.graph, tr)
    pv_ = parts["val"]; gv_ = np.asarray(pv_["gold_pos"]); gal_ = np.tile(np.asarray(pv_["img_idx"]), (len(gv_), 1))
    tri_sel = {}
    for w in (MATCH_TRI_W if bool(getattr(eng.graph, "tri2id", None)) else (0.0,)):
        Gw = _G_w(Gs, w)
        for lr in MATCH_RIDGE:
            tri_sel[(w, lr)] = float(reciprocal_ranks(gal_, _pick(_match_scores(core, Gw, lr, pv_["q_idx"]), gal_), gv_).mean())
    w_tri = max(tri_sel, key=tri_sel.get)[0]          # chọn trên VAL bằng match-only (rẻ, không cần GNN)
    G = _G_w(Gs, w_tri); del Gs
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    K, z = CFG.rerank_k, znorm_per_row
    pv, pt = parts["val"], parts["test"]
    gv = np.asarray(pv["gold_pos"])
    best = {"gm": (-1.0, None), "cm": (-1.0, None)}
    val_match_only = {}
    for lr in MATCH_RIDGE:
        Sv = _match_scores(core, G, lr, pv["q_idx"])
        val_match_only[lr] = float(reciprocal_ranks(np.tile(np.asarray(pv["img_idx"]), (len(gv), 1)),
                                                    _pick(Sv, np.tile(np.asarray(pv["img_idx"]), (len(gv), 1))), gv).mean())
        for gm in MATCH_GAMMA:
            cand = _match_cands(eng, pv["q_idx"], pv["img_idx"], Sv, gm, K)
            sc, sg, spp = _graph_scores(model, eng, h, pv["q_idx"], cand)
            sm = z(_pick(Sv, cand))
            for b in MATCH_BETA:
                for tag in ("gm", "cm"):
                    m = float(reciprocal_ranks(cand, _final(tag, b, sc, sg, spp, sm, alpha, lam), gv).mean())
                    if m > best[tag][0] + 1e-9:
                        best[tag] = (m, (lr, gm, b))
    gt, groups = np.asarray(pt["gold_pos"]), np.array(pt["groups"])
    rr_clip = PER_QUERY[(ab, seed)]["rr_clip"]
    for tag, mname, suffix in (("gm", "CLIP+Graph+Match", "+match"), ("cm", "CLIP+Match", "+clipmatch")):
        lr, gm, b = best[tag][1]
        St = _match_scores(core, G, lr, pt["q_idx"])
        cand = _match_cands(eng, pt["q_idx"], pt["img_idx"], St, gm, K)
        sc, sg, spp = _graph_scores(model, eng, h, pt["q_idx"], cand)
        fin = _final(tag, b, sc, sg, spp, z(_pick(St, cand)), alpha, lam)
        res[mname] = retrieval_metrics(cand, fin, gt)
        for grp in ("direct", "indirect"):
            m = groups == grp
            if m.sum() >= 5:
                res[f"{mname} [{grp}] (n={int(m.sum())})"] = retrieval_metrics(cand[m], fin[m], gt[m])
        PER_QUERY[(ab + suffix, seed)] = dict(rr=reciprocal_ranks(cand, fin, gt), rr_clip=rr_clip, groups=groups, lam=lam)
    # ---- bộ quan hệ khó + truy vấn phản thực tế (tham số giữ nguyên từ VAL) ----
    key = globals().get("REL_GAL_KEY")
    name = globals().get("REL_RENAME", {}).get(ab, ab)
    if key is not None and key in globals().get("REL_GALLERIES", {}) and (name, seed) in REL_PQ[key]:
        for tag, suffix in (("gm", "+match"), ("cm", "+clipmatch")):
            lr, gm, b = best[tag][1]
            rr = []
            for q, cand in REL_GALLERIES[key]:
                cand = np.asarray(cand, np.int64)[None]
                sc, sg, spp = _graph_scores(model, eng, h, np.array([int(q)]), cand)
                sm = z(_pick(_match_scores(core, G, lr, [int(q)]), cand))
                fin = _final(tag, b, sc, sg, spp, sm, alpha, lam)[0]
                rr.append(1.0 / (1 + int((fin > fin[0]).sum())))
            base_ = REL_PQ[key][(name, seed)]
            REL_PQ[key][(name + suffix, seed)] = dict(rr=np.array(rr), rr_clip=base_["rr_clip"], groups=base_["groups"], lam=lam)
        if key in globals().get("REL_CF", {}):
            for tag, suffix in (("gm", "+match"), ("cm", "+clipmatch")):
                lr, gm, b = best[tag][1]
                rows = []
                for x in REL_CF[key]:
                    cand = np.asarray(x["cand"], np.int64); row = {}
                    for t in ("text", "cf"):
                        qf = torch.tensor(REL_CF_FEATS[x[t]][None])
                        cids = [n for n, _, _ in parse_query(x[t], eng.graph)]
                        s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
                        pm = torch.tensor(path_score_matrix(eng.graph, cids, cand)[None]).to(DEVICE)
                        s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
                        s_path = model.path_score(pm).cpu().numpy()
                        sm = _match_text_scores(core, G, lr, x[t], REL_CF_FEATS[x[t]])[cand][None]
                        fin = _final(tag, b, z(s_clip), z(s_gnn), z(s_path), z(sm), alpha, lam)[0]
                        row[f"rank_{t}"] = 1 + int((fin > fin[0]).sum())
                    rows.append(row)
                REL_CF_RES[key][(name + suffix, seed)] = pd.DataFrame(rows)
    MATCH_LOG.append(dict(n_img=eng.graph.n_img, ablation=name, seed=seed, w_tri=w_tri,
                          ridge_gm=best["gm"][1][0], gamma_gm=best["gm"][1][1], beta_gm=best["gm"][1][2], val_gm=best["gm"][0],
                          ridge_cm=best["cm"][1][0], gamma_cm=best["cm"][1][1], beta_cm=best["cm"][1][2], val_cm=best["cm"][0],
                          **{f"val_match_only_r{lr}": v for lr, v in val_match_only.items()}, sec=time.time() - t0))
    print(f"    [match] {name} seed {seed}: test MRR CLIP+Graph+Match {res['CLIP+Graph+Match']['MRR']:.4f} "
          f"(λr,γ,β={best['gm'][1]}) | CLIP+Match {res['CLIP+Match']['MRR']:.4f} (λr,γ,β={best['cm'][1]}) "
          f"| match-only val {max(val_match_only.values()):.4f} | w_tri={w_tri} | {time.time() - t0:.0f}s")


def match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res):
    if not globals().get("MATCH_ON", False):
        return
    try:
        _match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res)
    except Exception as ex:                       # nhánh phụ — không được làm hỏng thí nghiệm chính
        print("    [match] LỖI — bỏ qua nhánh khớp ở lần chạy này:", repr(ex)); traceback.print_exc()


## 1.14 Phân tích 5 thành công / 5 thất bại + fidelity trên ≥10 truy vấn

In [ ]:
def analyse_cases(art, records, n_success=5, n_fail=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, (cand, sc, sg, sp_) = art["parts"]["test"], art["raw"]
    full = combine(sc, sg, sp_, art["alpha"], art["lam"])
    gold = np.array(part["gold_pos"])

    def rank_of(S):
        o = np.argsort(-S, 1); r = np.take_along_axis(cand, o, 1)
        hit = r == gold[:, None]
        return np.where(hit.any(1), hit.argmax(1) + 1, 10**9)

    r_clip, r_full = rank_of(sc), rank_of(full)
    delta = r_clip - r_full                                   # >0 = đồ thị giúp
    succ = [i for i in np.argsort(-delta) if r_full[i] <= 10][:n_success]
    fail = [i for i in np.argsort(delta) if r_clip[i] < r_full[i]][:n_fail]

    cases = []
    for kind, idxs in (("THÀNH CÔNG", succ), ("THẤT BẠI", fail)):
        for i in idxs:
            qi = part["q_idx"][i]
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            expl, s, det = explain_and_score(model, graph, int(gold[i]), cids)
            cases.append(dict(kind=kind, query=eng.queries[qi]["text"][:110],
                              group=eng.queries[qi]["group"],
                              rank_clip=int(r_clip[i]), rank_graph=int(r_full[i]),
                              explanation=expl, detail=det))
    return cases


def fidelity_study(art, n_queries=CFG.n_explain_queries, top_m_paths_min=CFG.top_m_paths):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part = art["parts"]["test"]
    rows = []
    for i in range(len(part["q_idx"])):
        qi = part["q_idx"][i]
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        expl, s, det = explain_and_score(model, graph, int(part["gold_pos"][i]), cids)
        if det["n_paths"] == 0:
            continue
        if det["n_paths"] <= top_m_paths_min:   # quá ít đường đi ⇒ giải thích = toàn bộ, đối chứng vô nghĩa
            continue
        rows.append(dict(group=eng.queries[qi]["group"], **det,
                         drop_explanation=det["score_full"] - det["score_without_explanation"],
                         drop_random=det["score_full"] - det["score_without_random"]))
        if len(rows) >= n_queries:
            break
    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không có truy vấn nào có đường đi — kiểm tra lại bước dựng đồ thị.")
        return df, {}
    summary = dict(
        n_queries=len(df),
        mean_paths=float(df.n_paths.mean()),
        fidelity_minus=float(df.drop_explanation.mean()),
        fidelity_minus_random_control=float(df.drop_random.mean()),
        fidelity_plus_gap=float((df.score_full - df.score_only_explanation).mean()),
        sparsity=float(df.sparsity.mean()),
        validity=float(df.validity.mean()),
    )
    summary["necessity_ratio"] = (summary["fidelity_minus"] /
                                  max(summary["fidelity_minus_random_control"], 1e-9))
    return df, summary

In [ ]:
@torch.no_grad()
def final_rank_fidelity(art, n_queries=50, top_m=CFG.top_m_paths, n_rand=10):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, alpha, lam = art["parts"]["test"], art["alpha"], art["lam"]
    model.eval()
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    cand_all, clip_all = eng.clip_candidates(part["q_idx"], part["img_idx"], CFG.rerank_k)
    rng = np.random.RandomState(0)
    z = znorm_per_row
    sc = lambda ps: gain * float((paths_to_counts(ps) * w).sum())
    rows = []

    for i, qi in enumerate(part["q_idx"]):
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        gold = int(part["gold_pos"][i])
        cand = cand_all[i]
        hit = np.where(cand == gold)[0]
        if not cids or len(hit) == 0:
            continue
        paths = enumerate_paths(graph, gold, cids)
        if len(paths) <= top_m:          # ít đường đi quá thì đối chứng ngẫu nhiên vô nghĩa
            continue
        j = int(hit[0])
        pm = eng.path_feats(cand[None], np.array([qi]))[0]
        sp_raw = gain * (pm * w).sum(-1)
        s_gnn = model.gnn_score(h, eng.txt_feat[[qi]].to(DEVICE),
                                torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_clip = clip_all[i][None]

        def rank_with(kept):
            sp = sp_raw.copy()
            sp[j] = sc(kept)
            fin = combine(z(s_clip), z(s_gnn), z(sp[None]), alpha, lam)[0]
            return int((fin > fin[j]).sum()) + 1

        full = sc(paths)
        contrib = np.array([full - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])
        top = set(np.argsort(-contrib)[:top_m].tolist())

        r_rand = np.mean([
            rank_with([p for k, p in enumerate(paths)
                       if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])
            for _ in range(n_rand)])
        rows.append(dict(group=eng.queries[qi]["group"], n_paths=len(paths),
                         rank_full=rank_with(paths),
                         rank_drop_expl=rank_with([p for k, p in enumerate(paths) if k not in top]),
                         rank_drop_rand=float(r_rand),
                         rank_only_expl=rank_with([paths[k] for k in top])))
        if len(rows) >= n_queries:
            break

    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không truy vấn nào có > top_m đường đi.")
        return df, {}
    summ = dict(n_queries=len(df),
                rank_full=df.rank_full.mean(),
                rank_after_drop_explanation=df.rank_drop_expl.mean(),
                rank_after_drop_random=df.rank_drop_rand.mean(),
                rank_with_only_explanation=df.rank_only_expl.mean(),
                frac_rank_worse_when_drop_expl=float((df.rank_drop_expl > df.rank_full).mean()),
                frac_rank_worse_when_drop_rand=float((df.rank_drop_rand > df.rank_full).mean()))
    return df, summ

## 1.15 Demo: nhập truy vấn → top-k + giải thích

In [ ]:
@torch.no_grad()
def demo_query(text, art, k=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    gallery = art["parts"]["test"]["img_idx"]
    hits = parse_query(text, graph)
    print(f"\nTruy vấn: {text!r}")
    print(f"Khái niệm nhận ra: {[h[1] for h in hits] or '(không có)'}")
    cids = [n for n, _, _ in hits]

    # mã hoá truy vấn tự do: dùng CLIP nếu có, nếu không lấy caption gần nhất làm proxy
    q_emb = encode_free_text(text, eng)
    sim = (q_emb @ eng.img_feat[gallery].T).numpy()[0]
    top = np.argsort(-sim)[:CFG.rerank_k]
    cand = gallery[top]
    pm = torch.tensor(np.stack([_path_feats_free(graph, cand, cids)])).to(DEVICE)
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    s_gnn = model.gnn_score(h, q_emb.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    s = combine(znorm_per_row(sim[top][None]), znorm_per_row(s_gnn),
                znorm_per_row(s_path), art["alpha"], art["lam"])[0]
    order = np.argsort(-s)[:k]
    for rank, j in enumerate(order, 1):
        img_i = int(cand[j])
        rec = eng.records[img_i]
        expl, _, det = explain_and_score(model, graph, img_i, cids)
        print(f"\n  #{rank}  image_id={rec['image_id']}  score={s[j]:.4f}")
        print(f"      caption: {rec['caption'][:90]}")
        print(f"      đối tượng: {rec['objects'][:8]}")
        if expl:
            print(f"      GIẢI THÍCH ({det['n_paths']} đường đi, hiển thị {len(expl)}):")
            for e in expl:
                print(f"        • {e['text']}   [đóng góp {e['contribution']:+.4f}]")
        else:
            print("      (không có đường đi đồ thị — xếp hạng chỉ dựa trên CLIP)")


def _proxy_text(text, eng):
    toks = set(_norm(text).split())
    best, bi = -1, 0
    for i, q in enumerate(eng.queries):
        ov = len(toks & set(_norm(q["text"]).split()))
        if ov > best:
            best, bi = ov, i
    return eng.txt_feat[bi:bi + 1]


def encode_free_text(text, eng):
    """CLIP thật chỉ dùng khi đặc trưng ảnh cũng là CLIP thật; synthetic thì dùng proxy."""
    if dataset_name == "synthetic":
        return _proxy_text(text, eng)
    try:
        import open_clip
        if not hasattr(encode_free_text, "_m"):
            m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model,
                                                            pretrained=CFG.clip_weights)
            encode_free_text._m = m.to(DEVICE).eval()
            encode_free_text._t = open_clip.get_tokenizer(CFG.clip_model)
        with torch.no_grad():
            f = encode_free_text._m.encode_text(encode_free_text._t([text]).to(DEVICE))
        return F.normalize(f, dim=-1).float().cpu()
    except Exception:
        return _proxy_text(text, eng)

def _path_feats_free(graph, cand, cids):
    out = np.zeros((len(cand), len(PATTERNS)), np.float32)
    if not cids:
        return out
    cols = np.array(sorted(set(cids)))
    for p, name in enumerate(PATTERNS):
        out[:, p] = np.asarray(graph.R[name][cand][:, cols].sum(1)).ravel()
    return np.log1p(out)

## 1.16 Self-test — chạy trước khi tin bất kỳ con số nào

Ba kiểm tra này chính là ba câu hỏi vấn đáp dễ bị hỏi nhất. Chạy được và in ✓ thì
nhóm có bằng chứng ngay tại chỗ.

In [ ]:
def self_test(records, split, img_feat, txt_feat):
    import copy
    ok = []

    # (1) RÒ RỈ — cấy một quan hệ chỉ tồn tại ở ảnh TEST, đồ thị train không được có
    rec2 = copy.deepcopy(records)
    pos = {r["image_id"]: i for i, r in enumerate(rec2)}
    A, B = "zzsecretobja", "zzsecretobjb"
    for iid in split["test"][:40]:
        r = rec2[pos[iid]]
        r["objects"] = sorted(set(r["objects"]) | {A, B})
        r["relations"] = list(r["relations"]) + [(A, "juggles", B)]
    gt = MultimodalGraph(rec2, split["train"])
    ga = MultimodalGraph(rec2, split["train"] + split["test"])
    e = lambda G, M: (M[G.obj2id[A], G.obj2id[B]] != 0) if A in G.obj2id else False
    c1 = (not e(gt, gt.A_scene)) and e(ga, ga.A_scene)
    c2 = (not e(gt, gt.A_related)) and e(ga, ga.A_related)
    ok.append(("Cạnh scene + đồng xuất hiện chỉ xây từ train", c1 and c2))

    # (2) TRUNG THỰC — điểm đường đi tính bằng ma trận phải khớp liệt kê từng đường đi
    graph = MultimodalGraph(records, split["train"])
    qs = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, qs)
    bad = n = 0
    for qi in range(min(300, len(qs))):
        cids = [x for x, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        n += 1
        m = eng.path_feats(np.array([[qi]]), np.array([qi]))[0, 0]
        v = paths_to_counts(enumerate_paths(graph, qi, cids))
        bad += int(not np.allclose(m, v, atol=1e-5))
    ok.append((f"Điểm đường đi khớp giải thích liệt kê ({n - bad}/{n} cặp)", bad == 0))

    # (3) SPLIT — không chồng lấn và tái lập được
    s2 = make_split(records, seed=CFG.split_seed)
    ok.append(("Split tái lập được từ seed", s2["sha1"] == split["sha1"]))

    # (4) v5 — bộ ba cấp ảnh: nhãn chỉ từ caption TRAIN, ảnh train nhận dự đoán ngoài-fold
    if globals().get("TRIPLE_INFO") and any(r.get("img_triples") for r in records):
        ok.append(("Bộ ba cấp ảnh: nhãn huấn luyện chỉ từ ảnh TRAIN, ảnh train dùng dự đoán ngoài-fold",
                   bool(TRIPLE_INFO.get("train_only")) and bool(TRIPLE_INFO.get("oof"))))
        g3 = MultimodalGraph(records, split["train"])
        ok.append((f"Node bộ ba có trong đồ thị ({g3.n_tri} node, {g3.A_triple.nnz} cạnh ảnh–bộ ba)",
                   g3.n_tri > 0 and g3.A_triple.nnz > 0))

    print()
    for name, passed in ok:
        print(f"  {'✓' if passed else '✗ THẤT BẠI'}  {name}")
    assert all(p for _, p in ok), "Self-test không đạt — dừng lại và sửa trước khi chạy tiếp."
    return True

---
## 1.17 Thực thi (Flickr8k)

Từ đây trở xuống là phần chạy thật. Muốn chạy nhanh để thử, giảm `CFG.n_images`.

In [ ]:
# ============ FLICKR8K: nạp dữ liệu, rút đối tượng/quan hệ, CLIP, split ============
import nltk
for _pkg in ("wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    try: nltk.download(_pkg, quiet=True)
    except Exception: pass
from nltk.stem import WordNetLemmatizer

_LEM = WordNetLemmatizer(); _LEM_CACHE = {}
try:
    from nltk.tag import PerceptronTagger
    _TAGGER = PerceptronTagger()          # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)
except Exception as e:
    print("⚠ Không tải được POS tagger → dùng cách rút danh từ thô:", type(e).__name__)
    _TAGGER = None

def _lemma(w):
    r = _LEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "n")
        except Exception: r = w[:-1] if (w.endswith("s") and len(w) > 3) else w
        _LEM_CACHE[w] = r
    return r

NOT_OBJECTS = set("front background foreground top bottom side end corner air group area region place space "
                  "picture photo image one other something someone anything thing stuff view middle edge "
                  "part way lot pair couple bunch number while each both several many few kind type".split())
_SKIP_P = set("is are was were be been being has have had while as that who which and or".split())


def extract_objects_relations(captions):
    """Danh từ NHÌN THẤY ĐƯỢC (lọc WordNet) → lớp đối tượng. Quan hệ = (danh từ, động từ/giới từ,
    danh từ kế tiếp). Heuristic — nên đọc thử ~30 mẫu để báo cáo độ chính xác."""
    objs, rels = set(), set()
    for cap in captions:
        toks = re.findall(r"[A-Za-z]+", cap)
        if not toks:
            continue
        if _TAGGER is not None:
            tags = _TAGGER.tag(toks)
        else:
            tags = [(t, "NN" if len(t) > 2 and t.lower() not in STOP else "DT") for t in toks]
        last_n, preds = None, []
        for w, t in tags:
            wl = w.lower()
            if t in ("NN", "NNS") and len(wl) > 2 and wl not in NOT_OBJECTS and wl not in STOP:
                n = _lemma(wl)
                if n in NOT_OBJECTS or not is_visual_noun(n):
                    continue                     # tính từ/động từ bị tag nhầm NN: pink, walking, stand...
                objs.add(n)
                if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                    rels.add((last_n, " ".join(preds), n))
                last_n, preds = n, []
            elif (t.startswith("VB") or t in ("IN", "TO")) and wl not in _SKIP_P and last_n is not None:
                preds.append(wl)
    return objs, rels


def load_flickr8k():
    """Chỉ rút nhãn-từ-caption (cap_objects/cap_relations). objects/relations THẬT dùng cho
    đồ thị được gán sau bởi apply_flickr_annotation (tagger hoặc oracle)."""
    from datasets import load_dataset as hf_load
    dsd = hf_load("jxie/flickr8k")
    records, parts, iid = [], [], 0
    for hf_name, tag in (("train", "train"), ("validation", "val"), ("test", "test")):
        d = dsd[hf_name]; parts.append(d)
        for row in d.remove_columns("image"):
            allc = [str(row[f"caption_{k}"]).strip() for k in range(5)]
            objs, rels = extract_objects_relations(allc[1:])   # caption 1..4; caption_0 là truy vấn
            records.append(dict(image_id=iid, caption=allc[0], captions=allc,
                                cap_objects=sorted(objs), cap_relations=sorted(rels),
                                objects=[], relations=[], url="", split=tag))
            iid += 1
    print(f"Flickr8k: {len(records)} ảnh | {np.mean([len(r['cap_objects']) for r in records]):.1f} danh từ/ảnh | "
          f"{np.mean([len(r['cap_relations']) for r in records]):.1f} quan hệ/ảnh (rút từ caption)")
    for r in records[:3]:
        print("  •", r["caption"], "→", r["cap_objects"][:8], r["cap_relations"][:3])
    return records, parts


def encode_flickr8k(records, parts):
    cache = CFG.out_dir / "cache" / f"clip_flickr8k_{CFG.clip_model}_{CFG.clip_weights}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    img, txt = [], []
    with torch.no_grad():
        for d in parts:                                  # đúng thứ tự train → val → test như records
            for s in range(0, len(d), CFG.clip_bs):
                ims = d[s:s + CFG.clip_bs]["image"]
                x = torch.stack([preprocess(im.convert("RGB")) for im in ims]).to(DEVICE)
                img.append(F.normalize(model.encode_image(x), dim=-1).float().cpu().numpy())
        caps = [r["caption"] for r in records]
        for s in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[s:s + CFG.clip_bs]).to(DEVICE)
            txt.append(F.normalize(model.encode_text(tok), dim=-1).float().cpu().numpy())
    del model; torch.cuda.empty_cache()
    img, txt = np.concatenate(img), np.concatenate(txt)
    assert len(img) == len(records) == len(txt)
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


# =====================================================================================
# NHÃN ẢNH BẰNG CLIP ZERO-SHOT (thay cho nhãn rút từ caption của chính ảnh)
#   - Từ vựng: danh từ nhìn thấy được trong caption TRAIN, tần suất ≥ min_obj_freq.
#   - Điểm: cos(ảnh, prompt-ensemble); chuẩn hoá theo từng từ bằng mean/std trên ảnh TRAIN.
#   - (cách chuẩn hoá, k) chọn trên VAL theo F1 so với danh từ trong caption val.
#   - Caption của ảnh val/test KHÔNG bao giờ vào đồ thị; caption test chỉ dùng để BÁO CÁO P/R/F1.
# =====================================================================================
@torch.no_grad()
def clip_text_features(texts, bs=512):
    import open_clip
    if not hasattr(clip_text_features, "_m"):
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        clip_text_features._m = m.to(DEVICE).eval()
        clip_text_features._t = open_clip.get_tokenizer(CFG.clip_model)
    m, tok = clip_text_features._m, clip_text_features._t
    out = [F.normalize(m.encode_text(tok(texts[s:s + bs]).to(DEVICE)), dim=-1).float().cpu().numpy()
           for s in range(0, len(texts), bs)]
    return np.concatenate(out)


def _prf(pred, truth):
    tp = sum(len(p & t) for p, t in zip(pred, truth))
    P = tp / max(sum(map(len, pred)), 1); R = tp / max(sum(map(len, truth)), 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def clip_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vset = set(vocab)

    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    S = img_feat @ T.T                                          # [N, V]
    mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên train
    scores = {"raw": S, "z": (S - mu) / sd, "raw+z": S / S[tr].std() + (S - mu) / sd}

    def tags(Sc, idx, k):
        top = np.argpartition(-Sc[idx], k, axis=1)[:, :k]
        return [{vocab[j] for j in row} for row in top]
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]

    grid, best = [], None
    for mode, Sc in scores.items():
        for k in CFG.tag_k_grid:
            P, R, F1 = _prf(tags(Sc, va, k), truth(va))
            grid.append(dict(norm=mode, k=k, val_P=P, val_R=R, val_F1=F1))
            if best is None or F1 > best[0]:
                best = (F1, mode, k)
    _, mode, k = best
    all_tags = tags(scores[mode], np.arange(len(records)), k)
    P, R, F1 = _prf([all_tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm=mode, k=k, test_P=P, test_R=R, test_F1=F1, grid=grid)
    return all_tags, info


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    """Gán records[i]['objects'] / ['relations'] dùng cho đồ thị.
    relations chỉ giữ cho ảnh TRAIN (dùng để xây cạnh scene thống kê)."""
    mode = mode or CFG.flickr_annot
    tr = set(split["train"])
    info = dict(annotation=mode)
    if mode == "tagger":
        global TAG_INFO
        if not all("tags" in r for r in records) or "TAG_INFO" not in globals():
            tags, TAG_INFO = clip_tagger(records, split, img_feat, text_encoder)
            for r, t in zip(records, tags):
                r["tags"] = sorted(t)
        info.update(TAG_INFO)
    for r in records:
        r["objects"] = list(r["tags"]) if mode == "tagger" else list(r["cap_objects"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    msg = f"Nhãn ảnh = {mode}"
    if mode == "tagger":
        msg += (f" | vocab={info['vocab_size']} | chuẩn hoá={info['norm']}, k={info['k']} (chọn trên val)"
                f" | TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    print(msg)
    return info


# Split chính thức của HF; bản ngẫu nhiên cũ giữ lại cho trường hợp synthetic.
if "_make_split_random" not in globals():
    _make_split_random = make_split

def make_split(records, seed=CFG.split_seed):
    if not all("split" in r for r in records):
        return _make_split_random(records, seed)
    split = {k: sorted(r["image_id"] for r in records if r["split"] == k) for k in ("train", "val", "test")}
    split["seed"] = "official-hf-split"
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split chính thức HF (sha1={split['sha1']}): train={len(split['train'])} "
          f"val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])) and not (set(split["val"]) & set(split["test"]))
    return split


# Truy vấn Flickr có số nhiều ("dogs") → phải lemmatize như lúc rút đối tượng.
def parse_query(text, graph):
    toks = [_lemma(t) for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    # v5: bộ ba (s, p, o) tách từ truy vấn → node bộ ba cấp ảnh (mẫu P6)
    if getattr(graph, "n_tri", 0) and "query_triples" in globals():
        for key in query_triples(text):
            nid = graph.tri2id.get(key)
            if nid is not None and nid not in seen:
                seen.add(nid); hits.append((nid, " ".join(key), " ".join(key)))
    return hits

## 1.2b Quan hệ cấp ảnh (v5)

Lần chạy trước cho thấy `none ≈ no_relations ≈ random_edges` trên cả ba bộ dữ liệu. Lý do: mọi cạnh quan hệ (scene / related / isa) là **thống kê cấp lớp** gộp từ train, nên hai ảnh cùng chứa `dog` nhận đúng cùng một tập đường đi và quan hệ không phân biệt được ứng viên.

Bản này thêm **node bộ ba cấp ảnh** ⟨chủ thể, vị từ, đối tượng⟩: mỗi ảnh có tập bộ ba riêng, do một linear probe (khởi tạo từ văn bản CLIP, học trên caption **train**, ảnh train nhận dự đoán ngoài-fold) dự đoán từ đặc trưng CLIP của chính ảnh. Mẫu đường đi mới `P6: ảnh –thể_hiện→ ⟨s p o⟩` khớp với bộ ba tách từ truy vấn và vẫn cộng tính, nên giải thích và fidelity giữ nguyên cơ chế. Nhãn đối tượng cũng chuyển từ CLIP zero-shot sang probe cùng kiểu.

Đối chứng mới: `no_triples` (= `none` của bản trước), `shuffled_triples` (ảnh nhận bộ ba của ảnh khác). Đánh giá mới: **bộ quan hệ khó**, gallery gồm ảnh đúng và các ảnh test có cùng đối tượng nhưng không có quan hệ của truy vấn (theo caption test, chỉ dùng làm nhãn chấm).


In [ ]:
# =====================================================================================
# 1.2b  QUAN HỆ CẤP ẢNH (bản v5) — vì sao cần
#   Lần chạy trước: 'none' ≈ 'no_relations' ≈ 'random_edges' trên cả Flickr8k, AG, MSR-VTT.
#   Nguyên nhân: mọi cạnh quan hệ (scene / related / isa) là thống kê CẤP LỚP gộp từ train —
#   hai ảnh cùng có 'dog' nhận ĐÚNG cùng tập đường đi 'dog → …', nên quan hệ không phân biệt
#   được ứng viên; nó chỉ là một kiểu mở rộng truy vấn mà CLIP đã làm tốt hơn.
#   Bản này thêm:
#     (1) Nhãn đối tượng bằng LINEAR PROBE khởi tạo từ văn bản CLIP (học trên caption TRAIN,
#         ảnh train dùng dự đoán NGOÀI-FOLD) thay cho zero-shot (F1 test cũ 0.25).
#     (2) Node BỘ BA (chủ thể, vị từ, đối tượng) CẤP ẢNH: mỗi ảnh có tập bộ ba riêng do probe
#         dự đoán từ đặc trưng CLIP của CHÍNH ẢNH (không dùng caption val/test).
#         Mẫu đường đi mới P6: ảnh –thể_hiện→ ⟨s p o⟩, khớp với bộ ba tách từ truy vấn.
#     (3) Đối chứng mới: no_triples (= 'none' của bản trước), shuffled_triples (bộ ba của
#         ảnh KHÁC ⇒ kiểm tra tín hiệu có thật sự gắn với ảnh không).
#     (4) Bộ đánh giá QUAN HỆ KHÓ: gallery = ảnh đúng + ảnh test có CÙNG đối tượng nhưng
#         KHÔNG có quan hệ của truy vấn (theo caption test — chỉ dùng làm NHÃN CHẤM).
# =====================================================================================
import shutil

# --- dùng lại cache CLIP của lần chạy trước nếu output cũ được Add Input ---
for _f in glob.glob("/kaggle/input/**/cache/clip_flickr8k_*.npz", recursive=True):
    _dst = CFG.out_dir / "cache" / Path(_f).name
    if not _dst.exists():
        shutil.copy2(_f, _dst)
        print("Dùng lại cache CLIP từ input:", _dst.name)

PREPS = set("in on at with by for from into onto over under near behind beside across through along "
            "around against toward towards up down out off past above below between inside outside "
            "beneath underneath atop upon".split())
TRIVIAL_PREDS = {"of", "for", "by"}          # 'body of water', 'pair of shoe' không phải quan hệ thị giác
_VLEM_CACHE = {}


def _verb_lemma(w):
    r = _VLEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "v")
        except Exception: r = w
        _VLEM_CACHE[w] = r
    return r


def canon_triple(s, p, o):
    """('dog', 'running after', 'ball') → ('dog', 'run', 'ball'); ('dog', 'in', 'snow') giữ giới từ."""
    toks = p.split()
    if not toks or s == o:
        return None
    t0 = toks[0]
    cp = t0 if t0 in PREPS else _verb_lemma(t0)
    if cp in TRIVIAL_PREDS:
        return None
    return (s, cp, o)


def caption_triples(caps):
    _, rels = extract_objects_relations(caps)
    return {t for t in (canon_triple(*r) for r in rels) if t is not None}


_QT_CACHE = {}


def query_triples(text):
    """Bộ ba chuẩn hoá tách từ VĂN BẢN TRUY VẤN — cùng bộ rút quan hệ dùng cho caption train."""
    if text not in _QT_CACHE:
        _QT_CACHE[text] = sorted(caption_triples([text]))
    return _QT_CACHE[text]


# ---------------------------------------------------------------------------------
# Linear probe khởi tạo từ văn bản CLIP: logit = e^s·cos(ảnh, w_v) + b_v, w_v khởi tạo = text_v,
# phạt ‖W − T‖² để từ hiếm vẫn bám prior văn bản. Ảnh TRAIN: dự đoán NGOÀI-FOLD (k-fold).
# ---------------------------------------------------------------------------------
def _fit_probe(X, Y, T, steps=None, lr=None, wd_init=None, seed=0):
    steps = steps or CFG.probe_steps; lr = lr or CFG.probe_lr
    wd_init = CFG.probe_wd_init if wd_init is None else wd_init
    torch.manual_seed(seed)
    X, Y, T = (torch.as_tensor(a, dtype=torch.float32, device=DEVICE) for a in (X, Y, T))
    W = nn.Parameter(T.clone())
    s = nn.Parameter(torch.tensor(math.log(30.0), device=DEVICE))
    prior = Y.mean(0).clamp(1e-4, 1 - 1e-4)
    with torch.no_grad():
        b0 = torch.log(prior / (1 - prior)) - (s.exp() * (X @ T.T)).mean(0)
    b = nn.Parameter(b0)
    opt = torch.optim.Adam([dict(params=[W], lr=lr), dict(params=[b, s], lr=lr * 10)])
    for _ in range(steps):
        logit = s.exp() * (X @ W.T) + b
        loss = F.binary_cross_entropy_with_logits(logit, Y) + wd_init * (W - T).pow(2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    W, b, s = W.detach(), b.detach(), s.detach()
    return lambda Xq: torch.sigmoid(s.exp() * (torch.as_tensor(Xq, dtype=torch.float32, device=DEVICE) @ W.T)
                                    + b).cpu().numpy()


def cross_fit_probe(X, Y_tr, T, tr, folds=None, seed=0):
    """X [N, D] mọi ảnh; Y_tr [len(tr), V] nhãn CHỈ của ảnh train. Trả P [N, V]:
    ảnh train ← mô hình KHÔNG thấy ảnh đó (ngoài-fold); ảnh val/test ← mô hình học trên toàn bộ train."""
    folds = folds or CFG.probe_folds
    tr = np.asarray(tr)
    P = np.zeros((len(X), T.shape[0]), np.float32)
    order = np.random.RandomState(seed).permutation(len(tr))
    for f in np.array_split(order, folds):
        keep = np.setdiff1d(order, f)
        pred = _fit_probe(X[tr[keep]], Y_tr[keep], T, seed=seed)
        P[tr[f]] = pred(X[tr[f]])
    rest = np.setdiff1d(np.arange(len(X)), tr)
    if len(rest):
        P[rest] = _fit_probe(X[tr], Y_tr, T, seed=seed)(X[rest])
    return P


def _multi_hot(sets, vocab_index, n_rows):
    Y = np.zeros((n_rows, len(vocab_index)), np.float32)
    for i, s in enumerate(sets):
        for t in s:
            j = vocab_index.get(t)
            if j is not None:
                Y[i, j] = 1.0
    return Y


def _topk_sets(P, idx, vocab, k, tau=0.0):
    out = []
    for i in idx:
        row = P[i]
        top = np.argpartition(-row, min(k, len(row) - 1))[:k]
        out.append({vocab[j] for j in top if row[j] >= tau})
    return out


def probe_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vix = {w: j for j, w in enumerate(vocab)}
    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Y_tr = _multi_hot([records[i]["cap_objects"] for i in tr], vix, len(tr))
    P = cross_fit_probe(img_feat, Y_tr, T, tr)
    vset = set(vocab)
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]
    grid, best = [], None
    for k in CFG.tag_k_grid:
        Pv, Rv, Fv = _prf(_topk_sets(P, va, vocab, k), truth(va))
        grid.append(dict(norm="probe", k=k, val_P=Pv, val_R=Rv, val_F1=Fv))
        if best is None or Fv > best[0]:
            best = (Fv, k)
    k = best[1]
    tags = _topk_sets(P, np.arange(len(records)), vocab, k)
    Pt, Rt, Ft = _prf([tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm="probe(ngoài-fold)", k=k, test_P=Pt, test_R=Rt, test_F1=Ft,
                grid=grid, vocab=vocab)
    return tags, info


if "_apply_flickr_annotation_base" not in globals():
    _apply_flickr_annotation_base = apply_flickr_annotation


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    mode = mode or CFG.flickr_annot
    if mode != "probe":
        return _apply_flickr_annotation_base(records, split, img_feat, mode, text_encoder)
    global PROBE_INFO
    if not all("probe_tags" in r for r in records) or "PROBE_INFO" not in globals():
        tags, PROBE_INFO = probe_tagger(records, split, img_feat, text_encoder)
        for r, t in zip(records, tags):
            r["probe_tags"] = sorted(t)
    tr = set(split["train"])
    for r in records:
        r["objects"] = list(r["probe_tags"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    info = dict(annotation="probe", **{k: v for k, v in PROBE_INFO.items() if k != "vocab"})
    print(f"Nhãn ảnh = probe | vocab={info['vocab_size']} | k={info['k']} (chọn trên val) | "
          f"TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    return info


def fit_image_triples(records, split, img_feat, text_encoder=None):
    """Gán records[i]['img_triples'] = [((s, p, o), xác_suất)] cho MỌI ảnh.
    Nhãn huấn luyện = bộ ba chuẩn hoá trong caption 1..4 của ảnh TRAIN; ảnh train nhận dự đoán
    ngoài-fold; (k, τ) chọn trên VAL theo F1; caption test chỉ để BÁO CÁO P/R/F1."""
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    labels = {}
    cnt, surface = Counter(), defaultdict(Counter)
    for i in range(len(records)):
        ts = set()
        for s_, p_, o_ in records[i]["cap_relations"]:
            t = canon_triple(s_, p_, o_)
            if t is None:
                continue
            ts.add(t)
        labels[i] = ts
    tr_set = set(tr.tolist())
    for i in tr:
        for s_, p_, o_ in records[i]["cap_relations"]:
            t = canon_triple(s_, p_, o_)
            if t is not None:
                cnt[t] += 1; surface[t][p_] += 1
    obj_vocab = set(PROBE_INFO["vocab"]) if "PROBE_INFO" in globals() else \
        {o for i in tr for o in records[i]["cap_objects"]}
    vocab = sorted(t for t, c in cnt.items()
                   if c >= CFG.triple_min_count and t[0] in obj_vocab and t[2] in obj_vocab)
    vix = {t: j for j, t in enumerate(vocab)}
    prompts = [f"a photo of a {s} {surface[(s, p, o)].most_common(1)[0][0]} the {o}." for s, p, o in vocab]
    T = text_encoder(prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Y_tr = _multi_hot([labels[i] for i in tr], vix, len(tr))
    P = cross_fit_probe(img_feat, Y_tr, T, tr)
    vset = set(vocab)
    truth = lambda idx: [labels[i] & vset for i in idx]
    grid, best = [], None
    for k in CFG.triple_k_grid:
        for tau in CFG.triple_tau_grid:
            Pv, Rv, Fv = _prf(_topk_sets(P, va, vocab, k, tau), truth(va))
            grid.append(dict(k=k, tau=tau, val_P=Pv, val_R=Rv, val_F1=Fv))
            if best is None or Fv > best[0]:
                best = (Fv, k, tau)
    _, k, tau = best
    # zero-shot (chỉ văn bản, chưa học) để so sánh
    S0 = img_feat @ T.T
    zs = _topk_sets((S0 - S0[tr].mean(0)) / (S0[tr].std(0) + 1e-6), te, vocab, k)
    for i in range(len(records)):
        row = P[i]
        top = np.argpartition(-row, min(k, len(row) - 1))[:k]
        records[i]["img_triples"] = sorted(((vocab[j], float(row[j])) for j in top if row[j] >= tau),
                                           key=lambda x: -x[1])
    pred_te = [{t for t, _ in records[i]["img_triples"]} for i in te]
    Pt, Rt, Ft = _prf(pred_te, truth(te))
    _, _, Fz = _prf(zs, truth(te))
    qcov = np.mean([bool(set(query_triples(records[i]["caption"])) & vset) for i in te])
    info = dict(vocab_size=len(vocab), k=k, tau=tau, test_P=Pt, test_R=Rt, test_F1=Ft, zero_shot_test_F1=Fz,
                mean_triples_per_image=float(np.mean([len(r["img_triples"]) for r in records])),
                test_queries_with_vocab_triple=float(qcov), grid=grid,
                train_only=bool(Y_tr.shape[0] == len(tr_set)), oof=True,
                examples=[dict(caption=records[i]["caption"], triples=[" ".join(t) for t, _ in records[i]["img_triples"][:5]])
                          for i in te[:5]])
    print(f"Bộ ba cấp ảnh: vocab={len(vocab)} | k={k}, τ={tau} (chọn trên val) | "
          f"TEST P={Pt:.3f} R={Rt:.3f} F1={Ft:.3f} (zero-shot F1={Fz:.3f}) | "
          f"{info['mean_triples_per_image']:.2f} bộ ba/ảnh | truy vấn test có bộ ba trong vocab: {qcov:.1%}")
    return info


# ---------------------------------------------------------------------------------
# Bộ đánh giá QUAN HỆ KHÓ: cùng đối tượng, khác quan hệ.
# ---------------------------------------------------------------------------------
REL_GALLERIES, REL_PQ = {}, defaultdict(dict)
REL_GAL_KEY, REL_RENAME = None, {}


def _sample_negs(negs, rng):
    if len(negs) > CFG.rel_neg_max:
        negs = list(rng.choice(negs, CFG.rel_neg_max, replace=False))
    return negs


def build_rel_galleries_flickr(records, split, seed=0):
    """Truy vấn test có ≥1 bộ ba (s, p, o). Âm = ảnh test mà 5 caption nhắc ĐỦ mọi s, o của truy vấn
    nhưng KHÔNG có bộ ba nào của truy vấn. Caption test chỉ dùng làm NHÃN CHẤM."""
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    te = [pos[i] for i in split["test"]]
    nouns, tris = {}, {}
    for p in te:
        o, rels = extract_objects_relations(records[p]["captions"])
        nouns[p] = o
        tris[p] = {t for t in (canon_triple(*r) for r in rels) if t is not None}
    rng = np.random.RandomState(seed)
    gal, n_q = [], 0
    for p in te:
        qt = set(query_triples(records[p]["caption"]))
        if not qt:
            continue
        n_q += 1
        need = {t[0] for t in qt} | {t[2] for t in qt}
        negs = [p2 for p2 in te if p2 != p and need <= nouns[p2] and not (qt & tris[p2])]
        if len(negs) < CFG.rel_neg_min:
            continue
        gal.append((p, np.array([p] + _sample_negs(negs, rng), np.int64)))
    print(f"Bộ quan hệ khó (Flickr8k): {len(gal)}/{n_q} truy vấn test có bộ ba đủ ≥{CFG.rel_neg_min} âm "
          f"| trung bình {np.mean([len(g) - 1 for _, g in gal]):.1f} âm/truy vấn")
    return gal


def build_rel_galleries_ag(recs, split, seed=0):
    """Truy vấn test nhắc tới một quan hệ người–vật 'v o' có trong annotation của video đúng.
    Âm = video test có vật o nhưng KHÔNG có quan hệ 'v o' (theo annotation)."""
    pos = {r["image_id"]: i for i, r in enumerate(recs)}
    te = [pos[i] for i in split["test"]]
    objs = {p: set(recs[p]["objects"]) for p in te}
    rng = np.random.RandomState(seed)
    gal, n_q = [], 0
    for p in te:
        toks = [t for t in _norm(recs[p]["caption"]).split() if t not in STOP and len(t) > 1]
        verbs = set()
        for t in toks:
            v = _ag_lemma(t, "v"); verbs.add(AG_VERB_ALIAS.get(v, v))
        nouns = {AG_NOUN_ALIAS.get(_ag_lemma(t), _ag_lemma(t)) for t in toks}
        comps = {c for c in objs[p] if " " in c and c.split(" ", 1)[0] in verbs and c.split(" ", 1)[1] in nouns}
        if not comps:
            continue
        n_q += 1
        need = {c.split(" ", 1)[1] for c in comps}
        negs = [p2 for p2 in te if p2 != p and need <= objs[p2] and not (comps & objs[p2])]
        if len(negs) < CFG.rel_neg_min:
            continue
        gal.append((p, np.array([p] + _sample_negs(negs, rng), np.int64)))
    print(f"Bộ quan hệ khó (AG): {len(gal)}/{n_q} truy vấn test có quan hệ người–vật đủ ≥{CFG.rel_neg_min} âm "
          f"| trung bình {np.mean([len(g) - 1 for _, g in gal]):.1f} âm/truy vấn")
    return gal


@torch.no_grad()
def rel_probe_rr(model, eng, gal, alpha, lam):
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    z = znorm_per_row
    rr, rr_clip, groups = [], [], []
    for q, cand in gal:
        cand = np.asarray(cand, np.int64)
        qf = eng.txt_feat[[int(q)]]
        s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
        pm = torch.tensor(eng.path_feats(cand[None], np.array([q]))).to(DEVICE)
        s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_path = model.path_score(pm).cpu().numpy()
        fin = combine(z(s_clip), z(s_gnn), z(s_path), alpha, lam)[0]
        rr.append(1.0 / (1 + int((fin > fin[0]).sum())))
        rr_clip.append(1.0 / (1 + int((s_clip[0] > s_clip[0, 0]).sum())))
        groups.append(eng.queries[int(q)]["group"])
    return np.array(rr), np.array(rr_clip), np.array(groups)


def relprobe_hook(ab, seed, model, eng, parts, alpha, lam):
    """Gọi trong run_experiment sau khi đánh giá test (α, λ đã chọn trên VAL — không chọn lại)."""
    if REL_GAL_KEY is None or REL_GAL_KEY not in REL_GALLERIES:
        return
    rr, rr_clip, groups = rel_probe_rr(model, eng, REL_GALLERIES[REL_GAL_KEY], alpha, lam)
    name = REL_RENAME.get(ab, ab)
    store = REL_PQ[REL_GAL_KEY]
    store[(name, seed)] = dict(rr=rr, rr_clip=rr_clip, groups=groups, lam=lam)
    store.setdefault(("CLIP", seed), dict(rr=rr_clip, rr_clip=rr_clip, groups=groups, lam=None))
    print(f"    [quan hệ khó] {name} seed {seed}: MRR {rr.mean():.4f} (CLIP {rr_clip.mean():.4f}) "
          f"| R@1 {(rr == 1).mean():.4f} | n={len(rr)}")


def rel_probe_report(key, base="none", others=None, out_csv=None):
    pq = REL_PQ[key]
    names = list(dict.fromkeys(a for a, _ in pq))
    rows = []
    for a in names:
        runs = [v["rr"] for (b, s), v in pq.items() if b == a]
        rows.append(dict(model=a, n_queries=len(runs[0]), seeds=len(runs),
                         MRR=f"{np.mean([r.mean() for r in runs]):.4f} ± {np.std([r.mean() for r in runs]):.4f}",
                         R1=f"{np.mean([(r == 1).mean() for r in runs]):.4f}"))
    tab = pd.DataFrame(rows)
    others = others or [a for a in names if a != base]
    sig_ = pd.concat([paired_test(pq, base, b) for b in others if any(k[0] == b for k in pq)])
    sig_ = sig_[sig_.group == "all"] if len(sig_) else sig_
    if out_csv is not None:
        tab.to_csv(out_csv, index=False)
        sig_.to_csv(str(out_csv).replace(".csv", "_paired.csv"), index=False)
    return tab, sig_


In [ ]:
# =====================================================================================
# 1.2c  ĐỒ THỊ CẢNH TỪ MÔ HÌNH MÔ TẢ ẢNH ĐỘC LẬP VỚI CLIP (bản v7)
#   Chẩn đoán (CPU, trên đặc trưng đã cache): mọi node của bản v5/v6 được dự đoán từ chính vector CLIP
#   của ảnh ⇒ đồ thị không mang thông tin mới (+0,003 MRR). Cận trên khi mỗi ảnh có mô tả văn bản độc lập
#   (caption người viết, khớp cả câu): +0,03…+0,05 (ViT-L/14), +0,08…+0,10 (ViT-B/32).
#   Bản này: mỗi ảnh (kể cả val/test — mô hình chỉ NHÌN ẢNH, không dùng caption của bộ dữ liệu) được một mô hình
#   mô tả ảnh (BLIP) sinh vài câu mô tả; từ đó rút node cho đồ thị:
#     • node nội dung: từ đã lemma + cụm 2 từ kề nhau (đối tượng, thuộc tính, hành động: 'dog', 'black dog', 'run')
#     • node bộ ba ⟨s p o⟩ cấp ảnh (mẫu P6) và cạnh scene (thống kê trên ảnh TRAIN) rút từ CHÍNH các mô tả đó
#   Truy vấn được tách đúng cùng cách (parse_query) ⇒ khớp node bằng chuỗi, giải thích/fidelity giữ nguyên cơ chế.
#   Caption người viết của TRAIN chỉ còn dùng làm truy vấn huấn luyện; của VAL/TEST chỉ là truy vấn + nhãn chấm.
# =====================================================================================
CFG.vlm_name     = "Salesforce/blip-image-captioning-large"
CFG.vlm_bs       = 32
CFG.vlm_n_sample = 3          # số mô tả lấy mẫu (nucleus) thêm vào 1 mô tả beam search
CFG.vlm_max_tok  = 40


def content_grams(text):
    """Cùng cách tách với parse_query của Flickr: lemma từng từ (bỏ stopword) + cụm 2 từ kề nhau."""
    toks = [_lemma(t) for t in _norm(text).split() if t not in STOP and len(t) > 1]
    return set(toks) | {" ".join(p) for p in zip(toks, toks[1:])}


def vlm_captions(parts, n):
    cache = CFG.out_dir / "cache" / f"vlm_caps_flickr8k_{CFG.vlm_name.split('/')[-1]}_{n}.json"
    for f in glob.glob(f"/kaggle/input/**/cache/{cache.name}", recursive=True):
        if not cache.exists():
            shutil.copy2(f, cache)
    if cache.exists():
        print("Dùng lại mô tả ảnh đã sinh:", cache.name)
        return json.load(open(cache))
    from transformers import BlipProcessor, BlipForConditionalGeneration
    proc = BlipProcessor.from_pretrained(CFG.vlm_name)
    model = BlipForConditionalGeneration.from_pretrained(CFG.vlm_name).to(DEVICE).eval()
    if DEVICE == "cuda":
        model = model.half()
    torch.manual_seed(0)
    out, t0 = [], time.time()
    with torch.no_grad():
        for d in parts:                                  # đúng thứ tự train → val → test như records
            for s in range(0, len(d), CFG.vlm_bs):
                ims = [im.convert("RGB") for im in d[s:s + CFG.vlm_bs]["image"]]
                inp = proc(images=ims, return_tensors="pt").to(DEVICE)
                if DEVICE == "cuda":
                    inp["pixel_values"] = inp["pixel_values"].half()
                caps = [[] for _ in ims]
                gens = [model.generate(**inp, num_beams=3, max_new_tokens=CFG.vlm_max_tok)]
                gens += [model.generate(**inp, do_sample=True, top_p=0.9, max_new_tokens=CFG.vlm_max_tok)
                         for _ in range(CFG.vlm_n_sample)]
                for g in gens:
                    for i, t in enumerate(proc.batch_decode(g, skip_special_tokens=True)):
                        caps[i].append(t.strip())
                out += caps
                if (len(out) // CFG.vlm_bs) % 25 == 0:
                    print(f"  mô tả ảnh: {len(out)}/{n} | {time.time() - t0:.0f}s", flush=True)
    del model; torch.cuda.empty_cache()
    assert len(out) == n, (len(out), n)
    json.dump(out, open(cache, "w"))
    return out


if "_apply_flickr_annotation_v5" not in globals():
    _apply_flickr_annotation_v5 = apply_flickr_annotation


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    mode = mode or CFG.flickr_annot
    if mode not in ("vlm", "oracle_grams"):
        return _apply_flickr_annotation_v5(records, split, img_feat, mode, text_encoder)
    if not all(r.get("_node_src") == mode for r in records):
        # oracle_grams: gallery ĐÃ ĐƯỢC CHÚ THÍCH — caption 1..4 của ảnh (caption 0 là truy vấn, không bao giờ dùng làm node)
        caps = vlm_captions(hf_parts, len(records)) if mode == "vlm" else [r["captions"][1:] for r in records]
        for r, c in zip(records, caps):
            r["vlm_caps"] = c; r["_node_src"] = mode
            grams = set()
            for t in c:
                grams |= content_grams(t)
            _, rels = extract_objects_relations(c)
            r["vlm_nodes"], r["vlm_rels"] = sorted(grams), sorted(rels)
            r["vlm_triples"] = sorted({t for t in (canon_triple(*x) for x in rels) if t is not None})
    tr = set(split["train"])
    for r in records:
        r["objects"] = list(r["vlm_nodes"])
        r["relations"] = list(r["vlm_rels"]) if r["image_id"] in tr else []     # cạnh scene: thống kê CHỈ trên train
        r["img_triples"] = [(t, 1.0) for t in r["vlm_triples"]]                # bộ ba của CHÍNH ảnh (mô hình nhìn ảnh)
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    te = [pos[i] for i in split["test"]]
    # Độ phủ: bao nhiêu % từ/cụm từ của TRUY VẤN test có trong node của ảnh đúng (caption test chỉ dùng để ĐO)
    cov = np.mean([len(content_grams(records[i]["caption"]) & set(records[i]["vlm_nodes"])) /
                   max(len(content_grams(records[i]["caption"])), 1) for i in te])
    qt = [set(query_triples(records[i]["caption"])) for i in te]
    tcov = np.mean([bool(q & set(records[i]["vlm_triples"])) for q, i in zip(qt, te) if q] or [0.0])
    info = dict(annotation=mode, vlm=CFG.vlm_name if mode == "vlm" else "caption người viết 1–4", captions_per_image=len(records[0]["vlm_caps"]),
                mean_nodes_per_image=float(np.mean([len(r["vlm_nodes"]) for r in records])),
                mean_triples_per_image=float(np.mean([len(r["vlm_triples"]) for r in records])),
                test_query_gram_coverage=float(cov), test_query_triple_hit=float(tcov))
    print(f"Nhãn ảnh = {mode} ({info['vlm']}) | {info['captions_per_image']} mô tả/ảnh | "
          f"{info['mean_nodes_per_image']:.1f} node/ảnh | {info['mean_triples_per_image']:.1f} bộ ba/ảnh | "
          f"độ phủ từ/cụm từ của truy vấn test ở ảnh đúng: {cov:.1%} | truy vấn test có bộ ba khớp ảnh đúng: {tcov:.1%}")
    for r in records[-3:]:
        print(f"  • truy vấn : {r['caption'][:90]}\n      mô tả VLM: {r['vlm_caps'][:2]}\n"
              f"      bộ ba    : {[' '.join(t) for t in r['vlm_triples'][:4]]}")
    return info


def paired_vs_clip(pq, ab, n_boot=10000, seed=0):
    """Kiểm định cặp CLIP+Graph vs CLIP trên CÙNG truy vấn test (RR trung bình theo seed)."""
    seeds = sorted(s for a, s in pq if a == ab)
    rr = np.mean([pq[(ab, s)]["rr"] for s in seeds], 0); rc = pq[(ab, seeds[0])]["rr_clip"]
    groups = pq[(ab, seeds[0])]["groups"]; rng = np.random.RandomState(seed); rows = []
    for grp in ("all", "direct", "indirect"):
        m = np.ones(len(rr), bool) if grp == "all" else groups == grp
        d = (rr - rc)[m]
        if len(d) < 5:
            continue
        bs = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        rows.append(dict(config=ab, group=grp, n=int(m.sum()), MRR_clip=rc[m].mean(), MRR_graph=rr[m].mean(),
                         R1_clip=(rc[m] == 1).mean(), R1_graph=np.mean([(pq[(ab, s)]["rr"][m] == 1).mean() for s in seeds]),
                         diff=d.mean(), ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5)))
    return pd.DataFrame(rows)


In [ ]:
# HF_TOKEN (Kaggle Secrets) — không bắt buộc; thiếu secret thì vẫn tải ẩn danh.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    print("Không đọc được HF_TOKEN từ Kaggle Secrets → tải ẩn danh:", type(e).__name__)


In [ ]:
# v7: backbone chính ViT-L/14 (đặc trưng lấy từ cache của version trước); ViT-B/32 chạy ở cell đối chứng cuối Phần 1.
CFG.clip_model, CFG.clip_weights, CFG.clip_bs = "ViT-L-14", "laion2b_s32b_b82k", 128   # mô hình cuối (version 9); đổi thành "ViT-H-14", "laion2b_s32b_b79k", 64 để chạy ViT-H
for _a in ("_m", "_t"):
    clip_text_features.__dict__.pop(_a, None)
import pandas as pd
records, hf_parts = load_flickr8k()
dataset_name = "flickr8k"
split = make_split(records)
img_feat, txt_feat = encode_flickr8k(records, hf_parts)
annot_info = apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)
print(f"\nDữ liệu: {dataset_name} | {len(records)} ảnh | đặc trưng CLIP {img_feat.shape}")

TRIPLE_INFO = fit_image_triples(records, split, img_feat) if (CFG.use_triples and CFG.flickr_annot not in ("vlm", "oracle_grams")) else {}
if CFG.flickr_annot in ("tagger", "probe"):
    print("\nChọn (chuẩn hoá, k) của tagger trên VALIDATION:")
    print(pd.DataFrame(annot_info["grid"]).pivot_table(index="k", columns="norm",
                                                         values="val_F1").round(3).to_string())
    for r in records[-3:]:                       # vài ảnh TEST: nhãn tagger vs danh từ trong caption
        print(f"  • {r['caption'][:70]}\n      nhãn   : {r['objects']}\n      caption: {r['cap_objects']}"
              f"\n      bộ ba  : {[' '.join(t) + f' ({p:.2f})' for t, p in r.get('img_triples', [])[:4]]}")
with open(CFG.out_dir / "results" / "annotation_info.json", "w") as f:
    json.dump({k: v for k, v in annot_info.items()}, f, indent=2)
with open(CFG.out_dir / "results" / "triple_info.json", "w") as f:
    json.dump({k: v for k, v in TRIPLE_INFO.items()}, f, indent=2, ensure_ascii=False)

In [ ]:
self_test(records, split, img_feat, txt_feat)

In [ ]:
# Chọn CFG.learn_clip_w trên VALIDATION (1 seed, cấu hình 'none'). Không nhìn test.
def pick_learn_clip_w(records, split, img_feat, txt_feat, seed=0):
    rows = []
    for flag in (False, True):
        CFG.learn_clip_w = flag
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(records, split, img_feat, txt_feat, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        rows.append(dict(learn_clip_w=flag, val_MRR=max(r["MRR"] for r in grid), alpha=a, lam=l,
                         clip_w=round(F.softplus(model.clip_logit).item(), 3),
                         path_w=round(F.softplus(model.path_logit).item(), 3)))
    df_ = pd.DataFrame(rows); display(df_)
    CFG.learn_clip_w = bool(df_.loc[df_.val_MRR.idxmax(), "learn_clip_w"])
    print("→ CFG.learn_clip_w =", CFG.learn_clip_w, "(chọn theo VAL MRR)")
    return df_

clipw_flickr = pick_learn_clip_w(records, split, img_feat, txt_feat)

In [ ]:
# Thí nghiệm chính: 8 cấu hình ablation × 3 seed (6 cũ + no_triples + shuffled_triples), nhãn ảnh = CFG.flickr_annot
REL_GALLERIES["flickr"] = build_rel_galleries_flickr(records, split)
REL_GAL_KEY, REL_RENAME = "flickr", {}
all_res, art = run_experiment(records, split, img_feat, txt_feat, ablations=FLICKR_ABLATIONS)
REL_GAL_KEY = None
df = results_table(all_res)
df.to_csv(CFG.out_dir / "results" / f"main_results_{CFG.flickr_annot}.csv", index=False)
df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
display(df)

In [ ]:
# =====================================================================================
# ABLATION THEO GIẢI THÍCH: giải thích có đúng với tri thức ĐỘC LẬP với đồ thị không?
#
# Với mỗi ablation (seed đầu), giải thích của ẢNH ĐÚNG trên mọi truy vấn test = top-m đường đi
# QUAN HỆ (P1..P5, bỏ P0) theo đóng góp biên — đúng định nghĩa explain_and_score. Mỗi CẠNH
# trong đường đi được chấm bằng nguồn KHÔNG dùng để dựng đồ thị:
#   • scene / related  : NPMI của hai đầu cạnh trên tập nhãn tham chiếu của ảnh VAL+TEST
#                        (Flickr: danh từ trong caption 1..4 — caption 0 là truy vấn nên bỏ;
#                         AG: annotation của video). Đồ thị chỉ thống kê trên TRAIN ⇒ độc lập.
#                        Cạnh "hợp lý" ⇔ NPMI > 0 (hai thứ hay đi cùng nhau hơn ngẫu nhiên).
#   • isa              : đúng ⇔ đích nằm trong bao đóng hypernym WordNet (hoặc FALLBACK_ISA).
# Chỉ số:
#   edge_ok    % cạnh hợp lý            path_ok  % đường đi có MỌI cạnh hợp lý
#   grounded   % đường đi mà mọi node ĐỐI TƯỢNG trung gian có trong nhãn tham chiếu của ảnh đó
#   rand_path_ok  path_ok khi chọn NGẪU NHIÊN cùng số đường đi quan hệ của chính ảnh đó
#                 ⇒ (path_ok − rand_path_ok) = cơ chế chọn giải thích có ưu tiên đường đi đúng không
# Không rò rỉ: nhãn tham chiếu chỉ dùng để CHẤM, không huấn luyện / chọn siêu tham số.
# Chạy NGAY sau cell thí nghiệm chính (cần ART_BY_ABL của cell 28 bản đợt 4).
# =====================================================================================
from functools import lru_cache


def reference_sets(records, ids):
    key = "cap_objects" if "cap_objects" in records[0] else "objects"
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    return key, {i: set(records[pos[i]][key]) for i in ids}


def make_npmi(docs):
    n = len(docs)
    c1 = Counter(x for d in docs for x in d)
    c2 = Counter(p for d in docs for p in itertools.combinations(sorted(d), 2))
    def npmi(a, b):
        ab = c2.get(tuple(sorted((a, b))), 0)
        if ab == 0 or c1[a] == 0 or c1[b] == 0:
            return -1.0
        p_ab = ab / n
        return math.log(p_ab / ((c1[a] / n) * (c1[b] / n))) / max(-math.log(p_ab), 1e-9)
    return npmi


@lru_cache(maxsize=None)
def _isa_ok(a, b):
    if b in FALLBACK_ISA.get(a, []):
        return True
    wn = _wordnet()
    if wn is None:
        return False
    for s in wn.synsets(a.replace(" ", "_"), pos="n")[:3]:
        for h in s.closure(lambda x: x.hypernyms()):
            if b in {l.name().replace("_", " ").lower() for l in h.lemmas()}:
                return True
    return False


_EDGE_KIND = {"P1_scene": ["scene"], "P2_isa": ["isa"], "P3_related": ["related"],
              "P4_scene_isa": ["scene", "isa"], "P5_related_isa": ["related", "isa"]}


def _edges(p):
    nodes = p["nodes"]                      # [o, c] hoặc [o, o2, c]
    return list(zip(_EDGE_KIND[p["pattern"]], nodes[:-1], nodes[1:]))


def _rel_contrib(paths, model):
    """Đóng góp biên của từng đường đi = đúng công thức explain_and_score, nhưng O(P)."""
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    pidx = np.array([PIDX[p["pattern"]] for p in paths])
    pw = np.array([p["w"] for p in paths])
    tot = np.bincount(pidx, weights=pw, minlength=len(PATTERNS))
    return gain * w[pidx] * (np.log1p(tot[pidx]) - np.log1p(np.maximum(tot[pidx] - pw, 0)))


def explanation_grounding(art_by_abl, records, split, top_m=CFG.top_m_paths, n_rand=10, seed=0):
    key, ref = reference_sets(records, split["val"] + split["test"])
    npmi = make_npmi(list(ref.values()))
    rng = np.random.RandomState(seed)
    rows = []
    for ab, art in art_by_abl.items():
        g, eng, model, part = art["graph"], art["engine"], art["model"], art["parts"]["test"]
        name = lambda n: g.id2name.get(n, "")
        is_obj = lambda n: g.n_img <= n < g.n_img + g.n_obj
        edge_cache = {}

        def edge_ok(kind, a, b):
            k = (kind, a, b)
            if k not in edge_cache:
                edge_cache[k] = (_isa_ok(name(a), name(b)) if kind == "isa"
                                 else npmi(name(a), name(b)) > 0)
            return edge_cache[k]

        def path_ok(p):
            return all(edge_ok(*e) for e in _edges(p))

        for i, qi in enumerate(part["q_idx"]):
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            if not cids:
                continue
            gold = int(part["gold_pos"][i])
            rel = [p for p in enumerate_paths(g, gold, cids) if p["pattern"] in _EDGE_KIND]   # P1..P5
            if not rel:
                continue
            top = [rel[j] for j in np.argsort(-_rel_contrib(rel, model))[:top_m]]
            truth = ref.get(records[gold]["image_id"], set())
            k = len(top)
            rand_ok = np.mean([np.mean([path_ok(rel[j]) for j in rng.choice(len(rel), k, replace=False)])
                               for _ in range(n_rand)])
            edges = [e for p in top for e in _edges(p)]
            rows.append(dict(
                ablation=ab, group=part["groups"][i], n_rel_paths=len(rel), n_expl=k,
                edge_ok=np.mean([edge_ok(*e) for e in edges]),
                path_ok=np.mean([path_ok(p) for p in top]),
                rand_path_ok=rand_ok,
                grounded=np.mean([all(name(n) in truth for n in p["nodes"][:-1] if is_obj(n))
                                  for p in top]),
                mean_npmi=np.mean([npmi(name(a), name(b)) for kd, a, b in edges if kd != "isa"] or [np.nan])))
    df = pd.DataFrame(rows)
    print(f"Nhãn tham chiếu: '{key}' của {len(ref)} ảnh VAL+TEST | top_m={top_m}")
    return df


def _boot_ci(x, n_boot=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = x[np.random.RandomState(seed).randint(0, len(x), (n_boot, len(x)))].mean(1)
    return f"{x.mean():.3f} [{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]"


def grounding_table(df):
    out = []
    for grp in ("all", "direct", "indirect"):
        for ab in df.ablation.unique():
            d = df[(df.ablation == ab) & ((df.group == grp) | (grp == "all"))]
            if len(d) < 5:
                continue
            out.append(dict(ablation=ab, group=grp, n_queries=len(d),
                            rel_paths_per_q=round(d.n_rel_paths.mean(), 1),
                            edge_ok=_boot_ci(d.edge_ok), path_ok=_boot_ci(d.path_ok),
                            rand_path_ok=_boot_ci(d.rand_path_ok),
                            selection_lift=_boot_ci(d.path_ok - d.rand_path_ok),
                            grounded=_boot_ci(d.grounded), mean_npmi=round(d.mean_npmi.mean(), 3)))
    return pd.DataFrame(out)


# ------------------------------ Flickr8k ------------------------------
ART_FLICKR = dict(ART_BY_ABL)                     # giữ lại trước khi cell oracle / AG ghi đè
try:
    gdf = explanation_grounding(ART_FLICKR, records, split)
    gtab = grounding_table(gdf)
    gdf.to_csv(CFG.out_dir / "results" / "explanation_grounding_per_query.csv", index=False)
    gtab.to_csv(CFG.out_dir / "results" / "explanation_grounding.csv", index=False)
    display(gtab)
except Exception as e:                      # phân tích phụ — không để nó chặn phần còn lại
    print("⚠ explanation_grounding bỏ qua:", type(e).__name__, e)

In [ ]:
# Kiểm định CẶP giữa các ablation trên cùng truy vấn test (trung bình 3 seed / truy vấn).
# Chênh MRR 0.000x với std 0.002-0.003 thì KHÔNG kết luận được gì — cần CI cho hiệu số.
# Chạy NGAY sau cell thí nghiệm chính (cell oracle / AG sẽ ghi đè PER_QUERY).
def paired_test(pq, a="none", b="no_relations", n_boot=10000, seed=0):
    seeds = sorted({s for ab, s in pq if ab == a} & {s for ab, s in pq if ab == b})
    ra = np.mean([pq[(a, s)]["rr"] for s in seeds], 0)
    rb = np.mean([pq[(b, s)]["rr"] for s in seeds], 0)
    groups = pq[(a, seeds[0])]["groups"]
    rng = np.random.RandomState(seed)
    rows = []
    for grp in ("all", "direct", "indirect"):
        m = np.ones(len(ra), bool) if grp == "all" else groups == grp
        d = (ra - rb)[m]
        if len(d) < 5:
            continue
        boot = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append(dict(group=grp, n=int(m.sum()), MRR_a=ra[m].mean(), MRR_b=rb[m].mean(),
                         diff=d.mean(), ci_lo=lo, ci_hi=hi,
                         p_a_not_better=float((boot <= 0).mean()),
                         a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum())))
    return pd.DataFrame(rows).assign(a=a, b=b)


PQ_MAIN = dict(PER_QUERY)            # giữ lại trước khi cell sau ghi đè
sig = pd.concat([paired_test(PQ_MAIN, "none", b)
                 for b in ("no_triples", "shuffled_triples", "no_relations", "random_edges",
                           "no_scene", "no_related", "no_concept")
                 if any(k[0] == b for k in PQ_MAIN)])
sig.to_csv(CFG.out_dir / "results" / "paired_ablation_test.csv", index=False)
print("λ chọn trên val cho 'none':", [v["lam"] for k, v in PQ_MAIN.items() if k[0] == "none"],
      "(λ=1.0 ⇒ s_path KHÔNG tham gia điểm cuối)")
display(sig.round(4))

In [ ]:
# v5 — BỘ QUAN HỆ KHÓ trên Flickr8k: gallery = ảnh đúng + ảnh test cùng đối tượng, khác quan hệ.
# Đây là chỗ quan hệ CẤP ẢNH phải tạo khác biệt: nhãn đối tượng không còn phân biệt được ứng viên.
rel_tab, rel_sig = rel_probe_report("flickr", base="none",
                                    others=["CLIP", "no_triples", "shuffled_triples", "no_relations", "random_edges"],
                                    out_csv=CFG.out_dir / "results" / "relational_hard_negatives.csv")
display(rel_tab); display(rel_sig.round(4))
print("diff > 0 và ci_lo > 0 ⇒ 'none' (có bộ ba cấp ảnh) tốt hơn cấu hình so sánh một cách có ý nghĩa.")


In [ ]:
# Mô hình cuối của Flickr8k chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_FLICKR = list(FUSION_LOG)
sel_f, final_f = final_model_table(all_res, FUSION_FLICKR)
display(sel_f.round(4)); display(final_f)
display(pd.DataFrame(FUSION_FLICKR).round(3))
sig_ad = paired_test(dict(PER_QUERY), "none+adaptive", "none")
display(sig_ad.round(4))
for name, obj in (("final_model_val_selection.csv", sel_f), ("final_model_test.csv", final_f),
                  ("fusion_log.csv", pd.DataFrame(FUSION_FLICKR)), ("paired_adaptive_vs_global.csv", sig_ad)):
    obj.to_csv(CFG.out_dir / "results" / name, index=False)

In [ ]:
# So sánh với nhãn ORACLE (rút từ caption 1..4 của chính ảnh — cách của bản trước).
# Chỉ để định lượng "rò rỉ" của cách cũ: chênh lệch oracle − tagger chính là phần điểm
# đến từ việc ứng viên mang sẵn mô tả văn bản của chính nó. KHÔNG báo cáo là kết quả chính.
RUN_ORACLE = False      # v6: đã chạy ở version 2 (ViT-L/14)
if RUN_ORACLE:
    apply_flickr_annotation(records, split, img_feat, "oracle")
    oracle_res, _ = run_experiment(records, split, img_feat, txt_feat,
                                   ablations=("none", "no_relations"))
    df_oracle = results_table(oracle_res)
    df_oracle.to_csv(CFG.out_dir / "results" / "main_results_oracle.csv", index=False)
    apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)   # KHÔI PHỤC nhãn chính
    cmp_annot = pd.concat([df[df.ablation.isin(["none", "no_relations"])].assign(annotation=CFG.flickr_annot),
                           df_oracle.assign(annotation="oracle")])
    cmp_annot = cmp_annot[cmp_annot.model.isin(["CLIP", "CLIP+Graph"])]
    cmp_annot.to_csv(CFG.out_dir / "results" / "annotation_comparison.csv", index=False)
    display(cmp_annot[["annotation", "ablation", "model", "R@1", "R@5", "MRR"]])

In [ ]:
# Fidelity đo trên THỨ HẠNG của điểm cuối (CLIP + GNN + path)
rdf, rsumm = final_rank_fidelity(art)
rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
for k, v in rsumm.items():
    print(f"  {k:36s} {v:.3f}" if isinstance(v, float) else f"  {k:36s} {v}")
with open(CFG.out_dir / "results" / "xai_final_rank_summary.json", "w") as f:
    json.dump(rsumm, f, indent=2)

**Cách đọc bảng.**

1. `none / CLIP` vs `none / CLIP+Graph` — đồ thị có giúp không (Nhiệm vụ 1).
2. `none` vs `random_edges` / `no_relations` — cải thiện có đến từ **cấu trúc quan hệ** không.
   Chỉ kết luận "quan hệ có ích" khi `none` > `random_edges` **và** `none` > `no_relations`
   vượt quá độ lệch chuẩn giữa các seed.
3. `no_scene` / `no_concept` — loại cạnh nào (scene graph rút từ caption hay IsA WordNet) đóng góp.
4. Dòng `[indirect]` — nhóm duy nhất mà đường đi P1–P5 có thể giúp; P0 không tồn tại cho nhóm này.

Nếu `no_relations` vẫn ≥ `none`, kết luận trung thực là: lợi ích đến từ nhãn đối tượng (P0)
và GNN, còn quan hệ trong đồ thị hiện tại chưa giúp — kèm phân tích lý do ở mục case study.

In [ ]:
# Độ nhạy theo α (yêu cầu ở cột Mở rộng của Nhiệm vụ 2)
import pandas as pd
gdf = pd.DataFrame(art["grid"])
piv = gdf.pivot_table(index="alpha", columns="lam", values="MRR")
print("MRR trên VALIDATION theo (α, λ) — α=1.0 là CLIP thuần:")
print(piv.round(4).to_string())
piv.to_csv(CFG.out_dir / "results" / "alpha_sensitivity.csv")

In [ ]:
# Phân tích 5 thành công + 5 thất bại
cases = analyse_cases(art, records)
for c in cases:
    print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → CLIP+Graph {c['rank_graph']}")
    print(f"  truy vấn: {c['query']}")
    for e in c["explanation"]:
        print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    d = c["detail"]
    ctrl = d['score_full'] - d['score_without_random']
    print(f"  fidelity−: {d['score_full'] - d['score_without_explanation']:+.4f} | "
          f"đối chứng ngẫu nhiên: {'n/a (1 đường đi)' if ctrl != ctrl else f'{ctrl:+.4f}'} | "
          f"sparsity {d['sparsity']:.2f} | {d['n_paths']} đường đi")
with open(CFG.out_dir / "results" / "case_analysis.json", "w") as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)

In [ ]:
# Fidelity / validity / sparsity trên ≥10 truy vấn (cấp độ 3)
fdf, summ = fidelity_study(art)
fdf.to_csv(CFG.out_dir / "results" / "fidelity_per_query.csv", index=False)
print("Tổng hợp xAI:")
for k, v in summ.items():
    print(f"  {k:34s} {v:.4f}" if isinstance(v, float) else f"  {k:34s} {v}")
print("\nnecessity_ratio > 1 nghĩa là bỏ đường đi giải thích gây hại NHIỀU HƠN bỏ ngẫu nhiên")
print("cùng số lượng ⇒ giải thích thực sự là thứ mô hình dùng, không phải trang trí.")
with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
    json.dump(summ, f, indent=2)

In [ ]:
# Bộ ≥20 truy vấn đánh giá (cấp độ 2) — CHỈ lấy truy vấn của ảnh test
eval_qs = export_eval_queries(art["engine"].queries, art["graph"],
                              CFG.out_dir / "results" / "eval_queries_20.json", n=20,
                              allowed_ids=split["test"])

In [ ]:
# Demo
demo_query("a child playing football on the grass", art, k=3)
demo_query("a person cooking food in a kitchen", art, k=3)

In [ ]:
# =====================================================================================
# v7 — (a) kiểm định cặp CLIP+Graph vs CLIP; (b) đối chứng XÁO node VLM giữa các ảnh; (c) backbone ViT-B/32
# Chạy SAU mọi phân tích của mô hình chính (run_experiment ghi đè PER_QUERY / ART_BY_ABL).
# =====================================================================================
import copy
R_F = CFG.out_dir / "results"
vs_clip = [paired_vs_clip(PQ_MAIN, ab).assign(backbone=CFG.clip_model) for ab in ("none", "no_class_edges", "no_relations", "no_triples")
           if any(k[0] == ab for k in PQ_MAIN)]

# (b) mỗi ảnh nhận node + bộ ba VLM của một ảnh KHÁC cùng split ⇒ nếu hiệu quả đến từ nội dung riêng của ảnh thì phải mất
_rng = np.random.RandomState(0)
records_sh = copy.deepcopy(records)
_pos = {r["image_id"]: i for i, r in enumerate(records)}
for _k in ("train", "val", "test"):
    _idx = np.array([_pos[i] for i in split[_k]]); _perm = _rng.permutation(_idx)
    for a, b in zip(_idx, _perm):
        for f in ("objects", "relations", "img_triples"):
            records_sh[a][f] = copy.deepcopy(records[b][f])
REL_GAL_KEY, REL_RENAME = "flickr", {"none": "shuffled_nodes"}
sh_res, _ = run_experiment(records_sh, split, img_feat, txt_feat, ablations=("none",))
REL_GAL_KEY, REL_RENAME = None, {}
PQ_SH = {("shuffled_nodes" + ab[4:], s): v for (ab, s), v in PER_QUERY.items() if ab == "none" or ab.startswith("none+")}
sh_df = results_table(sh_res).assign(ablation="shuffled_nodes")
vs_clip.append(paired_vs_clip(PQ_SH, "shuffled_nodes").assign(backbone=CFG.clip_model))
sig_sh = paired_test({**PQ_MAIN, **PQ_SH}, "none", "shuffled_nodes")
sig_sh.to_csv(R_F / "paired_none_vs_shuffled_nodes.csv", index=False)
pd.concat([df, sh_df]).to_csv(R_F / "main_results_with_controls.csv", index=False)
rel_tab, rel_sig = rel_probe_report("flickr", base="none",
                                    others=["CLIP", "shuffled_nodes", "no_class_edges", "no_relations"],
                                    out_csv=R_F / "relational_hard_negatives.csv")
print("Bộ quan hệ khó (Flickr8k), kèm đối chứng shuffled_nodes:"); display(rel_tab); display(rel_sig.round(4))

# (c) cùng đồ thị VLM, backbone CLIP yếu hơn (ViT-B/32) — dùng đặc trưng đã cache ở version trước
_b = sorted(glob.glob(f"/kaggle/input/**/cache/clip_flickr8k_ViT-B-32_*_{len(records)}.npz", recursive=True)) + \
     sorted((CFG.out_dir / "cache").glob(f"clip_flickr8k_ViT-B-32_*_{len(records)}.npz"))
if _b:
    _z = np.load(_b[0]); imgB, txtB = _z["img"], _z["txt"]
    REL_GALLERIES["flickr_b32"] = REL_GALLERIES["flickr"]
    REL_GAL_KEY, REL_RENAME = "flickr_b32", {}
    b_res, _ = run_experiment(records, split, imgB, txtB, ablations=("none", "no_class_edges"))
    REL_GAL_KEY = None
    PQ_B = dict(PER_QUERY)
    b_df = results_table(b_res); b_df.to_csv(R_F / "main_results_vitb32.csv", index=False)
    display(b_df[b_df.model.isin(["CLIP", "CLIP+Graph"])])
    vs_clip += [paired_vs_clip(PQ_B, ab).assign(backbone="ViT-B-32") for ab in ("none", "no_class_edges")]
    relb_tab, relb_sig = rel_probe_report("flickr_b32", base="none", others=["CLIP", "no_class_edges"],
                                          out_csv=R_F / "relational_hard_negatives_vitb32.csv")
    display(relb_tab); display(relb_sig.round(4))
else:
    print("Không thấy cache CLIP ViT-B/32 trong /kaggle/input → bỏ qua phần (c).")

vs_clip = pd.concat(vs_clip)
vs_clip.to_csv(R_F / "paired_graph_vs_clip.csv", index=False)
print("CLIP+Graph vs CLIP (kiểm định cặp, bootstrap theo truy vấn):")
print(vs_clip.round(4).to_string())
print("\nnone vs shuffled_nodes:"); print(sig_sh.round(4).to_string())


In [ ]:
# =====================================================================================
# v9 — BÁO CÁO NHÁNH KHỚP (Flickr8k): CLIP+Graph+Match / CLIP+Match vs CLIP, vs CLIP+Graph, vs đối chứng
# =====================================================================================
R_F = CFG.out_dir / "results"
def _vs_names(pq, names, **tags):
    out = [paired_vs_clip(pq, n).assign(**tags) for n in names if any(k[0] == n for k in pq)]
    return pd.concat(out) if out else pd.DataFrame()
_mf = [_vs_names(PQ_MAIN, ["none", "none+clipmatch", "none+match", "no_class_edges+match", "no_relations+match"], backbone=CFG.clip_model),
       _vs_names(PQ_SH, ["shuffled_nodes", "shuffled_nodes+clipmatch", "shuffled_nodes+match"], backbone=CFG.clip_model)]
if "PQ_B" in globals():
    _mf.append(_vs_names(PQ_B, ["none", "none+clipmatch", "none+match"], backbone="ViT-B-32"))
match_vs_clip_f = pd.concat(_mf); match_vs_clip_f.to_csv(R_F / "match_vs_clip.csv", index=False)
print("Flickr8k — kiểm định cặp vs CLIP:"); print(match_vs_clip_f[match_vs_clip_f.group == "all"].round(4).to_string())
_P = {**PQ_MAIN, **PQ_SH}
_pairs = [("none+match", "none"), ("none+match", "none+clipmatch"), ("none+match", "shuffled_nodes+match"),
          ("none+match", "no_relations+match"), ("none+match", "no_class_edges+match"), ("none+clipmatch", "none")]
_sig = [paired_test(_P, a, b).assign(a=a, b=b) for a, b in _pairs
        if any(k[0] == a for k in _P) and any(k[0] == b for k in _P)]
if "PQ_B" in globals():
    _sig += [paired_test(PQ_B, a, b).assign(a=a, b=b, backbone="ViT-B-32") for a, b in _pairs[:2]
             if any(k[0] == a for k in PQ_B) and any(k[0] == b for k in PQ_B)]
match_sig_f = pd.concat(_sig); match_sig_f.to_csv(R_F / "match_paired.csv", index=False)
print("\nKiểm định cặp giữa các cấu hình:"); print(match_sig_f[match_sig_f.group == "all"].round(4).to_string())
if "flickr" in REL_PQ and any(k[0] == "none+match" for k in REL_PQ["flickr"]):
    _rt, _rs = rel_probe_report("flickr", base="none+match",
                                others=["CLIP", "none", "none+clipmatch", "shuffled_nodes+match", "no_relations+match"],
                                out_csv=R_F / "relational_hard_negatives_match.csv")
    print("\nBộ quan hệ khó (Flickr8k) với nhánh khớp:"); print(_rt.to_string()); print(_rs.round(4).to_string())
pd.DataFrame(MATCH_LOG).to_csv(R_F / "match_log.csv", index=False)


---
# PHẦN 2 — GNN của SCENIR thay GNN cổ điển (Flickr8k)

_Nhiệm vụ 2 · dùng lại toàn bộ mã của Phần 1._

Cùng `run_experiment`, cùng split, nhãn tagger và ứng viên CLIP; chỉ đổi `CFG.gnn_type = "scenir"` (mục 1.7.2).

In [ ]:
# Gói kết quả để nộp
print("\nTệp đã ghi:")
for p in sorted((CFG.out_dir / "results").glob("*")) + [CFG.out_dir / "split.json"]:
    print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")

---
# PHẦN 3 — Action Genome (dataset thứ hai)

_Cùng pipeline của Phần 1–2 trên video/frame Action Genome: liên kết khái niệm (dictionary vs semantic), 3 ablation × 3 seed, SCENIR._

In [ ]:
import shutil, os, glob, json
from pathlib import Path

# Dataset hieutrinh123/action-genome được Kaggle mount sẵn (chỉ đọc, đĩa cục bộ)
# ⇒ không cần chép/giải nén. Dò thư mục chứa ag_records.json; frame nằm ở <thư mục>/ag_frames/.
_ag_found = sorted(glob.glob("/kaggle/input/**/ag_records.json", recursive=True))
assert _ag_found, ("Không thấy ag_records.json trong /kaggle/input — Add Input → "
                   "dataset 'hieutrinh123/action-genome'.")
DATA_PATH = str(Path(_ag_found[0]).parent)
AG_CACHE_DIR = Path("/kaggle/working/output/action_genome/cache")   # = AG_OUT / "cache"
AG_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Nếu có cache CLIP của lần chạy trước được Add Input thì dùng lại, khỏi mã hoá frame.
for _f in glob.glob("/kaggle/input/**/clip_ag_*.npz", recursive=True):
    _dst = AG_CACHE_DIR / Path(_f).name
    if not _dst.exists():
        shutil.copy2(_f, _dst)

_n_jpg = len(os.listdir(Path(DATA_PATH) / "ag_frames"))
print("AG root:", DATA_PATH, "|", _n_jpg, "frame")

# v6: AG giữ backbone ViT-L/14 (dùng lại cache CLIP) và chỉ 3 ablation chính; đối chứng quan hệ chạy riêng ở cuối Phần 3.
CFG.clip_model, CFG.clip_weights, CFG.clip_bs = "ViT-L-14", "laion2b_s32b_b82k", 128   # mô hình cuối (version 9); đổi thành "ViT-H-14", "laion2b_s32b_b79k", 64 để chạy ViT-H
for _f in (clip_text_features, encode_free_text):          # v6: bỏ model CLIP ViT-B/32 đã cache ở Phần 1
    for _a in ("_m", "_t"):
        _f.__dict__.pop(_a, None)
CFG.ablations = ("none", "no_class_edges", "no_relations")


In [ ]:
# =====================================================================================
# v6 — HAI ĐỐI CHỨNG KIỂM TRA "CHÍNH QUAN HỆ mang thông tin" trên AG
#  (1) composite_swap: mỗi node quan hệ 'v o' của MỌI video bị thay bằng 'v′ o' (v′ ≠ v, lấy theo
#      phân bố vị từ của vật o trên TRAIN). Giữ nguyên vật và số node ⇒ nếu hiệu quả trên bộ quan hệ
#      khó mất đi thì thông tin nằm ở QUAN HỆ ĐÚNG, không phải ở việc có thêm node.
#  (2) Truy vấn phản thực tế: đổi vị từ trong truy vấn ('drinking … cup' → 'holding … cup') sao cho
#      video đúng KHÔNG có quan hệ mới. Mô hình hiểu quan hệ phải hạ hạng video đúng; CLIP gần như không.
# =====================================================================================
import copy
AG_VO = defaultdict(Counter)                     # vật → phân bố vị từ (CHỈ train), dựng khi có ag_records


def _ag_vo():
    if not AG_VO:
        tr_ids = set(ag_split["train"])
        for r in ag_records:
            if r["image_id"] in tr_ids:
                for c in r["objects"]:
                    if " " in c:
                        v, o = c.split(" ", 1); AG_VO[o][v] += 1
    return AG_VO


# ---------------- truy vấn phản thực tế: đổi vị từ ----------------
VERB_ING = dict(look="looking", hold="holding", carry="carrying", drink="drinking", eat="eating",
                sit="sitting", stand="standing", lie="lying", lean="leaning", touch="touching",
                wear="wearing", wipe="wiping", write="writing", twist="twisting")


def build_ag_counterfactuals(recs, gal):
    out = []
    for q, cand in gal:
        text, objs = recs[q]["caption"], set(recs[q]["objects"])
        toks = re.findall(r"[A-Za-z]+|[^A-Za-z]+", text)
        nouns = {AG_NOUN_ALIAS.get(_ag_lemma(t.lower()), _ag_lemma(t.lower())) for t in toks if t.isalpha()}
        for k, t in enumerate(toks):
            if not t.isalpha():
                continue
            v = _ag_lemma(t.lower(), "v"); v = AG_VERB_ALIAS.get(v, v)
            hit = sorted(c for c in objs if " " in c and c.split(" ", 1)[0] == v and c.split(" ", 1)[1] in nouns)
            if not hit:
                continue
            o = hit[0].split(" ", 1)[1]
            alts = [x for x, _ in _ag_vo()[o].most_common() if x != v and f"{x} {o}" not in objs and x in VERB_ING]
            if not alts:
                continue
            new = list(toks); new[k] = VERB_ING[alts[0]]
            out.append(dict(q=int(q), cand=np.asarray(cand), text=text, cf="".join(new), rel=hit[0],
                            cf_rel=f"{alts[0]} {o}"))
            break
    return out


REL_CF, REL_CF_FEATS, REL_CF_RES = {}, {}, defaultdict(dict)


def _init_ag_cf():
    """Dựng bộ truy vấn phản thực tế từ bộ quan hệ khó của AG (gọi một lần, lúc hook chạy lần đầu)."""
    _ag_vo()
    REL_CF["ag"] = build_ag_counterfactuals(ag_records, REL_GALLERIES["ag"])
    texts = sorted({x["text"] for x in REL_CF["ag"]} | {x["cf"] for x in REL_CF["ag"]})
    REL_CF_FEATS.update(zip(texts, clip_text_features(texts)))        # CÙNG bộ mã hoá cho cả 2 phiên bản
    log(f"truy vấn phản thực tế: {len(REL_CF['ag'])}/{len(REL_GALLERIES['ag'])} truy vấn của bộ quan hệ khó")
    for x in REL_CF["ag"][:4]:
        print(f"  • [{x['rel']} → {x['cf_rel']}]\n      gốc : {x['text'][:110]}\n      đổi : {x['cf'][:110]}")


@torch.no_grad()
def cf_ranks(model, eng, cfs, alpha, lam):
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    z = znorm_per_row
    out = []
    for x in cfs:
        cand = x["cand"]
        row = {}
        for tag in ("text", "cf"):
            qf = torch.tensor(REL_CF_FEATS[x[tag]][None])
            cids = [n for n, _, _ in parse_query(x[tag], eng.graph)]
            s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
            pm = torch.tensor(path_score_matrix(eng.graph, cids, cand)[None]).to(DEVICE)
            s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
            s_path = model.path_score(pm).cpu().numpy()
            fin = combine(z(s_clip), z(s_gnn), z(s_path), alpha, lam)[0]
            row[f"rank_{tag}"] = 1 + int((fin > fin[0]).sum())
            row[f"clip_rank_{tag}"] = 1 + int((s_clip[0] > s_clip[0, 0]).sum())
        out.append(row)
    return pd.DataFrame(out)


if "_relprobe_hook_v5" not in globals():
    _relprobe_hook_v5 = relprobe_hook


def relprobe_hook(ab, seed, model, eng, parts, alpha, lam):
    _relprobe_hook_v5(ab, seed, model, eng, parts, alpha, lam)
    if REL_GAL_KEY == "ag" and "ag" not in REL_CF and "ag" in REL_GALLERIES:
        _init_ag_cf()
    if REL_GAL_KEY in REL_CF:
        d = cf_ranks(model, eng, REL_CF[REL_GAL_KEY], alpha, lam)
        name = REL_RENAME.get(ab, ab)
        REL_CF_RES[REL_GAL_KEY][(name, seed)] = d
        REL_CF_RES[REL_GAL_KEY].setdefault(("CLIP", seed), d.assign(rank_text=d.clip_rank_text,
                                                                    rank_cf=d.clip_rank_cf))
        print(f"    [phản thực tế] {name} seed {seed}: video đúng bị hạ hạng ở "
              f"{(d.rank_cf > d.rank_text).mean():.1%} truy vấn (CLIP {(d.clip_rank_cf > d.clip_rank_text).mean():.1%})")


def cf_report(key, base="none", n_boot=10000):
    res = REL_CF_RES[key]
    names = list(dict.fromkeys(a for a, _ in res))
    rows, per = [], {}
    for a in names:
        ds = [d for (b, s), d in res.items() if b == a]
        drr = np.mean([1 / d.rank_text - 1 / d.rank_cf for d in ds], 0)      # RR(gốc) − RR(đổi vị từ), TB theo seed
        per[a] = drr
        rows.append(dict(model=a, n=len(drr), seeds=len(ds),
                         demoted=np.mean([(d.rank_cf > d.rank_text).mean() for d in ds]),
                         promoted=np.mean([(d.rank_cf < d.rank_text).mean() for d in ds]),
                         delta_RR=drr.mean()))
    tab = pd.DataFrame(rows)
    rng = np.random.RandomState(0); sig_rows = []
    for b in names:
        if b == base:
            continue
        d = per[base] - per[b]
        bs = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        sig_rows.append(dict(a=base, b=b, n=len(d), diff_delta_RR=d.mean(),
                             ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5)))
    return tab, pd.DataFrame(sig_rows)


In [ ]:
# =====================================================================================
# v8 — ĐỒ THỊ CẢNH KHÔNG–THỜI GIAN: Action Genome + nhãn hành động / bối cảnh của Charades
#   Chẩn đoán (CPU): chỉ với annotation AG (35 loại vật, 25 quan hệ), một ánh xạ tuyến tính truy vấn→annotation
#   đã đạt MRR 0,36–0,39 — ngang toàn bộ hệ thống GNN (0,37–0,38) ⇒ giới hạn là LƯỢNG THÔNG TIN trong đồ thị:
#   truy vấn mô tả hành động (mở cửa, đi lại, cầm lên…) mà annotation AG không có. AG được dựng trên Charades;
#   Charades có nhãn hành động (157 lớp, kèm mốc thời gian) và bối cảnh (phòng) do người chú thích cho từng video.
#   Thêm vào đồ thị, cho MỌI video (cùng điều kiện "gallery đã được chú thích" như annotation AG):
#     • node hành động 'động_từ vật' ('open door', 'put clothes'…) / động từ đơn ('sneeze') + vật của hành động
#     • node bối cảnh ('kitchen', 'living room'…)
#     • bộ ba THỜI GIAN cấp video ⟨a before b⟩ (a bắt đầu trước b ≥ 1 giây) — mẫu P6; truy vấn: thứ tự nhắc tới
#   KHÔNG dùng cột 'script' / 'descriptions' / 'objects' của Charades (văn bản gần với truy vấn).
#   Đối chứng (cell sau thí nghiệm chính): no_actions (đồ thị AG như version 4), shuffled_actions (nhãn Charades
#   của video KHÁC cùng split).
# =====================================================================================
import csv, zipfile, urllib.request

CH_URL = "https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip"
CH_SCENE = {"Kitchen": "kitchen", "Bedroom": "bedroom", "Living room": "living room", "Bathroom": "bathroom",
            "Home Office": "office", "Entryway": "entryway", "Dining room": "dining room", "Stairs": "stair",
            "Hallway": "hallway", "Laundry room": "laundry room", "Closet": "closet", "Recreation room": "recreation room",
            "Pantry": "pantry", "Garage": "garage", "Basement": "basement"}
CH, CH_ACTION_NODES = {}, set()


def _ch_dir():
    hits = glob.glob("/kaggle/input/**/Charades_v1_train.csv", recursive=True)
    if hits:
        return Path(hits[0]).parent
    d = (Path("/kaggle/working") if RUN_ON_KAGGLE else Path(".")) / "charades"
    if not (d / "Charades" / "Charades_v1_train.csv").exists():
        d.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(CH_URL, d / "Charades.zip")
        with zipfile.ZipFile(d / "Charades.zip") as z:
            z.extractall(d)
    return d / "Charades"


def _ch_init():
    """Nạp Charades một lần; mở rộng tập động từ của ag_parse_query và IsA 'động_từ vật' → 'động_từ'."""
    if CH:
        return
    d = _ch_dir()
    two = lambda fn: dict(l.strip().split(" ", 1) for l in open(d / fn, encoding="utf-8") if l.strip())
    verb = {k: AG_VERB_ALIAS.get(v.strip().lower(), v.strip().lower()) for k, v in two("Charades_v1_verbclasses.txt").items()}
    obj = {k: (None if v.strip() == "None" else ag_canon(v)) for k, v in two("Charades_v1_objectclasses.txt").items()}
    name = {}
    for l in open(d / "Charades_v1_mapping.txt", encoding="utf-8"):
        c, o, v = l.split()
        name[c] = (verb[v], obj[o])
    rows = {}
    for f in ("train", "test"):
        for row in csv.DictReader(open(d / f"Charades_v1_{f}.csv", encoding="utf-8")):
            rows[row["id"]] = row
    CH.update(rows=rows, name=name)
    _AG_VERBS.update(verb.values())
    for v, o in name.values():
        node = f"{v} {o}" if o else v
        CH_ACTION_NODES.add(node)
        if o:
            FALLBACK_ISA.setdefault(node, [v])
    print(f"Charades: {len(rows)} video | {len(CH_ACTION_NODES)} node hành động | tập động từ truy vấn: {len(_AG_VERBS)}")


def charades_info(video_id):
    row = CH["rows"].get(video_id)
    out = dict(nodes=set(), rels=set(), triples=[])
    if row is None:
        return out
    acts = []
    for x in (row.get("actions") or "").split(";"):
        p = x.split()
        if len(p) == 3 and p[0] in CH["name"]:
            v, o = CH["name"][p[0]]
            node = f"{v} {o}" if o else v
            acts.append((float(p[1]), node))
            out["nodes"].add(node)
            if o:
                out["nodes"].add(o); out["rels"].add((o, "afford", node))
    sc = next((v for k, v in CH_SCENE.items() if (row.get("scene") or "").startswith(k)), None)
    if sc:
        out["nodes"].add(sc)
    acts.sort()
    tri = {(a, "before", b) for i, (s1, a) in enumerate(acts) for s2, b in acts[i + 1:] if a != b and s2 >= s1 + 1.0}
    out["triples"] = sorted(tri)
    return out


def apply_ag_actions(recs, mode="full", seed=0):
    """Gán objects / relations / img_triples của từng video theo mode: full | none | shuffled (trong cùng split)."""
    src = list(range(len(recs)))
    if mode == "shuffled":
        rng = np.random.RandomState(seed)
        for k in ("train", "val", "test"):
            idx = np.array([i for i, r in enumerate(recs) if r["split"] == k])
            for a, b in zip(idx, rng.permutation(idx)):
                src[a] = int(b)
    for i, r in enumerate(recs):
        ch = recs[src[i]] if mode != "none" else None
        r["objects"] = sorted(set(r["ag_objects"]) | (set(ch["ch_nodes"]) if ch else set()))
        r["relations"] = sorted(set(map(tuple, r["ag_relations"])) | (set(map(tuple, ch["ch_rels"])) if ch else set()))
        r["img_triples"] = [(tuple(t), 1.0) for t in ch["ch_triples"]] if ch else []
    return recs


def ag_temporal_hits(names_in_order, graph, seen):
    """Bộ ba thời gian của TRUY VẤN: hai hành động được nhắc liên tiếp ⇒ ⟨trước before sau⟩ (nếu là node của đồ thị)."""
    hits = []
    if not getattr(graph, "n_tri", 0):
        return hits
    acts = [n for n in dict.fromkeys(names_in_order) if n in CH_ACTION_NODES]
    for a, b in zip(acts, acts[1:]):
        nid = graph.tri2id.get((a, "before", b))
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, f"{a} before {b}", f"{a} … {b}"))
    return hits


In [ ]:
# =====================================================================================
# CẤP ĐỘ 2 — DATASET THỨ HAI: ACTION GENOME (text-to-video)
# Chạy toàn bộ pipeline trên AG mà KHÔNG ghi đè kết quả Flickr8k.
# =====================================================================================
import os
import nltk
from contextlib import contextmanager
from nltk.stem import WordNetLemmatizer
import pandas as pd
for _p in ("wordnet", "omw-1.4"):
    try: nltk.download(_p, quiet=True)
    except Exception: pass

_AG_LEM = WordNetLemmatizer(); _AG_LEM_CACHE = {}

def _ag_lemma(w, pos="n"):
    k = (w, pos)
    if k not in _AG_LEM_CACHE:
        try: _AG_LEM_CACHE[k] = _AG_LEM.lemmatize(w, pos)
        except Exception: _AG_LEM_CACHE[k] = w
    return _AG_LEM_CACHE[k]

# ---- Từ vựng AG: phần lấy từ dữ liệu vs phần viết tay (ghi rõ trong báo cáo) ----
_ag_hits = glob.glob(f"{DATA_PATH}/**/ag_records.json", recursive=True)
assert _ag_hits, "Chưa Add Input output của notebook ag-download"
_ag_payload = json.load(open(_ag_hits[0]))

# (1) Quan hệ người–vật → động từ gốc. Ánh xạ thủ công 1-1 từ tên lớp AG; đã kiểm tra bên dưới.
#     Bỏ: quan hệ không gian, quan hệ phủ định/mơ hồ, covered_by (ngược chiều ngữ nghĩa).
AG_PRED2VERB = {"looking_at": "look", "holding": "hold", "carrying": "carry",
                "have_it_on_the_back": "carry", "drinking_from": "drink", "eating": "eat",
                "sitting_on": "sit", "standing_on": "stand", "lying_on": "lie", "leaning_on": "lean",
                "touching": "touch", "wearing": "wear", "wiping": "wipe", "writing_on": "write",
                "twisting": "twist"}
_pred_seen = Counter(p for r in _ag_payload["records"] for _, p, _ in r["triples"])
print("Predicate có trong dữ liệu nhưng KHÔNG dùng:",
      {p: n for p, n in _pred_seen.items() if p not in AG_PRED2VERB})
_bad = [p for p in AG_PRED2VERB if p not in _pred_seen]
assert not _bad, f"Khóa không tồn tại trong annotation AG: {_bad}"

# (2) Alias danh từ: SINH TỰ ĐỘNG từ tên lớp nhiều tên của AG (vd 'cup/glass/bottle')
_ag_classes = {c for r in _ag_payload["records"] for c in r["objects"]}
AG_NOUN_ALIAS = {_ag_lemma(alt.strip().lower()): _ag_lemma(c.split("/")[0].strip().lower())
                 for c in _ag_classes if "/" in c for alt in c.split("/")[1:]}
print("Alias danh từ sinh từ dữ liệu:", AG_NOUN_ALIAS)

# (3) Alias động từ: VIẾT TAY, căn cứ là tần suất động từ trong mô tả của tập TRAIN
AG_VERB_ALIAS = {"watch": "look", "grab": "hold"}
_AG_VERBS = set(AG_PRED2VERB.values())
_train_verbs = Counter(
    _ag_lemma(t, "v") for r in _ag_payload["records"] if r["split"] == "train"
    for t in _norm(r["caption"]).split()
    if t not in STOP and _ag_lemma(t, "v") != t)        # xấp xỉ: token có dạng biến tố động từ
print("Động từ phổ biến trên TRAIN chưa được ánh xạ:",
      [(v, n) for v, n in _train_verbs.most_common(40)
       if v not in _AG_VERBS and v not in AG_VERB_ALIAS][:20])


def ag_canon(cls):
    """'cup/glass/bottle' → 'cup'; 'groceries' → 'grocery' (khớp lemma phía truy vấn)."""
    return _ag_lemma(cls.split("/")[0].strip().lower().replace("_", " "))


def load_action_genome():
    root = Path(_ag_hits[0]).parent
    out = []
    for i, r in enumerate(_ag_payload["records"]):
        objs = {ag_canon(c) for c in r["objects"]} - {"person"}   # video nào cũng có người
        comps, rels = set(), set()
        for kind, pred, cls in r["triples"]:
            v, o = AG_PRED2VERB.get(pred), ag_canon(cls)
            if v is None or o == "person":
                continue
            c = f"{v} {o}"                       # node ghép, vd 'hold cup'
            comps.add(c)
            rels.add((o, "afford", c))           # cup –afford→ hold cup (gộp CHỈ trên train)
        out.append(dict(image_id=i, video_id=r["video_id"], split=r["split"],
                        caption=r["caption"], captions=r["descriptions"],
                        objects=sorted(objs | comps), relations=sorted(rels), url="",
                        frames=[str(root / "ag_frames" / f) for f in r["frames"]]))
    _ch_init()                                   # v8: nhãn hành động / bối cảnh / thứ tự thời gian của Charades
    for r_ in out:
        ch_ = charades_info(r_["video_id"])
        r_["ag_objects"], r_["ag_relations"] = list(r_["objects"]), list(r_["relations"])
        r_["ch_nodes"], r_["ch_rels"], r_["ch_triples"] = sorted(ch_["nodes"]), sorted(ch_["rels"]), ch_["triples"]
    apply_ag_actions(out, "full")
    print(f"  + Charades: {np.mean([len(r['ch_nodes']) for r in out]):.1f} node hành động/bối cảnh mỗi video | "
          f"{np.mean([len(r['ch_triples']) for r in out]):.1f} bộ ba thời gian mỗi video")
    print(f"Action Genome: {len(out)} video | "
          f"{np.mean([len(r['objects']) for r in out]):.1f} node/video | "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/video | "
          f"{np.mean([len(r['frames']) for r in out]):.1f} frame/video")
    for r in out[:3]:
        print("  •", r["caption"][:80], "→", r["objects"][:8])
    return out


def encode_action_genome(recs):
    """CLIP từng frame → mean-pool theo thời gian → L2. Văn bản = mô tả Charades."""
    cache = CFG.out_dir / "cache" / f"clip_ag_{CFG.clip_model}_{CFG.clip_weights}_{len(recs)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model,
                                                                 pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    items = [(i, p) for i, r in enumerate(recs) for p in r["frames"]]
    missing = [p for _, p in items if not os.path.exists(p)]
    if len(missing) > 0.01 * len(items) and "ag_ensure_frames" in globals():
        print(f"Thiếu {len(missing)}/{len(items)} frame (cache CLIP không khớp) → chép frame từ Drive…")
        _DONE.unlink(missing_ok=True)            # bỏ dấu 'chỉ annotation' để chép đủ
        ag_ensure_frames()
        missing = [p for _, p in items if not os.path.exists(p)]
    assert len(missing) <= 0.01 * len(items), \
        f"{len(missing)}/{len(items)} frame không tồn tại (vd {missing[:2]}) — kiểm tra DATA_PATH"
    class FrameDS(Dataset):
        def __len__(self): return len(items)
        def __getitem__(self, k):
            try: im = Image.open(items[k][1]).convert("RGB")
            except Exception: im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    feats = []
    with torch.no_grad():
        from tqdm.auto import tqdm
        log(f"CLIP: mã hoá {len(items)} frame của {len(recs)} video (batch {CFG.clip_bs})")
        for x in tqdm(DataLoader(FrameDS(), batch_size=CFG.clip_bs, num_workers=2),
                      desc="CLIP frame", unit="batch"):
            feats.append(F.normalize(model.encode_image(x.to(DEVICE)), dim=-1).float().cpu().numpy())
        feats = np.concatenate(feats)
        img = np.zeros((len(recs), feats.shape[1]), np.float32)
        np.add.at(img, np.array([i for i, _ in items]), feats)
        img /= np.linalg.norm(img, axis=1, keepdims=True).clip(1e-8)
        caps = [r["caption"] for r in recs]
        # Mô tả Charades thường dài hơn 77 token ⇒ open_clip cắt cụt. Tách theo câu, mã hoá từng câu,
        # lấy trung bình. Giữ cả bản cắt cụt; chọn bản nào bằng MRR CLIP trên VAL (không nhìn test).
        pieces, owner = [], []
        for i, t in enumerate(caps):
            sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip().split()) >= 2] or [t]
            pieces += sents; owner += [i] * len(sents)
        enc = lambda xs: np.concatenate([F.normalize(model.encode_text(tokenizer(xs[s:s + CFG.clip_bs]).to(DEVICE)),
                                                     dim=-1).float().cpu().numpy()
                                         for s in range(0, len(xs), CFG.clip_bs)])
        txt_trunc = enc(caps)
        E = enc(pieces); txt_chunk = np.zeros_like(txt_trunc)
        np.add.at(txt_chunk, np.array(owner), E)
        txt_chunk /= np.linalg.norm(txt_chunk, axis=1, keepdims=True)
        va = np.array([i for i, r in enumerate(recs) if r["split"] == "val"])
        mrr = lambda T: reciprocal_ranks(np.tile(va, (len(va), 1)), T[va] @ img[va].T, va).mean()
        m_t, m_c = mrr(txt_trunc), mrr(txt_chunk)
        assert max(m_t, m_c) > 0.02, \
            f"CLIP VAL MRR {max(m_t, m_c):.4f} ≈ ngẫu nhiên — frame hỏng hoặc ảnh/văn bản lệch thứ tự"
        txt = txt_chunk if m_c > m_t else txt_trunc
        log(f"Văn bản AG — VAL MRR CLIP: cắt cụt {m_t:.4f} | theo câu {m_c:.4f} → dùng "
            f"{'theo câu' if m_c > m_t else 'cắt cụt'}")
    log(f"CLIP xong: ảnh {img.shape}, văn bản {txt.shape} → cache {cache.name}")
    del model; torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


def ag_make_split(recs, seed=None):
    """Split lấy từ ag_records.json: train/test gốc của AG, val tách bằng seed cố định."""
    sp_ = {k: sorted(r["image_id"] for r in recs if r["split"] == k) for k in ("train", "val", "test")}
    sp_["seed"] = "ag-official-train-test + val seed 20240101"
    payload = json.dumps({k: sp_[k] for k in ("train", "val", "test")}, sort_keys=True)
    sp_["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(sp_, f)
    print(f"Split AG (sha1={sp_['sha1']}): train={len(sp_['train'])} "
          f"val={len(sp_['val'])} test={len(sp_['test'])}")
    assert not (set(sp_["train"]) & set(sp_["test"])) and not (set(sp_["val"]) & set(sp_["test"]))
    return sp_


def ag_parse_query(text, graph):
    """'A person is drinking from a glass' → ['drink', 'cup'] → khớp 'drink cup' và 'cup'."""
    toks = []
    for t in _norm(text).split():
        if t in STOP or len(t) < 2:
            continue
        v = _ag_lemma(t, "v"); v = AG_VERB_ALIAS.get(v, v)
        if v in _AG_VERBS:
            toks.append(v); continue
        n = _ag_lemma(t, "n"); toks.append(AG_NOUN_ALIAS.get(n, n))
    cands = [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    if "ag_temporal_hits" in globals():          # v8: bộ ba thời gian theo thứ tự nhắc tới trong truy vấn
        big = {i for i in range(len(toks) - 1) if " ".join(toks[i:i + 2]) in CH_ACTION_NODES}      # hành động dạng 'động_từ vật'
        order = sorted([(i, " ".join(toks[i:i + 2])) for i in big] +
                       [(i, toks[i]) for i in range(len(toks)) if i not in big and (i - 1) not in big])
        hits += ag_temporal_hits([g for _, g in order], graph, seen)
    return hits


@contextmanager
def use_dataset(out_dir, **overrides):
    """Tạm thay hàm toàn cục + thư mục output; khôi phục nguyên trạng khi ra khỏi khối."""
    g = globals()
    saved = {k: g.get(k) for k in overrides}
    saved_out = CFG.out_dir
    g.update(overrides)
    CFG.out_dir = Path(out_dir)
    for sub in ("cache", "results"):
        (CFG.out_dir / sub).mkdir(parents=True, exist_ok=True)
    _GROUP_CACHE.clear()
    try:
        yield
    finally:
        for k, v in saved.items():
            if v is None: g.pop(k, None)
            else: g[k] = v
        CFG.out_dir = saved_out
        _GROUP_CACHE.clear()


# (4) IsA cho node ghép 'động từ vật' → khái niệm hành động 'động từ' (vd 'hold cup' –là_một→ 'hold').
#     WordNet chỉ phủ 31/202 lớp AG nên tầng khái niệm gần như rỗng; đây là tri thức NGOÀI
#     (cấu trúc tên lớp), không phải thống kê ⇒ an toàn cho test. Truy vấn "someone holding
#     something" giờ có đường đi P2/P4/P5 tới 'hold' thay vì không khớp node nào.
#     Ghi vào FALLBACK_ISA toàn cục để mọi cell AG sau (semantic linking) dùng cùng một đồ thị;
#     tên 'động từ vật' không bao giờ xuất hiện trong Flickr8k nên không ảnh hưởng phần trước.
AG_ACTION_ISA = {f"{v} {ag_canon(cls)}": [v]
                 for r in _ag_payload["records"] for _, pred, cls in r["triples"]
                 for v in [AG_PRED2VERB.get(pred)] if v is not None and ag_canon(cls) != "person"}
FALLBACK_ISA.update(AG_ACTION_ISA)
print(f"IsA hành động: {len(AG_ACTION_ISA)} node ghép → {len(set(AG_PRED2VERB.values()))} khái niệm động từ")

# ------------------------------------ LOG TIẾN ĐỘ ------------------------------------
# Mọi dòng có [mm:ss từ đầu cell | +ss so với dòng trước] và flush ngay (Colab hay giữ buffer).
# Các hàm nặng của pipeline được BỌC lại qua use_dataset(...) — chỉ có hiệu lực trong khối
# with, tự khôi phục khi ra khỏi khối, không sửa cell nào khác.
_T0 = _TL = time.time()


def log(msg):
    global _TL
    now = time.time()
    m, s_ = divmod(int(now - _T0), 60)
    print(f"[{m:02d}:{s_:02d} | +{now - _TL:5.1f}s] {msg}", flush=True)
    _TL = now


@contextmanager
def step(title):
    log(f"▶ {title}")
    t = time.time()
    yield
    log(f"✔ {title} ({time.time() - t:.1f}s)")


_N_ABL = len(CFG.ablations); _N_SEED = len(CFG.seeds)
_PROG = dict(abl=None, i_abl=0, i_seed=0)
_orig = dict(prepare=prepare, train_graph_scorer=train_graph_scorer, tune_on_val=tune_on_val,
             evaluate_split=evaluate_split, self_test=self_test)


def _prepare_logged(records, split, img_feat, txt_feat, ablation="none", seed=0):
    if ablation != _PROG["abl"]:
        _PROG.update(abl=ablation, i_abl=_PROG["i_abl"] + 1, i_seed=0)
    _PROG["i_seed"] += 1
    tag = f"[ablation {_PROG['i_abl']}/{_N_ABL} '{ablation}' | seed {_PROG['i_seed']}/{_N_SEED}]"
    with step(f"{tag} dựng đồ thị + bộ truy vấn"):
        out = _orig["prepare"](records, split, img_feat, txt_feat, ablation, seed)
    g = out[0]
    log(f"    nodes={g.n_nodes} | scene={g.A_scene.nnz} related={g.A_related.nnz} isa={g.A_isa.nnz}"
        f" | đường đi P1..P5={sum(g.R[p].nnz for p in PATTERNS[1:])}")
    return out


def _train_logged(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    with step(f"  huấn luyện GNN seed={seed} ({CFG.epochs} epoch, {len(train_q_idx)} truy vấn)"):
        model, hist = _orig["train_graph_scorer"](engine, train_q_idx, train_img_idx,
                                                  seed=seed, verbose=verbose)
    gates = model.edge_gates()
    log(f"    loss {hist[0]:.3f} → {hist[-1]:.3f} | clip_w {F.softplus(model.clip_logit).item():.2f}"
        f" | path_w {F.softplus(model.path_logit).item():.2f}"
        f" | w_path " +
        " ".join(f"{x:.3f}" for x in model.pattern_weights().detach().cpu().numpy()) +
        (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4]) if gates else ""))
    return model, hist


def _tune_logged(model, engine, val_q, val_gallery, verbose=True):
    with step(f"  quét (α, λ) trên VAL ({len(CFG.alpha_grid) * len(CFG.lambda_grid)} cặp)"):
        (a, l), grid = _orig["tune_on_val"](model, engine, val_q, val_gallery, verbose=False)
    best = max(grid, key=lambda r: r["MRR"])
    log(f"    chọn α={a}, λ={l} | VAL MRR={best['MRR']:.4f} (CLIP thuần: "
        f"{next(r['MRR'] for r in grid if r['alpha'] == 1.0):.4f})"
        + ("  ⚠ λ=1 ⇒ s_path không tham gia điểm cuối" if l == 1.0 else ""))
    return (a, l), grid


def _eval_logged(model, engine, part, alpha, lam, by_group=True):
    res, raw = _orig["evaluate_split"](model, engine, part, alpha, lam, by_group)
    log(f"    TEST: CLIP MRR={res['CLIP']['MRR']:.4f} → CLIP+Graph MRR={res['CLIP+Graph']['MRR']:.4f}")
    return res, raw


def _self_test_logged(*a, **k):
    with step("self-test (rò rỉ / khớp giải thích / split) — dựng đồ thị 3 lần, hơi lâu"):
        return _orig["self_test"](*a, **k)


# ------------------------------------ CHẠY ------------------------------------
AG_OUT = Path("/kaggle/working/output/action_genome")
log(f"Bắt đầu Action Genome | ablations={CFG.ablations} | seeds={CFG.seeds} | epochs={CFG.epochs}"
    f" | device={DEVICE}")
with use_dataset(AG_OUT, parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged,
                 self_test=_self_test_logged):
    R = CFG.out_dir / "results"

    with step("1/9 nạp annotation AG"):
        ag_records = load_action_genome()
    with step("2/9 split"):
        ag_split = make_split(ag_records)
    with step("3/9 đặc trưng CLIP (dùng cache nếu có)"):
        ag_img, ag_txt = encode_action_genome(ag_records)
        if "ag_save_clip_cache_to_drive" in globals():
            ag_save_clip_cache_to_drive()
    with step("4/9 self-test"):
        self_test(ag_records, ag_split, ag_img, ag_txt)
    if "pick_learn_clip_w" in globals():
        with step("4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'none', seed 0)"):
            clipw_ag = pick_learn_clip_w(ag_records, ag_split, ag_img, ag_txt)

    with step(f"5/9 thí nghiệm chính: {_N_ABL} ablation × {_N_SEED} seed = {_N_ABL * _N_SEED} lần huấn luyện"):
        _PROG.update(abl=None, i_abl=0, i_seed=0)
        REL_GALLERIES["ag"] = build_rel_galleries_ag(ag_records, ag_split)      # v5: bộ quan hệ khó
        REL_GAL_KEY, REL_RENAME = "ag", {}
        ag_all_res, ag_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                            ablations=CFG.ablations)
        REL_GAL_KEY = None
    ag_df = results_table(ag_all_res)
    import pickle
    with open(R / "per_query_all_ablations.pkl", "wb") as f:      # cho kiểm định cặp về sau
        pickle.dump(dict(PER_QUERY), f)
    ag_df.to_csv(R / "main_results.csv", index=False)
    display(ag_df)
    PQ_AG = dict(PER_QUERY) if "PER_QUERY" in globals() else {}   # cho cell kiểm định cặp

    ag_piv = pd.DataFrame(ag_art["grid"]).pivot_table(index="alpha", columns="lam", values="MRR")
    ag_piv.to_csv(R / "alpha_sensitivity.csv")
    print("\nMRR trên VALIDATION theo (α, λ):\n", ag_piv.round(4).to_string())

    with step("6/9 phân tích 5 thành công / 5 thất bại"):
        ag_cases = analyse_cases(ag_art, ag_records)
    for c in ag_cases:
        print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → "
              f"CLIP+Graph {c['rank_graph']}\n  truy vấn: {c['query']}")
        for e in c["explanation"]:
            print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    with open(R / "case_analysis.json", "w") as f:
        json.dump(ag_cases, f, ensure_ascii=False, indent=2)

    with step("7/9 fidelity trên điểm đường đi"):
        ag_fdf, ag_summ = fidelity_study(ag_art)
    ag_fdf.to_csv(R / "fidelity_per_query.csv", index=False)
    with open(R / "xai_summary.json", "w") as f:
        json.dump(ag_summ, f, indent=2)
    print("\nxAI (điểm đường đi):", {k: round(v, 4) if isinstance(v, float) else v
                                     for k, v in ag_summ.items()})

    if "final_rank_fidelity" in globals():
        with step("8/9 fidelity trên thứ hạng điểm cuối"):
            ag_rdf, ag_rsumm = final_rank_fidelity(ag_art)
        ag_rdf.to_csv(R / "fidelity_final_rank.csv", index=False)
        with open(R / "xai_final_rank_summary.json", "w") as f:
            json.dump(ag_rsumm, f, indent=2)
        print("xAI (thứ hạng điểm cuối):", {k: round(v, 3) if isinstance(v, float) else v
                                           for k, v in ag_rsumm.items()})

    with step("9/9 xuất bộ truy vấn đánh giá + demo"):
        export_eval_queries(ag_art["engine"].queries, ag_art["graph"],
                            R / "eval_queries_20.json", n=20, allowed_ids=ag_split["test"])
        demo_query("a person drinking from a glass while sitting on the couch", ag_art, k=3)
        demo_query("someone watching tv and holding a phone", ag_art, k=3)

# ------------------- Bảng tổng hợp cấp độ 2: hai bộ dữ liệu -------------------
if "df" in globals():
    both = pd.concat([df.assign(dataset="flickr8k"), ag_df.assign(dataset="action_genome")])
    both = both[both.ablation == "none"][["dataset", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]]
    both.to_csv(Path("/kaggle/working/results") / "level2_two_datasets.csv", index=False)
    display(both)
log("XONG toàn bộ cell Action Genome")

In [ ]:
# =====================================================================================
# v8 — ĐỐI CHỨNG cho đồ thị không–thời gian trên AG: no_actions (đồ thị AG như version 4) và
#       shuffled_actions (nhãn Charades của video KHÁC) + kiểm định cặp + bộ quan hệ khó + phản thực tế
# =====================================================================================
import copy
R_AG = AG_OUT / "results"
ag_vs = [paired_vs_clip(PQ_AG, ab).assign(dataset="AG") for ab in CFG.ablations]
ctrl_df, PQ_CTRL = [], {}
with use_dataset(AG_OUT / "controls", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    for _name, _mode in (("no_actions", "none"), ("shuffled_actions", "shuffled")):
        _recs = apply_ag_actions(copy.deepcopy(ag_records), _mode)
        _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
        REL_GAL_KEY, REL_RENAME = "ag", {"none": _name}
        _res, _ = run_experiment(_recs, ag_split, ag_img, ag_txt, ablations=("none",))
        REL_GAL_KEY, REL_RENAME = None, {}
        ctrl_df.append(results_table(_res).assign(ablation=_name))
        _pq = {(_name + ab[4:], s): v for (ab, s), v in PER_QUERY.items() if ab == "none" or ab.startswith("none+")}
        PQ_CTRL.update(_pq); ag_vs.append(paired_vs_clip(_pq, _name).assign(dataset="AG"))
_N_ABL = len(CFG.ablations)

pd.concat([ag_df] + ctrl_df).to_csv(R_AG / "main_results_with_controls.csv", index=False)
ag_vs = pd.concat(ag_vs); ag_vs.to_csv(R_AG / "paired_graph_vs_clip.csv", index=False)
print("AG — CLIP+Graph vs CLIP (kiểm định cặp):"); print(ag_vs[ag_vs.group == "all"].round(4).to_string())
sig_ctrl = pd.concat([paired_test({**PQ_AG, **PQ_CTRL}, "none", b) for b in ("no_actions", "shuffled_actions")] +
                     [paired_test(PQ_AG, "none", b) for b in CFG.ablations if b != "none"])
sig_ctrl.to_csv(R_AG / "paired_ablation_test.csv", index=False)
print("\nAG — đồ thị đầy đủ (none) vs từng cấu hình:"); print(sig_ctrl[sig_ctrl.group == "all"].round(4).to_string())
rel_tab_ag, rel_sig_ag = rel_probe_report("ag", base="none",
                                          others=["CLIP", "no_actions", "shuffled_actions", "no_class_edges", "no_relations"],
                                          out_csv=R_AG / "relational_hard_negatives.csv")
print("\nBộ quan hệ khó (AG):"); print(rel_tab_ag.to_string()); print(rel_sig_ag.round(4).to_string())
if REL_CF_RES.get("ag"):
    cf_tab, cf_sig = cf_report("ag")
    cf_tab.to_csv(R_AG / "counterfactual_verb_swap.csv", index=False); cf_sig.to_csv(R_AG / "counterfactual_verb_swap_paired.csv", index=False)
    print("\nTruy vấn phản thực tế:"); print(cf_tab.round(4).to_string()); print(cf_sig.round(4).to_string())


In [ ]:
# =====================================================================================
# v9 — BÁO CÁO NHÁNH KHỚP (Action Genome)
# =====================================================================================
R_AG = AG_OUT / "results"
_P = {**PQ_AG, **PQ_CTRL}
_names = ["none", "none+clipmatch", "none+match", "no_class_edges+match", "no_relations+match",
          "no_actions", "no_actions+match", "shuffled_actions", "shuffled_actions+match"]
match_vs_clip_ag = pd.concat([paired_vs_clip(_P, n).assign(dataset="AG") for n in _names if any(k[0] == n for k in _P)])
match_vs_clip_ag.to_csv(R_AG / "match_vs_clip.csv", index=False)
print("AG — kiểm định cặp vs CLIP:"); print(match_vs_clip_ag[match_vs_clip_ag.group == "all"].round(4).to_string())
_pairs = [("none+match", "none"), ("none+match", "none+clipmatch"), ("none+match", "no_actions+match"),
          ("none+match", "shuffled_actions+match"), ("none+match", "no_relations+match"),
          ("none+match", "no_class_edges+match"), ("none+clipmatch", "none")]
match_sig_ag = pd.concat([paired_test(_P, a, b).assign(a=a, b=b) for a, b in _pairs
                          if any(k[0] == a for k in _P) and any(k[0] == b for k in _P)])
match_sig_ag.to_csv(R_AG / "match_paired.csv", index=False)
print("\nKiểm định cặp giữa các cấu hình:"); print(match_sig_ag[match_sig_ag.group == "all"].round(4).to_string())
if any(k[0] == "none+match" for k in REL_PQ.get("ag", {})):
    _rt, _rs = rel_probe_report("ag", base="none+match",
                                others=["CLIP", "none", "none+clipmatch", "no_actions+match", "shuffled_actions+match",
                                        "no_relations+match", "no_class_edges+match"],
                                out_csv=R_AG / "relational_hard_negatives_match.csv")
    print("\nBộ quan hệ khó (AG) với nhánh khớp:"); print(_rt.to_string()); print(_rs.round(4).to_string())
if any(k[0] == "none+match" for k in REL_CF_RES.get("ag", {})):
    _ct, _cs = cf_report("ag", base="none+match")
    _ct.to_csv(R_AG / "counterfactual_verb_swap_match.csv", index=False)
    _cs.to_csv(R_AG / "counterfactual_verb_swap_match_paired.csv", index=False)
    print("\nTruy vấn phản thực tế với nhánh khớp:"); print(_ct.round(4).to_string()); print(_cs.round(4).to_string())
pd.DataFrame(MATCH_LOG).to_csv(R_AG / "match_log.csv", index=False)


### 3.1 Chọn τ trên VALIDATION

### 3.2 Đánh giá (Evaluation)

### 3.3 Ví dụ liên kết (đọc tay để kiểm tra chất lượng)

### 3.4 Chạy đầy đủ với τ tốt nhất: 3 ablation × 3 seed

### 3.5 So sánh dictionary vs semantic trên TEST (cùng split)

---
# PHẦN 4 — Cấp độ 4(b): truy vấn văn bản → video (MSVD + MSR-VTT)

> **Gộp từ `project03_level4b.ipynb`.** Phần này là một pipeline độc lập với Phần 1–3: nó **định nghĩa lại** `CFG`, `make_split`, `scatter_sum`, `znorm_per_row`… nên **chỉ chạy sau khi Phần 1–3 đã chạy xong** (hoặc khởi động lại runtime rồi chạy riêng từ cell `import` đầu tiên của Phần 4). Không chạy lại các cell của Phần 1–3 sau khi đã chạy Phần 4.


> **v5:** phần này không chạy lại trong bản này (mọi cell code có `%%script true`). Xoá dòng đó ở đầu cell để chạy.

### Cấp độ 4(b) — Truy vấn văn bản → video có xếp hạng lại và giải thích bằng đồ thị

**Phạm vi.** Notebook này chỉ làm **Level 4(b): mở rộng text-to-video** của hệ thống truy vấn có đồ thị. Không đề xuất thuật toán mới (Level 4(a)), không khảo sát người dùng (Level 4(c)), không làm image-to-text. Phần nền tảng text→image nằm ở `image/` (notebook của nhóm, Flickr8k + Action Genome).

**Pipeline** (mục 3.3 của master instruction):
1. truy vấn văn bản;
2. CLIP text encoder đóng băng;
3. lấy mẫu 12 khung hình / video;
4. CLIP image encoder đóng băng;
5. gộp khung hình thành embedding video (cách gộp chọn trên VAL);
6. top-K = 50 video ứng viên theo cosine;
7. xếp hạng lại bằng điểm đồ thị;
8. trả top-K cùng giải thích là các số hạng đồ thị đã thực sự cộng vào điểm.

**Mô hình đồ thị được tái sử dụng, không phải đóng góp mới:**
* **QC-RGAT** (v4) là mô hình chính vì điểm của nó tách chính xác thành các số hạng.
* **GraphSAGE có kiểu cạnh** (mô hình cuối của v3) là baseline GNN đã học.
* Cả hai được đánh giá, **không mặc định** cái nào hơn CLIP-only hay chỉ-contains.

**Dữ liệu và trạng thái từng benchmark:**

| Tập | Vai trò | Đã nhìn trước | Trạng thái kết quả |
|---|---|---|---|
| MSVD train (1200 video) | dựng đồ thị, huấn luyện GNN | — | — |
| MSVD val (100 video) | chọn cấu hình | — | phát triển |
| MSVD test (670 video) | đánh giá | 4 lần (v1–v4) | **EXPLORATORY** |
| MSR-VTT 1k-A | — | 1 lần (v3) | không dùng |
| MSR-VTT fresh1k (seed 20261025) | — | 1 lần (v4) | không dùng (đặt `MSVD_V4_CONFIRM_SET=fresh1k` thì chạy lại, ghi EXPLORATORY) |
| **MSR-VTT fresh1k-B** (seed 20261026) | **xác nhận**, đánh giá **một lần** sau `freeze()` | 0 | **CONFIRMATORY** |

fresh1k-B gồm 1000 video train-9k, rời 1k-A, rời 200 video dev và rời fresh1k. Nó cùng miền với fresh1k, mà số liệu fresh1k đã ảnh hưởng tới thiết kế notebook này; điều đó được khai báo trong báo cáo.

**Phần thêm so với v4** (các ô "Level 4(b)"):
* Ablation **huấn luyện lại** theo họ quan hệ:
  - G7: bỏ cạnh vai `agent_of`/`object_of`;
  - G8: bỏ cạnh `svo`/`prep`.
* Bộ truy vấn **trực tiếp / gián tiếp ≥ 20 / nhóm**: chọn theo luật đăng ký trước, dựng **trước** khi chấm mô hình, báo metric riêng.
* xAI theo **thứ hạng**: giải thích là các số hạng đồ thị ủng hộ (contains và đường đi quan hệ). Xoá chúng rồi đo Δ điểm, Δ hạng, top-1 và top-10, so với xoá ngẫu nhiên.
* Phân tích **5 thành công + 5 thất bại**: chọn theo luật, phân tầng nguyên nhân.
* **Demo**: nhập văn bản tự do, nhận top-K với điểm CLIP, điểm đồ thị, đường đi giải thích và thumbnail.
* `data_inventory.json`, `data_leakage_audit.md`, `audit_report.md`, `level4b_conclusions.md` và `artifacts_manifest.json`, **tự sinh** từ chính lần chạy.

**Chạy trên Kaggle:**
1. Settings: GPU T4, **Internet On**.
2. *(Khuyến nghị)* Add Input output của lần chạy v4 để dùng lại cache đặc trưng MSVD.
3. Save & Run All.
4. Tải `/kaggle/working/project03_level4b_review.zip`.

**Biến môi trường:**
* `MSVD_V4_STAGES` (mặc định `V1,V7`);
* `MSVD_L4_SEEDS` (mặc định `0,1,2,3,4`);
* `MSVD_L4_EPOCHS`;
* `MSVD_V4_CONFIRM_SET`: `fresh1kB` (mặc định), `fresh1k` hoặc `1kA`. Hai giá trị sau ⇒ EXPLORATORY.

**Nguyên tắc.**
* Test và tập ngoài chỉ được mở sau `freeze()`.
* Caption của test và tập ngoài không bao giờ dùng để dựng từ vựng hay quan hệ, huấn luyện hoặc chọn cấu hình.
* PASS của kiểm thử phần mềm **không** phải kết luận khoa học.
* Thí nghiệm không chạy được thì ghi `NOT RUN` kèm lý do.


## 4.1 Cấu hình

Mọi siêu tham số nằm ở cell dưới. Phần dữ liệu, CLIP, lấy mẫu frame và tagger giữ nguyên như v1/v2, nên cache đặc trưng MSVD của v2 dùng lại được. Các khối mới của v3 gồm:
* vị từ: họ có vị từ tự do, prompt, ngưỡng support;
* người chấm: mẫu, ngưỡng precision, file nhãn;
* benchmark xác nhận: MSR-VTT ở commit ghim;
* thống kê: Holm.

## 4.2 Giao thức đăng ký trước (viết TRƯỚC khi chạm MSR-VTT fresh1k-B)

**Đã biết trước khi viết notebook này** (v4, Kaggle, fresh1k):
* v4 hơn CLIP-only: ΔMRR +0,0109, CI [+0,0072; +0,0145].
* v4 **không** hơn chỉ-contains, không vị từ, vị từ hoán vị, rewired, attention tĩnh hay v3.
* Bỏ mọi cạnh quan hệ lúc suy luận làm MRR **tăng** nhẹ.
* Xoá giải thích (chỉ đường đi quan hệ) làm điểm giảm nhưng **không** đổi thứ hạng nào.

Vì vậy notebook này **không kỳ vọng** quan hệ giúp. Mục tiêu là đo lại một cách trung thực trên tập chưa nhìn, kèm các đối chứng còn thiếu.

**Họ so sánh chính** (một phía, `a > b`, MRR trên mọi caption, bootstrap theo video, Holm–Bonferroni α = 0,05). Chỉ **fresh1k-B** mới là xác nhận; test MSVD là thăm dò.

| | So sánh | Câu hỏi |
|---|---|---|
| H1 | G3 vs G0 | đồ thị (QC-RGAT) có hơn CLIP-only? |
| H2 | G3 vs G1 | cạnh quan hệ có đóng góp ngoài contains? |
| H3 | G3 vs G2 | khớp vị từ có ích? |
| H4 | G3 vs G4 | ghép **đúng** vị từ vào **đúng** cạnh có ích? |
| H5 | G3 vs G5 | topology thật có hơn rewiring có ràng buộc? |
| H6 | G3 vs G6 | attention theo truy vấn có hơn attention tĩnh? |
| H7 | G3 vs R3 | QC-RGAT có hơn GraphSAGE có kiểu cạnh (v3)? |
| H8 | G3 vs G7 | họ cạnh **vai** (`agent_of`, `object_of`) có giá trị? (G7 huấn luyện lại không có họ này) |
| H9 | G3 vs G8 | họ cạnh **`svo`/`prep`** có giá trị? (G8 huấn luyện lại không có họ này) |

**Luật kết luận:**
* "Quan hệ có giá trị thật" chỉ được kết luận khi **H2, H4 và H5 cùng được ủng hộ** và đối chứng đạt audit.
* Một họ cạnh "có giá trị" khi H8 hoặc H9 tương ứng được ủng hộ.

**So sánh phụ** (không vào họ Holm, chỉ báo cáo kèm CI):
* G3 vs G0 theo nhóm truy vấn trực tiếp / gián tiếp;
* trên bộ truy vấn đăng ký trước (≥ 20 / nhóm).

**Chọn cấu hình** (chỉ VAL):
* Incumbent là T = 1 (đơn giản hơn).
* T = 2 chỉ được nhận khi Δ MRR val ≥ 0,002 **và** p Holm < 0,10.
* Cố định trước: d = 64, lr = 2e-3, k = 50.
* α (trọng số CLIP trong điểm cuối) chọn trên VAL trong lưới có α = 1 (CLIP-only). Nếu α = 1 thắng trên VAL thì đồ thị không được dùng ở điểm cuối, và kết quả được báo đúng như vậy.

**Ngân sách:** 5 seed × 15 epoch cho mọi thí nghiệm. Trên cùng một tập đánh giá, mọi thí nghiệm dùng chung tập ứng viên CLIP top-k.

**Bộ truy vấn đăng ký trước** (mục 7.3). Lấy từ caption của tập đánh giá, chỉ dựa vào hai thứ: nhãn CLIP zero-shot của video đúng và cạnh đồ thị dựng từ train. **Không** dựa vào điểm của mô hình.
* **Trực tiếp:** mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.
* **Gián tiếp:** truy vấn nêu một quan hệ (bộ ba chủ thể–vị từ–đối tượng); nhãn video chỉ phủ **một phần** khái niệm của truy vấn; có ít nhất một cạnh quan hệ nối khái niệm còn thiếu với một nhãn của video. Đó là các quan hệ dự kiến phải khai thác.
* Mỗi nhóm lấy 20 truy vấn (mỗi video tối đa 1) bằng seed cố định.


## 4.3 Hàng rào chống rò rỉ

* **`CaptionStore`**: caption không được lưu trong `RECORDS`. Mọi lần đọc caption, hoặc dùng đặc trưng của caption, đều phải nêu **mục đích**; một ma trận quyết định split nào được dùng cho mục đích nào.
  * `vocab`, `relations`, `annotation` và `gnn_train` chỉ dùng train; `tune` và `tagger_select` chỉ dùng val.
  * Caption test MSVD và tập ngoài (`ext`, MSR-VTT) chỉ dùng để `encode`, `query_input` và (sau đóng băng) `eval`, `report`, `display`.
  * Truy cập sai sẽ ném `PermissionError` ngay.
* **`TestGate`**: test MSVD và tập ngoài chỉ mở sau `freeze()`, và mọi lần chạm đều được ghi log kèm tên split. Ở `dev_mode`, test MSVD bị khoá; tập ngoài khi đó là video **train-9k** của MSR-VTT, rời hẳn 1k-A.
* **`provenance()`**: mỗi artifact được ghi lại nguồn, split caption và split video đã dùng; khai báo sai loại thì `assert` thất bại.

## 4.4 Dữ liệu: MSVD (phát triển) + MSR-VTT 1k-A (xác nhận) — Cổng A

**MSVD.** Như v1/v2. Video thật nằm trong `MSVD_Videos.zip`. Notebook dò `MSVD_ROOT`, rồi `/kaggle/input`, rồi cache, rồi tải từ HF ở commit ghim. Không có video thì **dừng**.

**MSR-VTT** (`friedrichor/MSR-VTT`, commit ghim). Dùng `msrvtt_test_1k.json` (1000 video, caption chuẩn JSFusion), `raw_data/MSRVTT_data.json` (20 caption/video) và `MSRVTT_Videos.zip` (2,2 GB, mp4).
* Video tập ngoài được nối **sau** MSVD trong `RECORDS` với split `ext`. Vì vậy cache MSVD giữ nguyên tên và thứ tự.
* Truy vấn chính là mọi caption của 1000 video; truy vấn phụ là caption chuẩn 1k-A. Nếu caption chuẩn không nằm trong 20 caption thì nó được thêm vào và đếm riêng.
* Không nạp được MSR-VTT thì tập xác nhận ghi **NOT RUN** kèm lý do; notebook không thay bằng tập khác.

## 4.5 Split chính thức và manifest

Split MSVD giữ đúng `msvd_{train,val,test}.json` (1200 / 100 / 670 video), chia **theo video**. sha1 chỉ tính trên ba split MSVD, nên **trùng với v2**; kiểm tra C7 xác nhận điều này. Tập ngoài `ext` có sha1 riêng và không được trùng video nào với MSVD.

## 4.6 Lấy mẫu khung hình và đặc trưng CLIP đóng băng (Cổng B)

* **Chính sách lấy mẫu `uniform-segment-midpoint`:** video được chia thành `F_max = 12` đoạn bằng nhau, mỗi đoạn lấy frame ở giữa: `idx_k = floor((k + 0.5)/12 · n)`, với `n` là số frame **giải mã được thực sự**. Số này được đếm bằng `grab()`, vì `CAP_PROP_FRAME_COUNT` của AVI có thể sai. Ở lượt đọc thứ hai, notebook chỉ `retrieve()` đúng 12 frame cần lấy, nên bộ nhớ chỉ chứa 12 frame/video.
* Mọi frame được mã hoá **một lần** bằng CLIP-image đóng băng, chuẩn hoá L2, và lưu cache `[N_video, 12, D]` (float16) cùng chỉ số frame, timestamp, fps và thời lượng. Cache được dò ở `msvd_l4/cache` và ở `/kaggle/input/**/cache`.
* Mọi caption được mã hoá **một lần** bằng CLIP-text của cùng mô hình. Bước này không học gì, chỉ để cache đặc trưng; việc dùng đặc trưng caption về sau vẫn phải theo luật của `CaptionStore`.
* Thumbnail của frame giữa video (rộng 160 px) được lưu để demo hiển thị kết quả.

### 4.6b Gộp theo thời gian (chọn trên VAL)

Notebook dùng một embedding duy nhất cho mỗi video, gộp từ `F ∈ {4, 8, 12}` frame con, cách đều nhau trong 12 frame đã lấy mẫu. Có hai cách gộp:

* `mean`: chuẩn hoá L2 từng frame, lấy trung bình, rồi chuẩn hoá lại.
* `centrality`: trọng số `softmax(cos(frame, trung bình)/τ)`. Cách này giảm trọng số các frame lạc chủ đề, chẳng hạn frame chuyển cảnh hoặc frame tối; trọng số không phụ thuộc truy vấn, nên mỗi video vẫn chỉ có một vector.

Cặp `(F, cách gộp)` được chọn theo **MRR text→video của CLIP thuần trên VAL**, xếp hạng trong toàn bộ gallery val. Dòng `F=1` (frame giữa) chỉ được in ra để tham chiếu và không được chọn, để bảo đảm embedding thật sự gộp từ nhiều frame.

## 4.7 Từ vựng (chỉ TRAIN), quan hệ, tagger và mẫu cho người chấm

Parser quan hệ của v2 được **giữ nguyên**, để đo đúng chất lượng của nó. Mỗi quan hệ là `(nguồn, họ, đích, vị từ)`:
* `agent_of`: chủ thể → động từ;
* `object_of`: đồ vật → động từ. Ở hai họ này vị từ chính là động từ đích;
* `svo`: danh từ –cụm động từ→ danh từ, ví dụ `man –ride→ horse`;
* `prep`: danh từ –giới từ→ danh từ, ví dụ `cat –on→ table`.

**Kiểm chứng tự động (Nhiệm vụ B), không cần người chấm.**
* spaCy `en_core_web_sm` là parser phụ thuộc cú pháp, độc lập với heuristic POS. Nó trích lại quan hệ cùng schema trên **chính caption train**:
  * chủ ngữ (`nsubj`, kể cả kế thừa qua `conj`, `xcomp`, `acl`) → `agent_of`;
  * tân ngữ (`dobj`, `nsubjpass`) → `object_of`;
  * chủ–động–tân → `svo`;
  * giới từ gắn vào **danh từ** → `prep`.
* Một quan hệ heuristic được xác nhận khi spaCy trích cùng (nguồn, họ, đích), cùng động từ đầu (svo) hoặc cùng giới từ (prep). Tỉ lệ xác nhận được tính cho từng quan hệ, từng họ và từng ô (họ × mức phổ biến), kèm khoảng Wilson.
* Độ phủ ngược (spaCy tìm được mà heuristic cũng tìm được) chỉ là so sánh tương đối, **không** phải recall thật.
* Đây là nhãn **silver**: hai parser có thể cùng sai, nên báo cáo không gọi nó là precision.

**Mẫu phân tầng** (họ × mức phổ biến; 360 dòng) vẫn được xuất kèm cột `auto_parser_agrees`. Nếu có file nhãn người chấm (`relation_annotation*`, **tuỳ chọn**), notebook tính thêm precision, Wilson và Cohen's kappa.

## 4.8 Đặc tả đồ thị v3 (schema)

**Node.** Canonical inventory **cố định**: video MSVD + video tập ngoài, lớp `(lemma, POS)` từ train, khái niệm theo synset WordNet. Ablation chỉ **tắt cạnh**.

**Cạnh.** Mỗi bản ghi cạnh trong `graph_edges_v3.csv` gồm `(src, relation_family, relation, predicate, dst, direction, weight, confidence, source, status)`. Mỗi dòng là **một** quan hệ: một cặp có nhiều vị từ thành nhiều dòng.
* Trường `predicate` **không bao giờ** chứa tên loại quan hệ; có kiểm tra G2 cho điều này.
* `weight` là support (tần suất), **không** phải độ tin cậy ngữ nghĩa.
* `confidence` là `silver <tỉ lệ> (k/n)`: tỉ lệ caption train của quan hệ được parser độc lập xác nhận. Nếu có nhãn người chấm thì là `human <precision>`; không có cả hai thì là `unverified`.
* Cạnh bị loại vẫn được ghi với `status = excluded:<lý do>`.

**Mẫu đường đi.** Gồm 12 mẫu của v2 cộng `P1m_svo` và `P1m_prep`: đường đi `video → a –f[p]→ b` chỉ được tính khi truy vấn nhắc tới vị từ `p`.

### 4.8b Taxonomy (giữ nguyên cách dựng của v2)
Khái niệm được khoá theo synset WordNet (ví dụ `animal.n.01`), nên danh từ và động từ không bao giờ bị trộn với nhau. Có bốn biến thể trên **cùng** một tập node (tập node là hợp của các biến thể):

| Biến thể | Cách chọn hypernym |
|---|---|
| `heuristic` | quy tắc v1, lấy sense đầu tiên |
| `filtered` | `heuristic` bỏ các khái niệm "hub" (hơn 15% video train với tới) và khái niệm danh từ quá nông |
| `sense` | chọn sense bằng Extended Lesk trên ngữ cảnh caption **train**; mapping **mơ hồ** thì không nối `isa` và được ghi rõ |
| `none` | không có `isa` |

Mọi mapping (synset, gloss, điểm Lesk, cờ mơ hồ, hub) được xuất ra `graph_taxonomy_audit.csv`.

Ở v3, `filtered` thắng val nhưng thua trên test đã nhìn, nên **không** còn là ứng viên. Stage V4 so sánh `none` (mặc định, đơn giản hơn) với taxonomy không lọc (`heuristic`). `heuristic` chỉ được giữ khi thắng theo luật đăng ký trước. Kiểm tra mơ hồ, tương thích danh/động từ, hypernym quá chung, hub và chu trình vẫn được ghi trong audit.

### 4.8c Đồ thị v3 và hai đối chứng ngẫu nhiên có ràng buộc
**`GraphV3`** giữ topology cặp của v2, nên relation-family-only (G2) trùng với `v2_base`. Mỗi cặp (nguồn, họ, đích) mang **phân phối vị từ** `{vị từ: số video train}`.

**Chính sách cạnh:**
* `all`: như v2;
* `support`: chỉ vị từ có ≥ 5 video train;
* `agreement`: chỉ vị từ của quan hệ được parser độc lập xác nhận ở ≥ 50% caption train của nó (kiểm chứng tự động);
* `precision`: chỉ ô đã kiểm chứng bằng nhãn người (tuỳ chọn).

Một cặp không còn vị từ hợp lệ thì bị loại và lưu lại.

**Đối chứng:**
* **Rewiring** (G5): directed double-edge swap trong từng họ và nhóm kiểu node. Phép này giữ bậc vào/ra, trọng số và **đa tập phân phối vị từ**, vì vị từ đi theo cạnh. `contains` không đổi.
* **Hoán vị vị từ** (G4): hoán vị phân phối vị từ giữa các cạnh **cùng họ** svo/prep. Topology, trọng số và bậc giống hệt; chỉ việc ghép vị từ ↔ cạnh bị phá.

`path_mode` gồm `weighted`, `count`, `degnorm`, `dedup` như v2, và `pred_match` (= `weighted` cộng mẫu P1m khớp vị từ). Engine, `compute_R` và phần giải thích dùng **chung** các ma trận này.

## 4.9 Mô hình v3

**`TypedSAGEv3`.** Công thức là `h'_v = W₀h_v + Σ_t σ(g_t)·[W_t·AGG_t(h_u) + U_t·AGG_t(e_uv)]`.
* `e_uv` là biểu diễn vị từ của cạnh. Nó chỉ có ở các loại cạnh svo, svo_inv, prep, prep_inv, và chỉ khi `pred_mode ≠ none`.
* `AGG_t` dùng **cùng hệ số** cho `h_u` và `e_uv`, nên trọng số cạnh chỉ nhân một lần.
* Vị từ của agent_of/object_of chính là node động từ đích, nên không đưa vào message, tránh trùng thông tin.

**`pred_mode`:**
* `none` = relation-family-only, đúng mô hình v2;
* `clip` (phương án A) = CLIP-text **đóng băng** của vị từ theo prompt, qua một phép chiếu tuyến tính;
* `clip_adapt` (phương án B) = A cộng MLP dư **dùng chung** cho mọi vị từ, lớp cuối khởi tạo 0. Không có ID riêng cho từng vị từ, nên vị từ hiếm hoặc chưa gặp vẫn mã hoá được.

`mp_weight` (`binary` / `weighted_mean` / `weighted_symnorm`) và `init` như v2.
### 4.9b Mô hình v4 — QC-RGAT (GNN quan hệ có attention điều kiện theo truy vấn)

**Vì sao đổi mô hình.** Ở v3, truy vấn chỉ đi vào GNN dưới dạng **một vector CLIP toàn cục**. Cạnh quan hệ nối lớp với lớp và là thống kê **dùng chung** cho mọi video, nên mọi video có cùng nhãn đều nhận cùng "ngữ cảnh quan hệ". Vì vậy quan hệ chỉ làm mượt embedding lớp, và kết quả Kaggle v3 cho G3 ≈ G2 ≈ G4 ≈ G5. Ở v4, **chính truy vấn quyết định cạnh quan hệ nào được truyền tin**.

**Công thức** (T = 1 hoặc 2 lớp; chọn trên VAL):
* `H_q⁽⁰⁾(c) = E_c` và `H_q⁽ᵗ⁺¹⁾(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q⁽ᵗ⁾(u)`. Ở đây `z_{t,τ}` là toán tử của loại quan hệ τ, phép nhân theo chiều kiểu CompGCN/DistMult.
* `a_q = σ(β_τ + κ_τ·log w + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ'_τ·cos(e_p, e_P(q)))` gồm năm thành phần:
  * cổng theo loại quan hệ do truy vấn chọn (`W_g u_q`);
  * **vai** của node nguồn trong truy vấn (chủ thể / động từ / đối tượng / tân ngữ giới từ, do parser caption áp lên chính truy vấn);
  * **khớp vị từ** giữa cạnh và truy vấn, cả khớp đúng lẫn khớp mềm bằng CLIP-text;
  * support của cạnh (`κ_τ·log w`);
  * độ lệch `β_τ`.
* `s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v,c] · H_q⁽ᵀ⁾(c)⟩`. Video **chỉ nhận** qua nhãn của chính nó (inductive). Điểm cuối = `α·z(cos CLIP) + (1−α)·z(s)`, với α chọn trên VAL.

**Khởi tạo và học các cổng.** γ_τ khởi tạo bằng 1, tức prior "ưu tiên cạnh có vị từ khớp truy vấn"; dữ liệu có thể làm γ giảm. Các cổng vô hướng (β, κ, ρ, γ, γ') học với lr × `scalar_lr_mult`, như cổng của v3. Nếu khởi tạo γ = 0 và học với lr thường, γ gần như không đổi sau huấn luyện. Khi đó G2, G4, G6 cho kết quả trùng G3, và các đối chứng không còn đo được gì; lần thử dev đã gặp đúng trường hợp này.

**Giải thích từ chính cơ chế.** Attention chỉ phụ thuộc truy vấn và thuộc tính tĩnh của cạnh, nên phép truyền là **tuyến tính theo E**. Vì vậy `s` tách **chính xác** thành tổng các số hạng: thị giác + từng nhãn contains + từng **đường đi quan hệ** (`u –τ[p]→ c ←chứa– v`, có thêm một bước khi T = 2). Giải thích là top-m số hạng quan hệ. Xoá một cạnh làm điểm đổi **đúng bằng** tổng đóng góp của các đường đi qua cạnh đó (T = 1); điều này có kiểm thử Q3 và X3.

**Đối chứng tách từng thành phần** (cùng mô hình, cùng giao thức huấn luyện):
* G1 `v4_rel=False`: không cạnh quan hệ;
* G2 `v4_pred=False`: không khớp vị từ;
* G4: hoán vị vị từ giữa các cạnh cùng họ;
* G5: rewiring có ràng buộc;
* G6 `v4_qcond=False`: attention tĩnh, không theo truy vấn — tức là quan hệ như v3;
* R3: chính mô hình v3 cuối.

Liên hệ tài liệu: NBFNet (Zhu et al., NeurIPS 2021) với message điều kiện theo truy vấn; KGAT (Wang et al., KDD 2019) với attention nhận biết quan hệ; CompGCN (Vashishth et al., ICLR 2020) với toán tử quan hệ. Thuộc họ **GAT quan hệ** của cấp độ 1.


## 4.10 Engine và hai giao thức suy luận
* **strict inductive** (mặc định): video val, test và tập ngoài **chỉ nhận** message qua `contains_inv`, nên không ảnh hưởng node dùng chung hay video khác. Có unit test cho cả video test MSVD và video MSR-VTT.
* **transductive**: chỉ để báo cáo.

Engine mang thêm chỉ số vị từ của từng cạnh (`edge_pidx`) và bảng phân phối vị từ (`pmix`) cho GNN. Nó cũng có ma trận truy vấn × (vị từ, node) cho mẫu P1m. Vị từ của truy vấn được khớp khi mọi token của vị từ có mặt trong truy vấn, ví dụ `ride on` khớp "a man is riding on a horse".

## 4.11 Chỉ số đánh giá

* Mỗi **caption là một truy vấn**, có đúng một video dương. Notebook báo cáo R@1, R@5, R@10, MRR, nDCG@10, MedR và `MRR_video_macro` (macro-MRR theo video).
* Thứ hạng tính trên **toàn bộ gallery**: top-K được sắp lại, ngoài top-K giữ thứ hạng CLIP.
* Kiểm định cặp dùng **bootstrap theo cụm video**. Họ so sánh chính được hiệu chỉnh **Holm–Bonferroni**; luật chọn từng stage cũng hiệu chỉnh Holm trên các ứng viên của stage.
* Notebook báo cáo thêm: candidate recall, nhóm direct/indirect/other, khái niệm hiếm, vị từ hiếm/phổ biến/không có vị từ, số node/cạnh/bậc, thời gian chạy và bộ nhớ GPU.

## 4.12 Giải thích v3
* Đường đi được liệt kê theo **chiều** cạnh, kèm loại cạnh và **vị từ** ở bước quan hệ.
  * Mẫu P1m chỉ dùng vị từ có trong truy vấn.
  * Giải thích hợp lệ khi: cạnh có thật, đúng loại, đúng chiều, đúng kiểu node; vị từ thuộc **đúng** cạnh theo **đúng** chiều và có trong truy vấn.
  * Có đối chứng âm: sửa pred_id thì phải bị phát hiện.
* Điểm liệt kê phải khớp ma trận ở mọi `path_mode`.
* Fidelity báo riêng ba mức: thay đổi **điểm**, **thứ hạng**, **quyết định** top-1 / top-10.
* Có phần đóng góp theo mẫu đường đi và độ nhạy theo seed (Jaccard).
* Xoá cạnh giải thích được so với xoá cạnh **đối chứng** cùng loại.
* Notebook báo cáo trọng số hiệu dụng `(1−α)(1−λ)` của nhánh đường đi.

## 4.13 Khung thí nghiệm
Mỗi thí nghiệm có: `experiment_id`, stage, giả thuyết, thiết kế đầy đủ, seed, checkpoint fp16, số cạnh theo loại, số node, số tham số, (α, λ) chọn trên val, thời gian chạy, đỉnh bộ nhớ GPU, trạng thái. Thí nghiệm không chạy được ghi `NOT RUN` kèm lý do.

`decide()` áp luật đăng ký trước trên **val**; incumbent là phương án đơn giản hơn. `PRIMARY_COMPARISONS` là họ so sánh chính ở mục 4.2.

## 4.14 Self-test v3 (mục 12)
| mục 12 | kiểm tra | id |
|---|---|---|
| 1 | embedding vị từ tác động forward; pred_mode none (≡ v2) **không** bị ảnh hưởng | P1 |
| 2 | hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác | P2 |
| 3 | rewiring giữ kiểu node, họ, bậc, trọng số, đa tập vị từ | G6 |
| 4 | hoán vị vị từ không đổi topology / trọng số | G7 |
| 5 | mỗi ablation tắt đúng thành phần; inventory cố định | AB, G1 |
| 6 | trọng số chỉ nhân một lần (khớp tính tay, bất biến tỉ lệ) | G4 |
| 7 | node không có hàng xóm không sinh NaN; cô lập inductive | G4, G5 |
| 8 | không rò rỉ caption train/val/test/tập ngoài | C1, C1b, C2, C3, C8, C9 |
| 9 | giải thích đúng vị từ, đúng chiều; điểm khớp ma trận | G3, X1 |
| 10 | số trong báo cáo khớp CSV đầu ra | R1 |

Một kiểm thử phần mềm PASS **không** có nghĩa giả thuyết khoa học đã được xác nhận.

## 4.15 Level 4(b): bộ truy vấn đăng ký trước, xAI theo thứ hạng, phân tích ca, demo, kiểm kê

Các hàm dưới đây chỉ là **định nghĩa**, được gọi ở phần THỰC THI. Mọi file đầu ra sinh từ chính lần chạy; không có số liệu điền tay.

---
## 4.16 Thực thi

Thứ tự chạy:
1. Dữ liệu: MSVD và MSR-VTT → split → đặc trưng → pooling.
2. Từ vựng, vị từ, tagger, mẫu cho người chấm và nhãn.
3. Đồ thị v3, schema, audit → self-test.
4. Các stage V0–V7 trên VAL → **đóng băng**.
5. **Bộ truy vấn đăng ký trước** (trực tiếp / gián tiếp) → đánh giá test MSVD (thăm dò) và MSR-VTT fresh1k-B (xác nhận).
6. Phân tích quan hệ → xAI → metric bộ truy vấn → xAI theo thứ hạng → 5 + 5 ca → demo.
7. Xuất file → báo cáo, kết luận → kiểm kê dữ liệu, audit rò rỉ, nghiệm thu, 7 câu kết luận, manifest → nén zip.

In [ ]:
# ============ CHUYỂN PHẦN: giải phóng bộ nhớ GPU/RAM của phần A trước khi chạy phần B ============
import gc
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as _e:
    print("Bỏ qua dọn GPU:", _e)
gc.collect()
print("Phần A xong. Kết quả: /kaggle/working/results/ và /kaggle/working/output/action_genome/results/")


---
# PHẦN B — pipeline ảnh + video (Visual Genome, ARO, SugarCrepe, COCO-spatial, MSVD)


# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị (notebook final)

Notebook tự chứa, chạy một lần từ đầu đến cuối trên Kaggle (GPU T4, khoảng 3 giờ: video ~2–2,5 giờ + ảnh ~30 phút), bao phủ các yêu cầu cấp 1, 2, 3 (phần giải thích) và cấp 4 hướng (a), (b) của đề cương.
**Ngoài phạm vi notebook này:** bộ 20 truy vấn gián tiếp và MMKG tự xây (đang hoãn).

| Mục | Nội dung | Yêu cầu đề cương |
|---|---|---|
| 1–5 | Thiết lập · dữ liệu nhúng · lõi đồ thị · dữ liệu + luật chống rò rỉ · mô hình đóng băng (CLIP ViT-L/14, RelTR) | Mục 4, 6 |
| 4v | **Video** MSVD (text-to-video): notebook Level4B v5 — CLIP + PE-Core + đồ thị bằng chứng (hub, cầu nối) + Qwen2.5-VL | Cấp 4 (b) |
| 6 | **TN1** văn bản → ảnh, full-gallery: CLIP top-100 → điểm đồ thị (đường đi trên scene graph VG) → xếp hạng lại α | Cấp 1 (nhánh A) |
| 6b | **TN1b** GraphSAGE (GNN đã học) làm điểm đồ thị, cùng pipeline; ablation cạnh ngẫu nhiên / bỏ cạnh | Cấp 1 (thuật toán đã học) |
| 6c | **TN1c** SCENIR (ICML 2025), Graph Autoencoder không giám sát, cùng pipeline; ablation | Cấp 2 (phương pháp từ 2024) |
| 7 | **TN2** ARO VG-Relation + SugarCrepe: ảnh → caption đúng, scene graph RelTR từ điểm ảnh | Cấp 2 (≥ 2 tập), vai trò đồ thị |
| 7b | **TN2b** cạnh hình học suy từ box trên COCO-spatial (What'sUp) | Cấp 4 (a): tập công khai thứ hai |
| 8 | **XAI**: fidelity (xoá bằng chứng, lật ảnh), validity (khớp GT), sufficiency, sparsity + 10 ví dụ | Cấp 3 |
| 10 | Demo `search()`: nhập truy vấn → top-k ảnh + quan hệ giải thích | Mục 8 |
| 10b | 5 truy vấn thành công + 5 thất bại (văn bản → ảnh), luật chọn cố định | Cấp 1 |
| 11a | Độ nhạy α · so sánh cách tạo cạnh · xuất file split | Cấp 2 (mở rộng), mục 8 |
| 11 | Bảng tổng hợp · đối chiếu tái lập · **bảng đối chiếu yêu cầu đề cương** | — |

**Nguyên tắc chống rò rỉ:** không huấn luyện CLIP / RelTR; GNN (GraphSAGE, SCENIR) chỉ học trên ảnh không thuộc tập đánh giá; α chọn trên phần TUNE;
mọi ảnh Visual Genome thuộc VAL (RelTR chưa huấn luyện trên chúng), **không dùng heldout**; ngưỡng, lưới tham số và quy tắc quyết định đã đăng ký trước
(`v5_vgcoco_pilot/PILOT_PREREGISTRATION_VI.md`, `v6_compositional_benchmarks/PREREGISTRATION_VI.md`, `XAI_PREREGISTRATION_VI.md`,
`final_pipeline/GNN_PREREGISTRATION_VI.md`, `SCENIR_VERIFICATION_VI.md`, `LEVEL4A_PREREGISTRATION_VI.md`).

**Kaggle:** gắn datasets `dannywu375/visualgenome`, `mathurinache/visual-genome`, `clkmuhammed/microsoft-coco-2017-common-objects-in-context`; bật GPU và Internet
(cài `open_clip`, `sentence-transformers`; tải trọng số CLIP, mã + checkpoint RelTR, JSON của ARO / SugarCrepe / What'sUp).
Chạy thử không cần GPU: đặt `P3_SMOKE=1` (dữ liệu giả — không phải kết quả).

## 1. Thiết lập

In [ ]:
# ============================== 1. THIẾT LẬP: thư viện · chế độ chạy · hằng số đã đăng ký trước · tiện ích ==============================
import os, sys, json, csv, time, re, random, hashlib, subprocess, argparse, warnings, glob
from collections import Counter, defaultdict
import numpy as np
warnings.filterwarnings("ignore")

T0 = time.time()
SMOKE = os.environ.get("P3_SMOKE") == "1"            # chạy thử với dữ liệu giả + mô hình giả (KHÔNG phải kết quả)
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.environ.get("P3_OUT", "/kaggle/working/p3_out" if ON_KAGGLE else "./p3_out")
WORK = os.environ.get("P3_WORK", "/kaggle/working/p3_work" if ON_KAGGLE else "./p3_work")
FIG = f"{OUT}/figures"
for _d in (OUT, WORK, FIG): os.makedirs(_d, exist_ok=True)
_LOG = open(f"{OUT}/log.txt", "w", encoding="utf-8")

# Hằng số khoá trước khi chạy (giống hệt các bản đăng ký trước của v5, v6, v6-XAI)
SEEDS = (0, 1, 2)                     # 3 seed: lần rút caption (TN1) / cách chia TUNE-CHECK (TN2)
K_TOP = 100                           # CLIP top-K trước khi xếp hạng lại
ALPHAS = [round(i / 10, 1) for i in range(11)]          # lưới α, có α = 1 (chỉ CLIP)
W_GRID = sorted((a / 4, b / 4, (4 - a - b) / 4) for a in range(5) for b in range(5 - a))   # trọng số 3 kiểu đường đi, bước 0.25
W_OBJ = (0.0, 0.0, 1.0)
TAU = 0.05                            # độ dốc sigmoid của cạnh hình học
FLOOR = 1e-30                         # sàn log như v3
N_BOOT = 10000
SEED_RAND = 20261116
N_RAND_DEL = 10                       # số lần xoá ngẫu nhiên (đối chứng fidelity)
assert len(W_GRID) == 15 and len(ALPHAS) == 11


def elapsed():
    return f"{time.time() - T0:.0f}s"


def p3_log(*a):
    s = f"[{time.time() - T0:8.1f}s] " + " ".join(str(x) for x in a)
    print(s, flush=True); _LOG.write(s + "\n"); _LOG.flush()


def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    p3_log("$", " ".join(cmd)[:150], "->", r.returncode, (r.stderr or "")[-200:].replace("\n", " "))
    return r.returncode


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()


def stream_array(path, chunk=1 << 22):
    """đọc tuần tự một mảng JSON rất lớn (relationships.json ~743 MB) mà không nạp cả file."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = f.read(chunk); pos = 0
        while pos < len(buf) and buf[pos] in " \t\r\n﻿": pos += 1
        assert buf[pos] == "[", "top-level JSON is not an array"
        pos += 1
        while True:
            while True:
                while pos < len(buf) and buf[pos] in " \t\r\n,": pos += 1
                if pos < len(buf): break
                more = f.read(chunk)
                if not more: return
                buf = more; pos = 0
            if buf[pos] == "]": return
            while True:
                try:
                    obj, end = dec.raw_decode(buf, pos); break
                except json.JSONDecodeError:
                    more = f.read(chunk)
                    if not more: raise
                    buf = buf[pos:] + more; pos = 0
            pos = end
            yield obj


def fetch(dst, url=None, gdrive=None):
    """tải file công khai (benchmark JSON, checkpoint RelTR) một lần."""
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        if gdrive:
            try:
                import gdown  # noqa
            except ImportError:
                sh([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            sh([sys.executable, "-m", "gdown", gdrive, "-O", dst])
        else:
            sh(["wget", "-q", "-O", dst, url])
    assert os.path.exists(dst) and os.path.getsize(dst) > 0, f"tải thất bại: {url or gdrive}"
    return dst


p3_log(f"Chế độ {'SMOKE (dữ liệu giả)' if SMOKE else 'REAL'} | out={OUT}")


## 2. Dữ liệu nhúng (sinh tự động bởi `build_notebook.py`)
Danh sách id cho luật chống rò rỉ, bảng VG150, số tham chiếu của các lần chạy riêng trước (đối chiếu tái lập).

In [ ]:
# ============================== 2. DỮ LIỆU NHÚNG — nạp từ dataset hieutrinh123/project3-merged-assets ==============================
# Nội dung y hệt cell "2. DỮ LIỆU NHÚNG" gốc (danh sách id chống rò rỉ, bảng, notebook video v5, REF), chuyển ra file
# p3_embedded_data.py vì Kaggle giới hạn mã nguồn notebook 1 MB. Kiểm tra MD5 để chắc nội dung không đổi.
import glob as _g, hashlib as _h
_p = sorted(set(_g.glob("/kaggle/input/*/p3_embedded_data.py") + _g.glob("/kaggle/input/*/*/p3_embedded_data.py")
                + _g.glob("/kaggle/input/*/*/*/p3_embedded_data.py") + _g.glob("./p3_embedded_data.py")))
assert _p, "Thiếu dataset hieutrinh123/project3-merged-assets — Add Input dataset này"
_src = open(_p[0], encoding="utf-8").read()
assert _h.md5(_src.encode()).hexdigest() == "9aacd7a4b43507e253dbf50b72cfe66c", "p3_embedded_data.py khác bản gốc"
exec(compile(_src, _p[0], "exec"))


## 3. Lõi đồ thị
Scene graph là đồ thị có hướng: đỉnh = đối tượng (lớp VG150), cạnh = vị từ (và cạnh hình học suy từ box). Điểm đồ thị của một ứng viên là độ khớp của các **đường đi** từ truy vấn tới ảnh; giải thích chính là các cạnh / đỉnh đã khớp, nên giải thích đến từ cơ chế xếp hạng.

In [ ]:
# ============================== 3. LÕI ĐỒ THỊ: chuẩn hoá · scene graph · tách caption · điểm đồ thị · kết hợp · thống kê ==============================
# Hàm thuần (không đọc file, không mô hình). Gộp nguyên văn logic của g0b_core (phần dùng chung), v5_core và v6_core; chỉ đổi tên hàm trùng.

# ---- 3.1 Bảng VG150 + quan hệ vị từ (tương đương / kéo theo) ----
CLS_IDX = {c: i for i, c in enumerate(TABLES["VG_CLASSES"]) if i > 0}        # chỉ số = cột xác suất lớp của RelTR (cột 0 = 'N/A')
PRED_IDX = {p: i for i, p in enumerate(TABLES["VG_PREDS"]) if i > 0}         # chỉ số = cột xác suất vị từ của RelTR
CANON = {"wears": "wearing", "laying on": "lying on", "watching": "looking at"}   # vị từ tương đương
ENT_RAW = {"on": ["sitting on", "standing on", "lying on", "laying on", "walking on", "parked on", "mounted on", "riding"],
           "in": ["walking in"], "holding": ["carrying"]}                        # vị từ kéo theo: ảnh có p' ⇒ thoả truy vấn p
canon = lambda p: CANON.get(p, p)
ENT = {canon(p): sorted({canon(q) for q in L}) for p, L in ENT_RAW.items()}
FAM = {p: {p} | set(ENT.get(p, [])) for p in {canon(p) for p in PRED_IDX}}


def norm_name(n):
    """tên đối tượng → lớp VG150 (số nhiều đơn giản: shoes → shoe), như v3."""
    n = " ".join(str(n).lower().strip().split())
    if n in CLS_IDX: return n
    for cut in (1, 2):
        if n.endswith("s") and n[:-cut] in CLS_IDX: return n[:-cut]
    return None


def norm_pred(p):
    p = " ".join(str(p).lower().strip().split())
    return canon(p) if p in PRED_IDX else None


def nname(d):
    if d.get("name") is not None: return norm_name(d["name"])
    for n in d.get("names") or []:
        r = norm_name(n)
        if r: return r
    return None


# ---- 3.2 Split (cố định bằng sha1, tái lập được) ----
def dev_g0_split(i):          # pool G0: DEV (35%) / IV khoá (65%)
    return "DEV" if int(hashlib.sha1(("p3-dir2:" + str(i)).encode()).hexdigest(), 16) % 1000 < 350 else "IV"


def p5_split(i):              # Thí nghiệm 1: chia DEV-G0 làm đôi
    return "P-TUNE" if int(hashlib.sha1(("p5-pilot:" + str(i)).encode()).hexdigest(), 16) % 2 == 0 else "P-CHECK"


def v6_split(key, s):         # Thí nghiệm 2: chia theo ảnh, 3 seed
    return "TUNE" if int(hashlib.sha1(f"v6-{s}:{key}".encode()).hexdigest(), 16) % 2 == 0 else "CHECK"


class IdGuard:
    """Hàng rào rò rỉ: chỉ id thuộc tập cho phép mới được đọc / nạp ảnh."""

    def __init__(self, allowed, name):
        self.allowed, self.name, self.n_checked = {int(i) for i in allowed}, name, 0

    def check(self, ids, purpose=""):
        ids = [int(i) for i in ids]
        bad = [i for i in ids if i not in self.allowed]
        if bad: raise PermissionError(f"GUARD[{self.name}]: {len(bad)} id ngoài tập cho phép ({purpose}): {bad[:5]}")
        self.n_checked += len(ids)


# ---- 3.3 Scene graph GT của một ảnh (đỉnh = lớp, cạnh = bộ ba VG150) ----
class Img:
    __slots__ = ("id", "triples", "between", "classes")

    def __init__(s, i):
        s.id = i; s.triples = set(); s.between = defaultdict(list); s.classes = set()


def build_image(i, rel_entries, obj_entries, stats):
    """Dựng scene graph từ annotation VG: bỏ quan hệ trùng, self-loop, thiếu object_id; chuẩn hoá về VG150."""
    im = Img(i)
    for e in obj_entries:
        for o in e.get("objects", []):
            c = nname(o)
            if c: im.classes.add(c)
    seen_rid, seen_key = set(), set()
    for rec in rel_entries:
        for r in rec.get("relationships", []):
            stats["rel_total"] += 1
            rid = r.get("relationship_id")
            if rid in seen_rid: stats["rel_dup_relationship_id"] += 1; continue
            seen_rid.add(rid)
            su, ob = r.get("subject") or {}, r.get("object") or {}
            sid, oid = su.get("object_id", su.get("id")), ob.get("object_id", ob.get("id"))
            if sid is None or oid is None: stats["rel_missing_object_id"] += 1; continue
            if int(sid) == int(oid): stats["rel_self_loop"] += 1; continue
            p_raw = " ".join(str(r.get("predicate", "")).lower().split())
            key = (int(sid), p_raw, int(oid))
            if key in seen_key: stats["rel_duplicate_same_triple_ids"] += 1; continue
            seen_key.add(key)
            s150, o150 = nname(su), nname(ob)
            if s150 and o150:
                im.classes.add(s150); im.classes.add(o150)
                im.between[(s150, o150)].append(p_raw)
                p150 = norm_pred(p_raw)
                if p150: im.triples.add((s150, p150, o150)); stats["edges_vg150"] += 1
    return im


# ---- 3.4 Tách caption thành thực thể + bộ ba (luật, tất định) ----
PRED_PHRASES = {"riding": "riding", "rides": "riding", "ride": "riding", "holding": "holding", "holds": "holding",
                "sitting on": "sitting on", "sits on": "sitting on", "standing on": "standing on", "stands on": "standing on",
                "wearing": "wearing", "wears": "wearing", "next to": "near", "beside": "near", "near": "near",
                "on top of": "on", "on": "on", "in": "in", "inside": "in", "in front of": "in front of", "behind": "behind",
                "under": "under", "underneath": "under", "with": "with", "has": "has", "have": "has", "eating": "eating", "eats": "eating",
                "playing": "playing", "plays": "playing", "looking at": "looking at", "looks at": "looking at",
                "carrying": "carrying", "carries": "carrying", "walking on": "walking on", "walking in": "walking in",
                "laying on": "lying on", "lying on": "lying on", "parked on": "parked on", "hanging from": "hanging from",
                "flying in": "flying in", "covered in": "covered in", "above": "above", "over": "above"}


def _spans(tokens, lookup, maxn=3):
    out, i = [], 0
    while i < len(tokens):
        hit = None
        for n in range(min(maxn, len(tokens) - i), 0, -1):
            v = lookup(" ".join(tokens[i:i + n]))
            if v is not None: hit = (i, i + n, v); break
        if hit: out.append(hit); i = hit[1]
        else: i += 1
    return out


def parse_caption(cap):
    """→ (E: tập lớp, T: tập bộ ba). subject = thực thể gần nhất bên trái vị từ, object = gần nhất bên phải."""
    toks = re.findall(r"[a-z0-9]+", str(cap).lower())
    ents, preds = _spans(toks, norm_name), _spans(toks, PRED_PHRASES.get)
    T = set()
    for (a, b, p) in preds:
        left, right = [e for e in ents if e[1] <= a], [e for e in ents if e[0] >= b]
        if left and right:
            T.add((max(left, key=lambda e: e[1])[2], p, min(right, key=lambda e: e[0])[2]))
    return frozenset(e[2] for e in ents), frozenset(T)


# ---- 3.5 Điểm đồ thị trên scene graph GT (Thí nghiệm 1): ba kiểu đường đi truy vấn → ảnh ----
def match_detail(Tq, Eq, im):
    """s_rel: bộ ba khớp cạnh (cho phép kéo theo) · s_pair: cặp lớp nối bởi cạnh bất kỳ · s_obj: lớp có mặt. Kèm bằng chứng (giải thích)."""
    edges, pairs = [], []
    for (s, p, o) in sorted(Tq):
        for q in sorted(FAM.get(p, {p})):
            if (s, q, o) in im.triples: edges.append(f"{s}|{p}|{o}<={s}|{q}|{o}"); break
        if (s, o) in im.between or (o, s) in im.between: pairs.append(f"{s}|{o}")
    nodes = sorted(c for c in Eq if c in im.classes)
    nt = len(Tq)
    return dict(s_rel=len(edges) / nt if nt else 0.0, s_pair=len(pairs) / nt if nt else 0.0,
                s_obj=len(nodes) / len(Eq) if Eq else 0.0, edges=edges, pairs=pairs, nodes=nodes)


def candidate_components(parsed, topk, graphs):
    Q, K = topk.shape
    out = np.zeros((Q, K, 3))
    for qi, (Eq, Tq) in enumerate(parsed):
        if not Eq and not Tq: continue
        for kj in range(K):
            d = match_detail(Tq, Eq, graphs[topk[qi, kj]])
            out[qi, kj] = (d["s_rel"], d["s_pair"], d["s_obj"])
    return out


def rand_permutation(n, seed=SEED_RAND):
    """hoán vị cố định không có điểm bất động (đối chứng 'đồ thị của ảnh khác')."""
    order = list(range(n)); random.Random(seed).shuffle(order)
    perm = [0] * n
    for pos, j in enumerate(order): perm[j] = order[(pos + 1) % n]
    return perm


def shuffle_predicates(im):
    e = sorted(im.triples); preds = [p for (_, p, _) in e]
    random.Random(int(im.id)).shuffle(preds)
    new = Img(im.id); new.triples = {(s, p, o) for (s, _, o), p in zip(e, preds)}; new.between = im.between; new.classes = im.classes
    return new


# ---- 3.6 Kết hợp CLIP + đồ thị và xếp hạng (Thí nghiệm 1) ----
def zrows(x):
    mu, sd = x.mean(1, keepdims=True), x.std(1, keepdims=True)
    return np.where(sd > 0, (x - mu) / np.where(sd > 0, sd, 1), 0.0)


def fuse_scores(zc, comps, w, alpha):
    g = comps @ np.asarray(w, dtype=np.float64)
    flat = g.max(1) == g.min(1)                  # điểm đồ thị bằng nhau ⇒ giữ thứ tự CLIP
    zg = zrows(g); zg[flat] = 0.0
    return alpha * zc + (1 - alpha) * zg


def rank_of_relevant(scores, rel_pos, clip_rank):
    Q, K = scores.shape
    pos = np.arange(K)[None, :]; rp = np.where(rel_pos >= 0, rel_pos, 0)
    sr = scores[np.arange(Q), rp][:, None]
    better = (scores > sr).sum(1) + ((scores == sr) & (pos < rp[:, None])).sum(1)
    return np.where(rel_pos >= 0, 1 + better, clip_rank).astype(np.int64)


def select_params(grid_mrr):
    return min(grid_mrr, key=lambda k: (-round(grid_mrr[k], 12), -k[1], k[0]))


def tune_grid(zc, comps, rel_pos, clip_rank, w_list, alphas):
    return {(tuple(w), a): float(np.mean(1.0 / rank_of_relevant(fuse_scores(zc, comps, w, a), rel_pos, clip_rank)))
            for w in w_list for a in alphas}


def clip_topk(cos, k):
    Q, N = cos.shape; K = min(k, N)
    order = np.argsort(-cos, axis=1, kind="stable"); topk = order[:, :K]
    pos_rel = np.array([int(np.where(order[i] == i)[0][0]) for i in range(Q)])
    return topk, np.take_along_axis(cos, topk, 1), np.where(pos_rel < K, pos_rel, -1), pos_rel + 1


def metrics_from_ranks(r):
    r = np.asarray(r, dtype=np.float64)
    return dict(R1=float((r <= 1).mean()), R5=float((r <= 5).mean()), R10=float((r <= 10).mean()), MRR=float((1 / r).mean()),
                nDCG10=float(np.where(r <= 10, 1 / np.log2(r + 1), 0.0).mean()))


def bootstrap_delta(d_by_name, n_boot=N_BOOT, seed=SEED_RAND, chunk=1000):
    names = list(d_by_name); n = len(d_by_name[names[0]])
    rng = np.random.default_rng(seed); means = {k: [] for k in names}; done = 0
    while done < n_boot:
        b = min(chunk, n_boot - done); idx = rng.integers(0, n, size=(b, n))
        for k in names: means[k].append(np.asarray(d_by_name[k])[idx].mean(1))
        done += b
    return {k: dict(mean=float(np.mean(d_by_name[k])), lo=float(np.percentile(np.concatenate(means[k]), 2.5)),
                    hi=float(np.percentile(np.concatenate(means[k]), 97.5))) for k in names}


def mean_std(xs):
    a = np.asarray(xs, dtype=np.float64)
    return dict(mean=float(a.mean()), std=float(a.std(ddof=1)) if len(a) > 1 else 0.0, per_seed=[float(x) for x in a])


# ---- 3.7 Điểm đồ thị trên scene graph RelTR (Thí nghiệm 2, 3): cạnh vị từ + cạnh hình học ----
ARO_RX = re.compile(r"^the (.+?) is (.+) the (.+)$")
GEO = {"to the left of": "L", "to the right of": "R"}
ARO_PRED = {"on": "on", "on top of": "on", "wearing": "wearing", "in": "in", "inside": "in", "behind": "behind",
            "in front of": "in front of", "near": "near", "next to": "near", "beside": "near", "by": "near", "of": "of",
            "sitting on": "sitting on", "above": "above", "below": "under", "under": "under", "underneath": "under",
            "holding": "holding", "at": "at", "with": "with", "riding": "riding", "standing on": "standing on", "eating": "eating",
            "walking on": "walking on", "parked on": "parked on", "watching": "looking at", "looking at": "looking at",
            "lying on": "lying on", "walking in": "walking in", "covering": "covering"}


def aro_parse(cap):
    m = ARO_RX.match(cap)
    if not m: return (None, "other", None, None)
    X, r, Y = norm_name(m.group(1)), m.group(2), norm_name(m.group(3))
    if not X or not Y: return (None, "other", r, None)
    if r in GEO: return (X, "geo", GEO[r], Y)
    if r in ARO_PRED: return (X, "pred", ARO_PRED[r], Y)
    return (None, "other", r, None)


def aro_subset(rel_name):
    return "geo" if rel_name in GEO else ("pred" if rel_name in ARO_PRED else "other")


def rel_score(F, X, p, Y, pr=None):
    """cạnh vị từ: log max_j P(sub=X) · P(pred=p) · P(obj=Y) trên 200 truy vấn bộ ba của RelTR (điểm 'rel' của v3)."""
    pr = F["pr"] if pr is None else pr
    v = F["ps"][:, CLS_IDX[X]].astype(np.float64) * pr[:, PRED_IDX[p]].astype(np.float64) * F["po"][:, CLS_IDX[Y]].astype(np.float64)
    j = int(np.argmax(v))
    return float(np.log(max(v[j], FLOOR))), dict(kind="pred", j=j, p=float(v[j]), sub_box=[float(x) for x in F["sb"][j]],
                                                 obj_box=[float(x) for x in F["ob"][j]])


def geo_score(F, X, Y, d, nogeo=False):
    """cạnh hình học: max_{u≠v} P(u=X) · P(v=Y) · σ((cx_v − cx_u)/τ) trên 400 hộp đối tượng (subject + object của 200 truy vấn)."""
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    a, b, cx = P[:, CLS_IDX[X]], P[:, CLS_IDX[Y]], B[:, 0]
    if nogeo: s = np.ones((len(cx), len(cx)))
    else:
        diff = (cx[None, :] - cx[:, None]) if d == "L" else (cx[:, None] - cx[None, :])
        s = 1.0 / (1.0 + np.exp(-diff / TAU))
    M = a[:, None] * b[None, :] * s; np.fill_diagonal(M, 0.0)
    u, v = np.unravel_index(int(np.argmax(M)), M.shape)
    return float(np.log(max(M[u, v], FLOOR))), dict(kind="geo", u=int(u), v=int(v), p=float(M[u, v]),
                                                    x_box=[float(x) for x in B[u]], y_box=[float(x) for x in B[v]])


def aro_graph_score(F, parsed, pr=None, nogeo=False):
    X, kind, r, Y = parsed
    if kind == "geo": return geo_score(F, X, Y, r, nogeo=nogeo)
    if kind == "pred": return rel_score(F, X, r, Y, pr=pr)
    return None, None


def triples_graph_score(F, T, pr=None):
    if not T: return None, None
    vals, ev = [], []
    for (s, p, o) in sorted(T):
        if p not in PRED_IDX: return None, None
        v, e = rel_score(F, s, p, o, pr=pr); vals.append(v); ev.append(dict(triple=f"{s}|{p}|{o}", **e))
    return float(np.mean(vals)), ev


def shuffled_pr(F, key):
    perm = np.arange(F["pr"].shape[1])
    perm[1:] = 1 + np.random.RandomState(int(key) % (2 ** 31)).permutation(F["pr"].shape[1] - 1)
    return F["pr"][:, perm]


def correct(delta):
    d = np.asarray(delta, dtype=np.float64)
    return np.where(d > 0, 1.0, np.where(d == 0, 0.5, 0.0))


def fuse2(dc, dg, valid, alpha, sc, sg):
    dc, dg = np.asarray(dc, float), np.asarray(dg, float)
    g = np.where(valid, dg, 0.0) / sg if sg > 0 else np.zeros_like(dg)
    return np.where(valid, alpha * dc / sc + (1 - alpha) * g, dc / sc)


def tune_alpha(dc, dg, valid):
    sc = float(np.std(dc)) or 1.0
    sg = (float(np.std(np.asarray(dg)[valid])) if np.any(valid) else 1.0) or 1.0
    acc = {a: float(correct(fuse2(dc, dg, valid, a, sc, sg)).mean()) for a in ALPHAS}
    return min(acc, key=lambda a: (-round(acc[a], 12), -a)), sc, sg, acc


def bootstrap_diff_by_cluster(x, y, clusters, n_boot=N_BOOT, seed=SEED_RAND):
    x, y = np.asarray(x, float), np.asarray(y, float)
    keys, inv = np.unique(np.asarray(clusters).astype(str), return_inverse=True); K = len(keys)
    sx, sy, cnt = np.bincount(inv, x, K), np.bincount(inv, y, K), np.bincount(inv, minlength=K).astype(float)
    rng = np.random.default_rng(seed); out = np.empty(n_boot)
    for b0 in range(0, n_boot, 1000):
        b = min(1000, n_boot - b0); idx = rng.integers(0, K, size=(b, K))
        out[b0:b0 + b] = (sx[idx].sum(1) - sy[idx].sum(1)) / cnt[idx].sum(1)
    return dict(mean=float(x.mean() - y.mean()), lo=float(np.percentile(out, 2.5)), hi=float(np.percentile(out, 97.5)))


print(f"Lõi đồ thị sẵn sàng: {len(CLS_IDX)} lớp, {len(PRED_IDX)} vị từ VG150 | {elapsed()}")


## 4. Dữ liệu và luật chống rò rỉ

In [ ]:
# ============================== 4. DỮ LIỆU: tìm input · tải benchmark · lọc chống rò rỉ · scene graph GT · caption COCO ==============================
# Luật rò rỉ (khoá trước):
#  - Mọi ảnh Visual Genome dùng tới phải thuộc VAL của v3 (chỉ số lọc ≥ 83000 ⇒ RelTR chưa huấn luyện trên chúng; heldout KHÔNG dùng).
#  - TN1 chỉ dùng DEV-G0: VAL trừ ảnh benchmark v3, cache v4 và 99 ảnh Phase 5 (danh sách G0, kiểm MD5).
#  - SugarCrepe: ảnh COCO không thuộc VG, hoặc thuộc VG và nằm trong VAL.
assert hashlib.md5(",".join(map(str, VAL_IDS)).encode()).hexdigest() == VAL_MD5 and len(VAL_IDS) == 15128
assert hashlib.md5(",".join(map(str, G0_IDS)).encode()).hexdigest() == G0_MD5 and len(G0_IDS) == 13586
VALSET = set(VAL_IDS)
DEV_G0 = sorted(i for i in G0_IDS if dev_g0_split(i) == "DEV")
assert len(DEV_G0) == 4752 and set(DEV_G0) <= VALSET
GUARD_VAL = IdGuard(VAL_IDS, "VAL")
GUARD_DEV = IdGuard(DEV_G0, "DEV-G0")
p3_log(f"VAL {len(VAL_IDS)} ảnh | DEV-G0 {len(DEV_G0)} ảnh (TN1)")


def find_inputs():
    if os.environ.get("P3_PATHS"): return json.loads(os.environ["P3_PATHS"])
    found, vgdirs, cocoval = defaultdict(list), [], []
    for dp, dn, fn in os.walk("/kaggle/input", followlinks=True):
        b = os.path.basename(dp)
        if b.startswith("VG_100K"):
            if "dannywu375" in dp or "/input/visualgenome/" in dp: vgdirs.append(dp)
            dn[:] = []; continue
        if b == "val2017" and any(f.endswith(".jpg") for f in fn[:50]):
            cocoval.append(dp); dn[:] = []; continue
        if b in ("train2017", "test2017", "unlabeled2017"):
            dn[:] = []; continue
        for f in fn:
            if f in ("image_data.json", "relationships.json", "objects.json", "captions_train2017.json", "captions_val2017.json"):
                found[f].append(os.path.join(dp, f))
    # merged: Kaggle có thể gắn dataset ở /kaggle/input/<owner>/<slug>/ hoặc /kaggle/input/<slug>/ (không có tên chủ sở hữu)
    pick = lambda name, slug: sorted(p for p in found[name] if f"/{slug}/" in p or f"/input/{slug.split('/')[-1]}/" in p)
    r = dict(image_data=pick("image_data.json", "dannywu375/visualgenome"), rel=pick("relationships.json", "mathurinache/visual-genome"),
             obj_dw=pick("objects.json", "dannywu375/visualgenome"), cap_train=sorted(found["captions_train2017.json"]),
             cap_val=sorted(found["captions_val2017.json"]))
    for k, v in r.items(): assert v, f"thiếu input {k} — đã thấy: { {kk: vv[:3] for kk, vv in found.items()} }"
    r = {k: v[0] for k, v in r.items()}
    r.update(vg_dirs=sorted(vgdirs), coco_val=sorted(cocoval)[0])
    return r


P = find_inputs()
SHA = {k: sha256(P[k]) for k in ("image_data", "rel", "obj_dw")}
if not SMOKE:
    for k, v in G0_SHA.items(): assert SHA[k] == v, f"input {k} khác manifest G0"
p3_log("Input VG khớp SHA-256 của G0" if not SMOKE else "SMOKE: bỏ kiểm SHA", "|", {k: P[k] for k in ("image_data", "rel", "obj_dw")})

# ---- benchmark công khai (tải lúc chạy) ----
if SMOKE:
    BENCH = json.loads(os.environ["P3_BENCH"])
else:
    BENCH = dict(aro=fetch(f"{WORK}/visual_genome_relation.json", gdrive=ARO_GDRIVE),
                 replace_rel=fetch(f"{WORK}/replace_rel.json", url=SC_URL + "replace_rel.json"),
                 swap_obj=fetch(f"{WORK}/swap_obj.json", url=SC_URL + "swap_obj.json"))
BSHA = {k: sha256(v) for k, v in BENCH.items()}
p3_log("Benchmark:", BSHA)

# ---- image_data: ánh xạ COCO ↔ VG; chỉ mục file ảnh VG thuộc VAL ----
COCO_OF_VG, VG_OF_COCO = {}, {}
for e in json.load(open(P["image_data"], encoding="utf-8")):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if e.get("coco_id") is not None:
        VG_OF_COCO[int(e["coco_id"])] = i
        if i in VALSET: COCO_OF_VG[i] = int(e["coco_id"])
VG_JPG = {}
for d in P["vg_dirs"]:
    for f in os.listdir(d):
        if f.endswith(".jpg") and f[:-4].isdigit() and int(f[:-4]) in VALSET: VG_JPG.setdefault(int(f[:-4]), os.path.join(d, f))
p3_log(f"Ảnh VG thuộc VAL có file: {len(VG_JPG)}")

# ---- mục benchmark, lọc rò rỉ TRƯỚC khi mở ảnh ----
ITEMS2, EXCL2 = [], Counter()
for k, x in enumerate(json.load(open(BENCH["aro"], encoding="utf-8"))):
    vid = int(x["image_id"])
    if vid not in VALSET: EXCL2["aro_not_VAL"] += 1; continue
    if vid not in VG_JPG: EXCL2["aro_no_jpg"] += 1; continue
    box = (int(x["bbox_x"]), int(x["bbox_y"]), int(x["bbox_x"]) + int(x["bbox_w"]), int(x["bbox_y"]) + int(x["bbox_h"]))
    ITEMS2.append(dict(bench="aro", item=f"aro{k}", key=str(vid), vg_id=vid, path=VG_JPG[vid], crop=box, true=x["true_caption"],
                       false=x["false_caption"], subset=aro_subset(x["relation_name"]), rel=x["relation_name"],
                       xid=int(x["primary_object_id"]), yid=int(x["relation_info"]["object"])))
for b in ("replace_rel", "swap_obj"):
    for k, x in json.load(open(BENCH[b], encoding="utf-8")).items():
        coco = int(x["filename"].split(".")[0]); vid = VG_OF_COCO.get(coco)
        if vid is not None and vid not in VALSET: EXCL2[f"{b}_vg_not_VAL"] += 1; continue
        path = os.path.join(P["coco_val"], x["filename"])
        if not os.path.exists(path): EXCL2[f"{b}_no_jpg"] += 1; continue
        ITEMS2.append(dict(bench=b, item=f"{b}{k}", key=f"coco{coco}", vg_id=vid, path=path, crop=None, true=x["caption"],
                           false=x["negative_caption"], subset="vg_VAL" if vid else "not_in_VG", rel="", xid=None, yid=None))
GUARD_VAL.check([it["vg_id"] for it in ITEMS2 if it["vg_id"] is not None], "benchmark items")
p3_log("Mục benchmark:", dict(Counter(i["bench"] for i in ITEMS2)), "| loại:", dict(EXCL2))

# ---- annotation VG: chỉ cho ảnh cần (DEV-G0 cho TN1; ảnh ARO cho box GT của XAI) ----
NEED_REL = set(DEV_G0)
NEED_OBJ = set(DEV_G0) | {it["vg_id"] for it in ITEMS2 if it["bench"] == "aro"}
GUARD_VAL.check(NEED_OBJ, "annotation")
REL, OBJ = defaultdict(list), defaultdict(list)
for e in stream_array(P["rel"]):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if i in NEED_REL: REL[i].append(e)                 # mọi ảnh khác bị bỏ ngay, không lưu, không đếm
for e in stream_array(P["obj_dw"]):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if i in NEED_OBJ: OBJ[i].append(e)
GTBOX = {int(o["object_id"]): (int(o["x"]), int(o["y"]), int(o["x"]) + int(o["w"]), int(o["y"]) + int(o["h"]))
         for i in NEED_OBJ for e in OBJ.get(i, []) for o in e.get("objects", [])}
p3_log(f"Annotation: relationships {len(REL)} ảnh, objects {len(OBJ)} ảnh, {len(GTBOX)} hộp GT | {elapsed()}")

# ---- caption COCO cho TN1 ----
CAPS = defaultdict(list)
for k in ("cap_train", "cap_val"):
    for a in json.load(open(P[k], encoding="utf-8"))["annotations"]:
        CAPS[int(a["image_id"])].append((int(a["id"]), " ".join(str(a["caption"]).split())))
for v in CAPS.values(): v.sort()
EXCL1 = Counter()
ELIG1 = []
for i in DEV_G0:
    if i not in COCO_OF_VG: continue
    if len(CAPS.get(COCO_OF_VG[i], [])) < 3: EXCL1["lt3_captions"] += 1; continue
    if i not in VG_JPG: EXCL1["no_jpg"] += 1; continue
    ELIG1.append(i)
GUARD_DEV.check(ELIG1, "TN1 ảnh")
p3_log(f"TN1: {len(ELIG1)} ảnh DEV-G0 có caption COCO | loại {dict(EXCL1)} | {elapsed()}")


## 4v. Cấp 4 (b) — Mở rộng video MSVD (text-to-video)
Chạy nguyên vẹn notebook video Level4B v5 "EvidenceGraph" (đã chạy trên Kaggle, mã trùng khớp) trong tiến trình riêng, **trước** phần ảnh để GPU còn trống cho Qwen2.5-VL. Kết quả được đưa vào bảng tổng hợp ở mục 11. Khoảng 2–2,5 giờ.

## 4u. Phần C — Level 4(b) v2-graphfix (text → video, MSVD + MSR-VTT)
Notebook `lamvo30ai/project03-level4b-v2-graphfix` (version 1) được **nhúng nguyên văn** và chạy trong tiến trình riêng (papermill), giống cách mục 4v chạy notebook video v5, để không đụng biến của notebook này. Nó dựng đồ thị v3 có vị từ, TypedSAGE / QC-RGAT, tagger học từ caption train, cạnh ConceptNet, xAI theo thứ hạng; và tạo **cache đặc trưng frame CLIP** của MSVD / MSR-VTT mà notebook video v5 (mục 4v) dùng lại.

Chế độ chạy (`P3_RUN_L4B_V2`): `auto` (mặc định) — nếu output của notebook này đã được Add Input (thấy cache `frames_msvd_*.npz`) thì **không chạy lại**, chỉ chép kết quả; nếu không thấy thì chạy đầy đủ (~2,4 giờ trên T4). `1` = luôn chạy, `0` = không chạy.


In [ ]:
# ============================== 4u. PHẦN C — Level 4(b) v2-graphfix: nhúng nguyên văn, chạy trong tiến trình riêng ==============================
# Phải chạy TRƯỚC mục 4v: notebook video v5 dùng lại cache đặc trưng frame mà notebook này tạo ra (qua MSVD_L4_CACHE_DIRS).
_p2 = sorted(set(glob.glob("/kaggle/input/*/project03-level4b-v2-graphfix.ipynb") + glob.glob("/kaggle/input/*/*/project03-level4b-v2-graphfix.ipynb")
                 + glob.glob("/kaggle/input/*/*/*/project03-level4b-v2-graphfix.ipynb")))
L4B_V2_NB_JSON = open(_p2[0], encoding="utf-8").read() if _p2 else ""   # notebook gốc, nạp từ dataset hieutrinh123/project3-merged-assets (giới hạn 1 MB)
L4B_V2_NB_MD5 = 'f4b127817568994b192bbd399f9fc44b'
L4B_V2_DIR = "/kaggle/working/project03_level4b_v2" if ON_KAGGLE else f"{WORK}/project03_level4b_v2"
L4B_V2_RES = f"{OUT}/video_v2"
os.makedirs(L4B_V2_RES, exist_ok=True)
RES_L4B_V2 = dict(status="not run")
_t_v2 = time.time()


def _copy_v2_results(src_dir):
    n = 0
    for f in glob.glob(f"{src_dir}/*.csv") + glob.glob(f"{src_dir}/*.json") + glob.glob(f"{src_dir}/*.md"):
        try:
            open(f"{L4B_V2_RES}/{os.path.basename(f)}", "wb").write(open(f, "rb").read()); n += 1
        except Exception:
            pass
    return n


try:
    _mode = os.environ.get("P3_RUN_L4B_V2", "auto")
    # chỉ dò vài mức thư mục cố định (không rglob cả /kaggle/input — COCO/VG có hàng trăm nghìn file)
    _cached = sorted(set(glob.glob("/kaggle/input/*/project03_level4b_v2/cache/frames_msvd_*.npz") +
                         glob.glob("/kaggle/input/*/cache/frames_msvd_*.npz") +
                         glob.glob("/kaggle/input/*/*/project03_level4b_v2/cache/frames_msvd_*.npz"))) if ON_KAGGLE else []
    if SMOKE:
        RES_L4B_V2 = dict(status="skipped (SMOKE)")
    elif _mode == "0" or (_mode == "auto" and _cached):
        _root = os.path.dirname(os.path.dirname(_cached[0])) if _cached else None
        _n = _copy_v2_results(f"{_root}/results") if _root else 0
        RES_L4B_V2 = dict(status="reused" if _cached else "skipped", cache=_cached[:3], results_copied=_n, source=_root)
        p3_log("Phần C: không chạy lại —", ("dùng output đã Add Input: " + _root) if _root else "P3_RUN_L4B_V2=0", f"| chép {_n} file kết quả")
    else:
        _nb_in = f"{WORK}/level4b_v2_graphfix_input.ipynb"
        _nb_out = f"{L4B_V2_RES}/project03_level4b_v2_graphfix_executed.ipynb"
        assert hashlib.md5(L4B_V2_NB_JSON.encode()).hexdigest() == L4B_V2_NB_MD5, "notebook v2-graphfix nhúng bị thay đổi"
        open(_nb_in, "w", encoding="utf-8").write(L4B_V2_NB_JSON)
        try:
            import papermill as pm
        except ImportError:
            sh([sys.executable, "-m", "pip", "install", "-q", "papermill"]); import papermill as pm
        _old_out = os.environ.get("MSVD_L4_OUT")
        os.environ["MSVD_L4_OUT"] = L4B_V2_DIR          # chỉ cho tiến trình v2; khôi phục ngay sau đó để 4v (v5) dùng thư mục của nó
        try:
            p3_log("Phần C: bắt đầu chạy notebook Level 4(b) v2-graphfix (tiến trình riêng)…")
            pm.execute_notebook(_nb_in, _nb_out, cwd="/kaggle/working" if ON_KAGGLE else WORK,
                                kernel_name="python3", log_output=True, progress_bar=False)
        finally:
            if _old_out is None:
                os.environ.pop("MSVD_L4_OUT", None)
            else:
                os.environ["MSVD_L4_OUT"] = _old_out
        _n = _copy_v2_results(f"{L4B_V2_DIR}/results")
        RES_L4B_V2 = dict(status="ok", out_dir=L4B_V2_DIR, results_copied=_n)
    # cache của Phần C → notebook video v5 (mục 4v) tìm thấy qua MSVD_L4_CACHE_DIRS
    _cdirs = [d for d in os.environ.get("MSVD_L4_CACHE_DIRS", "").split(":") if d]
    for _d in ([f"{L4B_V2_DIR}/cache"] + [os.path.dirname(p) for p in _cached[:1]]):
        if os.path.isdir(_d) and _d not in _cdirs:
            _cdirs.append(_d)
    os.environ["MSVD_L4_CACHE_DIRS"] = ":".join(_cdirs)
    RES_L4B_V2["cache_dirs"] = _cdirs
except Exception as ex:
    RES_L4B_V2 = dict(status="error", error=repr(ex)[:1500])
    p3_log("Phần C: LỖI — các phần sau vẫn chạy tiếp:", repr(ex)[:300])
RES_L4B_V2["minutes"] = round((time.time() - _t_v2) / 60, 1)
p3_log("Phần C:", {k: v for k, v in RES_L4B_V2.items() if k != "error"})


In [ ]:
# ============================== 4v. CẤP 4 (b) — MỞ RỘNG VIDEO (text-to-video, MSVD): chạy nguyên vẹn notebook video v5 đã kiểm chứng ==============================
# Notebook video Level4B v5 "EvidenceGraph" (project/code/project03_level4b/video_v5/, đã chạy trên Kaggle — mã trùng khớp) được NHÚNG nguyên văn
# và chạy trong một tiến trình riêng (papermill) để không đụng biến của notebook này. Chạy TRƯỚC phần ảnh để GPU còn trống cho Qwen2.5-VL-7B.
# Ô này KHÔNG import torch (không khởi tạo CUDA ở tiến trình chính). Lỗi ở nhánh video được ghi lại, phần ảnh vẫn chạy tiếp.
VIDEO_DIR = f"{OUT}/video"
os.makedirs(VIDEO_DIR, exist_ok=True)
RES_VIDEO = dict(status="not run")
_t_video = time.time()
try:
    _nb_in = f"{WORK}/video_v5_input.ipynb"
    _nb_out = f"{VIDEO_DIR}/Project03_Level4B_v5_EvidenceGraph_executed.ipynb"
    if SMOKE:   # chạy thử: notebook giả chỉ ghi một summary giả đúng cấu trúc (KHÔNG phải kết quả MSVD)
        _fake = {"cells": [{"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
            "import json, os\n", f"d = {VIDEO_DIR!r} + '/fake_run/results'\n", "os.makedirs(d, exist_ok=True)\n",
            "json.dump({'test': [{'model': 'M0_clip', 'set': 'test_all', 'MRR': 0.5, 'R@1': 0.4, 'R@5': 0.7, 'R@10': 0.8, 'n_queries': 10},"
            "{'model': 'M4_full_graph', 'set': 'test_all', 'MRR': 0.6, 'R@1': 0.5, 'R@5': 0.8, 'R@10': 0.9, 'n_queries': 10}],"
            "'comparisons': [], 'fidelity': {}, 'test_looks_msvd': 'smoke'}, open(d + '/v5_summary.json', 'w'))\n"]}],
            "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}}, "nbformat": 4, "nbformat_minor": 5}
        json.dump(_fake, open(_nb_in, "w"))
        _cwd = VIDEO_DIR
    else:
        open(_nb_in, "w", encoding="utf-8").write(VIDEO_NB_JSON)
        assert hashlib.md5(VIDEO_NB_JSON.encode()).hexdigest() == VIDEO_NB_MD5, "notebook video nhúng bị thay đổi"
        _cwd = "/kaggle/working" if ON_KAGGLE else VIDEO_DIR
    try:
        import papermill as pm
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "papermill"]); import papermill as pm
    p3_log("Video: bắt đầu chạy notebook video v5 (tiến trình riêng)…")
    pm.execute_notebook(_nb_in, _nb_out, cwd=_cwd, kernel_name="python3", log_output=True, progress_bar=False)
    _hits = sorted(glob.glob(f"{_cwd}/**/results/v5_summary.json", recursive=True), key=os.path.getmtime)
    assert _hits, "không thấy v5_summary.json sau khi chạy notebook video"
    _vs = json.load(open(_hits[-1], encoding="utf-8"))
    _res_dir = os.path.dirname(_hits[-1])
    for f in glob.glob(f"{_res_dir}/*.csv") + glob.glob(f"{_res_dir}/*.json"):
        try:
            open(f"{VIDEO_DIR}/{os.path.basename(f)}", "wb").write(open(f, "rb").read())
        except Exception:
            pass
    RES_VIDEO = dict(status="ok", summary_path=_hits[-1], test=_vs.get("test", []), comparisons=_vs.get("comparisons", []),
                     fidelity=_vs.get("fidelity", {}), test_looks_msvd=_vs.get("test_looks_msvd"), vlm=_vs.get("vlm", {}))
except Exception as ex:
    RES_VIDEO = dict(status="error", error=repr(ex)[:1500])
    p3_log("Video: LỖI — phần ảnh vẫn chạy tiếp:", repr(ex)[:300])
RES_VIDEO["minutes"] = round((time.time() - _t_video) / 60, 1)
if RES_VIDEO["status"] == "ok":
    _tv = {r["model"]: r for r in RES_VIDEO["test"] if r.get("set") == "test_all"}
    p3_log("Video xong:", {k: round(v["MRR"], 4) for k, v in _tv.items()}, f"| {RES_VIDEO['minutes']} phút | lần nhìn test MSVD: {RES_VIDEO['test_looks_msvd']}")


## 5. Mô hình đóng băng (CLIP, RelTR)

In [ ]:
# ============================== 5. MÔ HÌNH ĐÓNG BĂNG: CLIP ViT-L/14 · RelTR (scene graph từ điểm ảnh) — không huấn luyện ==============================
NQ = 200   # số truy vấn bộ ba của RelTR

if SMOKE:
    from PIL import Image   # chỉ để có kiểu; ảnh giả không được mở
    MODEL_INFO = "fake (smoke)"
    _h = lambda s: int(hashlib.md5(str(s).encode()).hexdigest(), 16) % 2 ** 32

    def clip_image_paths(paths):
        v = np.stack([np.random.default_rng(_h(p)).standard_normal(16) for p in paths])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def clip_text(caps):
        v = np.stack([np.random.default_rng(_h(c)).standard_normal(16) for c in caps])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def reltr_clip_features(items, flip=False):
        """giả lập: cài bằng chứng cho caption ĐÚNG của mục ARO (để kiểm tra luồng xử lý, KHÔNG phải kết quả)."""
        F = []
        for it in items:
            r = np.random.default_rng(_h(it["item"]))
            ps, po, pr = r.random((NQ, 151)) * 0.01, r.random((NQ, 151)) * 0.01, r.random((NQ, 51)) * 0.01
            sb, ob = r.random((NQ, 4)) * 0.5 + 0.25, r.random((NQ, 4)) * 0.5 + 0.25
            par = aro_parse(it["true"]) if it["bench"] == "aro" else None
            if par and par[1] != "other":
                X, kind, rel, Y = par
                ps[0, CLS_IDX[X]] = po[0, CLS_IDX[Y]] = 0.9
                if kind == "pred": pr[0, PRED_IDX[rel]] = 0.9
                sb[0, :2], ob[0, :2] = ((0.2, 0.5), (0.8, 0.5)) if rel == "L" else ((0.8, 0.5), (0.2, 0.5))
                sb[0, 2:] = ob[0, 2:] = 0.1
            if flip: sb[:, 0], ob[:, 0] = 1 - sb[:, 0], 1 - ob[:, 0]
            img = r.standard_normal(16)
            F.append(dict(ps=ps.astype(np.float16), po=po.astype(np.float16), pr=pr.astype(np.float16), sb=sb.astype(np.float16),
                          ob=ob.astype(np.float16), img=img / np.linalg.norm(img)))
        return F

    def pil_of(it, flip=False):
        return Image.new("RGB", (100, 100), (128, 128, 128))

    def pil_path(p):
        return Image.new("RGB", (100, 100), (128, 128, 128))
else:
    try:
        import open_clip
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"]); import open_clip
    import torch, torchvision
    import torchvision.transforms as TT
    from PIL import Image, ImageOps
    from concurrent.futures import ThreadPoolExecutor
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    # RelTR: mã nguồn GitHub + checkpoint VG công khai (cùng cách v3)
    RDIR = f"{WORK}/RelTR"
    if not os.path.isdir(f"{RDIR}/models"):
        assert sh(["git", "clone", "-q", "--depth", "1", "https://github.com/yrcong/RelTR.git", RDIR]) == 0
    CK = fetch(f"{WORK}/reltr_vg_checkpoint0149.pth", gdrive=RELTR_GDRIVE)
    CK_SHA = sha256(CK)
    _orig = torchvision.models.resnet50
    torchvision.models.resnet50 = lambda *a, pretrained=False, **kw: _orig(*a, weights=None, **kw)
    sys.path.insert(0, RDIR)
    try:
        from models import build_model as _build
    finally:
        sys.path.remove(RDIR); torchvision.models.resnet50 = _orig
    _args = argparse.Namespace(lr_backbone=1e-5, dataset="vg", backbone="resnet50", dilation=False, position_embedding="sine",
                               enc_layers=6, dec_layers=6, dim_feedforward=2048, hidden_dim=256, dropout=0.1, nheads=8,
                               num_entities=100, num_triplets=NQ, pre_norm=False, aux_loss=False, device=DEVICE,
                               set_cost_class=1, set_cost_bbox=5, set_cost_giou=2, set_iou_threshold=0.7, bbox_loss_coef=5,
                               giou_loss_coef=2, rel_loss_coef=1, eos_coef=0.1, return_interm_layers=False)
    RELTR, _, _ = _build(_args)
    _sd = torch.load(CK, map_location="cpu", weights_only=False)
    p3_log("RelTR:", RELTR.load_state_dict(_sd["model"])); del _sd
    RELTR = RELTR.to(DEVICE).eval()
    RTF = TT.Compose([TT.Resize(800, max_size=1333), TT.ToTensor(), TT.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    CLIP, _, PRE = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_WEIGHTS)
    CLIP = CLIP.to(DEVICE).eval(); TOK = open_clip.get_tokenizer(CLIP_MODEL)
    MODEL_INFO = f"RelTR ckpt sha256 {CK_SHA} | CLIP {CLIP_MODEL}/{CLIP_WEIGHTS} | open_clip {open_clip.__version__} | torch {torch.__version__} | {DEVICE}"
    p3_log(MODEL_INFO)

    def pil_path(p):
        return Image.open(p).convert("RGB")

    def pil_of(it, flip=False):
        im = pil_path(it["path"])
        if it["crop"]: im = im.crop(it["crop"])                  # ARO: cắt theo bbox của mục (giao thức ARO)
        return ImageOps.mirror(im) if flip else im

    @torch.no_grad()
    def clip_image_paths(paths, bs=64):
        out = []
        with ThreadPoolExecutor(8) as ex:
            for s in range(0, len(paths), bs):
                x = list(ex.map(lambda p: PRE(pil_path(p)), paths[s:s + bs]))
                with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                    f = CLIP.encode_image(torch.stack(x).to(DEVICE)).float()
                out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def clip_text(caps, bs=256):
        out = []
        for s in range(0, len(caps), bs):
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                f = CLIP.encode_text(TOK(caps[s:s + bs]).to(DEVICE)).float()
            out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def reltr_clip_features(items, flip=False, bs=4):
        """RelTR: softmax rồi bỏ cột nền (cột cuối), 200 truy vấn bộ ba, hộp cxcywh chuẩn hoá — như v3. Kèm đặc trưng CLIP của cùng ảnh."""
        F = []
        for s in range(0, len(items), bs):
            pils = [pil_of(it, flip) for it in items[s:s + bs]]
            try:
                o = RELTR([RTF(im).to(DEVICE) for im in pils])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                parts = [RELTR([RTF(im).to(DEVICE)]) for im in pils]
                o = {k: torch.cat([q[k] for q in parts]) for k in ("sub_logits", "obj_logits", "rel_logits", "sub_boxes", "obj_boxes")}
            sm = lambda k: o[k].float().softmax(-1)[..., :-1].cpu().numpy().astype(np.float16)
            bx = lambda k: o[k].float().cpu().numpy().astype(np.float16)
            ps, po, pr, sb, ob = sm("sub_logits"), sm("obj_logits"), sm("rel_logits"), bx("sub_boxes"), bx("obj_boxes")
            assert ps.shape[1:] == (NQ, 151) and pr.shape[1:] == (NQ, 51)
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                ci = CLIP.encode_image(torch.stack([PRE(im) for im in pils]).to(DEVICE)).float()
            ci = torch.nn.functional.normalize(ci, dim=-1).cpu().numpy()
            for k in range(len(pils)):
                F.append(dict(ps=ps[k], po=po[k], pr=pr[k], sb=sb[k], ob=ob[k], img=ci[k]))
            if (s // bs) % 200 == 0: p3_log(f"  RelTR+CLIP{' (lật)' if flip else ''} {len(F)}/{len(items)}")
        return F


## 6. Thí nghiệm 1 — văn bản → ảnh, full-gallery (bộ chấm đường đi trên scene graph)

In [ ]:
# ============================== 6. THÍ NGHIỆM 1 — Văn bản → ảnh (nhánh A), full-gallery: CLIP top-K → điểm đồ thị → xếp hạng lại ==============================
# Đăng ký trước: v5_vgcoco_pilot/PILOT_PREREGISTRATION_VI.md. Truy vấn = caption COCO (người viết khác VG); đồ thị = scene graph GT của VG.
# Tham số (w, α) chọn trên P-TUNE; P-CHECK chỉ đánh giá. Đối chứng: G-obj (chỉ đối tượng), G-rand (đồ thị ảnh khác), G-shuf (hoán vị vị từ).
GSTATS = Counter()
GRAPH = {i: build_image(i, REL.get(i, []), OBJ.get(i, []), GSTATS) for i in ELIG1}
FEAT1 = dict(zip(ELIG1, clip_image_paths([VG_JPG[i] for i in ELIG1])))
SPL1 = {"P-TUNE": [i for i in ELIG1 if p5_split(i) == "P-TUNE"], "P-CHECK": [i for i in ELIG1 if p5_split(i) == "P-CHECK"]}
CAPTION = {(i, c): CAPS[COCO_OF_VG[i]][c][1] for i in ELIG1 for c in SEEDS}
METHODS1 = ("G-full", "G-obj", "G-rand", "G-shuf")
W_LISTS = {"G-full": W_GRID, "G-obj": [W_OBJ], "G-rand": W_GRID, "G-shuf": W_GRID}
p3_log(f"TN1: P-TUNE {len(SPL1['P-TUNE'])} / P-CHECK {len(SPL1['P-CHECK'])} ảnh | {elapsed()}")


def exp1_split(sp, c):
    ids = SPL1[sp]; caps = [CAPTION[(i, c)] for i in ids]
    T = clip_text(caps); I = np.stack([FEAT1[i] for i in ids])
    topk, cos_top, rel_pos, clip_rank = clip_topk(T @ I.T, K_TOP)
    parsed = [parse_caption(x) for x in caps]
    g_true = [GRAPH[i] for i in ids]
    perm = rand_permutation(len(ids))
    comps = {"G-full": candidate_components(parsed, topk, g_true),
             "G-rand": candidate_components(parsed, topk, [g_true[perm[j]] for j in range(len(ids))]),
             "G-shuf": candidate_components(parsed, topk, [shuffle_predicates(g) for g in g_true])}
    comps["G-obj"] = comps["G-full"]
    return dict(ids=ids, caps=caps, topk=topk, zc=zrows(cos_top), rel_pos=rel_pos, clip_rank=clip_rank, parsed=parsed, comps=comps,
                g_true=g_true, T=T, I=I)


RES1 = {"seeds": {}}
EXP1_KEEP = {}
for c in SEEDS:
    tune, check = exp1_split("P-TUNE", c), exp1_split("P-CHECK", c)
    params, alpha_curve = {}, {}
    for m in METHODS1:
        grid = tune_grid(tune["zc"], tune["comps"][m], tune["rel_pos"], tune["clip_rank"], W_LISTS[m], ALPHAS)
        params[m] = (select_params(grid), grid[select_params(grid)])
        alpha_curve[m] = {a: max(v for (w, aa), v in grid.items() if aa == a) for a in ALPHAS}   # độ nhạy α: MRR tốt nhất theo w, trên TUNE
    ranks, fused = {"CLIP": check["clip_rank"]}, {}
    for m in METHODS1:
        (w, a), _ = params[m]
        fused[m] = fuse_scores(check["zc"], check["comps"][m], w, a)
        ranks[m] = rank_of_relevant(fused[m], check["rel_pos"], check["clip_rank"])
    rr = {k: 1.0 / v for k, v in ranks.items()}
    has_T = np.array([len(T) > 0 for E, T in check["parsed"]])
    RES1["seeds"][c] = dict(
        params={m: dict(w=list(params[m][0][0]), alpha=params[m][0][1], tune_MRR=params[m][1]) for m in METHODS1},
        n_tune=len(tune["ids"]), n_check=len(check["ids"]), check={k: metrics_from_ranks(v) for k, v in ranks.items()},
        candidate_recall_at_K=float((check["clip_rank"] <= K_TOP).mean()), alpha_curve_tune_MRR=alpha_curve,
        parse_coverage=dict(entity=float(np.mean([len(E) > 0 for E, T in check["parsed"]])), triple=float(has_T.mean())),
        bootstrap=bootstrap_delta({"H1 full-CLIP": rr["G-full"] - rr["CLIP"], "H2 full-obj": rr["G-full"] - rr["G-obj"],
                                   "H3 full-rand": rr["G-full"] - rr["G-rand"], "H4 full-shuf": rr["G-full"] - rr["G-shuf"]}),
        subgroups={nm: dict(n=int(mk.sum()), **({k: metrics_from_ranks(v[mk]) for k, v in ranks.items()} if mk.any() else {}))
                   for nm, mk in (("co_bo_ba", has_T), ("khong_bo_ba", ~has_T))})
    if c == 0: EXP1_KEEP = dict(check=check, params=params, fused=fused, ranks=ranks)
    p3_log(f"TN1 seed {c}: MRR", {k: round(v["MRR"], 4) for k, v in RES1["seeds"][c]["check"].items()}, "| H1", RES1["seeds"][c]["bootstrap"]["H1 full-CLIP"])

H1 = {c: RES1["seeds"][c]["bootstrap"]["H1 full-CLIP"] for c in SEEDS}
RES1["decision"] = "GO" if all(H1[c]["lo"] > 0 for c in SEEDS) else "NO-GO"
RES1["mean_std"] = {k: {m: mean_std([RES1["seeds"][c]["check"][k][m] for c in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                    for k in ["CLIP", *METHODS1]}
RES1.update(n_eligible=len(ELIG1), excluded=dict(EXCL1), graph_stats=dict(GSTATS))
p3_log("TN1 quyết định (đăng ký trước):", RES1["decision"], "| MRR trung bình:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1["mean_std"].items()})


## 6b. Thí nghiệm 1b — GraphSAGE (GNN đã học)
Đăng ký trước: `final_pipeline/GNN_PREREGISTRATION_VI.md`. Đồ thị dị thể ảnh – khái niệm; học dự đoán liên kết trên ảnh không thuộc P-CHECK; suy diễn quy nạp một chiều; ablation cạnh ngẫu nhiên và bỏ cạnh.

In [ ]:
# ============================== 6b. THÍ NGHIỆM 1b — GraphSAGE (GNN đã học) làm điểm đồ thị trong cùng pipeline văn bản → ảnh ==============================
# Đăng ký trước: final_pipeline/GNN_PREREGISTRATION_VI.md. Đồ thị dị thể ảnh – khái niệm (lớp VG150, bộ ba VG150) từ scene graph GT của chính ảnh.
# Huấn luyện chỉ trên ảnh DEV-G0 KHÔNG thuộc P-CHECK; suy diễn quy nạp một chiều (ảnh P-CHECK nhận tin, không gửi tin). α chọn trên P-TUNE.
import torch, torch.nn as nn, torch.nn.functional as TF_
GDEV = "cuda" if torch.cuda.is_available() else "cpu"
SAGE_DIM, SAGE_EPOCHS, SAGE_LR, SAGE_WD, SAGE_DROP, SAGE_MSG_FRAC, SAGE_LOGIT_SCALE = 256, 300, 1e-3, 1e-5, 0.1, 0.8, 10.0

EVAL_IMGS = list(SPL1["P-CHECK"]); EVAL_SET = set(EVAL_IMGS)
TRAIN_IMGS = sorted(i for i in DEV_G0 if i in VG_JPG and i not in EVAL_SET)
GUARD_DEV.check(TRAIN_IMGS + EVAL_IMGS, "GraphSAGE")
assert not set(TRAIN_IMGS) & set(EVAL_IMGS)
for i in TRAIN_IMGS:
    if i not in GRAPH: GRAPH[i] = build_image(i, REL.get(i, []), OBJ.get(i, []), Counter())
_miss = [i for i in TRAIN_IMGS if i not in FEAT1]
FEAT1.update(dict(zip(_miss, clip_image_paths([VG_JPG[i] for i in _miss]))) if _miss else {})
IMG_NODES = TRAIN_IMGS + EVAL_IMGS
IMG_IDX = {i: k for k, i in enumerate(IMG_NODES)}
CLS_LIST = sorted(CLS_IDX)
TRI_LIST = sorted({t for i in TRAIN_IMGS for t in GRAPH[i].triples})          # chỉ bộ ba có trong ảnh huấn luyện
CON = {("c", c): k for k, c in enumerate(CLS_LIST)}
CON.update({("t", t): len(CLS_LIST) + k for k, t in enumerate(TRI_LIST)})
NI, NC = len(IMG_NODES), len(CON)
X_IMG = torch.tensor(np.stack([FEAT1[i] for i in IMG_NODES]), dtype=torch.float32)
X_CON = torch.tensor(clip_text([f"a photo of a {c}" for c in CLS_LIST] + [f"a photo of a {s} {p} a {o}" for s, p, o in TRI_LIST]), dtype=torch.float32)
p3_log(f"GraphSAGE: {len(TRAIN_IMGS)} ảnh huấn luyện, {len(EVAL_IMGS)} ảnh P-CHECK, {len(CLS_LIST)} lớp + {len(TRI_LIST)} bộ ba | {elapsed()}")


def image_edges(rand_seed=None):
    """danh sách (ảnh_idx, khái_niệm_idx). rand_seed ≠ None ⇒ cạnh ngẫu nhiên giữ nguyên bậc từng loại (ablation SAGE-rand)."""
    E = []
    rr = np.random.default_rng(rand_seed) if rand_seed is not None else None
    for i in IMG_NODES:
        im = GRAPH[i]
        cl = [CON[("c", c)] for c in sorted(im.classes) if ("c", c) in CON]
        tr = [CON[("t", t)] for t in sorted(im.triples) if ("t", t) in CON]
        if rr is not None:
            cl = list(rr.choice(len(CLS_LIST), size=len(cl), replace=False)) if cl else []
            tr = list(len(CLS_LIST) + rr.choice(len(TRI_LIST), size=min(len(tr), len(TRI_LIST)), replace=False)) if tr else []
        E += [(IMG_IDX[i], int(c)) for c in cl + tr]
    return np.array(E, dtype=np.int64).reshape(-1, 2)


def mean_adj(edges_img_recv, edges_con_recv, n):
    """ma trận thưa chuẩn hoá theo hàng (mean aggregator). Đỉnh khái niệm có chỉ số NI + k."""
    rows = np.concatenate([edges_img_recv[:, 0], NI + edges_con_recv[:, 1]])
    cols = np.concatenate([NI + edges_img_recv[:, 1], edges_con_recv[:, 0]])
    deg = np.bincount(rows, minlength=n).astype(np.float32)
    vals = 1.0 / np.maximum(deg[rows], 1.0)
    return torch.sparse_coo_tensor(torch.tensor(np.stack([rows, cols])), torch.tensor(vals), (n, n)).coalesce()


class SAGE(nn.Module):
    def __init__(self, d_img, d_con, d=SAGE_DIM, layers=2):
        super().__init__()
        self.pi, self.pc = nn.Linear(d_img, d), nn.Linear(d_con, d)
        self.ws = nn.ModuleList(nn.Linear(d, d) for _ in range(layers))
        self.wn = nn.ModuleList(nn.Linear(d, d, bias=False) for _ in range(layers))

    def forward(self, xi, xc, M):
        h = torch.cat([self.pi(xi), self.pc(xc)])
        for l in range(len(self.ws)):
            h = self.ws[l](h) + self.wn[l](torch.sparse.mm(M, h))
            if l < len(self.ws) - 1: h = TF_.dropout(TF_.relu(h), SAGE_DROP, self.training)
        return TF_.normalize(h, dim=-1)


def train_sage(seed, layers=2, rand=False):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    E = image_edges(SEED_RAND + seed if rand else None)
    is_train = np.array([IMG_NODES[a] not in EVAL_SET for a in E[:, 0]]) if len(E) else np.zeros(0, bool)
    Etr = E[is_train]
    xi, xc = X_IMG.to(GDEV), X_CON.to(GDEV)
    model = SAGE(xi.shape[1], xc.shape[1], layers=layers).to(GDEV)
    opt = torch.optim.Adam(model.parameters(), lr=SAGE_LR, weight_decay=SAGE_WD)
    is_cls = Etr[:, 1] < len(CLS_LIST)
    losses = []
    for ep in range(SAGE_EPOCHS):
        model.train()
        perm = rng.permutation(len(Etr)); cut = int(SAGE_MSG_FRAC * len(Etr))
        msg, sup = Etr[perm[:cut]], Etr[perm[cut:]]
        M = mean_adj(msg, msg, NI + NC).to(GDEV)          # chỉ ảnh huấn luyện gửi/nhận tin trong lúc học
        h = model(xi, xc, M)
        neg = np.where(sup[:, 1] < len(CLS_LIST), rng.integers(0, len(CLS_LIST), len(sup)),
                       len(CLS_LIST) + rng.integers(0, max(len(TRI_LIST), 1), len(sup)))
        a = torch.tensor(sup[:, 0], device=GDEV)
        pos = (h[a] * h[NI + torch.tensor(sup[:, 1], device=GDEV)]).sum(-1) * SAGE_LOGIT_SCALE
        ng = (h[a] * h[NI + torch.tensor(neg, device=GDEV)]).sum(-1) * SAGE_LOGIT_SCALE
        loss = TF_.binary_cross_entropy_with_logits(torch.cat([pos, ng]), torch.cat([torch.ones_like(pos), torch.zeros_like(ng)]))
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(float(loss))
    model.eval()
    with torch.no_grad():
        # suy diễn: khái niệm chỉ nhận tin từ ảnh huấn luyện; mọi ảnh (kể cả P-CHECK) nhận tin từ khái niệm của chính nó
        H = model(xi, xc, mean_adj(E, Etr, NI + NC).to(GDEV)).cpu().numpy()
    return H, dict(loss_first=losses[0], loss_last=losses[-1], n_edges=int(len(E)), n_train_edges=int(len(Etr)))


def query_concepts(parsed):
    E, T = parsed
    return [CON[("c", c)] for c in sorted(E) if ("c", c) in CON] + [CON[("t", t)] for t in sorted(T) if ("t", t) in CON]


def exp1_clip(sp, c):
    ids = SPL1[sp]; caps = [CAPTION[(i, c)] for i in ids]
    T = clip_text(caps); I = np.stack([FEAT1[i] for i in ids])
    topk, cos_top, rel_pos, clip_rank = clip_topk(T @ I.T, K_TOP)
    return dict(ids=ids, topk=topk, zc=zrows(cos_top), rel_pos=rel_pos, clip_rank=clip_rank, parsed=[parse_caption(x) for x in caps])


def sage_components(d, H):
    Q, K = d["topk"].shape
    out = np.zeros((Q, K, 1))
    for q in range(Q):
        cs = query_concepts(d["parsed"][q])
        if not cs: continue
        v = H[NI + np.array(cs)].mean(0); v = v / (np.linalg.norm(v) + 1e-12)
        out[q, :, 0] = H[[IMG_IDX[d["ids"][j]] for j in d["topk"][q]]] @ v
    return out


VARIANTS = {"SAGE": dict(layers=2, rand=False), "SAGE-rand": dict(layers=2, rand=True), "SAGE-0": dict(layers=0, rand=False)}
RES1G = {"seeds": {}}
for s in SEEDS:
    tune, check = exp1_clip("P-TUNE", s), exp1_clip("P-CHECK", s)
    ranks, info = {"CLIP": check["clip_rank"]}, {}
    for name, cfg in VARIANTS.items():
        H, tr = train_sage(s, **cfg)
        ct, cc = sage_components(tune, H), sage_components(check, H)
        grid = tune_grid(tune["zc"], ct, tune["rel_pos"], tune["clip_rank"], [(1.0,)], ALPHAS)
        (w, a) = select_params(grid)
        ranks[name] = rank_of_relevant(fuse_scores(check["zc"], cc, w, a), check["rel_pos"], check["clip_rank"])
        info[name] = dict(alpha=a, tune_MRR=grid[(w, a)], alpha_curve_tune_MRR={aa: grid[((1.0,), aa)] for aa in ALPHAS}, **tr)
    rr = {k: 1.0 / v for k, v in ranks.items()}
    RES1G["seeds"][s] = dict(train=info, check={k: metrics_from_ranks(v) for k, v in ranks.items()},
                             query_concept_coverage=float(np.mean([len(query_concepts(p)) > 0 for p in check["parsed"]])),
                             bootstrap=bootstrap_delta({"G1 SAGE-CLIP": rr["SAGE"] - rr["CLIP"], "G2 SAGE-rand": rr["SAGE"] - rr["SAGE-rand"],
                                                        "G3 SAGE-noedge": rr["SAGE"] - rr["SAGE-0"]}))
    p3_log(f"TN1b seed {s}: MRR", {k: round(v["MRR"], 4) for k, v in RES1G["seeds"][s]["check"].items()},
        "| α", {k: v["alpha"] for k, v in info.items()}, "| loss", {k: (round(v["loss_first"], 3), round(v["loss_last"], 3)) for k, v in info.items()})
RES1G["mean_std"] = {k: {m: mean_std([RES1G["seeds"][s]["check"][k][m] for s in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                     for k in ["CLIP", *VARIANTS]}
RES1G["decision"] = {h: ("DƯƠNG" if all(RES1G["seeds"][s]["bootstrap"][h]["lo"] > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                     for h in RES1G["seeds"][0]["bootstrap"]}
RES1G.update(n_train_images=len(TRAIN_IMGS), n_eval_images=len(EVAL_IMGS), n_class_nodes=len(CLS_LIST), n_triple_nodes=len(TRI_LIST),
             hyper=dict(dim=SAGE_DIM, epochs=SAGE_EPOCHS, lr=SAGE_LR, wd=SAGE_WD, dropout=SAGE_DROP, msg_frac=SAGE_MSG_FRAC, logit_scale=SAGE_LOGIT_SCALE))
p3_log("TN1b quyết định (đăng ký trước):", RES1G["decision"], "| MRR:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1G["mean_std"].items()}, "|", elapsed())


## 6c. Thí nghiệm 1c — SCENIR (ICML 2025), phương pháp từ 2024
Đối chiếu với mã gốc: `final_pipeline/SCENIR_VERIFICATION_VI.md`. Học không giám sát trên scene graph của ảnh huấn luyện; truy vấn = scene graph tách từ caption; ablation cạnh ngẫu nhiên / bỏ cạnh.

In [ ]:
# ============================== 6c. THÍ NGHIỆM 1c — SCENIR (ICML 2025) trong cùng pipeline văn bản → ảnh (phương pháp từ 2024 cho cấp 2) ==============================
# Xác minh và kế hoạch: final_pipeline/SCENIR_VERIFICATION_VI.md. Cài lại ARGVA của mã gốc (github.com/nickhaidos/scenir-icml2025, MIT) bằng PyTorch thuần:
# encoder GIN 2 nhánh (μ, log σ), feature decoder (MSE), edge decoder MLP + tích vô hướng, discriminator đối kháng; siêu tham số = config.json.
# Huấn luyện KHÔNG giám sát trên scene graph của ảnh huấn luyện (giống GraphSAGE, không chứa P-CHECK). Truy vấn = scene graph tách từ caption.
SC_CFG = dict(epochs=30, lr=1e-3, gamma=0.95, bs=64, latent=1000, layers=3, disc_hidden=32, edge_hidden=256, edge_out=32, disc_steps=5)
SC_EPS, SC_MAX_LOGSTD = 1e-15, 10.0

# ---- đặc trưng đỉnh: Sentence-Transformer 768 chiều như bài báo (giả định all-mpnet-base-v2); dự phòng CLIP-text ViT-L/14 (768) ----
SC_FEAT_SRC = None
if not SMOKE:
    try:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            sh([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"]); from sentence_transformers import SentenceTransformer
        _st = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=GDEV)
        SC_X = torch.tensor(_st.encode(CLS_LIST, convert_to_numpy=True, normalize_embeddings=False), dtype=torch.float32)
        SC_FEAT_SRC = "sentence-transformers/all-mpnet-base-v2"; del _st
    except Exception as ex:
        p3_log("SCENIR: không tải được Sentence-Transformer, dùng CLIP-text:", repr(ex)[:200])
if SC_FEAT_SRC is None:
    SC_X = torch.tensor(clip_text(CLS_LIST), dtype=torch.float32); SC_FEAT_SRC = f"CLIP-text {CLIP_MODEL}" if not SMOKE else "fake (smoke)"
SC_CI = {c: k for k, c in enumerate(CLS_LIST)}
p3_log(f"SCENIR: đặc trưng đỉnh {SC_FEAT_SRC}, {tuple(SC_X.shape)}")


def sg_of_pairs(pairs):
    """đồ thị lớp-level: đỉnh = lớp tham gia ≥ 1 quan hệ (bỏ đỉnh cô lập như bài báo), cạnh vô hướng (lưu hai chiều)."""
    pairs = sorted({tuple(sorted((a, b))) for a, b in pairs if a != b and a in SC_CI and b in SC_CI})
    nodes = sorted({c for p in pairs for c in p})
    loc = {c: k for k, c in enumerate(nodes)}
    ei = [(loc[a], loc[b]) for a, b in pairs] + [(loc[b], loc[a]) for a, b in pairs]
    return [SC_CI[c] for c in nodes], np.array(ei, dtype=np.int64).reshape(-1, 2).T


def image_sg(i, variant, seed):
    pairs = list(GRAPH[i].between.keys())
    nodes, ei = sg_of_pairs(pairs)
    if variant == "noedge":
        return nodes, np.zeros((2, 0), np.int64)
    if variant == "rand" and len(nodes) > 1:
        rr = np.random.default_rng(SEED_RAND + seed * 100003 + int(i))
        allp = [(a, b) for a in range(len(nodes)) for b in range(a + 1, len(nodes))]
        pick = [allp[k] for k in rr.choice(len(allp), size=min(ei.shape[1] // 2, len(allp)), replace=False)]
        ei = np.array([(a, b) for a, b in pick] + [(b, a) for a, b in pick], dtype=np.int64).reshape(-1, 2).T
    return nodes, ei


def query_sg(parsed):
    return sg_of_pairs([(s, o) for (s, p, o) in parsed[1]])


class GIN(nn.Module):
    """GINConv(Linear), ε = 0: h'_v = W·(h_v + Σ_{u→v} h_u)."""
    def __init__(self, a, b):
        super().__init__(); self.lin = nn.Linear(a, b)

    def forward(self, x, ei):
        agg = torch.zeros_like(x).index_add_(0, ei[1], x[ei[0]]) if ei.shape[1] else torch.zeros_like(x)
        return self.lin(x + agg)


class ScenirModel(nn.Module):
    def __init__(self, din):
        super().__init__()
        L, H = SC_CFG["layers"], SC_CFG["latent"]
        dims = [din] + [H] * L
        self.mu = nn.ModuleList(GIN(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.ls = nn.ModuleList(GIN(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.disc = nn.Sequential(nn.Linear(H, SC_CFG["disc_hidden"]), nn.ReLU(), nn.Linear(SC_CFG["disc_hidden"], SC_CFG["disc_hidden"]), nn.ReLU(),
                                  nn.Linear(SC_CFG["disc_hidden"], 1))
        self.fdec = nn.Sequential(nn.Linear(H, din), nn.ReLU(), nn.Linear(din, din))
        self.edec = nn.Sequential(nn.Linear(H, SC_CFG["edge_hidden"]), nn.ReLU(), nn.Linear(SC_CFG["edge_hidden"], SC_CFG["edge_out"]))

    @staticmethod
    def _run(layers, x, ei):
        for k, l in enumerate(layers):
            x = l(x, ei)
            if k < len(layers) - 1: x = TF_.relu(x)
        return x

    def encode(self, x, ei):
        self._mu, self._ls = self._run(self.mu, x, ei), self._run(self.ls, x, ei).clamp(max=SC_MAX_LOGSTD)
        return self._mu + torch.randn_like(self._ls) * torch.exp(self._ls) if self.training else self._mu

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.disc(z)) + SC_EPS).mean()

    def disc_loss(self, z):
        real, fake = torch.sigmoid(self.disc(torch.randn_like(z))), torch.sigmoid(self.disc(z.detach()))
        return -torch.log(real + SC_EPS).mean() - torch.log(1 - fake + SC_EPS).mean()

    def kl_loss(self):
        return -0.5 * torch.mean(torch.sum(1 + 2 * self._ls - self._mu ** 2 - self._ls.exp() ** 2, dim=1))

    def recon_loss(self, x, ei, neg):
        z = self.encode(x, ei)
        feat = TF_.mse_loss(self.fdec(z), x)
        zd = self.edec(z)
        dec = lambda e: torch.sigmoid((zd[e[0]] * zd[e[1]]).sum(-1))
        pos = -torch.log(dec(ei) + SC_EPS).mean() if ei.shape[1] else torch.zeros((), device=x.device)
        ngl = -torch.log(1 - dec(neg) + SC_EPS).mean() if neg.shape[1] else torch.zeros((), device=x.device)
        return pos + ngl + feat


def sc_batch(graphs):
    xs, eis, ptr, off = [], [], [0], 0
    for nodes, ei in graphs:
        xs.append(SC_X[nodes]); eis.append(ei + off); off += len(nodes); ptr.append(off)
    return torch.cat(xs), torch.tensor(np.concatenate(eis, axis=1) if eis else np.zeros((2, 0), np.int64)), np.array(ptr)


def sc_negatives(ei, ptr, rng):
    """cặp (u, v) cùng đồ thị, không tự nối, không trùng cạnh thật; số lượng = số cạnh dương (như batched_negative_sampling)."""
    n = ei.shape[1]
    if n == 0: return np.zeros((2, 0), np.int64)
    exist = set(map(tuple, ei.T.tolist()))
    g = np.searchsorted(ptr, ei[0], side="right") - 1
    out = []
    for gi in rng.choice(g, size=n):
        lo, hi = ptr[gi], ptr[gi + 1]
        for _ in range(10):
            u, v = rng.integers(lo, hi, 2)
            if u != v and (u, v) not in exist: out.append((u, v)); break
    return np.array(out, dtype=np.int64).reshape(-1, 2).T


def train_scenir(seed, variant):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    G = [image_sg(i, variant, seed) for i in TRAIN_IMGS]
    G = [g for g in G if len(g[0]) > 0]
    model = ScenirModel(SC_X.shape[1]).to(GDEV)
    d_opt = torch.optim.AdamW(model.disc.parameters(), lr=SC_CFG["lr"])
    e_opt = torch.optim.AdamW(list(model.mu.parameters()) + list(model.ls.parameters()) + list(model.fdec.parameters()) + list(model.edec.parameters()), lr=SC_CFG["lr"])
    sched = torch.optim.lr_scheduler.ExponentialLR(e_opt, gamma=SC_CFG["gamma"])
    hist = []
    for ep in range(SC_CFG["epochs"]):
        model.train(); perm = rng.permutation(len(G)); tot = 0.0
        for s in range(0, len(G), SC_CFG["bs"]):
            x, ei, ptr = sc_batch([G[k] for k in perm[s:s + SC_CFG["bs"]]])
            neg = torch.tensor(sc_negatives(ei.numpy(), ptr, rng)).to(GDEV); x, ei = x.to(GDEV), ei.to(GDEV)
            e_opt.zero_grad()
            z = model.encode(x, ei)
            for _ in range(SC_CFG["disc_steps"]):
                d_opt.zero_grad(); dl = model.disc_loss(z); dl.backward(); d_opt.step()
            loss = (1 / 6) * model.reg_loss(z) + 3 * model.recon_loss(x, ei, neg) + (1 / 3) * (1 / x.shape[0]) * model.kl_loss()
            loss.backward(); e_opt.step(); tot += float(loss)
        sched.step(); hist.append(tot)
    model.eval()
    return model, dict(n_train_graphs=len(G), loss_first=hist[0], loss_last=hist[-1])


@torch.no_grad()
def sc_embed(model, graphs):
    """nhúng đồ thị = Σ μ của các đỉnh (sum-pool), chuẩn hoá L2; đồ thị rỗng → vector 0."""
    out = np.zeros((len(graphs), SC_CFG["latent"]), np.float32)
    for k, (nodes, ei) in enumerate(graphs):
        if len(nodes) == 0: continue
        z = model.encode(SC_X[nodes].to(GDEV), torch.tensor(ei).to(GDEV)).sum(0)
        out[k] = (z / (z.norm() + 1e-12)).cpu().numpy()
    return out


SC_VARIANTS = {"SCENIR": "full", "SCENIR-rand": "rand", "SCENIR-noedge": "noedge"}
RES1S = {"seeds": {}, "node_features": SC_FEAT_SRC, "config": SC_CFG}
for s in SEEDS:
    tune, check = exp1_clip("P-TUNE", s), exp1_clip("P-CHECK", s)
    ranks, info = {"CLIP": check["clip_rank"]}, {}
    for name, var in SC_VARIANTS.items():
        model, tr = train_scenir(s, var)
        emb_img = dict(zip(SPL1["P-TUNE"] + SPL1["P-CHECK"],
                           sc_embed(model, [image_sg(i, var, s) for i in SPL1["P-TUNE"] + SPL1["P-CHECK"]])))
        qsg = lambda d: sc_embed(model, [query_sg(p) if var != "noedge" else (query_sg(p)[0], np.zeros((2, 0), np.int64)) for p in d["parsed"]])

        def comps_of(d, qe):
            Q, K = d["topk"].shape
            return np.stack([np.stack([emb_img[d["ids"][j]] for j in d["topk"][q]]) @ qe[q] for q in range(Q)])[:, :, None]
        ct, cc = comps_of(tune, qsg(tune)), comps_of(check, qsg(check))
        grid = tune_grid(tune["zc"], ct, tune["rel_pos"], tune["clip_rank"], [(1.0,)], ALPHAS)
        (w, a) = select_params(grid)
        ranks[name] = rank_of_relevant(fuse_scores(check["zc"], cc, w, a), check["rel_pos"], check["clip_rank"])
        info[name] = dict(alpha=a, tune_MRR=grid[(w, a)], alpha_curve_tune_MRR={aa: grid[((1.0,), aa)] for aa in ALPHAS}, **tr)
        del model
    rr = {k: 1.0 / v for k, v in ranks.items()}
    RES1S["seeds"][s] = dict(train=info, check={k: metrics_from_ranks(v) for k, v in ranks.items()},
                             query_graph_coverage=float(np.mean([len(query_sg(p)[0]) > 0 for p in check["parsed"]])),
                             bootstrap=bootstrap_delta({"S1 SCENIR-CLIP": rr["SCENIR"] - rr["CLIP"], "S2 SCENIR-rand": rr["SCENIR"] - rr["SCENIR-rand"],
                                                        "S3 SCENIR-noedge": rr["SCENIR"] - rr["SCENIR-noedge"]}))
    p3_log(f"TN1c seed {s}: MRR", {k: round(v["MRR"], 4) for k, v in RES1S["seeds"][s]["check"].items()}, "| α", {k: v["alpha"] for k, v in info.items()},
        "| loss", {k: (round(v["loss_first"], 2), round(v["loss_last"], 2)) for k, v in info.items()})
RES1S["mean_std"] = {k: {m: mean_std([RES1S["seeds"][s]["check"][k][m] for s in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                     for k in ["CLIP", *SC_VARIANTS]}
RES1S["decision"] = {h: ("DƯƠNG" if all(RES1S["seeds"][s]["bootstrap"][h]["lo"] > 0 for s in SEEDS) else "KHÔNG DƯƠNG") for h in RES1S["seeds"][0]["bootstrap"]}
p3_log("TN1c quyết định (đăng ký trước):", RES1S["decision"], "| MRR:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1S["mean_std"].items()}, "|", elapsed())


## 7. Thí nghiệm 2 — benchmark kết hợp công khai (ARO, SugarCrepe)

In [ ]:
# ============================== 7. THÍ NGHIỆM 2 — Benchmark kết hợp công khai (ARO VG-Relation, SugarCrepe): ảnh → chọn caption đúng ==============================
# Đăng ký trước: v6_compositional_benchmarks/PREREGISTRATION_VI.md. Đồ thị = scene graph RelTR suy từ điểm ảnh (KHÔNG dùng GT vì caption ARO sinh từ GT).
# Đối chứng: C-rand (đặc trưng mục khác), C-shuf (hoán vị vị từ), C-nogeo (bỏ vị trí). α chọn trên TUNE, đánh giá CHECK, 3 seed chia.
FEATS2 = reltr_clip_features(ITEMS2)
TT2, TF2 = clip_text([it["true"] for it in ITEMS2]), clip_text([it["false"] for it in ITEMS2])
V3B = set(V3_BENCH_IDS)
p3_log(f"TN2: đặc trưng {len(FEATS2)} ảnh | {elapsed()}")

RAND2 = {}
for b in ("aro", "replace_rel", "swap_obj"):
    idx = [n for n, it in enumerate(ITEMS2) if it["bench"] == b]
    perm = rand_permutation(len(idx))
    for j, n in enumerate(idx): RAND2[n] = idx[perm[j]]

ROWS2 = []
for n, it in enumerate(ITEMS2):
    F, Fr = FEATS2[n], FEATS2[RAND2[n]]
    row = dict(bench=it["bench"], item=it["item"], key=it["key"], subset=it["subset"], rel=it["rel"],
               dc=float(F["img"] @ TT2[n] - F["img"] @ TF2[n]), in_v3_bench=bool(it["vg_id"] in V3B))
    if it["bench"] == "aro":
        pt, pf = aro_parse(it["true"]), aro_parse(it["false"])
        valid = pt[1] != "other" and pf[1] != "other"
        if valid:
            prs = shuffled_pr(F, it["key"])
            gt, et = aro_graph_score(F, pt); gf, ef = aro_graph_score(F, pf)
            row.update(dg=gt - gf, dg_rand=aro_graph_score(Fr, pt)[0] - aro_graph_score(Fr, pf)[0],
                       dg_shuf=(aro_graph_score(F, pt, pr=prs)[0] - aro_graph_score(F, pf, pr=prs)[0]) if pt[1] == "pred" else np.nan,
                       dg_nogeo=(aro_graph_score(F, pt, nogeo=True)[0] - aro_graph_score(F, pf, nogeo=True)[0]) if pt[1] == "geo" else np.nan,
                       ev_true=json.dumps(et), ev_false=json.dumps(ef))
    else:
        Tt, Tf = parse_caption(it["true"])[1], parse_caption(it["false"])[1]
        valid = bool(Tt) and bool(Tf) and Tt != Tf
        if valid:
            prs = shuffled_pr(F, int(it["key"][4:]))
            gt, et = triples_graph_score(F, Tt); gf, ef = triples_graph_score(F, Tf)
            if gt is None or gf is None: valid = False
            else:
                row.update(dg=gt - gf, dg_rand=triples_graph_score(Fr, Tt)[0] - triples_graph_score(Fr, Tf)[0],
                           dg_shuf=triples_graph_score(F, Tt, pr=prs)[0] - triples_graph_score(F, Tf, pr=prs)[0], dg_nogeo=np.nan,
                           ev_true=json.dumps(et), ev_false=json.dumps(ef))
    row["valid"] = bool(valid)
    for k in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"): row.setdefault(k, np.nan)
    ROWS2.append(row)


def exp2_evaluate(rows, s):
    sp = np.array([v6_split(r["key"], s) for r in rows]); ck, tu = sp == "CHECK", sp == "TUNE"
    dc = np.array([r["dc"] for r in rows]); valid = np.array([r["valid"] for r in rows]); clusters = np.array([r["key"] for r in rows])
    out = dict(n_tune=int(tu.sum()), n_check=int(ck.sum()), coverage=float(valid.mean()))
    res = {"CLIP": correct(dc)}
    for var in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"):
        dg = np.array([r[var] for r in rows], dtype=float); v = valid & ~np.isnan(dg)
        if not v.any(): continue
        dgz = np.where(v, dg, 0.0)
        res[f"graph_only[{var}]"] = np.where(v, correct(dgz), 0.5)
        a, sc, sg, acc = tune_alpha(dc[tu], dgz[tu], v[tu])
        res[f"fusion[{var}]"] = correct(fuse2(dc, dgz, v, a, sc, sg)); out[f"alpha[{var}]"] = a
        if var == "dg": out["alpha_grid_tune"] = acc
    out["acc_check"] = {k: float(v[ck].mean()) for k, v in res.items()}
    out["acc_check_valid_only"] = {k: float(v[ck & valid].mean()) for k, v in res.items()} if (ck & valid).any() else {}
    out["n_check_valid"] = int((ck & valid).sum())
    comps = {"H fusion-CLIP": ("fusion[dg]", "CLIP"), "graph-rand": ("graph_only[dg]", "graph_only[dg_rand]"),
             "fusion-fusion_rand": ("fusion[dg]", "fusion[dg_rand]"), "graph-shuf": ("graph_only[dg]", "graph_only[dg_shuf]"),
             "graph-nogeo": ("graph_only[dg]", "graph_only[dg_nogeo]"), "graph-CLIP": ("graph_only[dg]", "CLIP")}
    out["bootstrap_check"] = {}
    for nm, (a, b) in comps.items():
        if a in res and b in res:
            m = ck.copy()
            if "shuf" in nm: m &= ~np.isnan(np.array([r["dg_shuf"] for r in rows], dtype=float)) & valid
            if "nogeo" in nm: m &= ~np.isnan(np.array([r["dg_nogeo"] for r in rows], dtype=float)) & valid
            if m.sum() > 1: out["bootstrap_check"][nm] = dict(n=int(m.sum()), **bootstrap_diff_by_cluster(res[a][m], res[b][m], clusters[m]))
    return out


GROUPS2 = {"ARO_all": lambda r: r["bench"] == "aro", "ARO_geo": lambda r: r["bench"] == "aro" and r["subset"] == "geo",
           "ARO_pred": lambda r: r["bench"] == "aro" and r["subset"] == "pred", "ARO_other": lambda r: r["bench"] == "aro" and r["subset"] == "other",
           "ARO_all_excl_v3bench": lambda r: r["bench"] == "aro" and not r["in_v3_bench"],
           "SC_replace_rel": lambda r: r["bench"] == "replace_rel", "SC_swap_obj": lambda r: r["bench"] == "swap_obj"}
RES2 = {"seeds": {s: {g: exp2_evaluate([r for r in ROWS2 if f(r)], s) for g, f in GROUPS2.items() if any(f(r) for r in ROWS2)} for s in SEEDS}}
RES2["mean_std"] = {g: {m: mean_std([RES2["seeds"][s][g]["acc_check"][m] for s in SEEDS]) for m in RES2["seeds"][0][g]["acc_check"]}
                    for g in RES2["seeds"][0]}
RES2["decision"] = {g: ("DƯƠNG" if all(RES2["seeds"][s][g]["bootstrap_check"].get("H fusion-CLIP", {}).get("lo", -1) > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                    for g in ("ARO_all", "SC_replace_rel") if g in RES2["seeds"][0]}
RES2.update(items=dict(Counter(i["bench"] for i in ITEMS2)), excluded=dict(EXCL2), overlap_v3_benchmark_items=sum(it["vg_id"] in V3B for it in ITEMS2))
for g in RES2["mean_std"]:
    p3_log(f"TN2 {g}:", {k: round(v["mean"] * 100, 1) for k, v in RES2["mean_std"][g].items() if k in ("CLIP", "graph_only[dg]", "fusion[dg]", "fusion[dg_rand]")})
p3_log("TN2 quyết định (đăng ký trước):", RES2["decision"], "|", elapsed())


## 7b. Thí nghiệm 2b — Cấp 4 (a): cạnh hình học trên COCO-spatial
Đăng ký trước: `final_pipeline/LEVEL4A_PREREGISTRATION_VI.md`. Cạnh trái / phải / trên / dưới suy từ box RelTR; đối chứng bỏ vị trí, đặc trưng mục khác, lật ảnh.

In [ ]:
# ============================== 7b. THÍ NGHIỆM 2b — Cấp 4 hướng (a): cạnh hình học trên tập công khai thứ hai (COCO-spatial, What'sUp EMNLP 2023) ==============================
# Đăng ký trước: final_pipeline/LEVEL4A_PREREGISTRATION_VI.md. Cùng RelTR, CLIP, cách kết hợp và chọn α như TN2. Lọc rò rỉ như SugarCrepe.
CSP_GDRIVE = "1TCEoM0mgFmz8T4cF7PQ3XJmO6JjtiQ-s"
CSP_PATH = BENCH["cocosp"] if SMOKE else fetch(f"{WORK}/coco_qa_two_obj.json", gdrive=CSP_GDRIVE)
CSP_SHA = sha256(CSP_PATH)
CSP_RX = re.compile(r"^A photo of an? (.+?) (?:to the )?(left of|right of|above|below) (?:an? |the )?(.+)$")
CSP_DIR = {"left of": ("x", 1), "right of": ("x", -1), "above": ("y", 1), "below": ("y", -1)}


def csp_name(phrase):
    return norm_name(phrase) or norm_name(phrase.split()[-1])


def csp_parse(cap):
    m = CSP_RX.match(cap)
    if not m: return None
    X, Y = csp_name(m.group(1)), csp_name(m.group(3))
    return (X, m.group(2), Y) if X and Y else ("?", m.group(2), "?")


def geo_axis_score(F, X, Y, rel, nogeo=False):
    """như geo_score nhưng cho cả trục ngang (left/right) và trục dọc (above/below): σ(sign·(c_v − c_u)/τ)."""
    axis, sign = CSP_DIR[rel]
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    a, b, c = P[:, CLS_IDX[X]], P[:, CLS_IDX[Y]], B[:, 0 if axis == "x" else 1]
    s = np.ones((len(c), len(c))) if nogeo else 1.0 / (1.0 + np.exp(-sign * (c[None, :] - c[:, None]) / TAU))
    M = a[:, None] * b[None, :] * s; np.fill_diagonal(M, 0.0)
    u, v = np.unravel_index(int(np.argmax(M)), M.shape)
    return float(np.log(max(M[u, v], FLOOR))), dict(kind="geo", u=int(u), v=int(v), p=float(M[u, v]), x_box=[float(x) for x in B[u]], y_box=[float(x) for x in B[v]])


ITEMS2B, EXCL2B = [], Counter()
for k, x in enumerate(json.load(open(CSP_PATH, encoding="utf-8"))):
    coco = int(x[0]); vid = VG_OF_COCO.get(coco)
    if vid is not None and vid not in VALSET: EXCL2B["vg_not_VAL"] += 1; continue
    path = os.path.join(P["coco_val"], f"{coco:012d}.jpg")
    if not os.path.exists(path): EXCL2B["no_jpg"] += 1; continue
    pt = csp_parse(x[1]); pf = csp_parse(x[2])
    if pt is None or pf is None: EXCL2B["unparsed"] += 1; continue
    ITEMS2B.append(dict(bench="cocosp", item=f"cocosp{k}", key=f"cocosp{coco}", vg_id=vid, path=path, crop=None, true=x[1], false=x[2],
                        pt=pt, pf=pf, axis=CSP_DIR[pt[1]][0], valid=pt[0] != "?" and pf[0] != "?"))
GUARD_VAL.check([it["vg_id"] for it in ITEMS2B if it["vg_id"] is not None], "COCO-spatial")
p3_log(f"TN2b COCO-spatial: {len(ITEMS2B)} mục giữ lại, loại {dict(EXCL2B)}, đồ thị chấm được {sum(it['valid'] for it in ITEMS2B)} | {elapsed()}")

FEATS2B = reltr_clip_features(ITEMS2B)
T2B, F2B = clip_text([it["true"] for it in ITEMS2B]), clip_text([it["false"] for it in ITEMS2B])
PERM2B = rand_permutation(len(ITEMS2B))
_flip_idx = [n for n, it in enumerate(ITEMS2B) if it["valid"]]
FLIP2B = {}
if _flip_idx:
    _hidx = [n for n in _flip_idx if ITEMS2B[n]["axis"] == "x"]; _vidx = [n for n in _flip_idx if ITEMS2B[n]["axis"] == "y"]
    FLIP2B.update(dict(zip(_hidx, reltr_clip_features([ITEMS2B[n] for n in _hidx], flip=True))))
    if _vidx:
        _orig_pil_of = pil_of
        if not SMOKE:
            pil_of = lambda it, flip=False: ImageOps.flip(pil_path(it["path"])) if flip else pil_path(it["path"])     # lật DỌC cho mục trên/dưới
        FLIP2B.update(dict(zip(_vidx, reltr_clip_features([ITEMS2B[n] for n in _vidx], flip=True))))
        pil_of = _orig_pil_of
p3_log(f"TN2b: đặc trưng {len(FEATS2B)} ảnh + {len(FLIP2B)} ảnh lật | {elapsed()}")

ROWS2B = []
for n, it in enumerate(ITEMS2B):
    F, Fr = FEATS2B[n], FEATS2B[PERM2B[n]]
    row = dict(item=it["item"], key=it["key"], axis=it["axis"], rel=it["pt"][1], valid=it["valid"], dc=float(F["img"] @ T2B[n] - F["img"] @ F2B[n]))
    if it["valid"]:
        (X, r, Y), (Xf, rf, Yf) = it["pt"], it["pf"]
        gt, et = geo_axis_score(F, X, Y, r); gf, ef = geo_axis_score(F, Xf, Yf, rf)
        row.update(dg=gt - gf, dg_rand=geo_axis_score(Fr, X, Y, r)[0] - geo_axis_score(Fr, Xf, Yf, rf)[0],
                   dg_nogeo=geo_axis_score(F, X, Y, r, nogeo=True)[0] - geo_axis_score(F, Xf, Yf, rf, nogeo=True)[0],
                   dg_shuf=np.nan, ev_true=json.dumps(et), ev_false=json.dumps(ef))
        if n in FLIP2B:
            Ff = FLIP2B[n]
            dgf = geo_axis_score(Ff, X, Y, r)[0] - geo_axis_score(Ff, Xf, Yf, rf)[0]
            dcf = float(Ff["img"] @ T2B[n] - Ff["img"] @ F2B[n])
            row.update(flip_graph_reversed=float(np.sign(dgf) == -np.sign(row["dg"]) and row["dg"] != 0),
                       flip_clip_reversed=float(np.sign(dcf) == -np.sign(row["dc"]) and row["dc"] != 0))
    for kk in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"): row.setdefault(kk, np.nan)
    ROWS2B.append(row)

GROUPS2B = {"COCOSP_all": lambda r: True, "COCOSP_leftright": lambda r: r["axis"] == "x", "COCOSP_abovebelow": lambda r: r["axis"] == "y"}
RES2B = {"seeds": {s: {g: exp2_evaluate([r for r in ROWS2B if f(r)], s) for g, f in GROUPS2B.items() if any(f(r) for r in ROWS2B)} for s in SEEDS}}
RES2B["mean_std"] = {g: {m: mean_std([RES2B["seeds"][s][g]["acc_check"][m] for s in SEEDS]) for m in RES2B["seeds"][0][g]["acc_check"]}
                     for g in RES2B["seeds"][0]}
RES2B["decision"] = {h: ("DƯƠNG" if all(RES2B["seeds"][s]["COCOSP_all"]["bootstrap_check"].get(k, {}).get("lo", -1) > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                     for h, k in (("A1 fusion-CLIP", "H fusion-CLIP"), ("A2 graph-nogeo", "graph-nogeo"), ("A3 graph-rand", "graph-rand"))}
_fl = [r for r in ROWS2B if "flip_graph_reversed" in r]
if _fl:
    fg, fc = np.array([r["flip_graph_reversed"] for r in _fl]), np.array([r["flip_clip_reversed"] for r in _fl])
    RES2B["counterfactual"] = dict(n=len(_fl), graph_reversed=float(fg.mean()), clip_reversed=float(fc.mean()),
                                   gap=bootstrap_diff_by_cluster(fg, fc, [r["key"] for r in _fl]))
RES2B.update(n_items=len(ITEMS2B), excluded=dict(EXCL2B), n_scorable=int(sum(it["valid"] for it in ITEMS2B)), sha256=CSP_SHA)
for g in RES2B["mean_std"]:
    p3_log(f"TN2b {g}:", {k: round(v["mean"] * 100, 1) for k, v in RES2B["mean_std"][g].items() if k in ("CLIP", "graph_only[dg]", "fusion[dg]", "graph_only[dg_rand]", "graph_only[dg_nogeo]")})
p3_log("TN2b quyết định (đăng ký trước):", RES2B["decision"], "| phản thực tế:", RES2B.get("counterfactual"), "|", elapsed())


## 8. Giải thích (XAI, cấp 3)
Đăng ký trước: `v6_compositional_benchmarks/XAI_PREREGISTRATION_VI.md`. Sparsity gồm kích thước bằng chứng (hằng số theo thiết kế) và **độ tập trung bằng chứng** (thay đổi theo mục).

In [ ]:
# ============================== 8. GIẢI THÍCH (XAI, cấp 3): fidelity · sufficiency · phản thực tế (lật ảnh) · validity (khớp GT) · sparsity ==============================
# Đăng ký trước: v6_compositional_benchmarks/XAI_PREREGISTRATION_VI.md. Đơn vị: mục ARO đồ thị chấm được, phần CHECK của seed 0.
# Giải thích = đúng phần tử tạo ra điểm đồ thị: truy vấn bộ ba j* (vị từ) hoặc cặp hộp u*, v* (trái/phải).
XI = [n for n, it in enumerate(ITEMS2) if it["bench"] == "aro" and aro_parse(it["true"])[1] != "other" and aro_parse(it["false"])[1] != "other"
      and v6_split(it["key"], 0) == "CHECK"]
XI.sort(key=lambda n: int(ITEMS2[n]["item"][3:]))
GEO_N = [n for n in XI if aro_parse(ITEMS2[n]["true"])[1] == "geo"]
FLIP = dict(zip(GEO_N, reltr_clip_features([ITEMS2[n] for n in GEO_N], flip=True)))
p3_log(f"XAI: {len(XI)} mục ({len(GEO_N)} trái/phải) | đặc trưng ảnh lật xong | {elapsed()}")


def zero_rows(F, kind, rows):
    G = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in F.items()}
    for r in rows:
        if kind == "pred": G["ps"][r] = 0; G["po"][r] = 0; G["pr"][r] = 0
        else: (G["ps"] if r < NQ else G["po"])[r % NQ] = 0
    return G


def keep_rows(F, kind, rows):
    return zero_rows(F, kind, [r for r in range(NQ if kind == "pred" else 2 * NQ) if r not in set(rows)])


ev_rows = lambda e: [e["j"]] if e["kind"] == "pred" else [e["u"], e["v"]]
sgn = lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
cxcywh = lambda b: (b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2)


def dgraph(F, it):
    gt, et = aro_graph_score(F, aro_parse(it["true"])); gf, ef = aro_graph_score(F, aro_parse(it["false"]))
    return gt - gf, et, ef


def evidence_concentration(F, parsed):
    """sparsity không cố định: tỷ lệ điểm của phần tử giải thích (max) trên tổng điểm mọi ứng viên (truy vấn bộ ba hoặc cặp hộp).
    Gần 1 ⇒ giải thích gọn (một bằng chứng áp đảo); gần 0 ⇒ bằng chứng dàn trải trên nhiều ứng viên."""
    X, kind, r, Y = parsed
    if kind == "pred":
        v = F["ps"][:, CLS_IDX[X]].astype(np.float64) * F["pr"][:, PRED_IDX[r]].astype(np.float64) * F["po"][:, CLS_IDX[Y]].astype(np.float64)
        return float(v.max() / max(v.sum(), FLOOR))
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    diff = (B[None, :, 0] - B[:, None, 0]) if r == "L" else (B[:, None, 0] - B[None, :, 0])
    M = P[:, CLS_IDX[X]][:, None] * P[:, CLS_IDX[Y]][None, :] / (1.0 + np.exp(-diff / TAU)); np.fill_diagonal(M, 0.0)
    return float(M.max() / max(M.sum(), FLOOR))


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - ix * iy
    return ix * iy / u if u > 0 else 0.0


def crop_norm(gt, crop):
    cw, ch = crop[2] - crop[0], crop[3] - crop[1]
    return ((gt[0] - crop[0]) / cw, (gt[1] - crop[1]) / ch, (gt[2] - crop[0]) / cw, (gt[3] - crop[1]) / ch)


ROWS3 = []
for n in XI:
    it, F = ITEMS2[n], FEATS2[n]
    kind = aro_parse(it["true"])[1]
    dg, et, ef = dgraph(F, it)
    dc = float(F["img"] @ TT2[n] - F["img"] @ TF2[n])
    row = dict(item=it["item"], n=n, vg_id=it["vg_id"], kind=kind, true=it["true"], false=it["false"], dg=dg, dc=dc,
               graph_correct=float(correct([dg])[0]), clip_correct=float(correct([dc])[0]), ev_true=json.dumps(et), ev_false=json.dumps(ef))
    if dg != 0:
        rows_ev = ev_rows(et if dg > 0 else ef)
        row["del_changed"] = float(sgn(dgraph(zero_rows(F, kind, rows_ev), it)[0]) != sgn(dg))
        pool = [r for r in range(NQ if kind == "pred" else 2 * NQ) if r not in set(rows_ev)]
        rr = np.random.default_rng(int(hashlib.md5(it["item"].encode()).hexdigest(), 16) % 2 ** 32)
        row["rand_changed"] = float(np.mean([sgn(dgraph(zero_rows(F, kind, list(rr.choice(pool, size=len(rows_ev), replace=False))), it)[0]) != sgn(dg)
                                             for _ in range(N_RAND_DEL)]))
        row["suff_kept"] = float(sgn(dgraph(keep_rows(F, kind, sorted(set(ev_rows(et)) | set(ev_rows(ef)))), it)[0]) == sgn(dg))
        row["evidence_fraction"] = len(rows_ev) / (NQ if kind == "pred" else 2 * NQ)
        row["evidence_concentration"] = evidence_concentration(F, aro_parse(it["true"] if dg > 0 else it["false"]))
        if dg > 0 and it["xid"] in GTBOX and it["yid"] in GTBOX:
            gx, gy = crop_norm(GTBOX[it["xid"]], it["crop"]), crop_norm(GTBOX[it["yid"]], it["crop"])
            bx_, by_ = (et["sub_box"], et["obj_box"]) if kind == "pred" else (et["x_box"], et["y_box"])
            row["iou_x"], row["iou_y"] = iou(cxcywh(bx_), gx), iou(cxcywh(by_), gy)
            row["valid_both"] = float(row["iou_x"] >= 0.5 and row["iou_y"] >= 0.5); row["valid_any"] = float(row["iou_x"] >= 0.5 or row["iou_y"] >= 0.5)
    if n in FLIP:
        dgf = dgraph(FLIP[n], it)[0]; dcf = float(FLIP[n]["img"] @ TT2[n] - FLIP[n]["img"] @ TF2[n])
        row["flip_graph_reversed"] = float(sgn(dgf) == -sgn(dg) and dg != 0); row["flip_clip_reversed"] = float(sgn(dcf) == -sgn(dc) and dc != 0)
    ROWS3.append(row)


def _col(name, rows):
    return np.array([r[name] for r in rows if name in r], float), [r["vg_id"] for r in rows if name in r]


RES3 = {}
for kind in ("all", "geo", "pred"):
    sub = [r for r in ROWS3 if kind == "all" or r["kind"] == kind]
    d, cl = _col("del_changed", sub); rnd, _ = _col("rand_changed", sub)
    o = dict(n=len(sub), graph_acc=float(np.mean([r["graph_correct"] for r in sub])), clip_acc=float(np.mean([r["clip_correct"] for r in sub])))
    if len(d):
        vb, _ = _col("valid_both", sub); va, _ = _col("valid_any", sub)
        o.update(fidelity_del_changed=float(d.mean()), fidelity_rand_changed=float(rnd.mean()), fidelity_gap=bootstrap_diff_by_cluster(d, rnd, cl),
                 sufficiency_kept=float(_col("suff_kept", sub)[0].mean()), sparsity_evidence_fraction=float(_col("evidence_fraction", sub)[0].mean()),
                 sparsity_concentration_mean=float(_col("evidence_concentration", sub)[0].mean()),
                 sparsity_concentration_median=float(np.median(_col("evidence_concentration", sub)[0])),
                 validity_both_iou50=float(vb.mean()) if len(vb) else None, validity_any_iou50=float(va.mean()) if len(va) else None, n_validity=len(vb))
    fg, fcl = _col("flip_graph_reversed", sub); fc, _ = _col("flip_clip_reversed", sub)
    if len(fg):
        o.update(counterfactual_graph_reversed=float(fg.mean()), counterfactual_clip_reversed=float(fc.mean()), n_counterfactual=len(fg),
                 counterfactual_gap=bootstrap_diff_by_cluster(fg, fc, fcl))
    RES3[kind] = o
p3_log("XAI:", json.dumps({k: {m: (round(v, 3) if isinstance(v, float) else v) for m, v in o.items() if not isinstance(v, dict)} for k, o in RES3.items()}, ensure_ascii=False)[:900])

# ---- 10 ví dụ (5 đồ thị đúng, 5 đồ thị sai; 5 mục đầu theo id — luật khai báo trước) ----
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
EX3 = [r for r in ROWS3 if r["graph_correct"] == 1.0][:5] + [r for r in ROWS3 if r["graph_correct"] == 0.0][:5]


def draw_explanation(r, path):
    it = ITEMS2[r["n"]]; im = pil_of(it); W, H = im.size
    fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(im); ax.axis("off")
    for ev, color, lab in ((json.loads(r["ev_true"]), "lime", "đúng"), (json.loads(r["ev_false"]), "red", "sai")):
        bxs = [ev["sub_box"], ev["obj_box"]] if ev["kind"] == "pred" else [ev["x_box"], ev["y_box"]]
        for b, nm in zip(bxs, ("X", "Y")):
            x0, y0, x1, y1 = cxcywh(b)
            ax.add_patch(mpatches.Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H, fill=False, ec=color, lw=2, ls="-" if lab == "đúng" else "--"))
            ax.text(x0 * W, y0 * H, f"{nm}({lab})", color=color, fontsize=8, va="bottom")
    ax.set_title(f"{r['item']} | {'ĐỒ THỊ ĐÚNG' if r['graph_correct'] == 1 else 'ĐỒ THỊ SAI'}\nđúng: {r['true']} (p={json.loads(r['ev_true'])['p']:.2e})\n"
                 f"sai: {r['false']} (p={json.loads(r['ev_false'])['p']:.2e})\nΔCLIP={r['dc']:+.3f}", fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


for k, r in enumerate(EX3): draw_explanation(r, f"{FIG}/xai_ex{k:02d}_{r['item']}.png")
p3_log(f"XAI: đã vẽ {len(EX3)} ví dụ vào {FIG} | {elapsed()}")


## 10. Demo: nhập truy vấn → top-k + giải thích
Gọi `show(search("a child holding a kite"))` để thử truy vấn khác.

In [ ]:
# ============================== 10. DEMO: nhập truy vấn → top-k ảnh + quan hệ/đường đi giải thích (đề cương mục 8) ==============================
# Dùng gallery P-CHECK của TN1 (ảnh VG có scene graph GT), tham số (w, α) đã chọn trên P-TUNE (seed 0). Không tinh chỉnh gì thêm.
DEMO_IDS = EXP1_KEEP["check"]["ids"]
DEMO_I = EXP1_KEEP["check"]["I"]
(DEMO_W, DEMO_A), _ = EXP1_KEEP["params"]["G-full"]


def search(query, k=5, alpha=None, w=None):
    """CLIP top-K theo cosine → điểm đồ thị (3 kiểu đường đi) → xếp hạng lại α·z(cos) + (1−α)·z(graph). Trả về top-k + giải thích."""
    alpha = DEMO_A if alpha is None else alpha; w = DEMO_W if w is None else w
    t = clip_text([query])[0]
    cos = DEMO_I @ t
    top = np.argsort(-cos, kind="stable")[:K_TOP]
    E, T = parse_caption(query)
    comps = np.array([[(lambda d: (d["s_rel"], d["s_pair"], d["s_obj"]))(match_detail(T, E, GRAPH[DEMO_IDS[j]])) for j in top]])
    final = fuse_scores(zrows(cos[top][None, :]), comps, w, alpha)[0]
    order = np.lexsort((np.arange(len(top)), -final))[:k]
    res = []
    for r, kk in enumerate(order):
        i = DEMO_IDS[top[kk]]; d = match_detail(T, E, GRAPH[i])
        res.append(dict(rank=r + 1, clip_rank=int(kk) + 1, image_id=i, score=float(final[kk]), cos=float(cos[top[kk]]),
                        matched_edges=d["edges"], matched_pairs=d["pairs"], matched_nodes=d["nodes"]))
    return dict(query=query, parsed_entities=sorted(E), parsed_triples=sorted("|".join(x) for x in T), alpha=alpha, w=list(w), results=res)


def show(out, path=None):
    n = len(out["results"])
    fig, axs = plt.subplots(1, n, figsize=(3.3 * n, 3.9)); axs = np.atleast_1d(axs)
    for ax, r in zip(axs, out["results"]):
        ax.imshow(pil_path(VG_JPG[r["image_id"]])); ax.axis("off")
        why = "; ".join(e.split("<=")[0] for e in r["matched_edges"]) or ("cặp: " + ", ".join(r["matched_pairs"]) if r["matched_pairs"] else "đối tượng: " + ", ".join(r["matched_nodes"]))
        ax.set_title(f"#{r['rank']} (CLIP #{r['clip_rank']}) id {r['image_id']}\n{why[:60]}", fontsize=7)
    fig.suptitle(f"'{out['query']}' | bộ ba: {out['parsed_triples']} | α={out['alpha']}", fontsize=8); fig.tight_layout()
    path = path or f"{FIG}/_last_search.png"
    fig.savefig(path, dpi=100); plt.close(fig)
    try:
        from IPython.display import Image as _IImg, display
        display(_IImg(filename=path))                       # hiện hình khi chạy trong notebook
    except Exception:
        pass


DEMO = []
for q in ["a man riding a horse on the beach", "a dog sitting on a bench", "a woman holding an umbrella on the street"]:
    out = search(q); DEMO.append(out); show(out, f"{FIG}/demo_{len(DEMO)}.png")
    p3_log(f"DEMO '{q}':", [(r["image_id"], r["clip_rank"], r["matched_edges"][:1]) for r in out["results"]])
# Dùng tương tác trong notebook:  show(search("a child holding a kite"))


## 10b. Năm truy vấn thành công và năm truy vấn thất bại (văn bản → ảnh)
Luật chọn cố định; phân tích chi tiết trong `final_pipeline/CASE_ANALYSIS_LEVEL1_VI.md`.

In [ ]:
# ============================== 10b. 5 TRUY VẤN THÀNH CÔNG + 5 THẤT BẠI (văn bản → ảnh) — luật chọn cố định ==============================
# Luật (CASE_ANALYSIS_LEVEL1_VI.md): seed 0, P-CHECK; thành công = 5 truy vấn tăng hạng nhiều nhất khi thêm đồ thị (hạng CLIP − hạng CLIP+đồ thị),
# thất bại = 5 truy vấn giảm nhiều nhất; hoà xếp theo id ảnh. Mỗi hình: ảnh đúng (kèm hạng trước/sau) + top-3 sau xếp hạng lại kèm bằng chứng đồ thị.
_ck, _rk = EXP1_KEEP["check"], EXP1_KEEP["ranks"]
_d = _rk["CLIP"] - _rk["G-full"]
_ord_up = sorted(range(len(_d)), key=lambda q: (-_d[q], _ck["ids"][q]))[:5]
_ord_dn = sorted(range(len(_d)), key=lambda q: (_d[q], _ck["ids"][q]))[:5]
CASES1 = []
for tag, L in (("thanh_cong", _ord_up), ("that_bai", _ord_dn)):
    for q in L:
        E, T = _ck["parsed"][q]
        order = np.lexsort((np.arange(_ck["topk"].shape[1]), -EXP1_KEEP["fused"]["G-full"][q]))[:3]
        top = []
        for kk in order:
            i = _ck["ids"][_ck["topk"][q, kk]]; dd = match_detail(T, E, GRAPH[i])
            top.append(dict(image_id=i, clip_pos=int(kk) + 1, relevant=bool(i == _ck["ids"][q]), edges=dd["edges"], nodes=dd["nodes"]))
        drel = match_detail(T, E, GRAPH[_ck["ids"][q]])
        CASES1.append(dict(kind=tag, query_image_id=_ck["ids"][q], caption=_ck["caps"][q], rank_clip=int(_rk["CLIP"][q]), rank_graph=int(_rk["G-full"][q]),
                           rank_obj_only=int(_rk["G-obj"][q]), parsed_triples=sorted("|".join(t) for t in T), parsed_entities=sorted(E),
                           relevant_evidence=dict(edges=drel["edges"], nodes=drel["nodes"]), top3=top))
        fig, axs = plt.subplots(1, 4, figsize=(13, 3.9))
        panels = [(_ck["ids"][q], f"ẢNH ĐÚNG: hạng CLIP {CASES1[-1]['rank_clip']} → {CASES1[-1]['rank_graph']}\nkhớp: {drel['nodes']} {[e.split('<=')[0] for e in drel['edges']]}")]
        panels += [(t["image_id"], f"#{n + 1} sau xếp hạng (CLIP #{t['clip_pos']}){' ✓' if t['relevant'] else ''}\nkhớp: {t['nodes']} {[e.split('<=')[0] for e in t['edges']]}")
                   for n, t in enumerate(top)]
        for ax, (i, title) in zip(axs, panels):
            ax.imshow(pil_path(VG_JPG[i])); ax.axis("off"); ax.set_title(title[:110], fontsize=7)
        fig.suptitle(f"[{tag}] '{_ck['caps'][q]}' | bộ ba tách được: {CASES1[-1]['parsed_triples']}", fontsize=8); fig.tight_layout()
        fig.savefig(f"{FIG}/case_{tag}_{len(CASES1):02d}_{_ck['ids'][q]}.png", dpi=95); plt.close(fig)
json.dump(CASES1, open(f"{OUT}/cases_text_to_image.json", "w"), indent=1, ensure_ascii=False, default=str)
p3_log(f"Ca văn bản → ảnh: {[(c['kind'], c['rank_clip'], c['rank_graph']) for c in CASES1]} | tăng {int((_d > 0).sum())}, giảm {int((_d < 0).sum())}, "
    f"không đổi {int((_d == 0).sum())}; trùng hạng với bản chỉ đối tượng: {float(np.mean(_rk['G-obj'] == _rk['G-full'])):.3f}")


## 11a. Phân tích bổ sung: độ nhạy α, so sánh cách tạo cạnh, file split

In [ ]:
# ============================== 11a. PHÂN TÍCH BỔ SUNG: độ nhạy α · so sánh cách tạo cạnh · xuất file split · hiển thị hình ==============================
from IPython.display import Image as _IImg, Markdown as _IMd, display as _idisp


def show_png(path, width=900):
    try:
        _idisp(_IImg(filename=path, width=width))
    except Exception:
        pass


# ---- 11a.1 Độ nhạy theo α (đề cương mục 7: "phân tích độ nhạy theo trọng số kết hợp α"), đo trên tập TUNE (validation) ----
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, res, key in (("TN1 luật (G-full)", RES1, None), ("TN1b GraphSAGE", RES1G, "SAGE"), ("TN1c SCENIR", RES1S, "SCENIR")):
    M = np.array([[(res["seeds"][s]["alpha_curve_tune_MRR"]["G-full"] if key is None else res["seeds"][s]["train"][key]["alpha_curve_tune_MRR"])[a]
                   for a in ALPHAS] for s in SEEDS])
    axs[0].errorbar(ALPHAS, M.mean(0), yerr=M.std(0, ddof=1), marker="o", ms=3, capsize=2, label=name)
axs[0].set_title("TN1 văn bản → ảnh: MRR trên P-TUNE theo α", fontsize=9); axs[0].set_xlabel("α (1 = chỉ CLIP)"); axs[0].set_ylabel("MRR")
for g, res in (("ARO_all", RES2), ("ARO_geo", RES2), ("ARO_pred", RES2), ("SC_replace_rel", RES2), ("COCOSP_all", RES2B)):
    if g not in res["seeds"][0] or "alpha_grid_tune" not in res["seeds"][0][g]: continue
    M = np.array([[res["seeds"][s][g]["alpha_grid_tune"][a] for a in ALPHAS] for s in SEEDS]) * 100
    axs[1].errorbar(ALPHAS, M.mean(0), yerr=M.std(0, ddof=1), marker="o", ms=3, capsize=2, label=g)
axs[1].set_title("TN2/2b ảnh → caption: độ chính xác (%) trên TUNE theo α", fontsize=9); axs[1].set_xlabel("α (1 = chỉ CLIP)")
for ax in axs: ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{FIG}/alpha_sensitivity.png", dpi=120); plt.close(fig)
show_png(f"{FIG}/alpha_sensitivity.png")

# ---- 11a.2 So sánh các cách tạo cạnh / kiểu đường đi (đề cương mục 7, mở rộng) ----
EDGE_TABLE = [("TN1", "Không đồ thị (CLIP)", RES1["mean_std"]["CLIP"]["MRR"], "MRR"),
              ("TN1", "Chỉ đỉnh đối tượng (G-obj)", RES1["mean_std"]["G-obj"]["MRR"], "MRR"),
              ("TN1", "Bộ ba + cặp + đối tượng (G-full)", RES1["mean_std"]["G-full"]["MRR"], "MRR"),
              ("TN1", "G-full, vị từ hoán vị (G-shuf)", RES1["mean_std"]["G-shuf"]["MRR"], "MRR"),
              ("TN1", "G-full, đồ thị ảnh khác (G-rand)", RES1["mean_std"]["G-rand"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE: ảnh – lớp – bộ ba", RES1G["mean_std"]["SAGE"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE, cạnh ngẫu nhiên", RES1G["mean_std"]["SAGE-rand"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE, bỏ cạnh", RES1G["mean_std"]["SAGE-0"]["MRR"], "MRR"),
              ("TN1c", "SCENIR: scene graph lớp", RES1S["mean_std"]["SCENIR"]["MRR"], "MRR"),
              ("TN1c", "SCENIR, cạnh ngẫu nhiên", RES1S["mean_std"]["SCENIR-rand"]["MRR"], "MRR"),
              ("TN1c", "SCENIR, bỏ cạnh", RES1S["mean_std"]["SCENIR-noedge"]["MRR"], "MRR")]
for g, res, lab in (("ARO_geo", RES2, "ARO trái/phải"), ("ARO_pred", RES2, "ARO vị từ"), ("COCOSP_all", RES2B, "COCO-spatial")):
    v = res["mean_std"][g]
    EDGE_TABLE += [(lab, "Cạnh vị từ / hình học (RelTR)", v["graph_only[dg]"], "acc"), (lab, "Đồ thị của mục khác", v["graph_only[dg_rand]"], "acc")]
    for m, d in (("graph_only[dg_nogeo]", "Bỏ vị trí (cạnh hình học = hằng)"), ("graph_only[dg_shuf]", "Vị từ hoán vị")):
        if m in v: EDGE_TABLE.append((lab, d, v[m], "acc"))
_md = ["| Thí nghiệm | Cách tạo cạnh / biến thể | Giá trị (trung bình ± std, 3 seed) |", "|---|---|---|"]
_md += [f"| {e} | {d} | {v['mean']:.4f} ± {v['std']:.4f} |" if u == "MRR" else f"| {e} | {d} | {100 * v['mean']:.1f}% ± {100 * v['std']:.1f} |" for e, d, v, u in EDGE_TABLE]
EDGE_MD = "\n".join(_md)
_idisp(_IMd("**So sánh các cách tạo cạnh** (TN1: MRR trên P-CHECK; TN2/2b: độ chính xác 'chỉ đồ thị' trên CHECK)\n\n" + EDGE_MD))

# ---- 11a.3 Xuất file split cố định (sản phẩm nộp: "tập chia dữ liệu cố định hoặc mã tạo split") ----
SPLIT_DIR = f"{OUT}/splits"; os.makedirs(SPLIT_DIR, exist_ok=True)
with open(f"{SPLIT_DIR}/tn1_vg_coco_split.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["vg_image_id", "coco_id", "split", "role"])
    for i in ELIG1: w.writerow([i, COCO_OF_VG[i], p5_split(i), "eval (P-CHECK)" if p5_split(i) == "P-CHECK" else "tune (P-TUNE) + GNN train"])
    for i in TRAIN_IMGS:
        if i not in set(ELIG1): w.writerow([i, "", "GNN-train-only", "GraphSAGE / SCENIR train"])
with open(f"{SPLIT_DIR}/tn2_tn2b_items_split.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["bench", "item", "image_key", "split_seed0", "split_seed1", "split_seed2"])
    for it in ITEMS2 + ITEMS2B: w.writerow([it["bench"], it["item"], it["key"]] + [v6_split(it["key"], s) for s in SEEDS])
json.dump(dict(rules={"VAL (v3)": "sha1(id) theo v3, chỉ số lọc ≥ 83000; heldout KHÔNG dùng",
                      "DEV-G0": "sha1('p3-dir2:'+id) mod 1000 < 350, trong pool G0 (VAL trừ benchmark v3, cache v4, 99 ảnh Phase 5)",
                      "TN1 P-TUNE/P-CHECK": "sha1('p5-pilot:'+id) mod 2", "TN2/2b TUNE/CHECK": "sha1(f'v6-{seed}:{image_key}') mod 2, seed ∈ {0,1,2}",
                      "lọc rò rỉ benchmark": "ảnh VG phải thuộc VAL; ảnh COCO phải không thuộc VG hoặc thuộc VG-VAL"},
               md5=dict(VAL=VAL_MD5, G0=G0_MD5), counts=dict(tn1=len(ELIG1), gnn_train=len(TRAIN_IMGS), tn2=len(ITEMS2), tn2b=len(ITEMS2B))),
          open(f"{SPLIT_DIR}/SPLIT_RULES.json", "w"), indent=1, ensure_ascii=False)
p3_log(f"Đã xuất file split vào {SPLIT_DIR}")

# ---- 11a.4 Hình chính (hiển thị trong notebook) ----
for p in [f"{FIG}/demo_1.png"] + sorted(glob.glob(f"{FIG}/case_*.png"))[:2] + sorted(glob.glob(f"{FIG}/xai_ex*.png"))[:2]:
    show_png(p, width=800)


## 11. Tổng hợp và đối chiếu yêu cầu đề cương

In [ ]:
# ============================== 11. TỔNG HỢP: bảng kết quả · đối chiếu tái lập với các lần chạy trước · lưu đầu ra ==============================
def pct(x): return f"{100 * x:.1f}"


lines = ["# Kết quả pipeline Project 3", "", f"Chế độ: {'SMOKE (dữ liệu giả — KHÔNG phải kết quả)' if SMOKE else 'REAL'} · {MODEL_INFO} · {elapsed()}", "",
         "## TN1 — Văn bản → ảnh, full-gallery (P-CHECK, trung bình ± std qua 3 seed caption)", "",
         "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1['decision']}**", "",
          "## TN1b — GraphSAGE (GNN đã học) trong cùng pipeline văn bản → ảnh (P-CHECK, 3 seed)", "",
          "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1G["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1G['decision']}** (SAGE-rand = cạnh ngẫu nhiên, SAGE-0 = bỏ cạnh)", "",
          f"## TN1c — SCENIR (ICML 2025) trong cùng pipeline văn bản → ảnh (P-CHECK, 3 seed; đặc trưng đỉnh: {RES1S['node_features']})", "",
          "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1S["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1S['decision']}**", "",
          "## TN2 — Benchmark kết hợp (độ chính xác CHECK %, trung bình ± std qua 3 seed chia)", "",
          "| Nhóm | CLIP | Chỉ đồ thị | CLIP + đồ thị | Đồ thị mục khác | Bỏ vị trí |", "|---|---|---|---|---|---|"]
for g, v in RES2["mean_std"].items():
    f = lambda m: f"{pct(v[m]['mean'])} ± {pct(v[m]['std'])}" if m in v else "—"
    lines.append(f"| {g} | {f('CLIP')} | {f('graph_only[dg]')} | {f('fusion[dg]')} | {f('graph_only[dg_rand]')} | {f('graph_only[dg_nogeo]')} |")
lines += ["", f"Quyết định đăng ký trước: **{RES2['decision']}**", "",
          f"## TN2b — Cấp 4 (a): cạnh hình học trên COCO-spatial ({RES2B['n_items']} mục sau lọc, {RES2B['n_scorable']} mục đồ thị chấm được; loại {RES2B['excluded']})", "",
          "| Nhóm | CLIP | Chỉ đồ thị | CLIP + đồ thị | Đồ thị mục khác | Bỏ vị trí |", "|---|---|---|---|---|---|"]
for g, v in RES2B["mean_std"].items():
    f = lambda m: f"{pct(v[m]['mean'])} ± {pct(v[m]['std'])}" if m in v else "—"
    lines.append(f"| {g} | {f('CLIP')} | {f('graph_only[dg]')} | {f('fusion[dg]')} | {f('graph_only[dg_rand]')} | {f('graph_only[dg_nogeo]')} |")
_cf = RES2B.get("counterfactual")
lines += ["", f"Quyết định đăng ký trước: **{RES2B['decision']}**" + (f" · Lật ảnh: đồ thị đảo {pct(_cf['graph_reversed'])}%, CLIP đảo {pct(_cf['clip_reversed'])}% (n={_cf['n']})" if _cf else ""), "",
          "## XAI (ARO, CHECK seed 0)", "",
          "| Nhóm | n | Lật ảnh: đồ thị đảo / CLIP đảo | Xoá bằng chứng: đổi / ngẫu nhiên | Validity cả hai / ít nhất một | Sufficiency | Sparsity: kích thước / độ tập trung |",
          "|---|---|---|---|---|---|---|"]
for k, o in RES3.items():
    cf = f"{pct(o['counterfactual_graph_reversed'])} / {pct(o['counterfactual_clip_reversed'])}" if "counterfactual_graph_reversed" in o else "—"
    lines.append(f"| {k} | {o['n']} | {cf} | {pct(o.get('fidelity_del_changed', 0))} / {pct(o.get('fidelity_rand_changed', 0))} | "
                 f"{pct(o['validity_both_iou50'] or 0)} / {pct(o['validity_any_iou50'] or 0)} | {pct(o.get('sufficiency_kept', 0))} (theo cấu trúc) | "
                 f"{pct(o.get('sparsity_evidence_fraction', 0))}% (thiết kế) / {o.get('sparsity_concentration_mean', 0):.2f} (trung vị {o.get('sparsity_concentration_median', 0):.2f}) |")

# ---- đối chiếu tái lập với số đã chạy riêng (v5, v6, v6-XAI) ----
REPRO = []
if not SMOKE:
    for k, ref in REF["exp1_MRR"].items():
        REPRO.append(("TN1 MRR " + k, ref, RES1["mean_std"][k]["MRR"]["mean"]))
    for g, d in REF["exp2_acc"].items():
        for m, ref in d.items(): REPRO.append((f"TN2 {g} {m}", ref, RES2["mean_std"][g][m]["mean"]))
    for k, d in REF["xai"].items():
        for m, ref in d.items(): REPRO.append((f"XAI {k} {m}", ref, RES3[k][m]))
    lines += ["", "## Đối chiếu tái lập với các lần chạy riêng trước đó", "", "| Chỉ số | Lần trước | Pipeline này | Chênh |", "|---|---|---|---|"]
    lines += [f"| {n} | {a:.4f} | {b:.4f} | {b - a:+.4f} |" for n, a, b in REPRO]
lines += ["", "## So sánh các cách tạo cạnh", "", EDGE_MD]

# ---- Cấp 4 (b): video MSVD (notebook video v5 chạy nguyên vẹn ở mục 4v) ----
lines += ["", f"## Cấp 4 (b) — Video MSVD (notebook Level4B v5, trạng thái: {RES_VIDEO['status']}, {RES_VIDEO.get('minutes')} phút)", ""]
if RES_VIDEO["status"] == "ok":
    lines += [f"Lần nhìn test MSVD: **{RES_VIDEO.get('test_looks_msvd')}** — số test chỉ là thăm dò, không phải xác nhận độc lập.", "",
              "| Mô hình | Tập | R@1 | R@5 | R@10 | MRR | n truy vấn |", "|---|---|---|---|---|---|---|"]
    lines += [f"| {r['model']} | {r.get('set')} | {pct(r['R@1'])} | {pct(r['R@5'])} | {pct(r['R@10'])} | {r['MRR']:.4f} | {r.get('n_queries')} |" for r in RES_VIDEO["test"]]
    if RES_VIDEO["comparisons"]:
        lines += ["", "| So sánh | a vs b | ΔMRR [CI 95%] | p (Holm) |", "|---|---|---|---|"]
        lines += [f"| {c.get('comparison')} | {c.get('a')} vs {c.get('b')} | {c.get('diff', 0):+.4f} [{c.get('ci_lo', 0):+.4f}, {c.get('ci_hi', 0):+.4f}] | {c.get('p_holm')} |"
                  for c in RES_VIDEO["comparisons"]]
    if RES_VIDEO.get("fidelity"): lines += ["", f"Fidelity (video): {RES_VIDEO['fidelity']}"]
else:
    lines += [f"Nhánh video không hoàn tất: {RES_VIDEO.get('error', '')[:500]}"]

# ---- bảng đối chiếu yêu cầu đề cương (sinh tự động từ kết quả của chính lần chạy này) ----
_d = lambda dec: "DƯƠNG" if (dec == "GO" or (isinstance(dec, str) and dec.startswith("DƯƠNG"))) else "không dương"
CHECKLIST = [
    ("Cấp 1", "Baseline CLIP đóng băng trên ≥ 1 tập công khai", "✅", f"CLIP {CLIP_MODEL}; TN1 MRR {RES1['mean_std']['CLIP']['MRR']['mean']:.4f}"),
    ("Cấp 1", "Đồ thị từ dữ liệu có sẵn (scene graph VG)", "✅", "Scene graph GT (TN1), scene graph RelTR (TN2, 2b)"),
    ("Cấp 1", "≥ 1 thuật toán đã học (GraphSAGE)", "✅", f"TN1b: SAGE {RES1G['mean_std']['SAGE']['MRR']['mean']:.4f} vs CLIP {RES1G['mean_std']['CLIP']['MRR']['mean']:.4f} ({_d(RES1G['decision']['G1 SAGE-CLIP'])})"),
    ("Cấp 1", "Xếp hạng lại α·cos + (1−α)·graph, so CLIP vs CLIP+Graph cùng split", "✅", f"TN1 {RES1['decision']}; ARO {RES2['decision'].get('ARO_all')}"),
    ("Cấp 1", "Ablation bỏ cạnh / cạnh ngẫu nhiên", "✅", f"ARO: mục khác {pct(RES2['mean_std']['ARO_all']['graph_only[dg_rand]']['mean'])}%, bỏ vị trí {pct(RES2['mean_std']['ARO_all']['graph_only[dg_nogeo]']['mean'])}%"),
    ("Cấp 1", "Hiển thị quan hệ/đường đi giải thích", "✅", "Demo search() + hình XAI"),
    ("Cấp 1", "5 truy vấn thành công + 5 thất bại", "✅", f"{len(CASES1)} ca văn bản → ảnh (case_*.png) + 10 ca ARO (xai_ex*.png); phân tích: CASE_ANALYSIS_LEVEL1_VI.md"),
    ("Cấp 1", "R@1/5/10, MRR, ≥ 3 seed, α chọn trên validation", "✅", "Mọi thí nghiệm: 3 seed, α trên TUNE"),
    ("Cấp 2", "≥ 2 tập công khai", "✅", "Visual Genome + COCO, ARO, SugarCrepe, COCO-spatial"),
    ("Cấp 2", "1 phương pháp mới từ 2024, cùng pipeline/split", "✅", f"SCENIR (ICML 2025) TN1c: {RES1S['mean_std']['SCENIR']['MRR']['mean']:.4f} vs CLIP ({_d(RES1S['decision']['S1 SCENIR-CLIP'])})"),
    ("Cấp 2", "Bộ ≥ 20 truy vấn có nhóm gián tiếp, báo cáo tách nhóm", "⏸", "Hoãn (cần MMKG) — ngoài phạm vi notebook này"),
    ("Cấp 2", "Ablation xáo trộn / đồ thị ngẫu nhiên; so sánh cách tạo cạnh; độ nhạy α", "✅", "Bảng so sánh cạnh + alpha_sensitivity.png"),
    ("Cấp 3", "Fidelity định lượng ≥ 10 truy vấn", "✅", f"Lật ảnh {pct(RES3['all'].get('counterfactual_graph_reversed', 0))}% vs CLIP {pct(RES3['all'].get('counterfactual_clip_reversed', 0))}%; xoá {pct(RES3['all']['fidelity_del_changed'])}% vs ngẫu nhiên {pct(RES3['all']['fidelity_rand_changed'])}% (n={RES3['all']['n']})"),
    ("Cấp 3", "Validity", "✅", f"Khớp GT IoU≥0.5: cả hai {pct(RES3['all']['validity_both_iou50'] or 0)}%, ít nhất một {pct(RES3['all']['validity_any_iou50'] or 0)}%"),
    ("Cấp 3", "Sparsity", "✅", f"Kích thước {pct(RES3['all']['sparsity_evidence_fraction'])}% (thiết kế); độ tập trung trung bình {RES3['all']['sparsity_concentration_mean']:.2f}"),
    ("Cấp 3", "MMKG tự xây + data card", "⏸", "Hoãn — ngoài phạm vi notebook này"),
    ("Cấp 4 (a)", "Cải tiến của nhóm + giả thuyết + ablation + ≥ 2 tập công khai", "✅" if RES2B["decision"]["A1 fusion-CLIP"].startswith("DƯƠNG") else "🟡",
     f"Cạnh hình học: ARO trái/phải {pct(RES2['mean_std']['ARO_geo']['CLIP']['mean'])}→{pct(RES2['mean_std']['ARO_geo']['fusion[dg]']['mean'])}%; COCO-spatial {RES2B['decision']}"),
    ("Cấp 4 (b)", "Mở rộng video (text-to-video)", "✅" if RES_VIDEO["status"] == "ok" else "❌",
     (lambda tv: f"MSVD test: CLIP {tv['M0_clip']['MRR']:.4f} → đồ thị đầy đủ {tv['M4_full_graph']['MRR']:.4f} (lần nhìn test: {RES_VIDEO.get('test_looks_msvd')})"
      if RES_VIDEO["status"] == "ok" and "M0_clip" in tv and "M4_full_graph" in tv else f"trạng thái {RES_VIDEO['status']}")(
         {r["model"]: r for r in RES_VIDEO.get("test", []) if r.get("set") == "test_all"})),
    ("Mở rộng", "Chiều ảnh → văn bản (nhánh B)", "✅", "TN2, TN2b"),
    ("Nộp", "Mã tạo split / file split cố định", "✅", "splits/ (CSV + SPLIT_RULES.json)"),
    ("Nộp", "Notebook nhập truy vấn → top-k + giải thích", "✅", "Hàm search() (mục 10)")]
lines += ["", "## Đối chiếu yêu cầu đề cương", "", "| Cấp | Yêu cầu | Trạng thái | Bằng chứng (lần chạy này) |", "|---|---|---|---|"]
lines += [f"| {a} | {b} | {c} | {d} |" for a, b, c, d in CHECKLIST]
REPORT = "\n".join(lines)
open(f"{OUT}/RESULTS.md", "w", encoding="utf-8").write(REPORT)
json.dump(dict(exp1=RES1, exp1_graphsage=RES1G, exp1_scenir=RES1S, exp2=RES2, exp2b_cocospatial=RES2B, xai=RES3, video=RES_VIDEO, demo=DEMO, checklist=CHECKLIST, models=MODEL_INFO, smoke=SMOKE, inputs_sha256=SHA, bench_sha256=BSHA,
               reproduction=[dict(metric=n, previous=a, now=b) for n, a, b in REPRO], runtime_s=round(time.time() - T0, 1)),
          open(f"{OUT}/summary.json", "w"), indent=1, default=str, ensure_ascii=False)
for name, rows in (("exp2_items.csv", ROWS2), ("exp2b_cocospatial_items.csv", ROWS2B), ("xai_items.csv", ROWS3)):
    with open(f"{OUT}/{name}", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r})); w.writeheader(); w.writerows(rows)
try:
    _idisp(_IMd(REPORT))
except Exception:
    print(REPORT)
p3_log("Xong toàn bộ pipeline:", elapsed())


In [ ]:
# ============ TỔNG KẾT: nơi lưu kết quả của cả hai phần ============
import os, glob
for _p in ["/kaggle/working/results", "/kaggle/working/output/action_genome/results", "/kaggle/working/p3_out", "/kaggle/working/p3_out/video_v2"]:
    _fs = sorted(glob.glob(os.path.join(_p, "*")))
    print(f"{_p}: {len(_fs)} tệp")
    for _f in _fs[:40]:
        print("   ", os.path.basename(_f))

print("Phần C (Level 4(b) v2-graphfix):", {k: v for k, v in RES_L4B_V2.items() if k != "error"} if "RES_L4B_V2" in globals() else "chưa chạy")
